In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp -r "/content/drive/MyDrive/hiver_support_agent_checkpoint/data" .

In [3]:
!find data -type f | sort

data/checkpoints/cross_encoder_reranked_results.csv
data/checkpoints/retrieval_checkpoint.json
data/checkpoints/rrf_candidates.csv
data/checkpoints/structured_rag_corpus.csv
data/embeddings/historical_bge_large.npy
data/evaluation/golden_evaluation_set.csv
data/evaluation/golden_evaluation_set_frozen.csv
data/evaluation/retrieval_benchmark.csv
data/evaluation/top1_evidence_review.csv
data/evaluation/top5_historical_evidence.csv
data/evaluation/unlabeled_intent_sample_200.csv


In [4]:
import pandas as pd
import numpy as np

golden_eval = pd.read_csv(
    "data/evaluation/golden_evaluation_set_frozen.csv"
)

rag_corpus = pd.read_csv(
    "data/checkpoints/structured_rag_corpus.csv"
)

reranked_df = pd.read_csv(
    "data/checkpoints/cross_encoder_reranked_results.csv"
)

evidence_df = pd.read_csv(
    "data/evaluation/top5_historical_evidence.csv"
)

evidence_review = pd.read_csv(
    "data/evaluation/top1_evidence_review.csv"
)

historical_embeddings = np.load(
    "data/embeddings/historical_bge_large.npy"
)

print("=" * 80)
print("CHECKPOINT RESTORED")
print("=" * 80)

print("Golden cases:", len(golden_eval))
print("RAG cases:", len(rag_corpus))
print("Reranked rows:", len(reranked_df))
print("Evidence rows:", len(evidence_df))
print("BGE embeddings:", historical_embeddings.shape)

CHECKPOINT RESTORED
Golden cases: 200
RAG cases: 83218
Reranked rows: 10000
Evidence rows: 1000
BGE embeddings: (83218, 1024)


In [5]:
# ============================================================
# EVIDENCE REVIEW — BATCH 1
# ============================================================

BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .head(BATCH_SIZE)
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 1")
print("=" * 80)

print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 1
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
0,GOLD-0001,Contact @115821 regarding payment on an order....,PAYMENT_BILLING,AMZ-073280,0.615286,[user] will someone please contact mw urgently...,[USER] Oh no! Sorry to hear this! Without prov...,INVESTIGATION_OR_FOLLOWUP
1,GOLD-0002,@AmazonHelp LOL...you've missed the delivery d...,ORDER_DELIVERY,AMZ-032817,0.998420,"[user] yes, u missed the provided delivery date",[USER] Thanks for confirming! Who's the carrie...,INVESTIGATION_OR_FOLLOWUP
2,GOLD-0003,So bummed that my package was delayed even wit...,ORDER_DELIVERY,AMZ-077675,0.991000,hey [user] i don’t pay for prime 2 day shippin...,"[USER] Hi Madison, without going into personal...",GENERAL_SUPPORT
3,GOLD-0004,I love @115821 for my Christmas shopping gift ...,GENERAL_SOCIAL,AMZ-047124,0.957855,. [user] has to be the best thing ever for chr...,[USER] Can't beat skipping out on waiting in l...,GENERAL_SUPPORT
4,GOLD-0005,@AmazonHelp Bleg on the phone 😕 what version s...,DEVICE_TECHNICAL,AMZ-001463,0.859899,[user] will it work on my phone?,[USER] We appreciate you reaching out! Here ar...,GENERAL_SUPPORT
5,GOLD-0006,@115850 amazon stick from https://t.co/cvJQ06p...,SELLER_AUTHENTICITY,AMZ-082165,0.770269,[user] it was sold by amazon.,[USER] Thanks for confirming! Please reach us ...,INVESTIGATION_OR_FOLLOWUP
6,GOLD-0007,@117795 Unethical delivery setup. Worst cust c...,ORDER_DELIVERY,AMZ-006544,0.951277,[user] dishonest delivery setup. worst cust ca...,[USER] Please don't provide your order details...,INVESTIGATION_OR_FOLLOWUP
7,GOLD-0008,@AmazonHelp expected delivery yesterday and it...,ORDER_DELIVERY,AMZ-079961,0.999397,[user] it's not even out for delivery yet.,[USER] Unless we've provided notification othe...,INVESTIGATION_OR_FOLLOWUP
8,GOLD-0009,@667472 @115833 My 5 year old son ordered a Ni...,PURCHASE_CONTROL,AMZ-035797,0.450266,"dear [user] , alexa needs to enforce good mann...",[USER] We are always looking for ways to impro...,GENERAL_SUPPORT
9,GOLD-0010,#KaroMilkeLateDelivery is the mantra of @11585...,ORDER_DELIVERY,AMZ-048759,0.363737,[user] [user] this is d 3rd time with me the p...,[USER] Apologies for the unpleasant experience...,INVESTIGATION_OR_FOLLOWUP


In [6]:
# ============================================================
# EVIDENCE REVIEW — BATCH 1 LABELS
# ============================================================

batch1_labels = {
    "GOLD-0001": ("PARTIAL", "Related order/support issue, but historical evidence does not clearly address payment."),
    "GOLD-0002": ("RELEVANT", "Missed promised delivery date directly matches."),
    "GOLD-0003": ("RELEVANT", "Delayed package with Prime shipping is closely matched."),
    "GOLD-0004": ("RELEVANT", "Both are positive/social Amazon shopping interactions."),
    "GOLD-0005": ("RELEVANT", "Phone compatibility/version issue directly matches."),
    "GOLD-0006": ("PARTIAL", "Seller authenticity is related, but historical evidence does not directly verify authorization."),
    "GOLD-0007": ("RELEVANT", "Delivery setup/customer experience complaint is closely related."),
    "GOLD-0008": ("RELEVANT", "Expected delivery missed and package not out for delivery directly match."),
    "GOLD-0009": ("PARTIAL", "Child purchase/control issue has some connection to Alexa controls but is not directly addressed."),
    "GOLD-0010": ("RELEVANT", "Repeated package/delivery delay matches."),
    "GOLD-0011": ("RELEVANT", "Parent-item issue and prolonged support interaction directly match."),
    "GOLD-0012": ("RELEVANT", "Delivery delay/guaranteed delivery complaint matches."),
    "GOLD-0013": ("RELEVANT", "Amazon book purchase is directly related."),
    "GOLD-0014": ("RELEVANT", "Repeated Amazon logistics/delivery failure matches."),
    "GOLD-0015": ("IRRELEVANT", "Social commentary does not match historical order-status issue."),
    "GOLD-0016": ("RELEVANT", "Receiving ordered books/package is closely related."),
    "GOLD-0017": ("IRRELEVANT", "Account hacking/security does not match historical order complaint."),
    "GOLD-0018": ("RELEVANT", "Preorder release-day delivery is an almost exact match."),
    "GOLD-0019": ("RELEVANT", "Carrier/tracking issue directly matches."),
    "GOLD-0020": ("RELEVANT", "Marketplace selling/business inquiry and seller-support routing are closely related.")
}

for golden_id, (label, note) in batch1_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

print("=" * 80)
print("BATCH 1 SAVED")
print("=" * 80)

print(
    evidence_review["evidence_relevance"]
    .value_counts()
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("\nSaved:")
print("data/evaluation/top1_evidence_review.csv")

BATCH 1 SAVED
evidence_relevance
RELEVANT      15
PARTIAL        3
IRRELEVANT     2
Name: count, dtype: int64

Saved:
data/evaluation/top1_evidence_review.csv


/tmp/ipykernel_921/3117934686.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'PARTIAL' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evidence_review.loc[mask, "evidence_relevance"] = label
/tmp/ipykernel_921/3117934686.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Related order/support issue, but historical evidence does not clearly address payment.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evidence_review.loc[mask, "review_notes"] = note


In [7]:
# ============================================================
# FIX REVIEW COLUMN DTYPES
# ============================================================

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"]
    .fillna("")
    .astype(str)
)

evidence_review["review_notes"] = (
    evidence_review["review_notes"]
    .fillna("")
    .astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("Review columns converted to string dtype.")
print("Checkpoint updated successfully.")

Review columns converted to string dtype.
Checkpoint updated successfully.


In [8]:
# ============================================================
# EVIDENCE REVIEW — BATCH 2
# ============================================================

BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[BATCH_SIZE:2 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 2")
print("=" * 80)

print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 2
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
20,GOLD-0021,@AmazonHelp I still can not access my account....,ACCOUNT_ACCESS,AMZ-037921,0.980785,[user] that is the problem! i cant access on m...,[USER] I'm sorry! Use this instead: [URL] Our ...,GENERAL_SUPPORT
21,GOLD-0022,@AmazonHelp I've. Still awaiting a proper resp...,GENERAL_SUPPORT,AMZ-007596,0.999964,[user] i'm still awaiting a response to this.,"[USER] Sorry for the delay, please contact us ...",INVESTIGATION_OR_FOLLOWUP
22,GOLD-0023,@115851 @AmazonHelp @115850 #jeffbezos be hone...,GENERAL_SUPPORT,AMZ-066743,0.995596,"[user] [user] [user] #jeffbezos be honest, ear...",[USER] I’m extremely sorry about this experien...,INVESTIGATION_OR_FOLLOWUP
23,GOLD-0024,@117795 your delivery person wouldn't answer h...,ORDER_DELIVERY,AMZ-039408,0.917426,[user] why am i not able to cancel a delivery??,"[USER] If the order is shipped, you will not b...",CANCELLATION
24,GOLD-0025,@115850 #ReceivedDefectiveProduct #VerifiedBya...,RETURNS_REFUNDS,AMZ-000040,0.006808,[user] yesterday i had interacted with amazon ...,"[USER] Hey, please connect with us using the a...",INVESTIGATION_OR_FOLLOWUP
25,GOLD-0026,@AmazonHelp here is my order details 402-02424...,ORDER_TRACKING,AMZ-015334,0.965116,[user] i wasn't available at the time of deliv...,[USER] Please don't provide your order details...,INVESTIGATION_OR_FOLLOWUP
26,GOLD-0027,@AmazonHelp I'm confused... https://t.co/ikSK7...,ORDER_DELIVERY,AMZ-057599,0.993545,"[user] hi, slightly confused [url]",[USER] That's strange! Please reach out to us ...,GENERAL_SUPPORT
27,GOLD-0028,@115850 Tracking #: 5180712005991\nPlease upda...,ORDER_TRACKING,AMZ-012942,0.949117,[user] the amzl tracking and delivery experien...,[USER] We'd like to help out. Are you having a...,GENERAL_SUPPORT
28,GOLD-0029,@115821 / @AmazonHelp - Two day delivery is th...,ORDER_DELIVERY,AMZ-021308,0.974980,two day shipping isn’t really 2 days if estima...,@ [PHONE_NUMBER] /2) If we miss the date in ou...,GENERAL_SUPPORT
29,GOLD-0030,@AmazonHelp You’ve dropped the ball on this on...,ORDER_CANCELLATION,AMZ-032712,0.697051,[user] [user] dropped the ball on this one [url],[USER] Oh no! I'm sorry your order came that w...,REFUND_OR_COMPENSATION


In [9]:
batch2_labels = {
    "GOLD-0021": ("RELEVANT", "Same account-access problem; historical response provides a concrete recovery path."),
    "GOLD-0022": ("RELEVANT", "Same unresolved-support/follow-up situation; historical action is directly applicable."),
    "GOLD-0023": ("PARTIAL", "Both involve unresolved customer dissatisfaction, but the historical case is more generic and lacks the specific context of the complaint."),
    "GOLD-0024": ("RELEVANT", "Historical case directly addresses inability to cancel an order after shipment."),
    "GOLD-0025": ("PARTIAL", "Related to a defective/return issue, but historical response mainly redirects to further investigation."),
    "GOLD-0026": ("PARTIAL", "Related to delivery/tracking, but historical case focuses on an unavailable delivery attempt rather than the tracking request."),
    "GOLD-0027": ("RELEVANT", "Both are vague delivery-related problems where contacting support for investigation is an appropriate historical action."),
    "GOLD-0028": ("RELEVANT", "Both concern Amazon delivery/tracking experience; historical response provides a relevant support path."),
    "GOLD-0029": ("RELEVANT", "Directly related to delayed delivery and missed delivery expectations."),
    "GOLD-0030": ("PARTIAL", "Cancellation-related issue, but historical evidence focuses more on refund/compensation after an order problem."),
    "GOLD-0031": ("RELEVANT", "Same repeated pickup/return failure pattern; historical response directly addresses the unresolved pickup."),
    "GOLD-0032": ("RELEVANT", "Same delayed-order problem with multiple delayed deliveries; historical response is applicable."),
    "GOLD-0033": ("PARTIAL", "Pickup problem is related to order fulfillment, but the historical response is vague."),
    "GOLD-0034": ("RELEVANT", "Directly concerns poor delivery service and provides a relevant support response."),
    "GOLD-0035": ("RELEVANT", "Same late-delivery/escalation context; historical response addresses delivery expectations."),
    "GOLD-0036": ("RELEVANT", "Same Amazon delivery context and historical action provides a support/contact path."),
    "GOLD-0037": ("RELEVANT", "Same return-related situation; historical response gives a relevant return-support path."),
    "GOLD-0038": ("RELEVANT", "Historical case indicates the customer's concern was addressed, making it useful supporting evidence."),
    "GOLD-0039": ("RELEVANT", "Same Amazon Prime Video/content context; historical response is directly relevant."),
    "GOLD-0040": ("RELEVANT", "Direct refund request with historical response addressing refund handling.")
}

for golden_id, (label, note) in batch2_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

# Fix column dtypes before saving
evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 2 saved successfully")
print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch2_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 2 saved successfully
evidence_relevance
RELEVANT    15
PARTIAL      5
Name: count, dtype: int64


In [10]:
BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[2 * BATCH_SIZE:3 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 3")
print("=" * 80)
print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 3
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
40,GOLD-0041,Watched the Amazon delivery lady toss my packa...,ORDER_DELIVERY,AMZ-010028,0.973287,hahaha someone stole my amazon package,"[USER] Oh no! Just in case, please try these s...",GENERAL_SUPPORT
41,GOLD-0042,@AmazonHelp わかりますが、同じ表品を2セットで別はやっぱり嫌です…\n開けると「...,GENERAL_SUPPORT,AMZ-058491,0.024588,[user] ご丁寧にありがとうございます😊 2個使えれば十分なので返品はしまーせん！,[USER] さようでございましたか。かしこまりました。Amazonをご利用で何かございまし...,GENERAL_SUPPORT
42,GOLD-0043,"So mad at @115821 right now, I have prime, yet...",ORDER_DELIVERY,AMZ-031069,0.999595,i seriously can't remember when i actually got...,[USER] That's not the way we'd like you to fee...,INVESTIGATION_OR_FOLLOWUP
43,GOLD-0044,@AmazonHelp Amazon and today.,GENERAL_SUPPORT,AMZ-054012,0.997749,[user] [user] today means tomorrow. happens on...,[USER] Thank you for flagging this to us. I'll...,GENERAL_SUPPORT
44,GOLD-0045,I love @115821 one hour Morrison's delivery bu...,GENERAL_SUPPORT,AMZ-069432,0.683791,[user] i got an echo plus with one hour delive...,[USER] Awesome! That's one speedy delivery. We...,GENERAL_SUPPORT
45,GOLD-0046,@115850 Order 5568 not received timely even ex...,PAYMENT_BILLING,AMZ-020599,0.994615,[user] hi there i had ordered a product and it...,[USER] We have responded to you via DM. Kindly...,INVESTIGATION_OR_FOLLOWUP
46,GOLD-0047,@AmazonHelp both me and my roommate had separa...,ORDER_TRACKING,AMZ-011660,0.968253,[user] any reason why i got a notification of ...,[USER] I'm sorry for the issue with your deliv...,GENERAL_SUPPORT
47,GOLD-0048,I used to love @117795 because it always deliv...,ORDER_DELIVERY,AMZ-055562,0.917750,i don't understand why i pay for prime if my p...,[USER] I'm sorry for the poor service! We're h...,GENERAL_SUPPORT
48,GOLD-0049,"Another fine, quality delivery by @115821 's c...",ORDER_DELIVERY,AMZ-055451,0.983704,another guaranteed delivery by [user] [url],[USER] I'm sorry you haven't yet received your...,INVESTIGATION_OR_FOLLOWUP
49,GOLD-0050,@AmazonHelp I am at my Location for only tomorrow,ORDER_DELIVERY,AMZ-009234,0.949289,[user] the problem is i am here only for tomor...,"[USER] I get your concern. As stated, we're wo...",GENERAL_SUPPORT


In [11]:
batch3_labels = {
    "GOLD-0041": ("PARTIAL", "Both involve delivery/package problems, but the historical case is about a stolen package while the query is about a delivery person tossing the package. Related, but not the same resolution path."),
    "GOLD-0042": ("RELEVANT", "Same general support interaction and historical response appropriately closes the customer's concern."),
    "GOLD-0043": ("RELEVANT", "Directly related to delayed/poor Prime delivery service and provides relevant investigation/support evidence."),
    "GOLD-0044": ("RELEVANT", "Historical case concerns a delivery-date issue and provides an appropriate support follow-up."),
    "GOLD-0045": ("RELEVANT", "Same delivery-service context; historical response directly addresses the customer's delivery experience."),
    "GOLD-0046": ("PARTIAL", "Historical case involves an order not received on time, but the golden intent is payment/billing, so the evidence doesn't directly address the billing aspect."),
    "GOLD-0047": ("RELEVANT", "Both concern delivery notifications/attempts; historical response is useful for handling the delivery issue."),
    "GOLD-0048": ("RELEVANT", "Same Prime/delivery expectation problem; historical response directly addresses poor delivery service."),
    "GOLD-0049": ("RELEVANT", "Both involve a package not being received as expected; historical response provides relevant support/investigation."),
    "GOLD-0050": ("RELEVANT", "Same delivery timing/location constraint; historical response addresses the delivery-date concern."),
    "GOLD-0051": ("RELEVANT", "Both concern a technical problem across apps/browsers; historical troubleshooting/investigation is directly applicable."),
    "GOLD-0052": ("RELEVANT", "Directly addresses Prime delivery-date expectations and delayed delivery."),
    "GOLD-0053": ("RELEVANT", "Both concern Amazon Logistics/delivery timing problems; historical response is relevant evidence."),
    "GOLD-0054": ("RELEVANT", "Same carrier/delivery-window problem; historical response gives an appropriate reporting path."),
    "GOLD-0055": ("RELEVANT", "Same delivery-service context, with historical evidence addressing the customer's delivery experience."),
    "GOLD-0056": ("RELEVANT", "Same technical syncing/data-loss concern; historical response provides directly useful information about app deletion and sync."),
    "GOLD-0057": ("RELEVANT", "Same missed/delayed delivery situation; historical response provides an appropriate support/investigation path."),
    "GOLD-0058": ("RELEVANT", "Both concern failed delivery attempts and require delivery investigation."),
    "GOLD-0059": ("RELEVANT", "Same preorder/release-date delivery delay context; historical response is directly applicable."),
    "GOLD-0060": ("RELEVANT", "Same package showing out-for-delivery without successful receipt; historical response appropriately directs the customer to support.")
}

for golden_id, (label, note) in batch3_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 3 saved successfully")

print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch3_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 3 saved successfully
evidence_relevance
RELEVANT    18
PARTIAL      2
Name: count, dtype: int64


In [12]:
BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[3 * BATCH_SIZE:4 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 4")
print("=" * 80)
print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 4
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
60,GOLD-0061,@AmazonHelp Thanks for your early response. I ...,GENERAL_SUPPORT,AMZ-081133,0.998710,[user] i contacted customer service team and g...,[USER] We're glad your issue is resolved. Plea...,GENERAL_SUPPORT
61,GOLD-0062,"Here's what I ordered: two ""ten piece"" packs o...",ORDER_DELIVERY,AMZ-068626,0.873108,[user] i have two different packages for two d...,"[USER] Thanks for reaching out to us, Ava! To ...",GENERAL_SUPPORT
62,GOLD-0063,@115850 does not have provision to enter #GSTN...,PAYMENT_BILLING,AMZ-002485,0.810474,[user] please make all your products available...,[USER] We currently include only seller's GST ...,INVESTIGATION_OR_FOLLOWUP
63,GOLD-0064,@AmazonHelp I still have it! I’ve requested a ...,RETURNS_REFUNDS,AMZ-012809,0.888933,[user] sure i have send the details as request...,[USER] Thanks for the update. Our teams will r...,GENERAL_SUPPORT
64,GOLD-0065,@AmazonHelp If I cancel a purchase made with a...,ORDER_CANCELLATION,AMZ-032586,0.982492,[user] so if i buy the year and cancel will i ...,[USER] Here is some information about the Prim...,REFUND_OR_COMPENSATION
65,GOLD-0066,"@115830 very poor customer service today, cert...",GENERAL_SUPPORT,AMZ-017686,0.999345,[user] your customer service has really gone d...,[USER] I'm sorry you've had a negative experie...,GENERAL_SUPPORT
66,GOLD-0067,@AmazonHelp I think I have just replied about ...,GENERAL_SUPPORT,AMZ-007421,0.987583,[user] i have contacted on your contacts but i...,[USER] We'll take another look at it. Please s...,INVESTIGATION_OR_FOLLOWUP
67,GOLD-0068,@AmazonHelp Done,GENERAL_SUPPORT,AMZ-035305,0.993022,[user] done,[USER] Thanks for the update. We've received y...,INVESTIGATION_OR_FOLLOWUP
68,GOLD-0069,@AmazonHelp Still don't understand why my sist...,ORDER_DELIVERY,AMZ-065489,0.586906,"[user] hey, jeff my sisters package keeps gett...","[USER] Hi Stephen, we'll be pleased to help yo...",INVESTIGATION_OR_FOLLOWUP
69,GOLD-0070,@AmazonHelp Help form completed 2 days ago &am...,ACCOUNT_ACCESS,AMZ-013242,0.987085,[user] [user] still not getting response very ...,[USER] Our teams are working on your issue rig...,INVESTIGATION_OR_FOLLOWUP


In [13]:
batch4_labels = {
    "GOLD-0061": ("RELEVANT", "Same situation where the customer has already contacted support and the issue is being closed/resolved."),
    "GOLD-0062": ("RELEVANT", "Both concern multiple packages/orders and delivery handling; historical response provides a relevant support path."),
    "GOLD-0063": ("RELEVANT", "Directly relevant to GST information on Amazon products and seller-provided GST details."),
    "GOLD-0064": ("RELEVANT", "Same return/refund investigation context; customer has supplied requested information and the historical response describes the next step."),
    "GOLD-0065": ("RELEVANT", "Directly concerns cancellation of a Prime purchase and refund implications."),
    "GOLD-0066": ("PARTIAL", "Both involve customer-service dissatisfaction, but the historical case is generic and doesn't provide a specific solution."),
    "GOLD-0067": ("RELEVANT", "Same repeated-contact/unresolved support situation; historical response provides an investigation path."),
    "GOLD-0068": ("RELEVANT", "Same support follow-up workflow where the customer has completed the requested action."),
    "GOLD-0069": ("RELEVANT", "Same recurring delivery failure involving a package; historical response offers a direct support path."),
    "GOLD-0070": ("RELEVANT", "Directly related to account-access recovery and delayed response after submitting the help form."),
    "GOLD-0071": ("PARTIAL", "Both concern marketplace/customer-service dissatisfaction, but the historical case is too generic to provide strong evidence for the specific supplier complaint."),
    "GOLD-0072": ("RELEVANT", "Both are positive/social interactions with Amazon support; the historical exchange is directly aligned with the conversational intent."),
    "GOLD-0073": ("RELEVANT", "Both concern an unexpected delivery timing event; historical response acknowledges and addresses the delivery discrepancy."),
    "GOLD-0074": ("RELEVANT", "Same Prime/delivery-service dissatisfaction context and historical response addresses the delivery issue."),
    "GOLD-0075": ("PARTIAL", "Both involve dissatisfaction with Prime, but the historical response doesn't provide strong evidence specifically about controlling/canceling renewal."),
    "GOLD-0076": ("PARTIAL", "Related to Prime/customer dissatisfaction, but historical evidence doesn't directly address the decision to stop renewing."),
    "GOLD-0077": ("RELEVANT", "Refund-related issue with a historical response checking refund communication/status."),
    "GOLD-0078": ("RELEVANT", "Both are positive/social Amazon interactions around products and shopping; historical response is useful conversational evidence."),
    "GOLD-0079": ("RELEVANT", "Same delivery-expectation context; historical response explains shipping/delivery behavior."),
    "GOLD-0080": ("RELEVANT", "Directly concerns whether an exchange offer is available; historical response provides the applicable policy/status information.")
}

for golden_id, (label, note) in batch4_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 4 saved successfully")

print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch4_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 4 saved successfully
evidence_relevance
RELEVANT    16
PARTIAL      4
Name: count, dtype: int64


In [14]:
BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[4 * BATCH_SIZE:5 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 5")
print("=" * 80)
print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 5
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
80,GOLD-0081,@AmazonHelp third class help from you guys bee...,GENERAL_SUPPORT,AMZ-000858,0.912553,[user] some help you guys are.,[USER] Sorry for the delay in responding. We'd...,INVESTIGATION_OR_FOLLOWUP
81,GOLD-0082,@115850 I ordered an I phone for my fiance for...,PAYMENT_BILLING,AMZ-054093,0.967219,"[user] i have placed an order, i have entered ...",[USER] Let us look into it. Please reach out t...,INVESTIGATION_OR_FOLLOWUP
82,GOLD-0083,@AmazonHelp ...,GENERAL_SUPPORT,AMZ-015682,0.666563,[user] et donc ... ?,"[USER] la date affichée, est estimative, dès q...",GENERAL_SUPPORT
83,GOLD-0084,".@116316 interessant, dass Pakete mittlerweile...",ORDER_DELIVERY,AMZ-009495,0.925202,"[user] is das euer ernst, dass euer zusteller ...","[USER] Wenn du möchtest, sehen wir, das Amazon...",GENERAL_SUPPORT
84,GOLD-0085,@AmazonHelp There were 3 seperate charges... £...,PAYMENT_BILLING,AMZ-037726,0.982021,[user] yes but there are 2 different charges o...,[USER] We'd be more than happy to research thi...,INVESTIGATION_OR_FOLLOWUP
85,GOLD-0086,@AmazonHelp Is there an email I can write to l...,GENERAL_SUPPORT,AMZ-035699,0.978201,[user] is there a direct line or email i can use?,[USER] We'd love to help you. You're welcome t...,INVESTIGATION_OR_FOLLOWUP
86,GOLD-0087,@AmazonHelp Doutor Sono e It foram uns dos mai...,GENERAL_SOCIAL,AMZ-053428,0.108973,"[user] mañana os lo consulto por alli, gracias...",[USER] Con gusto Angel. Siempre que nos necesi...,GENERAL_SUPPORT
87,GOLD-0088,@AmazonHelp exp 10.27 arrived in PR 10.25 retu...,RETURNS_REFUNDS,AMZ-031493,0.795738,"[user] it says ""no tracking details"".","[USER] Hey, please get in touch - [URL] we'll ...",INVESTIGATION_OR_FOLLOWUP
88,GOLD-0089,@AmazonHelp Thanks!,GENERAL_SUPPORT,AMZ-008468,0.998726,[user] thanks!,[USER] No problem! Let us know if you need any...,GENERAL_SUPPORT
89,GOLD-0090,"@AmazonHelp I was sent the wrong item, who do ...",RETURNS_REFUNDS,AMZ-004284,0.994810,[user] i’ve been sent the wrong item. help?,[USER] I'm so sorry you received the wrong ite...,REFUND_OR_COMPENSATION


In [15]:
batch5_labels = {
    "GOLD-0081": ("RELEVANT", "Same unresolved customer-support situation; historical response provides an appropriate investigation/follow-up path."),
    "GOLD-0082": ("PARTIAL", "Historical case involves an order/payment-related investigation, but the specific billing issue is not sufficiently represented in the retrieved case."),
    "GOLD-0083": ("RELEVANT", "Both involve a customer asking about an estimated delivery date; historical response provides directly applicable information."),
    "GOLD-0084": ("RELEVANT", "Same delivery-driver/service issue; historical response provides a relevant support path."),
    "GOLD-0085": ("RELEVANT", "Directly related to multiple/duplicate charges and investigation of billing transactions."),
    "GOLD-0086": ("RELEVANT", "Directly answers the customer's need for a contact channel/email."),
    "GOLD-0087": ("RELEVANT", "Same general social/support conversation; historical response is appropriate conversational evidence."),
    "GOLD-0088": ("PARTIAL", "Both involve tracking/return-related delivery information, but the historical response mainly redirects to support rather than resolving the specific return issue."),
    "GOLD-0089": ("RELEVANT", "Exact type of interaction: customer thanks support and support closes the conversation."),
    "GOLD-0090": ("RELEVANT", "Directly matches wrong-item return/refund handling."),
    "GOLD-0091": ("PARTIAL", "Historical case discusses an exchange, while the golden intent is payment/billing; related commerce context but not strong billing evidence."),
    "GOLD-0092": ("RELEVANT", "Same escalation/manager-contact situation; historical response provides an investigation path."),
    "GOLD-0093": ("RELEVANT", "Same support follow-up context and historical response provides a concrete next step."),
    "GOLD-0094": ("RELEVANT", "Directly concerns an incorrectly delivered parcel and provides an appropriate response."),
    "GOLD-0095": ("RELEVANT", "Same resolved customer-service interaction; strong evidence for closure behavior."),
    "GOLD-0096": ("RELEVANT", "Customer reports an undelivered order; historical response provides the appropriate investigation/contact path."),
    "GOLD-0097": ("RELEVANT", "Same delivery problem after troubleshooting has already been attempted; historical response continues the investigation."),
    "GOLD-0098": ("RELEVANT", "Same order/tracking anomaly involving unexpected delivery communication."),
    "GOLD-0099": ("RELEVANT", "Both concern an order still showing an unresolved delivery status; historical response initiates investigation."),
    "GOLD-0100": ("RELEVANT", "Direct order-help request with a relevant historical support response.")
}

for golden_id, (label, note) in batch5_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 5 saved successfully")

print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch5_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 5 saved successfully
evidence_relevance
RELEVANT    17
PARTIAL      3
Name: count, dtype: int64


In [16]:
BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[5 * BATCH_SIZE:6 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 6")
print("=" * 80)
print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 6
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
100,GOLD-0101,@121284 will it accept commands to work with @...,DEVICE_TECHNICAL,AMZ-019430,0.926913,[user] is that functionality planned?,[USER] We do not currently have any informatio...,INVESTIGATION_OR_FOLLOWUP
101,GOLD-0102,"Crap day. Amazon delivery never turned up, wor...",ORDER_DELIVERY,AMZ-080381,0.970032,[user] took day off work for a delivery that n...,[USER] I am terribly sorry to hear that James....,INVESTIGATION_OR_FOLLOWUP
102,GOLD-0103,@115821 Why has your recent upgrade suddenly s...,ACCOUNT_ACCESS,AMZ-035875,0.994823,[user] gift card problem we had a problem rede...,[USER] Please connect with our support team he...,GENERAL_SUPPORT
103,GOLD-0104,@AmazonHelp We actually need to talk to a syst...,ACCOUNT_SECURITY,AMZ-030267,0.926039,[user] my account has been hacked and my email...,[USER] I'm sorry to hear this. Please contact ...,INVESTIGATION_OR_FOLLOWUP
104,GOLD-0105,@121284 My Echo Dot has reached my city😂 but n...,ORDER_DELIVERY,AMZ-052914,0.998054,[user] dispatched but not yet out for delivery.,[USER] If the order is dispatched with Amazon ...,GENERAL_SUPPORT
105,GOLD-0106,"@AmazonHelp Yep like 3 times, Fri, sat, &amp; ...",ORDER_DELIVERY,AMZ-076401,0.968927,[user] they are both through ups. now say dela...,[USER] I understand your frustration. Did we g...,GENERAL_SUPPORT
106,GOLD-0107,@AmazonHelp Can I get the money I paid for nex...,ORDER_DELIVERY,AMZ-040962,0.995958,[user] will i get a refund? i paid extra for n...,"[USER] When you can, please reach us via phone...",INVESTIGATION_OR_FOLLOWUP
107,GOLD-0108,"@AmazonHelp I already saw your help..,Now u wi...",GENERAL_SUPPORT,AMZ-024590,0.975625,[user] i have provided a phone number for furt...,[USER] Thanks for letting us know! Please give...,INVESTIGATION_OR_FOLLOWUP
108,GOLD-0109,@AmazonHelp I’ve now cancelled I think 9 order...,ORDER_CANCELLATION,AMZ-049754,0.960710,[user] i've just cancelled my order then re or...,[USER] That is great! Let us know if you have ...,GENERAL_SUPPORT
109,GOLD-0110,@AmazonHelp Can you help me with my order,GENERAL_SUPPORT,AMZ-023094,0.999566,[user] had a question about my order if you co...,[USER] We cannot access order or account infor...,INVESTIGATION_OR_FOLLOWUP


In [17]:
batch6_labels = {
    "GOLD-0101": ("RELEVANT", "Same device/feature capability question; historical response directly addresses whether the functionality is currently supported/planned."),
    "GOLD-0102": ("RELEVANT", "Same missed-delivery situation and customer impact; historical response is directly useful for handling the complaint."),
    "GOLD-0103": ("PARTIAL", "Both involve account/order-related problems, but the historical case is specifically about a gift-card issue rather than the account-access problem."),
    "GOLD-0104": ("RELEVANT", "Direct account-security compromise; historical response provides the appropriate support/escalation path."),
    "GOLD-0105": ("RELEVANT", "Same dispatched-but-not-yet-out-for-delivery situation; historical response directly explains delivery handling."),
    "GOLD-0106": ("RELEVANT", "Both involve delayed deliveries and repeated delivery attempts; historical response is useful for continued investigation."),
    "GOLD-0107": ("RELEVANT", "Directly concerns refund of an additional delivery charge; historical response provides the appropriate contact/investigation path."),
    "GOLD-0108": ("RELEVANT", "Same situation of providing contact information for further investigation/support."),
    "GOLD-0109": ("RELEVANT", "Same order-cancellation workflow; historical case shows the customer successfully cancelled/reordered."),
    "GOLD-0110": ("PARTIAL", "General order-help context is related, but the historical response is mostly a generic limitation rather than useful evidence for the specific question."),
    "GOLD-0111": ("RELEVANT", "Same gift-wrapping/product-service context; historical response provides useful information about gift handling."),
    "GOLD-0112": ("RELEVANT", "Directly concerns inappropriate package placement by delivery personnel; historical response provides escalation/investigation."),
    "GOLD-0113": ("RELEVANT", "Same resolved-support conversational context; historical response appropriately closes the interaction."),
    "GOLD-0114": ("RELEVANT", "Directly aligned with a customer thanking Amazon support; historical response is appropriate conversational evidence."),
    "GOLD-0115": ("RELEVANT", "Same poor-quality product/return investigation context; historical response provides an appropriate support path."),
    "GOLD-0116": ("RELEVANT", "Same tracking/account-information limitation; historical response explains the appropriate support boundary."),
    "GOLD-0117": ("RELEVANT", "Both concern an incomplete order/delivery; historical response provides the correct next step for missing items."),
    "GOLD-0118": ("RELEVANT", "Return-policy question with historical evidence providing a policy-information link."),
    "GOLD-0119": ("RELEVANT", "Same digital-content purchase question; historical response provides relevant purchasing information."),
    "GOLD-0120": ("PARTIAL", "Same marketplace-service dissatisfaction, but the historical response is generic and doesn't provide specific seller-performance evidence.")
}

for golden_id, (label, note) in batch6_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 6 saved successfully")

print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch6_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 6 saved successfully
evidence_relevance
RELEVANT    17
PARTIAL      3
Name: count, dtype: int64


In [18]:
BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[6 * BATCH_SIZE:7 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 7")
print("=" * 80)
print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 7
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
120,GOLD-0121,@AmazonHelp It doesn't appear to be. It appear...,ACCOUNT_SECURITY,AMZ-078824,0.978682,[user] is there some way i can send/report to ...,[USER] Thanks for reaching out to us today. Yo...,INVESTIGATION_OR_FOLLOWUP
121,GOLD-0122,@117804 Lindão!,GENERAL_SOCIAL,AMZ-066269,0.950811,"[user] obrigada, seu lindos! ❤️","[USER] De nada, Fernanda. Conte conosco sempre...",GENERAL_SUPPORT
122,GOLD-0123,@AmazonHelp Thank you! It arrived today. I'll ...,GENERAL_SOCIAL,AMZ-063046,0.999330,[user] it arrived today.,[USER] We're glad you received your package. L...,GENERAL_SUPPORT
123,GOLD-0124,"A 5,600 fue irresistible. Gracias, @116875! ht...",GENERAL_SOCIAL,AMZ-034578,0.118315,[user] son lo máximo ¡mil gracias!,[USER] Entrenamos para brindarte el mejor serv...,GENERAL_SUPPORT
124,GOLD-0125,@115850 @7092 @115850 start using UID Aadhar t...,ORDER_DELIVERY,AMZ-046673,0.094121,[user] can you check order no # [phone_number]...,[USER] Please don't provide your order details...,INVESTIGATION_OR_FOLLOWUP
125,GOLD-0126,@AmazonHelp OK Sur/mam... But my question is w...,RETURNS_REFUNDS,AMZ-033228,0.463279,"[user] not yet, where do i go to do that?","[USER] Hey, you can update your payment method...",GENERAL_SUPPORT
126,GOLD-0127,My Husband found his new nutritional drink Abb...,GENERAL_SOCIAL,AMZ-041011,0.070005,[user] found it thank you,[USER] Thanks for keeping us posted! Let us kn...,GENERAL_SUPPORT
127,GOLD-0128,@AmazonHelp Which email are you talking about?...,GENERAL_SUPPORT,AMZ-005324,0.996294,"[user] i've not received any new email, which ...",[USER] Kindly reply to the email sent by us an...,GENERAL_SUPPORT
128,GOLD-0129,Am I no longer able to access the @144771 with...,DIGITAL_CONTENT,AMZ-003625,0.804300,[user] i am not able to login,[USER] Kindly connect with us via [URL] ^SG,GENERAL_SUPPORT
129,GOLD-0130,"@AmazonHelp Iba a eso en un rato, que no he po...",GENERAL_SUPPORT,AMZ-044137,0.661529,[user] porque como muy pronto hasta mañana no ...,[USER] Puedes reportar la incidencia con nuest...,GENERAL_SUPPORT


In [19]:
batch7_labels = {
    "GOLD-0121": ("RELEVANT", "Direct account-security/reporting context; historical response provides an appropriate support path."),
    "GOLD-0122": ("RELEVANT", "Same positive/social interaction; historical response is appropriate conversational evidence."),
    "GOLD-0123": ("RELEVANT", "Customer confirms delivery; historical response directly acknowledges successful receipt."),
    "GOLD-0124": ("RELEVANT", "Same positive customer-support interaction; historical response is appropriate conversational evidence despite the low CE score."),
    "GOLD-0125": ("RELEVANT", "Historical case directly involves an order lookup and appropriately warns against sharing order details publicly."),
    "GOLD-0126": ("PARTIAL", "Both involve an unresolved customer issue, but the retrieved case is primarily about updating a payment method rather than the return/refund concern."),
    "GOLD-0127": ("RELEVANT", "Customer confirms the issue is resolved; historical response appropriately acknowledges the resolution."),
    "GOLD-0128": ("RELEVANT", "Same missing-email/follow-up situation; historical response provides a concrete next step."),
    "GOLD-0129": ("PARTIAL", "Both involve access/login problems, but the historical case doesn't specifically address the digital-content access context."),
    "GOLD-0130": ("RELEVANT", "Same support/reporting context; historical response provides an appropriate incident-reporting path."),
    "GOLD-0131": ("PARTIAL", "Customer indicates they were helped, but retrieved evidence shifts into troubleshooting and doesn't strongly support the exact social interaction."),
    "GOLD-0132": ("RELEVANT", "Historical response provides a direct Amazon support contact path for an ambiguous support request."),
    "GOLD-0133": ("RELEVANT", "Directly related to cancellation and potential authorization/refund behavior."),
    "GOLD-0134": ("RELEVANT", "Strong account-security evidence involving a suspected phishing/scam message; historical response directly addresses authenticity."),
    "GOLD-0135": ("RELEVANT", "Same failed-delivery situation; historical response provides an appropriate support path."),
    "GOLD-0136": ("RELEVANT", "Same return/product-condition investigation context; historical response is useful for handling the complaint."),
    "GOLD-0137": ("RELEVANT", "Directly concerns inappropriate package placement; historical response provides a concrete delivery-setting remedy."),
    "GOLD-0138": ("PARTIAL", "General customer-service dissatisfaction is related, but the historical case doesn't address the specific product/purchase issue."),
    "GOLD-0139": ("RELEVANT", "Direct refund-not-received situation; historical response directly addresses delayed refund handling."),
    "GOLD-0140": ("RELEVANT", "Same unexplained delivery problem; historical response provides a useful courier-feedback/escalation action.")
}

for golden_id, (label, note) in batch7_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 7 saved successfully")

print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch7_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 7 saved successfully
evidence_relevance
RELEVANT    16
PARTIAL      4
Name: count, dtype: int64


In [20]:
BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[7 * BATCH_SIZE:8 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 8")
print("=" * 80)
print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 8
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
140,GOLD-0141,So I order a tv from @115830 and is to be deli...,ORDER_DELIVERY,AMZ-070844,0.965121,[user] why does it say “one day delivery” and ...,[USER] Apologies this is now one of the busies...,INVESTIGATION_OR_FOLLOWUP
141,GOLD-0142,"@115821 US ""ships"" but @115850 India "" dispat...",GENERAL_SOCIAL,AMZ-047992,0.277978,[user] are taxes high in india ??,[USER] Taxes are charged on an order as per st...,GENERAL_SUPPORT
142,GOLD-0143,@AmazonHelp そうなのですね、予約なのでもう少し待ってみます。,PREORDER_DELIVERY,AMZ-044931,0.904968,[user] そうなのか…今回の件は諦めて以前予約注文したままにしてるので届くのを待ちたいと...,[USER] わかりづらい点があり、申し訳ございません。どうぞよろしくお願いいたします。ET,GENERAL_SUPPORT
143,GOLD-0144,@AmazonHelp I used call back earlier on this m...,GENERAL_SUPPORT,AMZ-021545,0.997005,"[user] the issue is resolved ,thanks for your ...",[USER] You're most welcome. Thanks for keeping...,GENERAL_SUPPORT
144,GOLD-0145,@115850 Mst pathetic xperience for prdct retun...,RETURNS_REFUNDS,AMZ-062843,0.883938,[user] worst experince ever..your amazon trans...,"[USER] I'm sorry, this wasn't intended. Did yo...",GENERAL_SUPPORT
145,GOLD-0146,"Credit where it's due, I've found @AmazonHelp ...",DIGITAL_CONTENT,AMZ-051887,0.818700,"[user] thanks, for the help my kindle is worki...",[USER] That's great to hear! Let us know if we...,GENERAL_SUPPORT
146,GOLD-0147,@119625 Why can’t you add cast info for South ...,DIGITAL_CONTENT,AMZ-056453,0.290710,[user] why you not adding malayalam movie sect...,[USER] Regional content is an integral part of...,GENERAL_SUPPORT
147,GOLD-0148,@AmazonHelp 4 Tage her und noch nicht mal vers...,ORDER_DELIVERY,AMZ-041032,0.484200,[user] wenn das mehrfach passiert und es dann ...,"[USER] Wir, vom Amazon.de Social Media Team, g...",GENERAL_SUPPORT
148,GOLD-0149,@AmazonHelp @115850 @115821 Amazon customer ca...,RETURNS_REFUNDS,AMZ-030303,0.802473,[user] i have an issue with returning an order...,[USER] here: [URL] and we'll take the necessar...,GENERAL_SUPPORT
149,GOLD-0150,My account is suspend by amazon and i was subm...,ACCOUNT_ACCESS,AMZ-000859,0.994879,[user] i have some issues with my amazon accou...,[USER] We'll surely help you out. Could you pl...,INVESTIGATION_OR_FOLLOWUP


In [21]:
batch8_labels = {
    "GOLD-0141": ("RELEVANT", "Same one-day/expected delivery timing issue; historical response provides useful context for delivery delays."),
    "GOLD-0142": ("RELEVANT", "Directly addresses taxes charged on Amazon orders, which is useful evidence for the customer's tax question."),
    "GOLD-0143": ("RELEVANT", "Same preorder/waiting-for-delivery context; historical response is applicable to the customer's situation."),
    "GOLD-0144": ("RELEVANT", "Exact resolved-support interaction; historical response confirms closure."),
    "GOLD-0145": ("RELEVANT", "Same return experience/customer-service problem; historical response provides an appropriate follow-up path."),
    "GOLD-0146": ("RELEVANT", "Same Kindle/device digital-content support context and confirms successful resolution."),
    "GOLD-0147": ("RELEVANT", "Directly related to regional movie/content availability and metadata."),
    "GOLD-0148": ("RELEVANT", "Same repeated delivery failure context; historical response provides relevant support handling."),
    "GOLD-0149": ("RELEVANT", "Directly concerns returning an order and provides a return-support path."),
    "GOLD-0150": ("RELEVANT", "Same account-access/support investigation context."),
    "GOLD-0151": ("RELEVANT", "Same refund/cashback resolution interaction; historical response confirms successful receipt."),
    "GOLD-0152": ("PARTIAL", "Historical evidence addresses a delivery problem, but the golden case combines missing delivery with account lock and payment concerns."),
    "GOLD-0153": ("PARTIAL", "Same customer-service dissatisfaction in digital content, but the historical response doesn't provide much specific content-related resolution evidence."),
    "GOLD-0154": ("RELEVANT", "Direct technical troubleshooting/resolution evidence; customer confirms the suggested solution worked."),
    "GOLD-0155": ("RELEVANT", "Same delivery-driver/package-placement problem; historical response explicitly investigates the issue."),
    "GOLD-0156": ("RELEVANT", "Direct order-tracking/progress request with a support response addressing the request."),
    "GOLD-0157": ("RELEVANT", "Same undelivered/late shipment situation; historical response provides the appropriate next step."),
    "GOLD-0158": ("RELEVANT", "Same delayed delivery/customer-service issue; historical response acknowledges the problem and provides support context."),
    "GOLD-0159": ("RELEVANT", "Same repeated-complaint/follow-up situation; historical response gives a concrete response-time expectation."),
    "GOLD-0160": ("RELEVANT", "Same unresolved delivery complaint; historical response provides relevant investigation/support context.")
}

for golden_id, (label, note) in batch8_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 8 saved successfully")

print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch8_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 8 saved successfully
evidence_relevance
RELEVANT    18
PARTIAL      2
Name: count, dtype: int64


In [22]:
BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[8 * BATCH_SIZE:9 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 9")
print("=" * 80)
print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 9
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
160,GOLD-0161,Bonjour @120533 @AmazonHelp 2eme fois que je s...,ORDER_DELIVERY,AMZ-053470,0.722689,[user] oui j'ai appelé on m'a dit qu'on me rap...,"[USER] D'accord, tenez-nous informé. ^BR",GENERAL_SUPPORT
161,GOLD-0162,Stayed home and did my #BlackFriday shopping o...,ORDER_TRACKING,AMZ-078516,0.901089,[user] both packages in this order are late. o...,[USER] I'd like a member of our Amazon Logisti...,INVESTIGATION_OR_FOLLOWUP
162,GOLD-0163,@AmazonHelp I have not. The given delivery dat...,ORDER_DELIVERY,AMZ-028061,0.994332,[user] i was not given a reason for the delay.,[USER] Oh my! Let's see if we can help further...,GENERAL_SUPPORT
163,GOLD-0164,@115850 I ordrd a large size fitness band. I'v...,ORDER_DELIVERY,AMZ-020922,0.976138,[user] i ordered a small fitbit and you’ve sen...,"[USER] Keep us updated on the re-order, Katie....",GENERAL_SUPPORT
164,GOLD-0165,@118919 my order Order Id: 405-3097032-312515...,ORDER_TRACKING,AMZ-007576,0.990905,[user] ordered redmi4 on 24th sept- order # [p...,[USER] Please don’t provide your order details...,INVESTIGATION_OR_FOLLOWUP
165,GOLD-0166,@115821 well done great job !! Box looks like ...,ORDER_DELIVERY,AMZ-083459,0.969066,when your neighbour takes in your delivery fro...,[USER] I apologize for the poor delivery exper...,INVESTIGATION_OR_FOLLOWUP
166,GOLD-0167,@AmazonHelp I have already filled out this for...,GENERAL_SUPPORT,AMZ-081688,0.999767,[user] i've filled in the form.,[USER] Thank you! Please allow us up to 12 hou...,GENERAL_SUPPORT
167,GOLD-0168,@115830 why does it need to be so hard to fin...,GENERAL_SUPPORT,AMZ-013895,0.936345,[user] why is it so hard to find ways to conta...,[USER] You can contact us from any of our Help...,INVESTIGATION_OR_FOLLOWUP
168,GOLD-0169,"@117086 Eu só queria dizer que te amo, só prom...",GENERAL_SOCIAL,AMZ-044496,0.998888,"[user] eu te amo ❤️ meus livros chegam rápido,...",[USER] Olá! Saiba que o amor é mútuo. Só o mel...,GENERAL_SUPPORT
169,GOLD-0170,@AmazonHelp I need the item delivered today. R...,ORDER_DELIVERY,AMZ-022109,0.988654,[user] i need the item for work.,[USER] Who does the tracking indicate as the c...,GENERAL_SUPPORT


In [23]:
batch9_labels = {
    "GOLD-0161": ("RELEVANT", "Same delivery follow-up situation; historical response appropriately continues the investigation/support process."),
    "GOLD-0162": ("RELEVANT", "Directly concerns multiple late packages and provides an Amazon Logistics escalation path."),
    "GOLD-0163": ("RELEVANT", "Same unexplained delivery-delay problem; historical response offers further assistance."),
    "GOLD-0164": ("RELEVANT", "Same order/product fulfillment context involving an incorrect item and reorder; historical response provides useful follow-up."),
    "GOLD-0165": ("RELEVANT", "Direct order-tracking context; historical response appropriately directs the customer away from publicly sharing order details."),
    "GOLD-0166": ("RELEVANT", "Same package delivery/neighbor handoff problem; historical response addresses the poor delivery experience."),
    "GOLD-0167": ("RELEVANT", "Same support-form completion/follow-up workflow; historical response gives a concrete expected response window."),
    "GOLD-0168": ("RELEVANT", "Directly addresses difficulty finding Amazon contact channels and provides the appropriate Help/contact route."),
    "GOLD-0169": ("RELEVANT", "Same positive/social interaction involving Amazon service; historical response is appropriate conversational evidence."),
    "GOLD-0170": ("PARTIAL", "Delivery-related, but the historical case focuses on identifying the carrier from tracking rather than guaranteeing delivery today."),
    "GOLD-0171": ("RELEVANT", "Directly concerns an unexpected Prime charge and provides an investigation path."),
    "GOLD-0172": ("RELEVANT", "Same Prime Pantry delivery-delay context; historical response is useful for handling the delay."),
    "GOLD-0173": ("PARTIAL", "Customer dissatisfaction is related, but the historical response focuses on asking for clarification rather than resolving the underlying complaint."),
    "GOLD-0174": ("RELEVANT", "Directly answers the customer's request for the UK contact number/call route."),
    "GOLD-0175": ("RELEVANT", "Same delivery/replacement problem; historical response provides a relevant diagnostic step before replacement."),
    "GOLD-0176": ("RELEVANT", "Directly concerns receiving the wrong mobile product and provides an appropriate support path."),
    "GOLD-0177": ("PARTIAL", "Related to product pricing/reordering, but the historical response discusses price fluctuation rather than the customer's specific reorder concern."),
    "GOLD-0178": ("RELEVANT", "Although the CE score is low, the retrieved case directly concerns duplicate charging and provides an investigation path."),
    "GOLD-0179": ("IRRELEVANT", "The golden query is about a return/printing problem, while the retrieved historical case is a generic positive shopping interaction with no useful return evidence."),
    "GOLD-0180": ("PARTIAL", "Both concern Amazon Household/account functionality, but the historical response addresses sharing options rather than recovering from accidental deletion.")
}

for golden_id, (label, note) in batch9_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 9 saved successfully")

print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch9_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 9 saved successfully
evidence_relevance
RELEVANT      15
PARTIAL        4
IRRELEVANT     1
Name: count, dtype: int64


In [24]:
BATCH_SIZE = 20

evidence_review_batch = (
    evidence_review
    .sort_values("golden_id")
    .iloc[9 * BATCH_SIZE:10 * BATCH_SIZE]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 10")
print("=" * 80)
print(f"Cases: {len(evidence_review_batch)}")

display(
    evidence_review_batch[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
)

TOP-1 HISTORICAL EVIDENCE — MANUAL REVIEW BATCH 10
Cases: 20


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type
180,GOLD-0181,@AmazonHelp Doesn't seem to.,GENERAL_SUPPORT,AMZ-014239,0.989168,[user] not yet.,[USER] What information or insight was provide...,INVESTIGATION_OR_FOLLOWUP
181,GOLD-0182,@AmazonHelp It says my parcel was delivered to...,ORDER_DELIVERY,AMZ-004149,0.999761,[user] my tracker says that a parcel has been ...,[USER] Oh no! I'm sorry for the trouble! Have ...,GENERAL_SUPPORT
182,GOLD-0183,@AmazonHelp — to cancel my order or just order...,ORDER_DELIVERY,AMZ-019687,0.993386,"[user] orders, then i may as well cancel prime.",[USER] I'm sorry for the inconvenience. Howeve...,GENERAL_SUPPORT
183,GOLD-0184,@AmazonHelp faced aweful experience from amazo...,ORDER_DELIVERY,AMZ-038873,0.915328,[user] you guys have some real problem with yo...,[USER] [URL] and we'll fix this for you. (2/2)^GD,GENERAL_SUPPORT
184,GOLD-0185,@AmazonHelp Thx Buddy👍😊💐,GENERAL_SOCIAL,AMZ-050433,0.691504,[user] thx ^_^,[USER] You're welcome! Let us know if you have...,GENERAL_SUPPORT
185,GOLD-0186,@AmazonHelp I have submitted the feedback from...,GENERAL_SUPPORT,AMZ-005552,0.993483,[user] i have shared details in email to amazo...,[USER] You can check the correspondences to yo...,INVESTIGATION_OR_FOLLOWUP
186,GOLD-0187,@115830 my safe place for parcels is my rear p...,ORDER_DELIVERY,AMZ-027694,0.962611,how is outside my front door in clear view of ...,[USER] I'm sorry that wasn't delivered to your...,GENERAL_SUPPORT
187,GOLD-0188,"@130626 , would love to be playing 2k18 now, b...",PREORDER_DELIVERY,AMZ-079880,0.995349,i would be playing #xenobladechronicles2 right...,[USER] Did you receive an e-mail regarding thi...,GENERAL_SUPPORT
188,GOLD-0189,@AmazonHelp So past me having to ship it back ...,RETURNS_REFUNDS,AMZ-026318,0.884919,[user] you provided the service to sell the pr...,[USER] Did you contact us directly here?: [URL...,INVESTIGATION_OR_FOLLOWUP
189,GOLD-0190,@AmazonHelp My orders no. 406-7064760-6917910 ...,ORDER_DELIVERY,AMZ-000808,0.999357,[user] delivery service is very poor,[USER] Sorry to know about the delivery issue....,INVESTIGATION_OR_FOLLOWUP


In [25]:
batch10_labels = {
    "GOLD-0181": ("RELEVANT", "Same unresolved-support/follow-up context; historical response is useful for continuing the investigation."),
    "GOLD-0182": ("RELEVANT", "Directly concerns a package marked delivered but apparently not received; historical response is highly applicable."),
    "GOLD-0183": ("PARTIAL", "Related to Prime/order cancellation, but the historical response doesn't directly address the specific cancellation/order-control question."),
    "GOLD-0184": ("RELEVANT", "Same delivery-service complaint and historical response provides a concrete support/fix path."),
    "GOLD-0185": ("RELEVANT", "Same simple support acknowledgment/thank-you interaction; useful conversational evidence."),
    "GOLD-0186": ("RELEVANT", "Same support follow-up after submitting information; historical response provides a concrete way to check correspondence."),
    "GOLD-0187": ("RELEVANT", "Direct package-placement/delivery-location problem; historical response addresses incorrect delivery placement."),
    "GOLD-0188": ("RELEVANT", "Same preorder/release-date context; historical response provides relevant delivery-status information."),
    "GOLD-0189": ("RELEVANT", "Same return-related customer complaint and historical response directs the customer to the appropriate support path."),
    "GOLD-0190": ("RELEVANT", "Directly concerns poor delivery service and provides relevant investigation/support evidence."),
    "GOLD-0191": ("PARTIAL", "Both concern delivery problems, but the golden intent is payment/billing and the retrieved evidence doesn't address a billing issue."),
    "GOLD-0192": ("PARTIAL", "Historical case concerns an ongoing investigation, but doesn't directly address the customer's specific purchase/payment question."),
    "GOLD-0193": ("PARTIAL", "Both involve digital/pre-release content, but the historical response focuses on shipping timing rather than digital-content availability."),
    "GOLD-0194": ("PARTIAL", "Delivery-related, but the historical case is extremely vague and doesn't provide useful evidence for the actual problem."),
    "GOLD-0195": ("PARTIAL", "Account-access context is related, but the retrieved response lacks a concrete solution to the specific access problem."),
    "GOLD-0196": ("RELEVANT", "Same delivery/customer-service dissatisfaction context; historical response provides an appropriate support response."),
    "GOLD-0197": ("RELEVANT", "Same undelivered-order situation; historical response gives a concrete reporting/escalation path."),
    "GOLD-0198": ("RELEVANT", "Directly answers the need for an Amazon contact channel and provides multiple contact options."),
    "GOLD-0199": ("PARTIAL", "Historical case concerns a package marked delivered, while the golden case is specifically about a digital code; related delivery language but weak evidence for the exact intent."),
    "GOLD-0200": ("RELEVANT", "Same delivery-status/dispatch timing situation; historical response gives a concrete expectation if delivery does not occur.")
}

for golden_id, (label, note) in batch10_labels.items():
    mask = evidence_review["golden_id"] == golden_id
    evidence_review.loc[mask, "evidence_relevance"] = label
    evidence_review.loc[mask, "review_notes"] = note

evidence_review["evidence_relevance"] = (
    evidence_review["evidence_relevance"].fillna("").astype(str)
)
evidence_review["review_notes"] = (
    evidence_review["review_notes"].fillna("").astype(str)
)

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("✅ Batch 10 saved successfully")

print(
    evidence_review
    .loc[evidence_review["golden_id"].isin(batch10_labels.keys()),
         "evidence_relevance"]
    .value_counts()
)

✅ Batch 10 saved successfully
evidence_relevance
RELEVANT    13
PARTIAL      7
Name: count, dtype: int64


In [26]:
# ============================================================
# FINAL TOP-1 EVIDENCE REVIEW — OVERALL METRICS
# ============================================================

import pandas as pd

review = pd.read_csv(
    "data/evaluation/top1_evidence_review.csv"
)

print("=" * 80)
print("FINAL TOP-1 HISTORICAL EVIDENCE EVALUATION")
print("=" * 80)

print(f"Total reviewed cases: {len(review)}")

counts = review["evidence_relevance"].value_counts()

relevant = counts.get("RELEVANT", 0)
partial = counts.get("PARTIAL", 0)
irrelevant = counts.get("IRRELEVANT", 0)

total = len(review)

print("\nEvidence distribution:")
print(f"RELEVANT    : {relevant:3d} ({relevant/total*100:.2f}%)")
print(f"PARTIAL     : {partial:3d} ({partial/total*100:.2f}%)")
print(f"IRRELEVANT  : {irrelevant:3d} ({irrelevant/total*100:.2f}%)")

useful = relevant + partial

print("\nKey metrics:")
print(f"Top-1 Relevant Rate       : {relevant/total*100:.2f}%")
print(f"Top-1 Useful Evidence Rate: {useful/total*100:.2f}%")
print(f"Top-1 Irrelevant Rate      : {irrelevant/total*100:.2f}%")

FINAL TOP-1 HISTORICAL EVIDENCE EVALUATION
Total reviewed cases: 200

Evidence distribution:
RELEVANT    : 160 (80.00%)
PARTIAL     :  37 (18.50%)
IRRELEVANT  :   3 (1.50%)

Key metrics:
Top-1 Relevant Rate       : 80.00%
Top-1 Useful Evidence Rate: 98.50%
Top-1 Irrelevant Rate      : 1.50%


In [27]:
intent_eval = pd.crosstab(
    review["intent"],
    review["evidence_relevance"],
    normalize="index"
).fillna(0) * 100

intent_eval = intent_eval.round(2)

display(intent_eval)

evidence_relevance,IRRELEVANT,PARTIAL,RELEVANT
intent,,,
ACCOUNT_ACCESS,0.00,44.44,55.56
ACCOUNT_SECURITY,25.00,0.00,75.00
DEVICE_TECHNICAL,0.00,0.00,100.00
DIGITAL_CONTENT,0.00,44.44,55.56
GENERAL_SOCIAL,5.88,5.88,88.24
GENERAL_SUPPORT,0.00,14.29,85.71
MARKETPLACE_SELLING,0.00,66.67,33.33
ORDER_CANCELLATION,0.00,25.00,75.00
ORDER_DELIVERY,0.00,8.70,91.30


In [28]:
score_analysis = (
    review
    .groupby("evidence_relevance")["ce_score"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

display(score_analysis)

,count,mean,median,min,max
evidence_relevance,,,,,
IRRELEVANT,3,0.1213,0.0679,0.0667,0.2293
PARTIAL,37,0.7758,0.9651,0.0068,0.9996
RELEVANT,160,0.8738,0.9753,0.0246,1.0000


In [29]:
# ============================================================
# CROSS-ENCODER THRESHOLD ANALYSIS
# Human-reviewed Top-1 evidence
# ============================================================

import numpy as np
import pandas as pd

review = pd.read_csv(
    "data/evaluation/top1_evidence_review.csv"
)

# Direct relevance only
review["is_relevant"] = (
    review["evidence_relevance"] == "RELEVANT"
).astype(int)

thresholds = [
    0.50, 0.60, 0.70, 0.75,
    0.80, 0.85, 0.90, 0.95, 0.97
]

rows = []

for threshold in thresholds:

    predicted_relevant = (
        review["ce_score"] >= threshold
    ).astype(int)

    tp = ((predicted_relevant == 1) &
          (review["is_relevant"] == 1)).sum()

    fp = ((predicted_relevant == 1) &
          (review["is_relevant"] == 0)).sum()

    fn = ((predicted_relevant == 0) &
          (review["is_relevant"] == 1)).sum()

    tn = ((predicted_relevant == 0) &
          (review["is_relevant"] == 0)).sum()

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0
    )

    coverage = predicted_relevant.mean()

    fpr = (
        fp / (fp + tn)
        if (fp + tn)
        else 0
    )

    rows.append({
        "threshold": threshold,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "coverage": round(coverage, 4),
        "false_positive_rate": round(fpr, 4)
    })

threshold_df = pd.DataFrame(rows)

print("=" * 80)
print("CROSS-ENCODER THRESHOLD ANALYSIS")
print("=" * 80)

display(threshold_df)

CROSS-ENCODER THRESHOLD ANALYSIS


,threshold,precision,recall,f1,coverage,false_positive_rate
0,0.50,0.8343,0.9125,0.8716,0.875,0.725
1,0.60,0.8314,0.8938,0.8614,0.860,0.725
2,0.70,0.8333,0.8438,0.8385,0.810,0.675
3,0.75,0.8375,0.8375,0.8375,0.800,0.650
4,0.80,0.8516,0.8250,0.8381,0.775,0.575
5,0.85,0.8533,0.8000,0.8258,0.750,0.550
6,0.90,0.8623,0.7438,0.7987,0.690,0.475
7,0.95,0.8390,0.6188,0.7122,0.590,0.475
8,0.97,0.8485,0.5250,0.6486,0.495,0.375


In [30]:
# ============================================================
# SAVE FINAL EVIDENCE EVALUATION SUMMARY
# ============================================================

evidence_summary = pd.DataFrame([
    {
        "metric": "top1_relevant_rate",
        "value": 0.80,
        "description": "Human-reviewed Top-1 evidence directly rated RELEVANT"
    },
    {
        "metric": "top1_useful_evidence_rate",
        "value": 0.985,
        "description": "Human-reviewed Top-1 evidence rated RELEVANT or PARTIAL"
    },
    {
        "metric": "top1_irrelevant_rate",
        "value": 0.015,
        "description": "Human-reviewed Top-1 evidence rated IRRELEVANT"
    },
    {
        "metric": "ce_threshold_best_f1",
        "value": 0.50,
        "description": "Best F1 threshold on the 200-case calibration set"
    },
    {
        "metric": "ce_threshold_safety_operating_point",
        "value": 0.90,
        "description": "Conservative candidate operating point for strong evidence"
    },
    {
        "metric": "ce_precision_at_0_90",
        "value": 0.8623,
        "description": "Precision for direct relevance at CE >= 0.90"
    },
    {
        "metric": "ce_recall_at_0_90",
        "value": 0.7438,
        "description": "Recall for direct relevance at CE >= 0.90"
    },
    {
        "metric": "ce_coverage_at_0_90",
        "value": 0.6900,
        "description": "Fraction of cases above CE >= 0.90"
    }
])

evidence_summary.to_csv(
    "data/evaluation/evidence_quality_summary.csv",
    index=False
)

print("✅ Final evidence evaluation summary saved")
display(evidence_summary)

✅ Final evidence evaluation summary saved


,metric,value,description
0,top1_relevant_rate,0.8000,Human-reviewed Top-1 evidence directly rated R...
1,top1_useful_evidence_rate,0.9850,Human-reviewed Top-1 evidence rated RELEVANT o...
2,top1_irrelevant_rate,0.0150,Human-reviewed Top-1 evidence rated IRRELEVANT
3,ce_threshold_best_f1,0.5000,Best F1 threshold on the 200-case calibration set
4,ce_threshold_safety_operating_point,0.9000,Conservative candidate operating point for str...
5,ce_precision_at_0_90,0.8623,Precision for direct relevance at CE >= 0.90
6,ce_recall_at_0_90,0.7438,Recall for direct relevance at CE >= 0.90
7,ce_coverage_at_0_90,0.6900,Fraction of cases above CE >= 0.90


In [31]:
# ============================================================
# TRUST CHECKER V1
# Evidence-grounded support automation
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

STRONG_CE_THRESHOLD = 0.90
CONDITIONAL_CE_THRESHOLD = 0.75

STRONG_TRUST_THRESHOLD = 0.80
CONDITIONAL_TRUST_THRESHOLD = 0.60


# ------------------------------------------------------------
# Evidence tier
# ------------------------------------------------------------

def evidence_tier(ce_score):

    if pd.isna(ce_score):
        return "WEAK"

    if ce_score >= STRONG_CE_THRESHOLD:
        return "STRONG"

    if ce_score >= CONDITIONAL_CE_THRESHOLD:
        return "CONDITIONAL"

    return "WEAK"


# ------------------------------------------------------------
# Resolution strength
# ------------------------------------------------------------

def resolution_strength(resolution_type):

    if pd.isna(resolution_type):
        return 0.30

    value = str(resolution_type).upper()

    # Strong historical resolution signal
    if any(x in value for x in [
        "RESOLUTION_CANDIDATE",
        "RESOLVED"
    ]):
        return 1.00

    # Investigation/follow-up can still be useful,
    # but does not prove a final resolution.
    if any(x in value for x in [
        "INVESTIGATION",
        "FOLLOWUP",
        "FOLLOW_UP"
    ]):
        return 0.60

    # Potential unresolved / weak evidence
    if any(x in value for x in [
        "UNRESOLVED",
        "WEAK",
        "OTHER"
    ]):
        return 0.30

    # Generic historical support interaction
    return 0.50


# ------------------------------------------------------------
# Calculate evidence consistency
# ------------------------------------------------------------

def evidence_consistency(evidence_df):

    if evidence_df is None or len(evidence_df) == 0:
        return 0.0

    scores = pd.to_numeric(
        evidence_df["ce_score"],
        errors="coerce"
    ).dropna()

    if len(scores) == 0:
        return 0.0

    # How many retrieved candidates have reasonably strong
    # semantic evidence?
    strong_fraction = (
        (scores >= CONDITIONAL_CE_THRESHOLD).mean()
    )

    # Average normalized CE signal
    mean_score = scores.mean()

    # Combine both
    consistency = (
        0.60 * strong_fraction +
        0.40 * mean_score
    )

    return float(np.clip(consistency, 0.0, 1.0))


# ------------------------------------------------------------
# Main Trust Checker
# ------------------------------------------------------------

def trust_check(
    evidence_df,
    risk_level="LOW"
):

    # --------------------------------------------------------
    # No evidence
    # --------------------------------------------------------

    if evidence_df is None or len(evidence_df) == 0:

        return {
            "trust_score": 0.0,
            "evidence_tier": "WEAK",
            "evidence_decision": "HUMAN",
            "automation_decision": "HUMAN",
            "reason": "No historical evidence retrieved."
        }


    evidence = evidence_df.copy()

    # --------------------------------------------------------
    # Ensure score exists
    # --------------------------------------------------------

    evidence["ce_score"] = pd.to_numeric(
        evidence["ce_score"],
        errors="coerce"
    )

    evidence = evidence.dropna(
        subset=["ce_score"]
    )

    if len(evidence) == 0:

        return {
            "trust_score": 0.0,
            "evidence_tier": "WEAK",
            "evidence_decision": "HUMAN",
            "automation_decision": "HUMAN",
            "reason": "Retrieved evidence has no valid CE scores."
        }


    # --------------------------------------------------------
    # Top-1 evidence
    # --------------------------------------------------------

    top1 = evidence.iloc[0]

    top1_ce = float(top1["ce_score"])

    top1_tier = evidence_tier(top1_ce)


    # --------------------------------------------------------
    # Resolution signal
    # --------------------------------------------------------

    resolution_score = resolution_strength(
        top1.get("resolution_type", None)
    )


    # --------------------------------------------------------
    # Multi-evidence consistency
    # --------------------------------------------------------

    top_k = evidence.head(5)

    consistency_score = evidence_consistency(
        top_k
    )


    # --------------------------------------------------------
    # Trust score
    #
    # CE score is the strongest signal.
    # Resolution and consistency provide supporting evidence.
    # --------------------------------------------------------

    trust_score = (
        0.60 * top1_ce +
        0.20 * resolution_score +
        0.20 * consistency_score
    )

    trust_score = float(
        np.clip(trust_score, 0.0, 1.0)
    )


    # --------------------------------------------------------
    # Evidence decision
    # --------------------------------------------------------

    if trust_score >= STRONG_TRUST_THRESHOLD:
        evidence_decision = "TRUSTED"

    elif trust_score >= CONDITIONAL_TRUST_THRESHOLD:
        evidence_decision = "CONDITIONAL"

    else:
        evidence_decision = "WEAK"


    # --------------------------------------------------------
    # HARD SAFETY GATE
    # --------------------------------------------------------

    risk = str(risk_level).upper()

    if risk == "HIGH":

        automation_decision = "HUMAN"

        reason = (
            "High-risk case requires human review "
            "regardless of retrieval confidence."
        )

    elif evidence_decision == "TRUSTED":

        automation_decision = "AUTO"

        reason = (
            "Strong historical evidence with sufficient "
            "retrieval consistency."
        )

    else:

        automation_decision = "HUMAN"

        reason = (
            "Historical evidence is insufficiently trusted "
            "for autonomous response."
        )


    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    return {
        "trust_score": round(trust_score, 4),
        "top1_ce_score": round(top1_ce, 4),
        "evidence_tier": top1_tier,
        "resolution_score": round(resolution_score, 4),
        "consistency_score": round(consistency_score, 4),
        "evidence_decision": evidence_decision,
        "automation_decision": automation_decision,
        "reason": reason
    }

In [32]:
# ============================================================
# INSPECT RERANKED RESULTS
# ============================================================

reranked = pd.read_csv(
    "data/checkpoints/cross_encoder_reranked_results.csv"
)

print("Shape:", reranked.shape)

print("\nColumns:")
print(reranked.columns.tolist())

display(reranked.head(3))

Shape: (10000, 11)

Columns:
['golden_id', 'case_id', 'ce_rank', 'ce_score', 'rrf_rank', 'rrf_score', 'bm25_rank', 'bge_rank', 'historical_weak_intent', 'golden_intent', 'intent_match']


,golden_id,case_id,ce_rank,ce_score,rrf_rank,rrf_score,bm25_rank,bge_rank,historical_weak_intent,golden_intent,intent_match
0,GOLD-0001,AMZ-073280,1,0.615286,23.0,0.013889,12.0,NaN,NaN,PAYMENT_BILLING,False
1,GOLD-0001,AMZ-057467,2,0.249812,13.0,0.014925,7.0,NaN,PAYMENT_BILLING,PAYMENT_BILLING,True
2,GOLD-0001,AMZ-058158,3,0.167556,4.0,0.016129,NaN,2.0,NaN,PAYMENT_BILLING,False


In [33]:
# ============================================================
# TRUST CHECKER V1
# Schema-aligned implementation
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

STRONG_CE_THRESHOLD = 0.90
CONDITIONAL_CE_THRESHOLD = 0.75

STRONG_TRUST_THRESHOLD = 0.80
CONDITIONAL_TRUST_THRESHOLD = 0.60


# ------------------------------------------------------------
# CE evidence tier
# ------------------------------------------------------------

def get_evidence_tier(ce_score):

    if pd.isna(ce_score):
        return "WEAK"

    if ce_score >= STRONG_CE_THRESHOLD:
        return "STRONG"

    if ce_score >= CONDITIONAL_CE_THRESHOLD:
        return "CONDITIONAL"

    return "WEAK"


# ------------------------------------------------------------
# Intent consistency
# ------------------------------------------------------------

def calculate_intent_consistency(
    evidence_df,
    predicted_intent
):

    if (
        evidence_df is None
        or len(evidence_df) == 0
        or not predicted_intent
    ):
        return 0.0

    if "historical_weak_intent" not in evidence_df.columns:
        return 0.0

    historical_intents = (
        evidence_df["historical_weak_intent"]
        .dropna()
        .astype(str)
        .str.upper()
    )

    if len(historical_intents) == 0:
        return 0.0

    predicted_intent = str(
        predicted_intent
    ).upper()

    match_rate = (
        historical_intents == predicted_intent
    ).mean()

    return float(match_rate)


# ------------------------------------------------------------
# Multi-evidence semantic consistency
# ------------------------------------------------------------

def calculate_semantic_consistency(
    evidence_df
):

    if evidence_df is None or len(evidence_df) == 0:
        return 0.0

    scores = pd.to_numeric(
        evidence_df["ce_score"],
        errors="coerce"
    ).dropna()

    if len(scores) == 0:
        return 0.0

    # Fraction of top-k evidence above the
    # conditional confidence threshold.
    strong_fraction = (
        scores >= CONDITIONAL_CE_THRESHOLD
    ).mean()

    # Mean CE score
    mean_score = scores.mean()

    consistency = (
        0.60 * strong_fraction +
        0.40 * mean_score
    )

    return float(
        np.clip(consistency, 0.0, 1.0)
    )


# ------------------------------------------------------------
# Main Trust Checker
# ------------------------------------------------------------

def trust_check_v1(
    evidence_df,
    predicted_intent,
    risk_level="LOW"
):

    # --------------------------------------------------------
    # No evidence
    # --------------------------------------------------------

    if evidence_df is None or len(evidence_df) == 0:

        return {
            "trust_score": 0.0,
            "evidence_tier": "WEAK",
            "intent_consistency": 0.0,
            "semantic_consistency": 0.0,
            "evidence_decision": "WEAK",
            "automation_decision": "HUMAN",
            "reason": "No historical evidence retrieved."
        }


    evidence = evidence_df.copy()

    # --------------------------------------------------------
    # Validate CE scores
    # --------------------------------------------------------

    evidence["ce_score"] = pd.to_numeric(
        evidence["ce_score"],
        errors="coerce"
    )

    evidence = evidence.dropna(
        subset=["ce_score"]
    )

    if len(evidence) == 0:

        return {
            "trust_score": 0.0,
            "evidence_tier": "WEAK",
            "intent_consistency": 0.0,
            "semantic_consistency": 0.0,
            "evidence_decision": "WEAK",
            "automation_decision": "HUMAN",
            "reason": "Retrieved evidence has no valid CE scores."
        }


    # --------------------------------------------------------
    # Top-1
    # --------------------------------------------------------

    top1 = evidence.iloc[0]

    top1_ce = float(
        top1["ce_score"]
    )

    tier = get_evidence_tier(
        top1_ce
    )


    # --------------------------------------------------------
    # Intent consistency
    # --------------------------------------------------------

    top_k = evidence.head(5)

    intent_consistency = (
        calculate_intent_consistency(
            top_k,
            predicted_intent
        )
    )


    # --------------------------------------------------------
    # Semantic consistency
    # --------------------------------------------------------

    semantic_consistency = (
        calculate_semantic_consistency(
            top_k
        )
    )


    # --------------------------------------------------------
    # Trust score
    #
    # CE score is the primary signal.
    # Intent agreement and multi-evidence consistency
    # provide supporting signals.
    # --------------------------------------------------------

    trust_score = (
        0.60 * top1_ce +
        0.20 * intent_consistency +
        0.20 * semantic_consistency
    )

    trust_score = float(
        np.clip(trust_score, 0.0, 1.0)
    )


    # --------------------------------------------------------
    # Evidence decision
    # --------------------------------------------------------

    if trust_score >= STRONG_TRUST_THRESHOLD:

        evidence_decision = "TRUSTED"

    elif trust_score >= CONDITIONAL_TRUST_THRESHOLD:

        evidence_decision = "CONDITIONAL"

    else:

        evidence_decision = "WEAK"


    # --------------------------------------------------------
    # HARD SAFETY GATE
    # --------------------------------------------------------

    risk = str(
        risk_level
    ).upper()

    if risk == "HIGH":

        automation_decision = "HUMAN"

        reason = (
            "High-risk case requires human review "
            "regardless of retrieval confidence."
        )

    elif evidence_decision == "TRUSTED":

        automation_decision = "AUTO"

        reason = (
            "Historical evidence is sufficiently "
            "strong and intent-consistent."
        )

    else:

        automation_decision = "HUMAN"

        reason = (
            "Historical evidence does not meet the "
            "autonomous-response trust criteria."
        )


    return {
        "trust_score": round(
            trust_score, 4
        ),

        "top1_ce_score": round(
            top1_ce, 4
        ),

        "evidence_tier": tier,

        "intent_consistency": round(
            intent_consistency, 4
        ),

        "semantic_consistency": round(
            semantic_consistency, 4
        ),

        "evidence_decision":
            evidence_decision,

        "automation_decision":
            automation_decision,

        "reason":
            reason
    }


print("✅ Trust Checker v1 loaded successfully")

✅ Trust Checker v1 loaded successfully


In [34]:
# ============================================================
# TEST TRUST CHECKER ON GOLD-0001
# ============================================================

golden_id = "GOLD-0001"

case_evidence = (
    reranked[
        reranked["golden_id"] == golden_id
    ]
    .sort_values("ce_rank")
    .head(5)
    .copy()
)

predicted_intent = "PAYMENT_BILLING"

result = trust_check_v1(
    evidence_df=case_evidence,
    predicted_intent=predicted_intent,
    risk_level="LOW"
)

print("=" * 80)
print("TRUST CHECKER V1 TEST")
print("=" * 80)

print(f"Golden ID       : {golden_id}")
print(f"Predicted intent: {predicted_intent}")

for key, value in result.items():
    print(f"{key:25}: {value}")

TRUST CHECKER V1 TEST
Golden ID       : GOLD-0001
Predicted intent: PAYMENT_BILLING
trust_score              : 0.4895
top1_ce_score            : 0.6153
evidence_tier            : WEAK
intent_consistency       : 0.5
semantic_consistency     : 0.1019
evidence_decision        : WEAK
automation_decision      : HUMAN
reason                   : Historical evidence does not meet the autonomous-response trust criteria.


In [35]:
# ============================================================
# HARD SAFETY GATE TEST
# ============================================================

high_risk_result = trust_check_v1(
    evidence_df=case_evidence,
    predicted_intent="PAYMENT_BILLING",
    risk_level="HIGH"
)

print("=" * 80)
print("HIGH-RISK SAFETY TEST")
print("=" * 80)

for key, value in high_risk_result.items():
    print(f"{key:25}: {value}")

HIGH-RISK SAFETY TEST
trust_score              : 0.4895
top1_ce_score            : 0.6153
evidence_tier            : WEAK
intent_consistency       : 0.5
semantic_consistency     : 0.1019
evidence_decision        : WEAK
automation_decision      : HUMAN
reason                   : High-risk case requires human review regardless of retrieval confidence.


In [36]:
# ============================================================
# TRUST CHECKER V1 — 200 CASE EVALUATION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

reranked = pd.read_csv(
    "data/checkpoints/cross_encoder_reranked_results.csv"
)

golden = pd.read_csv(
    "data/evaluation/golden_evaluation_set.csv"
)

# ------------------------------------------------------------
# Keep authoritative golden labels
# ------------------------------------------------------------

golden_labels = golden[
    [
        "golden_id",
        "intent",
        "risk_level",
        "automation_decision"
    ]
].copy()

# ------------------------------------------------------------
# Evaluate each golden case
# ------------------------------------------------------------

trust_results = []

for golden_id, group in reranked.groupby("golden_id"):

    group = (
        group
        .sort_values("ce_rank")
        .head(5)
        .copy()
    )

    label_row = golden_labels[
        golden_labels["golden_id"] == golden_id
    ]

    if len(label_row) == 0:
        continue

    label_row = label_row.iloc[0]

    predicted_intent = label_row["intent"]
    risk_level = label_row["risk_level"]

    result = trust_check_v1(
        evidence_df=group,
        predicted_intent=predicted_intent,
        risk_level=risk_level
    )

    result["golden_id"] = golden_id
    result["gold_intent"] = predicted_intent
    result["gold_risk_level"] = risk_level
    result["gold_automation_decision"] = (
        label_row["automation_decision"]
    )

    trust_results.append(result)


trust_eval = pd.DataFrame(
    trust_results
)

# ------------------------------------------------------------
# Reorder columns
# ------------------------------------------------------------

cols = [
    "golden_id",
    "gold_intent",
    "gold_risk_level",
    "gold_automation_decision",
    "trust_score",
    "top1_ce_score",
    "evidence_tier",
    "intent_consistency",
    "semantic_consistency",
    "evidence_decision",
    "automation_decision",
    "reason"
]

trust_eval = trust_eval[cols]

print("=" * 80)
print("TRUST CHECKER V1 — 200 CASE EVALUATION")
print("=" * 80)

print(f"Evaluated cases: {len(trust_eval)}")

display(
    trust_eval.head(10)
)

TRUST CHECKER V1 — 200 CASE EVALUATION
Evaluated cases: 200


,golden_id,gold_intent,gold_risk_level,gold_automation_decision,trust_score,top1_ce_score,evidence_tier,intent_consistency,semantic_consistency,evidence_decision,automation_decision,reason
0,GOLD-0001,PAYMENT_BILLING,MEDIUM,HUMAN,0.4895,0.6153,WEAK,0.5,0.1019,WEAK,HUMAN,Historical evidence does not meet the autonomo...
1,GOLD-0002,ORDER_DELIVERY,LOW,HUMAN,0.9986,0.9984,STRONG,1.0,0.9979,TRUSTED,AUTO,Historical evidence is sufficiently strong and...
2,GOLD-0003,ORDER_DELIVERY,LOW,HUMAN,0.9935,0.9910,STRONG,1.0,0.9945,TRUSTED,AUTO,Historical evidence is sufficiently strong and...
3,GOLD-0004,GENERAL_SOCIAL,LOW,AUTO,0.7669,0.9579,STRONG,0.0,0.9608,CONDITIONAL,HUMAN,Historical evidence does not meet the autonomo...
4,GOLD-0005,DEVICE_TECHNICAL,LOW,HUMAN,0.5675,0.8599,CONDITIONAL,0.0,0.2576,WEAK,HUMAN,Historical evidence does not meet the autonomo...
5,GOLD-0006,SELLER_AUTHENTICITY,LOW,AUTO,0.5223,0.7703,CONDITIONAL,0.0,0.3007,WEAK,HUMAN,Historical evidence does not meet the autonomo...
6,GOLD-0007,ORDER_DELIVERY,HIGH,HUMAN,0.8749,0.9513,STRONG,1.0,0.5209,TRUSTED,HUMAN,High-risk case requires human review regardles...
7,GOLD-0008,ORDER_DELIVERY,LOW,HUMAN,0.9990,0.9994,STRONG,1.0,0.9966,TRUSTED,AUTO,Historical evidence is sufficiently strong and...
8,GOLD-0009,PURCHASE_CONTROL,MEDIUM,HUMAN,0.2921,0.4503,WEAK,0.0,0.1099,WEAK,HUMAN,Historical evidence does not meet the autonomo...
9,GOLD-0010,ORDER_DELIVERY,LOW,HUMAN,0.4323,0.3637,WEAK,1.0,0.0701,WEAK,HUMAN,Historical evidence does not meet the autonomo...


In [37]:
# ============================================================
# TRUST CHECKER — AUTOMATION AGREEMENT
# ============================================================

comparison = pd.crosstab(
    trust_eval["gold_automation_decision"],
    trust_eval["automation_decision"],
    rownames=["Human Label"],
    colnames=["Trust Checker"]
)

print("=" * 80)
print("AUTOMATION DECISION CONFUSION MATRIX")
print("=" * 80)

display(comparison)


agreement = (
    trust_eval["gold_automation_decision"]
    ==
    trust_eval["automation_decision"]
).mean()

print(
    f"\nOverall automation agreement: "
    f"{agreement * 100:.2f}%"
)

AUTOMATION DECISION CONFUSION MATRIX


Trust Checker,AUTO,HUMAN
Human Label,,
AUTO,4,48
HUMAN,42,106



Overall automation agreement: 55.00%


In [38]:
# ============================================================
# RISK × TRUST CHECKER DECISION
# ============================================================

risk_decision = pd.crosstab(
    trust_eval["gold_risk_level"],
    trust_eval["automation_decision"],
    normalize="index"
).round(4) * 100

print("=" * 80)
print("TRUST CHECKER DECISION BY RISK")
print("=" * 80)

display(risk_decision)

TRUST CHECKER DECISION BY RISK


automation_decision,AUTO,HUMAN
gold_risk_level,,
HIGH,0.00,100.00
LOW,25.47,74.53
MEDIUM,27.94,72.06


In [40]:
print(trust_eval.shape)
display(trust_eval.head())

(200, 12)


,golden_id,gold_intent,gold_risk_level,gold_automation_decision,trust_score,top1_ce_score,evidence_tier,intent_consistency,semantic_consistency,evidence_decision,automation_decision,reason
0,GOLD-0001,PAYMENT_BILLING,MEDIUM,HUMAN,0.4895,0.6153,WEAK,0.5,0.1019,WEAK,HUMAN,Historical evidence does not meet the autonomo...
1,GOLD-0002,ORDER_DELIVERY,LOW,HUMAN,0.9986,0.9984,STRONG,1.0,0.9979,TRUSTED,AUTO,Historical evidence is sufficiently strong and...
2,GOLD-0003,ORDER_DELIVERY,LOW,HUMAN,0.9935,0.9910,STRONG,1.0,0.9945,TRUSTED,AUTO,Historical evidence is sufficiently strong and...
3,GOLD-0004,GENERAL_SOCIAL,LOW,AUTO,0.7669,0.9579,STRONG,0.0,0.9608,CONDITIONAL,HUMAN,Historical evidence does not meet the autonomo...
4,GOLD-0005,DEVICE_TECHNICAL,LOW,HUMAN,0.5675,0.8599,CONDITIONAL,0.0,0.2576,WEAK,HUMAN,Historical evidence does not meet the autonomo...


In [41]:
import os

os.makedirs("data/evaluation", exist_ok=True)

trust_eval.to_csv(
    "data/evaluation/trust_checker_evaluation.csv",
    index=False
)

print("Saved successfully:")
print("data/evaluation/trust_checker_evaluation.csv")

Saved successfully:
data/evaluation/trust_checker_evaluation.csv


In [42]:
# ============================================================
# TRUST CHECKER V1 — SIGNAL ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

trust_eval = pd.read_csv(
    "data/evaluation/trust_checker_evaluation.csv"
)

print("Shape:", trust_eval.shape)
print("\nColumns:")
print(trust_eval.columns.tolist())


# ------------------------------------------------------------
# BASIC GROUP STATISTICS
# ------------------------------------------------------------

signals = [
    "trust_score",
    "top1_ce_score",
    "intent_consistency",
    "semantic_consistency"
]

print("\n" + "=" * 80)
print("SIGNAL DISTRIBUTION: HUMAN AUTO vs HUMAN")
print("=" * 80)

for col in signals:

    print(f"\n--- {col} ---")

    stats = (
        trust_eval
        .groupby("gold_automation_decision")[col]
        .agg([
            "count",
            "mean",
            "median",
            "min",
            "max"
        ])
        .round(4)
    )

    display(stats)


# ------------------------------------------------------------
# RISK × HUMAN LABEL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("HUMAN AUTOMATION LABEL BY RISK")
print("=" * 80)

risk_label = pd.crosstab(
    trust_eval["gold_risk_level"],
    trust_eval["gold_automation_decision"],
    normalize="index"
).round(4) * 100

display(risk_label)


# ------------------------------------------------------------
# TRUST SCORE DISTRIBUTION BY LABEL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRUST SCORE BY HUMAN LABEL")
print("=" * 80)

display(
    trust_eval
    .groupby("gold_automation_decision")["trust_score"]
    .describe()
    .round(4)
)


# ------------------------------------------------------------
# SORT CASES BY TRUST SCORE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("HIGHEST TRUST CASES")
print("=" * 80)

display(
    trust_eval[
        [
            "golden_id",
            "gold_intent",
            "gold_risk_level",
            "gold_automation_decision",
            "trust_score",
            "top1_ce_score",
            "intent_consistency",
            "semantic_consistency",
            "automation_decision",
            "reason"
        ]
    ]
    .sort_values("trust_score", ascending=False)
    .head(30)
)


# ------------------------------------------------------------
# FALSE AUTO CASES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FALSE AUTO — CHECKER SAID AUTO, HUMAN SAID HUMAN")
print("=" * 80)

false_auto = trust_eval[
    (trust_eval["automation_decision"] == "AUTO") &
    (trust_eval["gold_automation_decision"] == "HUMAN")
].copy()

print("False AUTO cases:", len(false_auto))

display(
    false_auto[
        [
            "golden_id",
            "gold_intent",
            "gold_risk_level",
            "trust_score",
            "top1_ce_score",
            "intent_consistency",
            "semantic_consistency",
            "reason"
        ]
    ]
    .sort_values("trust_score", ascending=False)
    .head(50)
)


# ------------------------------------------------------------
# FALSE HUMAN CASES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FALSE HUMAN — CHECKER SAID HUMAN, HUMAN SAID AUTO")
print("=" * 80)

false_human = trust_eval[
    (trust_eval["automation_decision"] == "HUMAN") &
    (trust_eval["gold_automation_decision"] == "AUTO")
].copy()

print("False HUMAN cases:", len(false_human))

display(
    false_human[
        [
            "golden_id",
            "gold_intent",
            "gold_risk_level",
            "trust_score",
            "top1_ce_score",
            "intent_consistency",
            "semantic_consistency",
            "reason"
        ]
    ]
    .sort_values("trust_score", ascending=False)
)

Shape: (200, 12)

Columns:
['golden_id', 'gold_intent', 'gold_risk_level', 'gold_automation_decision', 'trust_score', 'top1_ce_score', 'evidence_tier', 'intent_consistency', 'semantic_consistency', 'evidence_decision', 'automation_decision', 'reason']

SIGNAL DISTRIBUTION: HUMAN AUTO vs HUMAN

--- trust_score ---


,count,mean,median,min,max
gold_automation_decision,,,,,
AUTO,52,0.6009,0.7349,0.0157,0.9971
HUMAN,148,0.7433,0.7916,0.0045,0.9998



--- top1_ce_score ---


,count,mean,median,min,max
gold_automation_decision,,,,,
AUTO,52,0.7731,0.9518,0.0246,1.0
HUMAN,148,0.8694,0.9706,0.0068,1.0



--- intent_consistency ---


...

,golden_id,gold_intent,gold_risk_level,gold_automation_decision,trust_score,top1_ce_score,intent_consistency,semantic_consistency,automation_decision,reason
181,GOLD-0182,ORDER_DELIVERY,HIGH,HUMAN,0.9998,0.9998,1.0,0.9997,HUMAN,High-risk case requires human review regardles...
42,GOLD-0043,ORDER_DELIVERY,MEDIUM,HUMAN,0.9996,0.9996,1.0,0.9990,AUTO,Historical evidence is sufficiently strong and...
7,GOLD-0008,ORDER_DELIVERY,LOW,HUMAN,0.9990,0.9994,1.0,0.9966,AUTO,Historical evidence is sufficiently strong and...
1,GOLD-0002,ORDER_DELIVERY,LOW,HUMAN,0.9986,0.9984,1.0,0.9979,AUTO,Historical evidence is sufficiently strong and...
59,GOLD-0060,ORDER_DELIVERY,LOW,HUMAN,0.9985,0.9982,1.0,0.9982,AUTO,Historical evidence is sufficiently strong and...
189,GOLD-0190,ORDER_DELIVERY,HIGH,HUMAN,0.9981,0.9994,1.0,0.9924,HUMAN,High-risk case requires human review regardles...
139,GOLD-0140,ORDER_DELIVERY,LOW,HUMAN,0.9977,0.9989,1.0,0.9920,AUTO,Historical evidence is sufficiently strong and...
170,GOLD-0171,PAYMENT_BILLING,HIGH,HUMAN,0.9976,0.9973,1.0,0.9960,HUMAN,High-risk case requires human review regardles...
31,GOLD-0032,ORDER_DELIVERY,LOW,AUTO,0.9971,0.9977,1.0,0.9923,AUTO,Historical evidence is sufficiently strong and...
99,GOLD-0100,ORDER_DELIVERY,MEDIUM,HUMAN,0.9969,0.9979,1.0,0.9911,AUTO,Historical evidence is sufficiently strong and...



FALSE AUTO — CHECKER SAID AUTO, HUMAN SAID HUMAN
False AUTO cases: 42


,golden_id,gold_intent,gold_risk_level,trust_score,top1_ce_score,intent_consistency,semantic_consistency,reason
42,GOLD-0043,ORDER_DELIVERY,MEDIUM,0.9996,0.9996,1.0000,0.9990,Historical evidence is sufficiently strong and...
7,GOLD-0008,ORDER_DELIVERY,LOW,0.9990,0.9994,1.0000,0.9966,Historical evidence is sufficiently strong and...
1,GOLD-0002,ORDER_DELIVERY,LOW,0.9986,0.9984,1.0000,0.9979,Historical evidence is sufficiently strong and...
59,GOLD-0060,ORDER_DELIVERY,LOW,0.9985,0.9982,1.0000,0.9982,Historical evidence is sufficiently strong and...
139,GOLD-0140,ORDER_DELIVERY,LOW,0.9977,0.9989,1.0000,0.9920,Historical evidence is sufficiently strong and...
99,GOLD-0100,ORDER_DELIVERY,MEDIUM,0.9969,0.9979,1.0000,0.9911,Historical evidence is sufficiently strong and...
106,GOLD-0107,ORDER_DELIVERY,MEDIUM,0.9966,0.9960,1.0000,0.9953,Historical evidence is sufficiently strong and...
78,GOLD-0079,ORDER_DELIVERY,LOW,0.9949,0.9989,1.0000,0.9779,Historical evidence is sufficiently strong and...
93,GOLD-0094,ORDER_DELIVERY,MEDIUM,0.9948,0.9939,1.0000,0.9922,Historical evidence is sufficiently strong and...
39,GOLD-0040,RETURNS_REFUNDS,MEDIUM,0.9943,0.9926,1.0000,0.9934,Historical evidence is sufficiently strong and...



FALSE HUMAN — CHECKER SAID HUMAN, HUMAN SAID AUTO
False HUMAN cases: 48


,golden_id,gold_intent,gold_risk_level,trust_score,top1_ce_score,intent_consistency,semantic_consistency,reason
94,GOLD-0095,GENERAL_SUPPORT,LOW,0.7999,0.9999,0.0,0.9998,Historical evidence does not meet the autonomo...
88,GOLD-0089,GENERAL_SUPPORT,LOW,0.7991,0.9987,0.0,0.9995,Historical evidence does not meet the autonomo...
60,GOLD-0061,GENERAL_SUPPORT,LOW,0.7989,0.9987,0.0,0.9984,Historical evidence does not meet the autonomo...
71,GOLD-0072,GENERAL_SOCIAL,LOW,0.7986,0.9982,0.0,0.9983,Historical evidence does not meet the autonomo...
115,GOLD-0116,ORDER_TRACKING,LOW,0.7961,0.9993,0.0,0.9826,Historical evidence does not meet the autonomo...
197,GOLD-0198,ACCOUNT_ACCESS,MEDIUM,0.7960,0.9954,0.0,0.9935,Historical evidence does not meet the autonomo...
79,GOLD-0080,PURCHASE_CONTROL,LOW,0.7955,0.9972,0.0,0.9857,Historical evidence does not meet the autonomo...
67,GOLD-0068,GENERAL_SUPPORT,LOW,0.7953,0.9930,0.0,0.9972,Historical evidence does not meet the autonomo...
17,GOLD-0018,PREORDER_DELIVERY,LOW,0.7909,0.9957,0.0,0.9672,Historical evidence does not meet the autonomo...
74,GOLD-0075,ORDER_DELIVERY,LOW,0.7904,0.9915,0.0,0.9773,Historical evidence does not meet the autonomo...


In [43]:
# ============================================================
# AUTO vs HUMAN × HUMAN-REVIEWED EVIDENCE QUALITY
# ============================================================

import pandas as pd

trust_eval = pd.read_csv(
    "data/evaluation/trust_checker_evaluation.csv"
)

evidence_review = pd.read_csv(
    "data/evaluation/top1_evidence_review.csv"
)

print("Trust evaluation:", trust_eval.shape)
print("Evidence review:", evidence_review.shape)

# ------------------------------------------------------------
# JOIN
# ------------------------------------------------------------

analysis_df = trust_eval.merge(
    evidence_review[
        [
            "golden_id",
            "evidence_relevance"
        ]
    ],
    on="golden_id",
    how="left"
)

print("\nJoined:", analysis_df.shape)

# ------------------------------------------------------------
# DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EVIDENCE RELEVANCE BY HUMAN AUTOMATION LABEL")
print("=" * 80)

table = pd.crosstab(
    analysis_df["gold_automation_decision"],
    analysis_df["evidence_relevance"],
    normalize="index"
).round(4) * 100

display(table)

# ------------------------------------------------------------
# COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COUNTS")
print("=" * 80)

counts = pd.crosstab(
    analysis_df["gold_automation_decision"],
    analysis_df["evidence_relevance"]
)

display(counts)

# ------------------------------------------------------------
# RISK × EVIDENCE × AUTOMATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RISK × EVIDENCE RELEVANCE × HUMAN LABEL")
print("=" * 80)

display(
    pd.crosstab(
        [
            analysis_df["gold_risk_level"],
            analysis_df["evidence_relevance"]
        ],
        analysis_df["gold_automation_decision"],
        normalize="index"
    ).round(4) * 100
)

Trust evaluation: (200, 12)
Evidence review: (200, 10)

Joined: (200, 13)

EVIDENCE RELEVANCE BY HUMAN AUTOMATION LABEL


evidence_relevance,IRRELEVANT,PARTIAL,RELEVANT
gold_automation_decision,,,
AUTO,3.85,11.54,84.62
HUMAN,0.68,20.95,78.38



COUNTS


evidence_relevance,IRRELEVANT,PARTIAL,RELEVANT
gold_automation_decision,,,
AUTO,2,6,44
HUMAN,1,31,116



RISK × EVIDENCE RELEVANCE × HUMAN LABEL


gold_automation_decision              AUTO   HUMAN
gold_risk_level evidence_relevance                
HIGH            IRRELEVANT            0.00  100.00
                PARTIAL               0.00  100.00
                RELEVANT              0.00  100.00
LOW             IRRELEVANT          100.00    0.00
                PARTIAL              35.29   64.71
                RELEVANT             49.43   50.57
MEDIUM          PARTIAL               0.00  100.00
                RELEVANT              1.85   98.15

In [44]:
# ============================================================
# AUTOMATION DECISION × RESOLUTION STATUS
# ============================================================

import pandas as pd

golden = pd.read_csv(
    "data/evaluation/golden_evaluation_set.csv"
)

trust_eval = pd.read_csv(
    "data/evaluation/trust_checker_evaluation.csv"
)

evidence_review = pd.read_csv(
    "data/evaluation/top1_evidence_review.csv"
)

# ------------------------------------------------------------
# JOIN GOLD LABELS
# ------------------------------------------------------------

analysis_df = (
    trust_eval[
        [
            "golden_id",
            "gold_automation_decision",
            "gold_risk_level"
        ]
    ]
    .merge(
        golden[
            [
                "golden_id",
                "resolution_status",
                "intent"
            ]
        ],
        on="golden_id",
        how="left"
    )
    .merge(
        evidence_review[
            [
                "golden_id",
                "evidence_relevance"
            ]
        ],
        on="golden_id",
        how="left"
    )
)

print("Shape:", analysis_df.shape)


# ============================================================
# 1. RESOLUTION STATUS × AUTOMATION
# ============================================================

print("\n" + "=" * 80)
print("RESOLUTION STATUS × HUMAN AUTOMATION LABEL")
print("=" * 80)

resolution_counts = pd.crosstab(
    analysis_df["resolution_status"],
    analysis_df["gold_automation_decision"]
)

display(resolution_counts)


print("\nPercentages:")

resolution_pct = pd.crosstab(
    analysis_df["resolution_status"],
    analysis_df["gold_automation_decision"],
    normalize="index"
).round(4) * 100

display(resolution_pct)


# ============================================================
# 2. RISK × RESOLUTION × AUTOMATION
# ============================================================

print("\n" + "=" * 80)
print("RISK × RESOLUTION STATUS × AUTOMATION")
print("=" * 80)

risk_resolution = pd.crosstab(
    [
        analysis_df["gold_risk_level"],
        analysis_df["resolution_status"]
    ],
    analysis_df["gold_automation_decision"]
)

display(risk_resolution)


# ============================================================
# 3. LOW-RISK CASES ONLY
# ============================================================

print("\n" + "=" * 80)
print("LOW-RISK: RESOLUTION STATUS × AUTOMATION")
print("=" * 80)

low_risk = analysis_df[
    analysis_df["gold_risk_level"] == "LOW"
]

low_resolution = pd.crosstab(
    low_risk["resolution_status"],
    low_risk["gold_automation_decision"]
)

display(low_resolution)


# ============================================================
# 4. LOW-RISK + RELEVANT EVIDENCE
# ============================================================

print("\n" + "=" * 80)
print("LOW-RISK + RELEVANT EVIDENCE")
print("=" * 80)

low_relevant = analysis_df[
    (analysis_df["gold_risk_level"] == "LOW") &
    (analysis_df["evidence_relevance"] == "RELEVANT")
]

display(
    pd.crosstab(
        low_relevant["resolution_status"],
        low_relevant["gold_automation_decision"]
    )
)


# ============================================================
# 5. RESOLUTION STATUS × RISK × EVIDENCE
# ============================================================

print("\n" + "=" * 80)
print("LOW-RISK + RELEVANT: DETAILED BREAKDOWN")
print("=" * 80)

display(
    low_relevant[
        [
            "golden_id",
            "intent",
            "resolution_status",
            "gold_automation_decision",
            "evidence_relevance"
        ]
    ]
    .sort_values(
        ["resolution_status", "gold_automation_decision"]
    )
)

Shape: (200, 6)

RESOLUTION STATUS × HUMAN AUTOMATION LABEL


gold_automation_decision,AUTO,HUMAN
resolution_status,,
ESCALATION_REQUIRED,0,6
INVESTIGATION_REQUIRED,0,121
NOT_APPLICABLE,17,0
PARTIALLY_RESOLVED,11,9
RESOLVED,19,2
RESOLVED_BY_CUSTOMER,5,0
UNRESOLVED,0,10



Percentages:


gold_automation_decision,AUTO,HUMAN
resolution_status,,
ESCALATION_REQUIRED,0.00,100.00
INVESTIGATION_REQUIRED,0.00,100.00
NOT_APPLICABLE,100.00,0.00
PARTIALLY_RESOLVED,55.00,45.00
RESOLVED,90.48,9.52
RESOLVED_BY_CUSTOMER,100.00,0.00
UNRESOLVED,0.00,100.00



RISK × RESOLUTION STATUS × AUTOMATION


...

gold_automation_decision,AUTO,HUMAN
resolution_status,,
INVESTIGATION_REQUIRED,0,44
NOT_APPLICABLE,17,0
PARTIALLY_RESOLVED,10,5
RESOLVED,19,0
RESOLVED_BY_CUSTOMER,5,0
UNRESOLVED,0,6



LOW-RISK + RELEVANT EVIDENCE


gold_automation_decision,AUTO,HUMAN
resolution_status,,
INVESTIGATION_REQUIRED,0,35
NOT_APPLICABLE,15,0
PARTIALLY_RESOLVED,8,5
RESOLVED,16,0
RESOLVED_BY_CUSTOMER,4,0
UNRESOLVED,0,4



LOW-RISK + RELEVANT: DETAILED BREAKDOWN


,golden_id,intent,resolution_status,gold_automation_decision,evidence_relevance
2,GOLD-0003,ORDER_DELIVERY,INVESTIGATION_REQUIRED,HUMAN,RELEVANT
4,GOLD-0005,DEVICE_TECHNICAL,INVESTIGATION_REQUIRED,HUMAN,RELEVANT
7,GOLD-0008,ORDER_DELIVERY,INVESTIGATION_REQUIRED,HUMAN,RELEVANT
12,GOLD-0013,ORDER_PURCHASE,INVESTIGATION_REQUIRED,HUMAN,RELEVANT
13,GOLD-0014,ORDER_DELIVERY,INVESTIGATION_REQUIRED,HUMAN,RELEVANT
...,...,...,...,...,...
150,GOLD-0151,GENERAL_SUPPORT,RESOLVED_BY_CUSTOMER,AUTO,RELEVANT
27,GOLD-0028,ORDER_TRACKING,UNRESOLVED,HUMAN,RELEVANT
33,GOLD-0034,ORDER_DELIVERY,UNRESOLVED,HUMAN,RELEVANT
53,GOLD-0054,ORDER_DELIVERY,UNRESOLVED,HUMAN,RELEVANT


In [45]:
# ============================================================
# HISTORICAL RESOLUTION PATTERN ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LOAD HISTORICAL RAG CORPUS
# ------------------------------------------------------------

rag_corpus = pd.read_csv(
    "data/checkpoints/structured_rag_corpus.csv"
)

print("RAG corpus shape:", rag_corpus.shape)

print("\nColumns:")
print(rag_corpus.columns.tolist())


# ============================================================
# 1. WHAT RESOLUTION TYPES EXIST?
# ============================================================

print("\n" + "=" * 80)
print("HISTORICAL RESOLUTION TYPE DISTRIBUTION")
print("=" * 80)

print(
    rag_corpus["resolution_type"]
    .value_counts(dropna=False)
)

print("\nPercentages:")

display(
    rag_corpus["resolution_type"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


# ============================================================
# 2. SHOW EXAMPLES
# ============================================================

print("\n" + "=" * 80)
print("EXAMPLES BY RESOLUTION TYPE")
print("=" * 80)

for resolution_type in rag_corpus["resolution_type"].dropna().unique():

    print(f"\n--- {resolution_type} ---")

    display(
        rag_corpus[
            rag_corpus["resolution_type"] == resolution_type
        ][
            [
                "case_id",
                "customer_problem",
                "support_action",
                "resolution_type",
                "evidence_quality"
            ]
        ]
        .head(5)
    )

RAG corpus shape: (83218, 6)

Columns:
['case_id', 'customer_problem', 'support_action', 'resolution_type', 'evidence_quality', 'source']

HISTORICAL RESOLUTION TYPE DISTRIBUTION
resolution_type
GENERAL_SUPPORT              53440
INVESTIGATION_OR_FOLLOWUP    27908
REFUND_OR_COMPENSATION        1130
CANCELLATION                   462
REPLACEMENT                    278
Name: count, dtype: int64

Percentages:


,proportion
resolution_type,
GENERAL_SUPPORT,64.22
INVESTIGATION_OR_FOLLOWUP,33.54
REFUND_OR_COMPENSATION,1.36
CANCELLATION,0.56
REPLACEMENT,0.33



EXAMPLES BY RESOLUTION TYPE

--- GENERAL_SUPPORT ---


,case_id,customer_problem,support_action,resolution_type,evidence_quality
0,AMZ-000001,[user] what in the world is a balance withheld...,"[USER] Sorry, I'm not quite sure what you're h...",GENERAL_SUPPORT,HIGH
1,AMZ-000002,[user] that's exactly what she did. bought it ...,"[USER] Sorry, we would be unable to change thi...",GENERAL_SUPPORT,HIGH
3,AMZ-000004,1/ [user] are you guys leaking email ids of us...,[USER] Our customers' security is our utmost i...,GENERAL_SUPPORT,HIGH
4,AMZ-000005,the sound quality of the deer hunter on amazon...,[USER] Let us know if you have any other quest...,GENERAL_SUPPORT,HIGH
6,AMZ-000007,[user] [user] you really want to keep my money...,[USER] I'm sorry for any trouble. Are you a se...,GENERAL_SUPPORT,MEDIUM



--- INVESTIGATION_OR_FOLLOWUP ---


...

,case_id,customer_problem,support_action,resolution_type,evidence_quality
46,AMZ-000047,[user] what is is it taking amazon to refund m...,[USER] I'm sorry about the delay with the refu...,REFUND_OR_COMPENSATION,HIGH
62,AMZ-000063,[user] disappointed in [user] bought the regri...,[USER] I'm sorry for the trouble! Please visit...,REFUND_OR_COMPENSATION,MEDIUM
210,AMZ-000211,[user] [user] was told a returnless refund wil...,"[USER] We understand your concern. However, fo...",REFUND_OR_COMPENSATION,HIGH
389,AMZ-000391,[user] [user] still no reply niether refund ??...,[USER] Please reply to the email sent by our t...,REFUND_OR_COMPENSATION,HIGH
417,AMZ-000419,[user] i have shared the details. your email d...,"[USER] I'm sorry for the delay in refunds, May...",REFUND_OR_COMPENSATION,HIGH



--- REPLACEMENT ---


,case_id,customer_problem,support_action,resolution_type,evidence_quality
165,AMZ-000166,[user] i want to buy mobile if i feel not sati...,[USER] All mobiles on Amazon.in are covered un...,REPLACEMENT,HIGH
1075,AMZ-001080,[user] dhl evidently put the wrong label on it...,[USER] Keep us posted on the arrival of the re...,REPLACEMENT,HIGH
1308,AMZ-001313,[user] yes i’ve had an email saying that they ...,[USER] Did they also confirm a replacement by ...,REPLACEMENT,HIGH
1418,AMZ-001423,dear amazon drivers: half &amp; half contains ...,[USER] Uh oh! Were you able to reach out to us...,REPLACEMENT,HIGH
1490,AMZ-001495,[user] no one's offering anything to really he...,[USER] I completely understand your frustratio...,REPLACEMENT,HIGH



--- CANCELLATION ---


,case_id,customer_problem,support_action,resolution_type,evidence_quality
207,AMZ-000208,[user] why have you taken a prime payment when...,[USER] Apologies - Did you receive an email co...,CANCELLATION,HIGH
485,AMZ-000487,[user] if i reorder i need to pay 3700 extra w...,"[USER] As informed earlier, we won't be able t...",CANCELLATION,HIGH
665,AMZ-000668,anyone else charged multiple times this week f...,[USER] Have you tried cancelling it through yo...,CANCELLATION,MEDIUM
839,AMZ-000843,[user] [user] because while the savings are ge...,"[USER] Has the order been cancelled, or is it ...",CANCELLATION,HIGH
1007,AMZ-001012,[user] crap. what should i do ? your telephoni...,"[USER] If you haven't canceled the order, you ...",CANCELLATION,HIGH


In [46]:
# ============================================================
# HISTORICAL RESOLUTION TENDENCY FROM TOP-5 RETRIEVAL
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

reranked = pd.read_csv(
    "data/checkpoints/cross_encoder_reranked_results.csv"
)

rag_corpus = pd.read_csv(
    "data/checkpoints/structured_rag_corpus.csv"
)

golden = pd.read_csv(
    "data/evaluation/golden_evaluation_set.csv"
)

print("Reranked:", reranked.shape)
print("RAG corpus:", rag_corpus.shape)
print("Golden:", golden.shape)


# ------------------------------------------------------------
# JOIN RESOLUTION TYPE INTO RETRIEVAL RESULTS
# ------------------------------------------------------------

retrieval_with_resolution = reranked.merge(
    rag_corpus[
        [
            "case_id",
            "resolution_type"
        ]
    ],
    on="case_id",
    how="left"
)

print("\nJoined retrieval:", retrieval_with_resolution.shape)

print("\nMissing resolution types:",
      retrieval_with_resolution["resolution_type"].isna().sum())


# ============================================================
# TOP-5 FEATURES PER GOLDEN CASE
# ============================================================

top5 = (
    retrieval_with_resolution
    .sort_values(["golden_id", "ce_rank"])
    .groupby("golden_id")
    .head(5)
    .copy()
)


def resolution_features(group):

    total = len(group)

    counts = group["resolution_type"].value_counts()

    investigation_rate = (
        counts.get("INVESTIGATION_OR_FOLLOWUP", 0) / total
    )

    general_support_rate = (
        counts.get("GENERAL_SUPPORT", 0) / total
    )

    refund_rate = (
        counts.get("REFUND_OR_COMPENSATION", 0) / total
    )

    cancellation_rate = (
        counts.get("CANCELLATION", 0) / total
    )

    replacement_rate = (
        counts.get("REPLACEMENT", 0) / total
    )

    return pd.Series({
        "investigation_rate": investigation_rate,
        "general_support_rate": general_support_rate,
        "refund_rate": refund_rate,
        "cancellation_rate": cancellation_rate,
        "replacement_rate": replacement_rate
    })


resolution_features_df = (
    top5
    .groupby("golden_id")
    .apply(resolution_features)
    .reset_index()
)

print("\nResolution features:")
display(resolution_features_df.head())


# ============================================================
# ADD GOLD LABELS
# ============================================================

analysis = resolution_features_df.merge(
    golden[
        [
            "golden_id",
            "intent",
            "risk_level",
            "resolution_status",
            "automation_decision"
        ]
    ],
    on="golden_id",
    how="left"
)

print("\nFinal analysis shape:", analysis.shape)


# ============================================================
# COMPARE AUTO vs HUMAN
# ============================================================

print("\n" + "=" * 80)
print("HISTORICAL RESOLUTION TENDENCY: AUTO vs HUMAN")
print("=" * 80)

features = [
    "investigation_rate",
    "general_support_rate",
    "refund_rate",
    "cancellation_rate",
    "replacement_rate"
]

display(
    analysis
    .groupby("automation_decision")[features]
    .agg(["mean", "median", "min", "max"])
    .round(4)
)


# ============================================================
# INVESTIGATION RATE DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("INVESTIGATION RATE BY HUMAN AUTOMATION LABEL")
print("=" * 80)

display(
    analysis
    .groupby("automation_decision")["investigation_rate"]
    .describe()
    .round(4)
)


# ============================================================
# HUMAN GOLD LABEL
# ============================================================

print("\n" + "=" * 80)
print("GOLD AUTOMATION × INVESTIGATION RATE BUCKET")
print("=" * 80)

analysis["investigation_bucket"] = pd.cut(
    analysis["investigation_rate"],
    bins=[-0.01, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=[
        "0-20%",
        "20-40%",
        "40-60%",
        "60-80%",
        "80-100%"
    ]
)

display(
    pd.crosstab(
        analysis["investigation_bucket"],
        analysis["automation_decision"],
        normalize="index"
    ).round(4) * 100
)


# ============================================================
# LOW-RISK ONLY
# ============================================================

print("\n" + "=" * 80)
print("LOW-RISK: INVESTIGATION RATE × AUTOMATION")
print("=" * 80)

low = analysis[
    analysis["risk_level"] == "LOW"
]

display(
    low
    .groupby("automation_decision")["investigation_rate"]
    .describe()
    .round(4)
)

Reranked: (10000, 11)
RAG corpus: (83218, 6)
Golden: (200, 16)

Joined retrieval: (10000, 12)

Missing resolution types: 0

Resolution features:


/tmp/ipykernel_921/3292244902.py:101: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(resolution_features)


,golden_id,investigation_rate,general_support_rate,refund_rate,cancellation_rate,replacement_rate
0,GOLD-0001,0.8,0.2,0.0,0.0,0.0
1,GOLD-0002,0.6,0.4,0.0,0.0,0.0
2,GOLD-0003,0.4,0.4,0.2,0.0,0.0
3,GOLD-0004,0.0,0.8,0.0,0.0,0.2
4,GOLD-0005,0.2,0.8,0.0,0.0,0.0



Final analysis shape: (200, 10)

HISTORICAL RESOLUTION TENDENCY: AUTO vs HUMAN


investigation_rate                  general_support_rate  \
                                  mean median  min  max                 mean   
automation_decision                                                            
AUTO                            0.1962    0.0  0.0  1.0               0.7731   
HUMAN                           0.4568    0.4  0.0  1.0               0.5176   

                                     refund_rate                   \
                    median  min  max        mean median  min  max   
automation_decision                                                 
AUTO                   1.0  0.0  1.0      0.0231    0.0  0.0  0.8   
HUMAN                  0.4  0.0  1.0      0.0162    0.0  0.0  0.4   

                    cancellation_rate                  replacement_rate  \
                                 mean median  min  max             mean   
automation_decision                                                       
AUTO                           0.0038    0.0  0.0  0.2           0.0038   
HUMAN                          0.0041    0.0  0.0  0.4           0.0054   

                                      
                    median  min  max  
automation_decision                   
AUTO                   0.0  0.0  0.2  
HUMAN                  0.0  0.0  0.2

...

,count,mean,std,min,25%,50%,75%,max
automation_decision,,,,,,,,
AUTO,52.0,0.1962,0.2977,0.0,0.0,0.0,0.4,1.0
HUMAN,148.0,0.4568,0.2741,0.0,0.2,0.4,0.6,1.0



GOLD AUTOMATION × INVESTIGATION RATE BUCKET


automation_decision,AUTO,HUMAN
investigation_bucket,,
0-20%,45.12,54.88
20-40%,15.38,84.62
40-60%,8.70,91.30
60-80%,12.50,87.50
80-100%,22.22,77.78



LOW-RISK: INVESTIGATION RATE × AUTOMATION


,count,mean,std,min,25%,50%,75%,max
automation_decision,,,,,,,,
AUTO,51.0,0.1804,0.2779,0.0,0.0,0.0,0.4,1.0
HUMAN,55.0,0.4509,0.2860,0.0,0.2,0.4,0.6,1.0


In [47]:
# ============================================================
# COMBINED AUTOMATION SIGNAL ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# We already created `analysis` in the previous cell.
# It contains:
# investigation_rate
# risk_level
# resolution_status
# automation_decision

# ------------------------------------------------------------
# LOW-RISK CASES
# ------------------------------------------------------------

low = analysis[
    analysis["risk_level"] == "LOW"
].copy()


# ------------------------------------------------------------
# INVESTIGATION RATE THRESHOLD ANALYSIS
# ------------------------------------------------------------

print("=" * 80)
print("LOW-RISK INVESTIGATION-RATE THRESHOLD ANALYSIS")
print("=" * 80)

thresholds = np.arange(0.0, 1.01, 0.10)

rows = []

for threshold in thresholds:

    # Candidate AUTO if investigation tendency is below threshold
    predicted_auto = (
        low["investigation_rate"] < threshold
    )

    actual_auto = (
        low["automation_decision"] == "AUTO"
    )

    tp = (predicted_auto & actual_auto).sum()
    fp = (predicted_auto & ~actual_auto).sum()
    fn = (~predicted_auto & actual_auto).sum()
    tn = (~predicted_auto & ~actual_auto).sum()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    rows.append({
        "threshold": threshold,
        "predicted_auto": int(predicted_auto.sum()),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "auto_precision": precision,
        "auto_recall": recall
    })

threshold_df = pd.DataFrame(rows)

display(
    threshold_df.round(4)
)


# ------------------------------------------------------------
# LOW-RISK + RELEVANT ONLY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOW-RISK + RELEVANT EVIDENCE")
print("=" * 80)

low_relevant = low[
    low["evidence_relevance"] == "RELEVANT"
].copy()

print("Cases:", len(low_relevant))

display(
    low_relevant
    .groupby("automation_decision")["investigation_rate"]
    .describe()
    .round(4)
)


# ------------------------------------------------------------
# LOW-RISK + RELEVANT + INVESTIGATION RATE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOW-RISK + RELEVANT — INVESTIGATION RATE BUCKETS")
print("=" * 80)

low_relevant["investigation_bucket"] = pd.cut(
    low_relevant["investigation_rate"],
    bins=[-0.01, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=[
        "0-20%",
        "20-40%",
        "40-60%",
        "60-80%",
        "80-100%"
    ]
)

display(
    pd.crosstab(
        low_relevant["investigation_bucket"],
        low_relevant["automation_decision"]
    )
)


print("\nPercentages:")

display(
    pd.crosstab(
        low_relevant["investigation_bucket"],
        low_relevant["automation_decision"],
        normalize="index"
    ).round(4) * 100
)

LOW-RISK INVESTIGATION-RATE THRESHOLD ANALYSIS


,threshold,predicted_auto,TP,FP,FN,TN,auto_precision,auto_recall
0,0.0,0,0,0,51,55,0.0000,0.0000
1,0.1,41,32,9,19,46,0.7805,0.6275
2,0.2,41,32,9,19,46,0.7805,0.6275
3,0.3,52,37,15,14,40,0.7115,0.7255
4,0.4,52,37,15,14,40,0.7115,0.7255
5,0.5,73,43,30,8,25,0.5890,0.8431
6,0.6,93,47,46,4,9,0.5054,0.9216
7,0.7,93,47,46,4,9,0.5054,0.9216
8,0.8,93,47,46,4,9,0.5054,0.9216
9,0.9,101,50,51,1,4,0.4950,0.9804



LOW-RISK + RELEVANT EVIDENCE


KeyError: 'evidence_relevance'

In [48]:
# ============================================================
# ADD HUMAN-REVIEWED EVIDENCE RELEVANCE
# ============================================================

evidence_review = pd.read_csv(
    "data/evaluation/top1_evidence_review.csv"
)

analysis = analysis.merge(
    evidence_review[
        [
            "golden_id",
            "evidence_relevance"
        ]
    ],
    on="golden_id",
    how="left"
)

print("Updated analysis shape:", analysis.shape)

print("\nMissing evidence labels:",
      analysis["evidence_relevance"].isna().sum())

display(
    analysis.head()
)

Updated analysis shape: (200, 12)

Missing evidence labels: 0


,golden_id,investigation_rate,general_support_rate,refund_rate,cancellation_rate,replacement_rate,intent,risk_level,resolution_status,automation_decision,investigation_bucket,evidence_relevance
0,GOLD-0001,0.8,0.2,0.0,0.0,0.0,PAYMENT_BILLING,MEDIUM,INVESTIGATION_REQUIRED,HUMAN,60-80%,PARTIAL
1,GOLD-0002,0.6,0.4,0.0,0.0,0.0,ORDER_DELIVERY,LOW,PARTIALLY_RESOLVED,HUMAN,40-60%,RELEVANT
2,GOLD-0003,0.4,0.4,0.2,0.0,0.0,ORDER_DELIVERY,LOW,INVESTIGATION_REQUIRED,HUMAN,20-40%,RELEVANT
3,GOLD-0004,0.0,0.8,0.0,0.0,0.2,GENERAL_SOCIAL,LOW,NOT_APPLICABLE,AUTO,0-20%,RELEVANT
4,GOLD-0005,0.2,0.8,0.0,0.0,0.0,DEVICE_TECHNICAL,LOW,INVESTIGATION_REQUIRED,HUMAN,0-20%,RELEVANT


In [49]:
# ============================================================
# LOW-RISK + RELEVANT EVIDENCE
# ============================================================

low_relevant = analysis[
    (analysis["risk_level"] == "LOW") &
    (analysis["evidence_relevance"] == "RELEVANT")
].copy()

print("LOW-RISK + RELEVANT cases:", len(low_relevant))

print("\nInvestigation rate by human automation label:")

display(
    low_relevant
    .groupby("automation_decision")["investigation_rate"]
    .describe()
    .round(4)
)


# ============================================================
# BUCKET ANALYSIS
# ============================================================

low_relevant["investigation_bucket"] = pd.cut(
    low_relevant["investigation_rate"],
    bins=[-0.01, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=[
        "0-20%",
        "20-40%",
        "40-60%",
        "60-80%",
        "80-100%"
    ]
)

print("\n" + "=" * 80)
print("LOW-RISK + RELEVANT — COUNTS")
print("=" * 80)

display(
    pd.crosstab(
        low_relevant["investigation_bucket"],
        low_relevant["automation_decision"]
    )
)

print("\nPercentages:")

display(
    (
        pd.crosstab(
            low_relevant["investigation_bucket"],
            low_relevant["automation_decision"],
            normalize="index"
        ) * 100
    ).round(2)
)

LOW-RISK + RELEVANT cases: 87

Investigation rate by human automation label:


,count,mean,std,min,25%,50%,75%,max
automation_decision,,,,,,,,
AUTO,43.0,0.1628,0.2664,0.0,0.0,0.0,0.3,0.8
HUMAN,44.0,0.4273,0.2944,0.0,0.2,0.4,0.6,1.0



LOW-RISK + RELEVANT — COUNTS


automation_decision,AUTO,HUMAN
investigation_bucket,,
0-20%,32,14
20-40%,4,12
40-60%,4,12
60-80%,3,2
80-100%,0,4



Percentages:


automation_decision,AUTO,HUMAN
investigation_bucket,,
0-20%,69.57,30.43
20-40%,25.00,75.00
40-60%,25.00,75.00
60-80%,60.00,40.00
80-100%,0.00,100.00


In [51]:
import pandas as pd
import numpy as np

GOLDEN_PATH = "/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/golden_evaluation_set_frozen.csv"

golden_df = pd.read_csv(GOLDEN_PATH)

print("Shape:", golden_df.shape)
print("\nColumns:")
print(golden_df.columns.tolist())

print("\nFirst 3 rows:")
display(golden_df.head(3))

Shape: (200, 16)

Columns:
['golden_id', 'case_id', 'root_tweet_id', 'turn_count', 'customer_turns', 'support_turns', 'duration_hours', 'max_gap_hours', 'customer_query', 'support_response', 'provisional_bucket', 'risk_level', 'annotation_notes', 'intent', 'resolution_status', 'automation_decision']

First 3 rows:


,golden_id,case_id,root_tweet_id,turn_count,customer_turns,support_turns,duration_hours,max_gap_hours,customer_query,support_response,provisional_bucket,risk_level,annotation_notes,intent,resolution_status,automation_decision
0,GOLD-0001,AMZ-012695,804063,2,1,1,0.430556,0.430556,Contact @115821 regarding payment on an order....,@311616 That is definitely not the experience ...,FOLLOWUP_OR_INVESTIGATION,MEDIUM,NaN,PAYMENT_BILLING,INVESTIGATION_REQUIRED,HUMAN
1,GOLD-0002,AMZ-072455,2873292,4,2,2,0.324167,0.188056,@AmazonHelp LOL...you've missed the delivery d...,@797722 I'm sorry for the recent delays. Pleas...,RESOLUTION_CANDIDATE,LOW,NaN,ORDER_DELIVERY,PARTIALLY_RESOLVED,HUMAN
2,GOLD-0003,AMZ-034151,2362699,2,1,1,0.375278,0.375278,So bummed that my package was delayed even wit...,@682197 Very sorry to hear. Were you provided ...,RESOLUTION_CANDIDATE,LOW,NaN,ORDER_DELIVERY,INVESTIGATION_REQUIRED,HUMAN


In [53]:
required_cols = [
    "golden_id",
    "customer_query",
    "intent",
    "risk_level",
    "automation_decision"
]

for col in required_cols:
    print(col, "→", col in golden_df.columns)

golden_id → True
customer_query → True
intent → True
risk_level → True
automation_decision → True


In [54]:
print("golden_df:", "golden_df" in globals())
print("tfidf_vectorizer:", "tfidf_vectorizer" in globals())
print("intent_model:", "intent_model" in globals())

golden_df: True
tfidf_vectorizer: False
intent_model: False


In [55]:
print("Variables available:")
for x in ["train_df", "weak_labels_df", "training_df", "labeled_df"]:
    print(x, "→", x in globals())

Variables available:
train_df → False
weak_labels_df → False
training_df → False
labeled_df → False


In [56]:
import os

DATA_DIR = "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"

for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/golden_evaluation_set.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/golden_evaluation_set_frozen.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/unlabeled_intent_sample_200.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/top1_evidence_review.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/top5_historical_evidence.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/retrieval_benchmark.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/structured_rag_corpus.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/cross_encoder_reranked_results.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/rrf_candidates.csv


In [57]:
import pandas as pd

RAG_PATH = "/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/structured_rag_corpus.csv"

rag_df = pd.read_csv(RAG_PATH)

print("Shape:", rag_df.shape)
print("\nColumns:")
print(rag_df.columns.tolist())

display(rag_df.head(3))

Shape: (83218, 6)

Columns:
['case_id', 'customer_problem', 'support_action', 'resolution_type', 'evidence_quality', 'source']


,case_id,customer_problem,support_action,resolution_type,evidence_quality,source
0,AMZ-000001,[user] what in the world is a balance withheld...,"[USER] Sorry, I'm not quite sure what you're h...",GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
1,AMZ-000002,[user] that's exactly what she did. bought it ...,"[USER] Sorry, we would be unable to change thi...",GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
2,AMZ-000003,[user] [user] i think u should cancel your ord...,"[USER] We would like to help you, Prakhar. Ple...",INVESTIGATION_OR_FOLLOWUP,MEDIUM,AmazonHelp historical support case


In [58]:
golden_ids = set(golden_df["case_id"].astype(str))

rag_df["case_id"] = rag_df["case_id"].astype(str)

print("Golden cases:", len(golden_ids))
print("RAG cases:", len(rag_df))
print("Golden cases present in RAG:",
      rag_df["case_id"].isin(golden_ids).sum())

Golden cases: 200
RAG cases: 83218
Golden cases present in RAG: 0


In [59]:
RERANK_PATH = "/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/cross_encoder_reranked_results.csv"

reranked_df = pd.read_csv(RERANK_PATH)

print("Shape:", reranked_df.shape)
print("\nColumns:")
print(reranked_df.columns.tolist())

display(reranked_df.head())

Shape: (10000, 11)

Columns:
['golden_id', 'case_id', 'ce_rank', 'ce_score', 'rrf_rank', 'rrf_score', 'bm25_rank', 'bge_rank', 'historical_weak_intent', 'golden_intent', 'intent_match']


,golden_id,case_id,ce_rank,ce_score,rrf_rank,rrf_score,bm25_rank,bge_rank,historical_weak_intent,golden_intent,intent_match
0,GOLD-0001,AMZ-073280,1,0.615286,23.0,0.013889,12.0,NaN,NaN,PAYMENT_BILLING,False
1,GOLD-0001,AMZ-057467,2,0.249812,13.0,0.014925,7.0,NaN,PAYMENT_BILLING,PAYMENT_BILLING,True
2,GOLD-0001,AMZ-058158,3,0.167556,4.0,0.016129,NaN,2.0,NaN,PAYMENT_BILLING,False
3,GOLD-0001,AMZ-000377,4,0.125092,3.0,0.016393,NaN,1.0,NaN,PAYMENT_BILLING,False
4,GOLD-0001,AMZ-044915,5,0.115635,30.0,0.013333,15.0,NaN,ORDER_PURCHASE,PAYMENT_BILLING,False


In [60]:
import numpy as np
import pandas as pd

# Work only with runtime-available retrieval signals
runtime_df = reranked_df.drop(
    columns=["golden_intent", "intent_match"]
).copy()

# Make sure ranking is correct
runtime_df = runtime_df.sort_values(
    ["golden_id", "ce_rank"]
)

# -----------------------------
# Feature extraction per case
# -----------------------------

def extract_evidence_features(group):

    top5 = group.head(5)

    ce = top5["ce_score"].astype(float)

    # Cross-encoder features
    top1_ce = ce.iloc[0]
    mean_ce = ce.mean()
    max_ce = ce.max()

    ce_strong_count = (ce >= 0.90).sum()
    ce_conditional_count = (ce >= 0.75).sum()

    # Retrieval consistency
    rrf_scores = top5["rrf_score"].astype(float)

    mean_rrf = rrf_scores.mean()

    # Historical weak-intent availability
    weak_intents = top5["historical_weak_intent"].dropna()

    weak_intent_coverage = len(weak_intents) / len(top5)

    # Historical investigation tendency
    # NOTE: resolution_type is not present here, so we
    # cannot calculate investigation_rate from this file.
    #
    # We'll merge that feature later from rag_df.

    return pd.Series({
        "top1_ce": top1_ce,
        "mean_ce": mean_ce,
        "max_ce": max_ce,
        "ce_strong_count": ce_strong_count,
        "ce_conditional_count": ce_conditional_count,
        "mean_rrf": mean_rrf,
        "weak_intent_coverage": weak_intent_coverage
    })


evidence_features = (
    runtime_df
    .groupby("golden_id")
    .apply(extract_evidence_features)
    .reset_index()
)

print("Feature shape:", evidence_features.shape)

display(evidence_features.head())

Feature shape: (200, 8)


/tmp/ipykernel_921/1867953527.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extract_evidence_features)


,golden_id,top1_ce,mean_ce,max_ce,ce_strong_count,ce_conditional_count,mean_rrf,weak_intent_coverage
0,GOLD-0001,0.615286,0.254676,0.615286,0.0,0.0,0.014934,0.4
1,GOLD-0002,0.998420,0.994704,0.998420,5.0,5.0,0.022889,1.0
2,GOLD-0003,0.991000,0.986164,0.991000,5.0,5.0,0.016734,0.6
3,GOLD-0004,0.957855,0.901911,0.957855,4.0,5.0,0.021089,0.4
4,GOLD-0005,0.859899,0.343932,0.859899,0.0,1.0,0.015403,0.0


In [61]:
# -----------------------------------------
# Merge historical resolution information
# -----------------------------------------

resolution_lookup = rag_df[
    [
        "case_id",
        "resolution_type"
    ]
].copy()

resolution_lookup["case_id"] = (
    resolution_lookup["case_id"].astype(str)
)

runtime_with_resolution = runtime_df.merge(
    resolution_lookup,
    on="case_id",
    how="left"
)

print("Rows after merge:", len(runtime_with_resolution))
print(
    "Missing resolution_type:",
    runtime_with_resolution["resolution_type"].isna().sum()
)

Rows after merge: 10000
Missing resolution_type: 0


In [62]:
def extract_resolution_features(group):

    top5 = group.head(5)

    resolution = top5["resolution_type"].fillna("UNKNOWN")

    n = len(top5)

    return pd.Series({
        "investigation_rate": (
            (resolution == "INVESTIGATION_OR_FOLLOWUP").sum() / n
        ),

        "general_support_rate": (
            (resolution == "GENERAL_SUPPORT").sum() / n
        ),

        "refund_rate": (
            (resolution == "REFUND_OR_COMPENSATION").sum() / n
        ),

        "cancellation_rate": (
            (resolution == "CANCELLATION").sum() / n
        ),

        "replacement_rate": (
            (resolution == "REPLACEMENT").sum() / n
        )
    })


resolution_features = (
    runtime_with_resolution
    .sort_values(["golden_id", "ce_rank"])
    .groupby("golden_id")
    .apply(extract_resolution_features)
    .reset_index()
)

print("Resolution feature shape:", resolution_features.shape)

display(resolution_features.head())

Resolution feature shape: (200, 6)


/tmp/ipykernel_921/2561284825.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extract_resolution_features)


,golden_id,investigation_rate,general_support_rate,refund_rate,cancellation_rate,replacement_rate
0,GOLD-0001,0.8,0.2,0.0,0.0,0.0
1,GOLD-0002,0.6,0.4,0.0,0.0,0.0
2,GOLD-0003,0.4,0.4,0.2,0.0,0.0
3,GOLD-0004,0.0,0.8,0.0,0.0,0.2
4,GOLD-0005,0.2,0.8,0.0,0.0,0.0


In [63]:
trust_features = evidence_features.merge(
    resolution_features,
    on="golden_id",
    how="inner"
)

print("Final feature shape:", trust_features.shape)

display(trust_features.head())

Final feature shape: (200, 13)


,golden_id,top1_ce,mean_ce,max_ce,ce_strong_count,ce_conditional_count,mean_rrf,weak_intent_coverage,investigation_rate,general_support_rate,refund_rate,cancellation_rate,replacement_rate
0,GOLD-0001,0.615286,0.254676,0.615286,0.0,0.0,0.014934,0.4,0.8,0.2,0.0,0.0,0.0
1,GOLD-0002,0.998420,0.994704,0.998420,5.0,5.0,0.022889,1.0,0.6,0.4,0.0,0.0,0.0
2,GOLD-0003,0.991000,0.986164,0.991000,5.0,5.0,0.016734,0.6,0.4,0.4,0.2,0.0,0.0
3,GOLD-0004,0.957855,0.901911,0.957855,4.0,5.0,0.021089,0.4,0.0,0.8,0.0,0.0,0.2
4,GOLD-0005,0.859899,0.343932,0.859899,0.0,1.0,0.015403,0.0,0.2,0.8,0.0,0.0,0.0


In [64]:
# Add the golden labels ONLY for analysis
analysis_df = trust_features.merge(
    golden_df[
        [
            "golden_id",
            "intent",
            "risk_level",
            "automation_decision"
        ]
    ],
    on="golden_id",
    how="left"
)

feature_cols = [
    "top1_ce",
    "mean_ce",
    "max_ce",
    "ce_strong_count",
    "ce_conditional_count",
    "mean_rrf",
    "weak_intent_coverage",
    "investigation_rate",
    "general_support_rate",
    "refund_rate",
    "cancellation_rate",
    "replacement_rate"
]

comparison = (
    analysis_df
    .groupby("automation_decision")[feature_cols]
    .agg(["mean", "median"])
    .round(3)
)

display(comparison)

top1_ce        mean_ce        max_ce         \
                       mean median    mean median   mean median   
automation_decision                                               
AUTO                  0.773  0.952   0.640  0.754  0.773  0.952   
HUMAN                 0.869  0.971   0.733  0.854  0.869  0.971   

                    ce_strong_count        ce_conditional_count         ...  \
                               mean median                 mean median  ...   
automation_decision                                                     ...   
AUTO                          1.923    1.0                2.654    3.0  ...   
HUMAN                         2.243    2.0                3.223    5.0  ...   

                    investigation_rate        general_support_rate         \
                                  mean median                 mean median   
automation_decision                                                         
AUTO                             0.196    0.0                0.773    1.0   
HUMAN                            0.457    0.4                0.518    0.4   

                    refund_rate        cancellation_rate         \
                           mean median              mean median   
automation_decision                                               
AUTO                      0.023    0.0             0.004    0.0   
HUMAN                     0.016    0.0             0.004    0.0   

                    replacement_rate         
                                mean median  
automation_decision                          
AUTO                           0.004    0.0  
HUMAN                          0.005    0.0  

[2 rows x 24 columns]

In [65]:
print("AUTO vs HUMAN")

for col in feature_cols:
    auto = analysis_df.loc[
        analysis_df["automation_decision"] == "AUTO",
        col
    ]

    human = analysis_df.loc[
        analysis_df["automation_decision"] == "HUMAN",
        col
    ]

    print(
        f"{col:28s} "
        f"AUTO={auto.mean():.3f} "
        f"HUMAN={human.mean():.3f}"
    )

AUTO vs HUMAN
top1_ce                      AUTO=0.773 HUMAN=0.869
mean_ce                      AUTO=0.640 HUMAN=0.733
max_ce                       AUTO=0.773 HUMAN=0.869
ce_strong_count              AUTO=1.923 HUMAN=2.243
ce_conditional_count         AUTO=2.654 HUMAN=3.223
mean_rrf                     AUTO=0.018 HUMAN=0.017
weak_intent_coverage         AUTO=0.154 HUMAN=0.399
investigation_rate           AUTO=0.196 HUMAN=0.457
general_support_rate         AUTO=0.773 HUMAN=0.518
refund_rate                  AUTO=0.023 HUMAN=0.016
cancellation_rate            AUTO=0.004 HUMAN=0.004
replacement_rate             AUTO=0.004 HUMAN=0.005


In [66]:
low_medium = analysis_df[
    analysis_df["risk_level"].isin(["LOW", "MEDIUM"])
]

print(
    low_medium.groupby("automation_decision")[feature_cols]
    .mean()
    .round(3)
)

                     top1_ce  mean_ce  max_ce  ce_strong_count  \
automation_decision                                              
AUTO                   0.773    0.640   0.773            1.923   
HUMAN                  0.872    0.732   0.872            2.254   

                     ce_conditional_count  mean_rrf  weak_intent_coverage  \
automation_decision                                                         
AUTO                                2.654     0.018                 0.154   
HUMAN                               3.221     0.018                 0.379   

                     investigation_rate  general_support_rate  refund_rate  \
automation_decision                                                          
AUTO                              0.196                 0.773        0.023   
HUMAN                             0.438                 0.539        0.015   

                     cancellation_rate  replacement_rate  
automation_decision                                   

In [67]:
from sklearn.metrics import roc_auc_score

# AUTO = 1, HUMAN = 0
y = (
    analysis_df["automation_decision"]
    .eq("AUTO")
    .astype(int)
)

print("Univariate AUC — all non-HIGH-risk cases\n")

auc_results = []

for col in feature_cols:
    x = analysis_df[col].astype(float)

    # AUC requires variation in the feature
    if x.nunique() < 2:
        continue

    auc = roc_auc_score(y, x)

    # AUC below 0.5 simply means the direction is reversed
    useful_auc = max(auc, 1 - auc)

    direction = "HIGHER → AUTO" if auc >= 0.5 else "LOWER → AUTO"

    auc_results.append({
        "feature": col,
        "raw_auc": round(auc, 3),
        "useful_auc": round(useful_auc, 3),
        "direction": direction
    })

auc_df = (
    pd.DataFrame(auc_results)
    .sort_values("useful_auc", ascending=False)
)

display(auc_df)

Univariate AUC — all non-HIGH-risk cases



,feature,raw_auc,useful_auc,direction
7,investigation_rate,0.246,0.754,LOWER → AUTO
8,general_support_rate,0.739,0.739,HIGHER → AUTO
6,weak_intent_coverage,0.304,0.696,LOWER → AUTO
4,ce_conditional_count,0.413,0.587,LOWER → AUTO
1,mean_ce,0.423,0.577,LOWER → AUTO
3,ce_strong_count,0.445,0.555,LOWER → AUTO
2,max_ce,0.459,0.541,LOWER → AUTO
0,top1_ce,0.459,0.541,LOWER → AUTO
5,mean_rrf,0.514,0.514,HIGHER → AUTO
11,replacement_rate,0.496,0.504,LOWER → AUTO


In [68]:
lm = analysis_df[
    analysis_df["risk_level"].isin(["LOW", "MEDIUM"])
].copy()

y_lm = (
    lm["automation_decision"]
    .eq("AUTO")
    .astype(int)
)

auc_results_lm = []

for col in feature_cols:
    x = lm[col].astype(float)

    if x.nunique() < 2:
        continue

    auc = roc_auc_score(y_lm, x)
    useful_auc = max(auc, 1 - auc)

    direction = (
        "HIGHER → AUTO"
        if auc >= 0.5
        else "LOWER → AUTO"
    )

    auc_results_lm.append({
        "feature": col,
        "raw_auc": round(auc, 3),
        "useful_auc": round(useful_auc, 3),
        "direction": direction
    })

auc_lm_df = (
    pd.DataFrame(auc_results_lm)
    .sort_values("useful_auc", ascending=False)
)

display(auc_lm_df)

,feature,raw_auc,useful_auc,direction
7,investigation_rate,0.258,0.742,LOWER → AUTO
8,general_support_rate,0.726,0.726,HIGHER → AUTO
6,weak_intent_coverage,0.323,0.677,LOWER → AUTO
4,ce_conditional_count,0.413,0.587,LOWER → AUTO
1,mean_ce,0.424,0.576,LOWER → AUTO
3,ce_strong_count,0.447,0.553,LOWER → AUTO
2,max_ce,0.464,0.536,LOWER → AUTO
0,top1_ce,0.464,0.536,LOWER → AUTO
11,replacement_rate,0.493,0.507,LOWER → AUTO
10,cancellation_rate,0.506,0.506,HIGHER → AUTO


In [69]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Only LOW + MEDIUM cases
model_df = analysis_df[
    analysis_df["risk_level"].isin(["LOW", "MEDIUM"])
].copy()

features_v2 = [
    "investigation_rate",
    "general_support_rate",
    "weak_intent_coverage",
    "top1_ce"
]

X = model_df[features_v2].fillna(0)
y = (
    model_df["automation_decision"]
    .eq("AUTO")
    .astype(int)
)

print("Samples:", len(model_df))
print("AUTO:", y.sum())
print("HUMAN:", (y == 0).sum())

model_v2 = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_proba = cross_val_predict(
    model_v2,
    X,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

oof_pred = (oof_proba >= 0.5).astype(int)

print("\n5-FOLD OUT-OF-FOLD RESULTS")
print("--------------------------------")
print("ROC-AUC:",
      round(roc_auc_score(y, oof_proba), 3))

print("PR-AUC:",
      round(average_precision_score(y, oof_proba), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y, oof_pred))

print("\nClassification Report:")
print(
    classification_report(
        y,
        oof_pred,
        target_names=["HUMAN", "AUTO"],
        zero_division=0
    )
)

Samples: 174
AUTO: 52
HUMAN: 122

5-FOLD OUT-OF-FOLD RESULTS
--------------------------------
ROC-AUC: 0.723
PR-AUC: 0.525

Confusion Matrix:
[[86 36]
 [16 36]]

Classification Report:
              precision    recall  f1-score   support

       HUMAN       0.84      0.70      0.77       122
        AUTO       0.50      0.69      0.58        52

    accuracy                           0.70       174
   macro avg       0.67      0.70      0.67       174
weighted avg       0.74      0.70      0.71       174



In [70]:
# Fit on all LOW/MEDIUM cases only
model_v2.fit(X, y)

coef_df = pd.DataFrame({
    "feature": features_v2,
    "coefficient": model_v2.named_steps["classifier"].coef_[0]
})

coef_df["absolute_coefficient"] = (
    coef_df["coefficient"].abs()
)

coef_df = coef_df.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(coef_df)

,feature,coefficient,absolute_coefficient
0,investigation_rate,-0.856021,0.856021
2,weak_intent_coverage,-0.611720,0.611720
1,general_support_rate,-0.155373,0.155373
3,top1_ce,-0.097543,0.097543


In [71]:
features_simple = [
    "investigation_rate",
    "general_support_rate"
]

X_simple = model_df[features_simple].fillna(0)
y_simple = (
    model_df["automation_decision"]
    .eq("AUTO")
    .astype(int)
)

model_simple = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

oof_simple = cross_val_predict(
    model_simple,
    X_simple,
    y_simple,
    cv=cv,
    method="predict_proba"
)[:, 1]

pred_simple = (oof_simple >= 0.5).astype(int)

print("2-FEATURE MODEL")
print("-----------------------------")
print("ROC-AUC:",
      round(roc_auc_score(y_simple, oof_simple), 3))

print("PR-AUC:",
      round(average_precision_score(y_simple, oof_simple), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_simple, pred_simple))

print("\nClassification Report:")
print(
    classification_report(
        y_simple,
        pred_simple,
        target_names=["HUMAN", "AUTO"],
        zero_division=0
    )
)

2-FEATURE MODEL
-----------------------------
ROC-AUC: 0.73
PR-AUC: 0.539

Confusion Matrix:
[[85 37]
 [16 36]]

Classification Report:
              precision    recall  f1-score   support

       HUMAN       0.84      0.70      0.76       122
        AUTO       0.49      0.69      0.58        52

    accuracy                           0.70       174
   macro avg       0.67      0.69      0.67       174
weighted avg       0.74      0.70      0.71       174



In [72]:
model_simple.fit(X_simple, y_simple)

simple_coef = pd.DataFrame({
    "feature": features_simple,
    "coefficient": model_simple.named_steps["classifier"].coef_[0]
})

display(simple_coef)

,feature,coefficient
0,investigation_rate,-0.703052
1,general_support_rate,0.142608


In [73]:
from sklearn.metrics import precision_score, recall_score

threshold_results = []

for threshold in np.arange(0.10, 0.96, 0.05):

    pred = (oof_simple >= threshold).astype(int)

    auto_count = pred.sum()

    if auto_count == 0:
        continue

    precision = precision_score(
        y_simple,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_simple,
        pred,
        zero_division=0
    )

    coverage = auto_count / len(pred)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "auto_cases": int(auto_count),
        "auto_coverage": round(coverage, 3),
        "auto_precision": round(precision, 3),
        "auto_recall": round(recall, 3)
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

,threshold,auto_cases,auto_coverage,auto_precision,auto_recall
0,0.10,169,0.971,0.296,0.962
1,0.15,165,0.948,0.297,0.942
2,0.20,163,0.937,0.301,0.942
3,0.25,143,0.822,0.322,0.885
4,0.30,142,0.816,0.317,0.865
5,0.35,110,0.632,0.391,0.827
6,0.40,105,0.603,0.410,0.827
7,0.45,83,0.477,0.458,0.731
8,0.50,73,0.420,0.493,0.692
9,0.55,71,0.408,0.507,0.692


In [74]:
threshold_df = threshold_df.sort_values(
    "threshold"
)

print("\nOOF probability distribution:")
print(
    pd.Series(oof_simple).describe(
        percentiles=[.1, .25, .5, .75, .9, .95]
    )
)


OOF probability distribution:
count    174.000000
mean       0.466258
std        0.195132
min        0.058678
10%        0.215667
25%        0.318251
50%        0.448504
75%        0.685301
90%        0.711744
95%        0.759851
max        0.759851
dtype: float64


In [75]:
print("\nTop AUTO-probability cases:")
display(
    model_df[
        [
            "golden_id",
            "risk_level",
            "automation_decision",
            "investigation_rate",
            "general_support_rate"
        ]
    ]
    .assign(
        oof_probability=oof_simple
    )
    .sort_values(
        "oof_probability",
        ascending=False
    )
    .head(20)
)


Top AUTO-probability cases:


,golden_id,risk_level,automation_decision,investigation_rate,general_support_rate,oof_probability
12,GOLD-0013,LOW,HUMAN,0.0,1.0,0.759851
15,GOLD-0016,LOW,AUTO,0.0,1.0,0.759851
17,GOLD-0018,LOW,AUTO,0.0,1.0,0.759851
150,GOLD-0151,LOW,AUTO,0.0,1.0,0.759851
92,GOLD-0093,MEDIUM,HUMAN,0.0,1.0,0.759851
112,GOLD-0113,LOW,AUTO,0.0,1.0,0.759851
129,GOLD-0130,LOW,HUMAN,0.0,1.0,0.759851
137,GOLD-0138,LOW,AUTO,0.0,1.0,0.759851
147,GOLD-0148,MEDIUM,HUMAN,0.0,1.0,0.759851
146,GOLD-0147,LOW,AUTO,0.0,1.0,0.759851


In [77]:
complexity_cols = [
    "golden_id",
    "turn_count",
    "customer_turns",
    "support_turns",
    "duration_hours",
    "max_gap_hours"
]

complexity_data = golden_df[complexity_cols].copy()

analysis_df = analysis_df.drop(
    columns=[
        "turn_count",
        "customer_turns",
        "support_turns",
        "duration_hours",
        "max_gap_hours"
    ],
    errors="ignore"
)

analysis_df = analysis_df.merge(
    complexity_data,
    on="golden_id",
    how="left"
)

print("Shape:", analysis_df.shape)

display(
    analysis_df[
        [
            "golden_id",
            "turn_count",
            "customer_turns",
            "support_turns",
            "duration_hours",
            "max_gap_hours",
            "risk_level",
            "automation_decision"
        ]
    ].head()
)

Shape: (200, 21)


,golden_id,turn_count,customer_turns,support_turns,duration_hours,max_gap_hours,risk_level,automation_decision
0,GOLD-0001,2,1,1,0.430556,0.430556,MEDIUM,HUMAN
1,GOLD-0002,4,2,2,0.324167,0.188056,LOW,HUMAN
2,GOLD-0003,2,1,1,0.375278,0.375278,LOW,HUMAN
3,GOLD-0004,2,1,1,0.051944,0.051944,LOW,AUTO
4,GOLD-0005,7,4,3,1.773889,1.313611,LOW,HUMAN


In [79]:
complexity_cols = [
    "golden_id",
    "turn_count",
    "customer_turns",
    "support_turns",
    "duration_hours",
    "max_gap_hours"
]

# Add complexity features to the 174-case model dataset
model_df = model_df.drop(
    columns=[
        "turn_count",
        "customer_turns",
        "support_turns",
        "duration_hours",
        "max_gap_hours"
    ],
    errors="ignore"
)

model_df = model_df.merge(
    golden_df[complexity_cols],
    on="golden_id",
    how="left"
)

print("Model dataset shape:", model_df.shape)

display(
    model_df[
        [
            "golden_id",
            "turn_count",
            "customer_turns",
            "support_turns",
            "duration_hours",
            "max_gap_hours",
            "risk_level",
            "automation_decision"
        ]
    ].head()
)

Model dataset shape: (174, 21)


,golden_id,turn_count,customer_turns,support_turns,duration_hours,max_gap_hours,risk_level,automation_decision
0,GOLD-0001,2,1,1,0.430556,0.430556,MEDIUM,HUMAN
1,GOLD-0002,4,2,2,0.324167,0.188056,LOW,HUMAN
2,GOLD-0003,2,1,1,0.375278,0.375278,LOW,HUMAN
3,GOLD-0004,2,1,1,0.051944,0.051944,LOW,AUTO
4,GOLD-0005,7,4,3,1.773889,1.313611,LOW,HUMAN


In [80]:
complexity_features = [
    "turn_count",
    "customer_turns",
    "support_turns",
    "duration_hours",
    "max_gap_hours"
]

y_complexity = (
    model_df["automation_decision"]
    .eq("AUTO")
    .astype(int)
)

complexity_auc_results = []

for col in complexity_features:

    x = model_df[col].astype(float).fillna(0)

    if x.nunique() < 2:
        continue

    auc = roc_auc_score(
        y_complexity,
        x
    )

    complexity_auc_results.append({
        "feature": col,
        "raw_auc": round(auc, 3),
        "useful_auc": round(max(auc, 1 - auc), 3),
        "direction": (
            "HIGHER → AUTO"
            if auc >= 0.5
            else "LOWER → AUTO"
        )
    })

complexity_auc_df = (
    pd.DataFrame(complexity_auc_results)
    .sort_values(
        "useful_auc",
        ascending=False
    )
)

display(complexity_auc_df)

,feature,raw_auc,useful_auc,direction
2,support_turns,0.459,0.541,LOWER → AUTO
0,turn_count,0.463,0.537,LOWER → AUTO
4,max_gap_hours,0.469,0.531,LOWER → AUTO
1,customer_turns,0.479,0.521,LOWER → AUTO
3,duration_hours,0.480,0.520,LOWER → AUTO


In [81]:
threshold_results = []

for threshold in np.arange(0.0, 1.01, 0.05):

    predicted_auto = (
        model_df["investigation_rate"] <= threshold
    ).astype(int)

    auto_count = predicted_auto.sum()

    if auto_count == 0:
        continue

    precision = precision_score(
        y_complexity,
        predicted_auto,
        zero_division=0
    )

    recall = recall_score(
        y_complexity,
        predicted_auto,
        zero_division=0
    )

    coverage = auto_count / len(model_df)

    threshold_results.append({
        "investigation_threshold": round(threshold, 2),
        "auto_cases": int(auto_count),
        "auto_coverage": round(coverage, 3),
        "auto_precision": round(precision, 3),
        "auto_recall": round(recall, 3)
    })

investigation_threshold_df = pd.DataFrame(
    threshold_results
)

display(investigation_threshold_df)

,investigation_threshold,auto_cases,auto_coverage,auto_precision,auto_recall
0,0.00,49,0.282,0.653,0.615
1,0.05,49,0.282,0.653,0.615
2,0.10,49,0.282,0.653,0.615
3,0.15,49,0.282,0.653,0.615
4,0.20,75,0.431,0.493,0.712
5,0.25,75,0.431,0.493,0.712
6,0.30,75,0.431,0.493,0.712
7,0.35,75,0.431,0.493,0.712
8,0.40,110,0.632,0.391,0.827
9,0.45,110,0.632,0.391,0.827


In [82]:
display(
    investigation_threshold_df
    .sort_values(
        ["auto_precision", "auto_coverage"],
        ascending=[False, False]
    )
)

,investigation_threshold,auto_cases,auto_coverage,auto_precision,auto_recall
0,0.00,49,0.282,0.653,0.615
1,0.05,49,0.282,0.653,0.615
2,0.10,49,0.282,0.653,0.615
3,0.15,49,0.282,0.653,0.615
4,0.20,75,0.431,0.493,0.712
5,0.25,75,0.431,0.493,0.712
6,0.30,75,0.431,0.493,0.712
7,0.35,75,0.431,0.493,0.712
8,0.40,110,0.632,0.391,0.827
9,0.45,110,0.632,0.391,0.827


In [83]:
# Test a simple conservative 2-signal policy

policy_results = []

for gs_threshold in np.arange(0.0, 1.01, 0.10):

    predicted_auto = (
        (model_df["investigation_rate"] == 0) &
        (model_df["general_support_rate"] >= gs_threshold)
    ).astype(int)

    auto_count = predicted_auto.sum()

    if auto_count == 0:
        continue

    precision = precision_score(
        y_complexity,
        predicted_auto,
        zero_division=0
    )

    recall = recall_score(
        y_complexity,
        predicted_auto,
        zero_division=0
    )

    coverage = auto_count / len(model_df)

    policy_results.append({
        "general_support_threshold": round(gs_threshold, 2),
        "auto_cases": int(auto_count),
        "auto_coverage": round(coverage, 3),
        "auto_precision": round(precision, 3),
        "auto_recall": round(recall, 3)
    })

policy_df = pd.DataFrame(policy_results)

display(policy_df)

,general_support_threshold,auto_cases,auto_coverage,auto_precision,auto_recall
0,0.0,49,0.282,0.653,0.615
1,0.1,49,0.282,0.653,0.615
2,0.2,49,0.282,0.653,0.615
3,0.3,48,0.276,0.646,0.596
4,0.4,48,0.276,0.646,0.596
5,0.5,48,0.276,0.646,0.596
6,0.6,47,0.270,0.660,0.596
7,0.7,47,0.270,0.660,0.596
8,0.8,47,0.270,0.660,0.596
9,0.9,44,0.253,0.659,0.558


In [84]:
display(
    policy_df.sort_values(
        ["auto_precision", "auto_coverage"],
        ascending=[False, False]
    )
)

,general_support_threshold,auto_cases,auto_coverage,auto_precision,auto_recall
6,0.6,47,0.270,0.660,0.596
7,0.7,47,0.270,0.660,0.596
8,0.8,47,0.270,0.660,0.596
9,0.9,44,0.253,0.659,0.558
10,1.0,44,0.253,0.659,0.558
0,0.0,49,0.282,0.653,0.615
1,0.1,49,0.282,0.653,0.615
2,0.2,49,0.282,0.653,0.615
3,0.3,48,0.276,0.646,0.596
4,0.4,48,0.276,0.646,0.596


In [85]:
# Conservative risk + investigation policy
predicted_auto = (
    (model_df["risk_level"] == "LOW") &
    (model_df["investigation_rate"] == 0)
).astype(int)

precision = precision_score(
    y_complexity,
    predicted_auto,
    zero_division=0
)

recall = recall_score(
    y_complexity,
    predicted_auto,
    zero_division=0
)

coverage = predicted_auto.mean()

print("LOW-RISK + NO-INVESTIGATION POLICY")
print("-----------------------------------")
print("AUTO candidates:", predicted_auto.sum())
print("Coverage:", round(coverage, 3))
print("AUTO precision:", round(precision, 3))
print("AUTO recall:", round(recall, 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_complexity, predicted_auto))

print("\nClassification Report:")
print(
    classification_report(
        y_complexity,
        predicted_auto,
        target_names=["HUMAN", "AUTO"],
        zero_division=0
    )
)

LOW-RISK + NO-INVESTIGATION POLICY
-----------------------------------
AUTO candidates: 41
Coverage: 0.236
AUTO precision: 0.78
AUTO recall: 0.615

Confusion Matrix:
[[113   9]
 [ 20  32]]

Classification Report:
              precision    recall  f1-score   support

       HUMAN       0.85      0.93      0.89       122
        AUTO       0.78      0.62      0.69        52

    accuracy                           0.83       174
   macro avg       0.82      0.77      0.79       174
weighted avg       0.83      0.83      0.83       174



In [86]:
print("Risk × actual automation label")

print(
    pd.crosstab(
        model_df["risk_level"],
        model_df["automation_decision"]
    )
)

Risk × actual automation label
automation_decision  AUTO  HUMAN
risk_level                      
LOW                    51     55
MEDIUM                  1     67


In [87]:
print("Risk × predicted automation")

print(
    pd.crosstab(
        model_df["risk_level"],
        predicted_auto,
        rownames=["risk_level"],
        colnames=["predicted_auto"]
    )
)

Risk × predicted automation
predicted_auto   0   1
risk_level            
LOW             65  41
MEDIUM          68   0


In [89]:
# -----------------------------------------
# FALSE AUTO cases — clean inspection
# -----------------------------------------

policy_pred = (
    (model_df["risk_level"] == "LOW") &
    (model_df["investigation_rate"] == 0)
).astype(int)

false_auto_ids = model_df.loc[
    (policy_pred == 1) &
    (model_df["automation_decision"] == "HUMAN"),
    "golden_id"
]

print("FALSE AUTO cases:", len(false_auto_ids))

# Pull the complete information directly from golden_df
false_auto_view = golden_df[
    golden_df["golden_id"].isin(false_auto_ids)
].copy()

# Add the runtime features
false_auto_view = false_auto_view.merge(
    trust_features,
    on="golden_id",
    how="left",
    suffixes=("", "_feature")
)

# Display only useful diagnostic columns
display(
    false_auto_view[
        [
            "golden_id",
            "risk_level",
            "intent",
            "automation_decision",
            "resolution_status",
            "investigation_rate",
            "general_support_rate",
            "top1_ce",
            "weak_intent_coverage",
            "customer_query",
            "support_response",
            "annotation_notes"
        ]
    ].sort_values("golden_id")
)

FALSE AUTO cases: 9


,golden_id,risk_level,intent,automation_decision,resolution_status,investigation_rate,general_support_rate,top1_ce,weak_intent_coverage,customer_query,support_response,annotation_notes
0,GOLD-0013,LOW,ORDER_PURCHASE,HUMAN,INVESTIGATION_REQUIRED,0.0,1.0,0.954412,0.0,"@117086 Oi, boa noite. Ontem eu comprei um liv...",@531056 Olá Thaís! Recomendamos você contatar ...,NaN
1,GOLD-0027,LOW,ORDER_DELIVERY,HUMAN,INVESTIGATION_REQUIRED,0.0,1.0,0.993545,0.0,@AmazonHelp I'm confused... https://t.co/ikSK7...,@790379 We'd like to look into the delivery fo...,NaN
2,GOLD-0029,LOW,ORDER_DELIVERY,HUMAN,INVESTIGATION_REQUIRED,0.0,1.0,0.974980,1.0,@115821 / @AmazonHelp - Two day delivery is th...,@375664 (2/2) details here: https://t.co/hQdly...,NaN
3,GOLD-0036,LOW,ORDER_DELIVERY,HUMAN,INVESTIGATION_REQUIRED,0.0,1.0,0.994308,0.0,"@AmazonHelp Es el vendido por amazon México, l...",@143578 Comprendo tu molestia. Debido que no t...,NaN
4,GOLD-0054,LOW,ORDER_DELIVERY,HUMAN,UNRESOLVED,0.0,1.0,0.232696,0.0,"@130687 Sólo necesito que, si decía de 16:30 a...","@694834 Hola Israel. Por favor, mantennos info...",Customer complains about delivery arriving out...
5,GOLD-0096,LOW,GENERAL_SUPPORT,HUMAN,INVESTIGATION_REQUIRED,0.0,1.0,0.545221,0.2,やっぱりネット通販なんか嫌い。ほんまに嫌い。\niTunesストアで買お。もう無理。Amaz...,@721758 Amazonでのお買い物において、お困りごとがありましたか？ EK,Customer expresses strong frustration with usi...
6,GOLD-0130,LOW,GENERAL_SUPPORT,HUMAN,INVESTIGATION_REQUIRED,0.0,1.0,0.661529,0.0,"@AmazonHelp Iba a eso en un rato, que no he po...","@123483 Josu, puedes contactarlos mediante los...",Customer indicates they will contact another s...
7,GOLD-0157,LOW,ORDER_DELIVERY,HUMAN,INVESTIGATION_REQUIRED,0.0,1.0,0.929374,0.0,Amazonで明日色ワールドエンド注文したんだけどいつ届くかな、、？\n#明日色ワールドエン...,@548448 商品のお届けは、ご案内する「お届け予定日」までにお届けするよう努めております...,NaN
8,GOLD-0191,LOW,PAYMENT_BILLING,HUMAN,INVESTIGATION_REQUIRED,0.0,1.0,0.023130,0.0,@AmazonHelp pour l'achat d'une enceinte Ultima...,@147592 Je vous remercie de nous l'avoir signa...,NaN


In [91]:
# -----------------------------------------
# Add current-query fields to model_df
# -----------------------------------------

query_cols = [
    "golden_id",
    "customer_query",
    "automation_decision",
    "risk_level"
]

query_data = golden_df[query_cols].copy()

# Remove duplicates if any
model_df = model_df.drop(
    columns=[
        "customer_query",
        "automation_decision_y",
        "risk_level_y"
    ],
    errors="ignore"
)

model_df = model_df.merge(
    query_data,
    on="golden_id",
    how="left",
    suffixes=("", "_gold")
)

# If existing columns survived with _gold suffixes, normalize them
if "automation_decision_gold" in model_df.columns:
    model_df["automation_decision"] = model_df["automation_decision_gold"]
    model_df.drop(columns=["automation_decision_gold"], inplace=True)

if "risk_level_gold" in model_df.columns:
    model_df["risk_level"] = model_df["risk_level_gold"]
    model_df.drop(columns=["risk_level_gold"], inplace=True)

print("Shape:", model_df.shape)

display(
    model_df[
        [
            "golden_id",
            "customer_query",
            "risk_level",
            "automation_decision"
        ]
    ].head()
)

Shape: (174, 22)


,golden_id,customer_query,risk_level,automation_decision
0,GOLD-0001,Contact @115821 regarding payment on an order....,MEDIUM,HUMAN
1,GOLD-0002,@AmazonHelp LOL...you've missed the delivery d...,LOW,HUMAN
2,GOLD-0003,So bummed that my package was delayed even wit...,LOW,HUMAN
3,GOLD-0004,I love @115821 for my Christmas shopping gift ...,LOW,AUTO
4,GOLD-0005,@AmazonHelp Bleg on the phone 😕 what version s...,LOW,HUMAN


In [92]:
query_df = model_df[
    [
        "golden_id",
        "customer_query",
        "automation_decision",
        "risk_level"
    ]
].copy()

query_df["customer_query"] = (
    query_df["customer_query"]
    .fillna("")
    .astype(str)
)

query_df["query_chars"] = (
    query_df["customer_query"].str.len()
)

query_df["query_words"] = (
    query_df["customer_query"]
    .str.split()
    .str.len()
)

query_df["has_url"] = (
    query_df["customer_query"]
    .str.contains(
        r"http|www\.|t\.co",
        case=False,
        regex=True
    )
    .astype(int)
)

query_df["has_question"] = (
    query_df["customer_query"]
    .str.contains(r"\?", regex=True)
    .astype(int)
)

print("Query feature shape:", query_df.shape)

display(query_df.head())

Query feature shape: (174, 8)


,golden_id,customer_query,automation_decision,risk_level,query_chars,query_words,has_url,has_question
0,GOLD-0001,Contact @115821 regarding payment on an order....,HUMAN,MEDIUM,140,21,0,0
1,GOLD-0002,@AmazonHelp LOL...you've missed the delivery d...,HUMAN,LOW,137,22,0,0
2,GOLD-0003,So bummed that my package was delayed even wit...,HUMAN,LOW,114,21,0,0
3,GOLD-0004,I love @115821 for my Christmas shopping gift ...,AUTO,LOW,52,9,0,0
4,GOLD-0005,@AmazonHelp Bleg on the phone 😕 what version s...,HUMAN,LOW,66,12,0,1


In [93]:
query_features = [
    "query_chars",
    "query_words",
    "has_url",
    "has_question"
]

display(
    query_df
    .groupby("automation_decision")[query_features]
    .mean()
    .round(3)
)

,query_chars,query_words,has_url,has_question
automation_decision,,,,
AUTO,96.135,15.288,0.173,0.288
HUMAN,123.311,20.402,0.205,0.172


In [94]:
query_auc_results = []

y_query = (
    query_df["automation_decision"]
    .eq("AUTO")
    .astype(int)
)

for col in query_features:

    x = query_df[col].astype(float)

    if x.nunique() < 2:
        continue

    auc = roc_auc_score(y_query, x)

    query_auc_results.append({
        "feature": col,
        "raw_auc": round(auc, 3),
        "useful_auc": round(max(auc, 1 - auc), 3),
        "direction": (
            "HIGHER → AUTO"
            if auc >= 0.5
            else "LOWER → AUTO"
        )
    })

query_auc_df = (
    pd.DataFrame(query_auc_results)
    .sort_values("useful_auc", ascending=False)
)

display(query_auc_df)

,feature,raw_auc,useful_auc,direction
1,query_words,0.360,0.640,LOWER → AUTO
0,query_chars,0.367,0.633,LOWER → AUTO
3,has_question,0.558,0.558,HIGHER → AUTO
2,has_url,0.484,0.516,LOWER → AUTO


In [96]:
# Add query features to model_df
model_df = model_df.drop(
    columns=[
        "query_chars",
        "query_words",
        "has_url",
        "has_question"
    ],
    errors="ignore"
)

model_df = model_df.merge(
    query_df[
        [
            "golden_id",
            "query_chars",
            "query_words",
            "has_url",
            "has_question"
        ]
    ],
    on="golden_id",
    how="left"
)

print("Shape:", model_df.shape)

display(
    model_df[
        [
            "golden_id",
            "risk_level",
            "investigation_rate",
            "query_words",
            "query_chars"
        ]
    ].head()
)

Shape: (174, 26)


,golden_id,risk_level,investigation_rate,query_words,query_chars
0,GOLD-0001,MEDIUM,0.8,21,140
1,GOLD-0002,LOW,0.6,22,137
2,GOLD-0003,LOW,0.4,21,114
3,GOLD-0004,LOW,0.0,9,52
4,GOLD-0005,LOW,0.2,12,66


In [97]:
query_policy_results = []

for word_threshold in range(5, 31, 2):

    predicted_auto = (
        (model_df["risk_level"] == "LOW") &
        (model_df["investigation_rate"] == 0) &
        (model_df["query_words"] <= word_threshold)
    ).astype(int)

    auto_count = predicted_auto.sum()

    if auto_count == 0:
        continue

    precision = precision_score(
        y_complexity,
        predicted_auto,
        zero_division=0
    )

    recall = recall_score(
        y_complexity,
        predicted_auto,
        zero_division=0
    )

    coverage = auto_count / len(model_df)

    query_policy_results.append({
        "word_threshold": word_threshold,
        "auto_cases": int(auto_count),
        "auto_coverage": round(coverage, 3),
        "auto_precision": round(precision, 3),
        "auto_recall": round(recall, 3)
    })

query_policy_df = pd.DataFrame(query_policy_results)

display(query_policy_df)

,word_threshold,auto_cases,auto_coverage,auto_precision,auto_recall
0,5,11,0.063,0.727,0.154
1,7,14,0.080,0.786,0.212
2,9,15,0.086,0.800,0.231
3,11,16,0.092,0.812,0.250
4,13,17,0.098,0.765,0.250
5,15,22,0.126,0.727,0.308
6,17,26,0.149,0.769,0.385
7,19,29,0.167,0.793,0.442
8,21,33,0.190,0.818,0.519
9,23,36,0.207,0.806,0.558


In [98]:
display(
    query_policy_df.sort_values(
        ["auto_precision", "auto_coverage"],
        ascending=[False, False]
    )
)

,word_threshold,auto_cases,auto_coverage,auto_precision,auto_recall
8,21,33,0.190,0.818,0.519
3,11,16,0.092,0.812,0.250
10,25,37,0.213,0.811,0.577
9,23,36,0.207,0.806,0.558
2,9,15,0.086,0.800,0.231
7,19,29,0.167,0.793,0.442
1,7,14,0.080,0.786,0.212
11,27,40,0.230,0.775,0.596
12,29,40,0.230,0.775,0.596
6,17,26,0.149,0.769,0.385


In [99]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# ---------------------------------------------------------
# FINAL CANDIDATE AUTOMATION POLICY
# ---------------------------------------------------------

model_df["predicted_automation"] = (
    (model_df["risk_level"] == "LOW") &
    (model_df["investigation_rate"] == 0) &
    (model_df["query_words"] <= 25)
).map({True: "AUTO", False: "HUMAN"})


# ---------------------------------------------------------
# GOLD LABEL
# ---------------------------------------------------------

model_df["gold_automation"] = model_df["automation_decision"]


# ---------------------------------------------------------
# CONFUSION MATRIX
# ---------------------------------------------------------

cm = confusion_matrix(
    model_df["gold_automation"],
    model_df["predicted_automation"],
    labels=["AUTO", "HUMAN"]
)

print("Confusion Matrix")
print(pd.DataFrame(
    cm,
    index=["Gold AUTO", "Gold HUMAN"],
    columns=["Pred AUTO", "Pred HUMAN"]
))


# ---------------------------------------------------------
# CLASSIFICATION REPORT
# ---------------------------------------------------------

print("\nClassification Report\n")

print(
    classification_report(
        model_df["gold_automation"],
        model_df["predicted_automation"],
        labels=["AUTO", "HUMAN"],
        digits=3
    )
)


# ---------------------------------------------------------
# AUTOMATION METRICS
# ---------------------------------------------------------

auto_mask = model_df["predicted_automation"] == "AUTO"

gold_auto = model_df["gold_automation"] == "AUTO"

tp = (auto_mask & gold_auto).sum()
fp = (auto_mask & ~gold_auto).sum()
fn = (~auto_mask & gold_auto).sum()

auto_cases = auto_mask.sum()
total_cases = len(model_df)

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
coverage = auto_cases / total_cases

print("======================================")
print("FINAL AUTOMATION POLICY")
print("======================================")
print("Policy: LOW risk + investigation_rate == 0 + query_words <= 25")
print()
print(f"Total cases       : {total_cases}")
print(f"AUTO cases        : {auto_cases}")
print(f"HUMAN cases       : {total_cases - auto_cases}")
print(f"AUTO coverage     : {coverage:.3f}")
print(f"AUTO precision    : {precision:.3f}")
print(f"AUTO recall       : {recall:.3f}")
print(f"False AUTO        : {fp}")
print(f"False HUMAN       : {fn}")

Confusion Matrix
            Pred AUTO  Pred HUMAN
Gold AUTO          30          22
Gold HUMAN          7         115

Classification Report

              precision    recall  f1-score   support

        AUTO      0.811     0.577     0.674        52
       HUMAN      0.839     0.943     0.888       122

    accuracy                          0.833       174
   macro avg      0.825     0.760     0.781       174
weighted avg      0.831     0.833     0.824       174

FINAL AUTOMATION POLICY
Policy: LOW risk + investigation_rate == 0 + query_words <= 25

Total cases       : 174
AUTO cases        : 37
HUMAN cases       : 137
AUTO coverage     : 0.213
AUTO precision    : 0.811
AUTO recall       : 0.577
False AUTO        : 7
False HUMAN       : 22


In [101]:
false_auto = model_df[
    (model_df["predicted_automation"] == "AUTO") &
    (model_df["gold_automation"] == "HUMAN")
].copy()

print("False AUTO cases:", len(false_auto))

display(
    false_auto[
        [
            "golden_id",
            "risk_level",
            "query_words",
            "query_chars",
            "investigation_rate",
            "general_support_rate",
            "predicted_automation",
            "gold_automation"
        ]
    ].sort_values(
        ["investigation_rate", "query_words"],
        ascending=[True, True]
    )
)

False AUTO cases: 7


,golden_id,risk_level,query_words,query_chars,investigation_rate,general_support_rate,predicted_automation,gold_automation
88,GOLD-0096,LOW,2,71,0.0,1.0,AUTO,HUMAN
137,GOLD-0157,LOW,3,58,0.0,1.0,AUTO,HUMAN
24,GOLD-0027,LOW,4,51,0.0,1.0,AUTO,HUMAN
167,GOLD-0191,LOW,13,85,0.0,1.0,AUTO,HUMAN
33,GOLD-0036,LOW,15,84,0.0,1.0,AUTO,HUMAN
114,GOLD-0130,LOW,15,68,0.0,1.0,AUTO,HUMAN
11,GOLD-0013,LOW,23,128,0.0,1.0,AUTO,HUMAN


In [102]:
import pandas as pd

golden_path = (
    "/content/drive/MyDrive/"
    "hiver_support_agent_checkpoint/data/"
    "evaluation/golden_evaluation_set.csv"
)

golden_full = pd.read_csv(golden_path)

print("Golden columns:")
print(golden_full.columns.tolist())

Golden columns:
['golden_id', 'case_id', 'root_tweet_id', 'turn_count', 'customer_turns', 'support_turns', 'duration_hours', 'max_gap_hours', 'customer_query', 'support_response', 'provisional_bucket', 'risk_level', 'annotation_notes', 'intent', 'resolution_status', 'automation_decision']


In [103]:
false_auto_queries = false_auto[["golden_id"]].merge(
    golden_full[
        [
            "golden_id",
            "customer_query",
            "support_response",
            "intent",
            "resolution_status",
            "risk_level",
            "automation_decision"
        ]
    ],
    on="golden_id",
    how="left"
)

display(
    false_auto_queries[
        [
            "golden_id",
            "intent",
            "risk_level",
            "resolution_status",
            "customer_query",
            "support_response"
        ]
    ]
)

,golden_id,intent,risk_level,resolution_status,customer_query,support_response
0,GOLD-0013,ORDER_PURCHASE,LOW,INVESTIGATION_REQUIRED,"@117086 Oi, boa noite. Ontem eu comprei um liv...",@531056 Olá Thaís! Recomendamos você contatar ...
1,GOLD-0027,ORDER_DELIVERY,LOW,INVESTIGATION_REQUIRED,@AmazonHelp I'm confused... https://t.co/ikSK7...,@790379 We'd like to look into the delivery fo...
2,GOLD-0036,ORDER_DELIVERY,LOW,INVESTIGATION_REQUIRED,"@AmazonHelp Es el vendido por amazon México, l...",@143578 Comprendo tu molestia. Debido que no t...
3,GOLD-0096,GENERAL_SUPPORT,LOW,INVESTIGATION_REQUIRED,やっぱりネット通販なんか嫌い。ほんまに嫌い。\niTunesストアで買お。もう無理。Amaz...,@721758 Amazonでのお買い物において、お困りごとがありましたか？ EK
4,GOLD-0130,GENERAL_SUPPORT,LOW,INVESTIGATION_REQUIRED,"@AmazonHelp Iba a eso en un rato, que no he po...","@123483 Josu, puedes contactarlos mediante los..."
5,GOLD-0157,ORDER_DELIVERY,LOW,INVESTIGATION_REQUIRED,Amazonで明日色ワールドエンド注文したんだけどいつ届くかな、、？\n#明日色ワールドエン...,@548448 商品のお届けは、ご案内する「お届け予定日」までにお届けするよう努めております...
6,GOLD-0191,PAYMENT_BILLING,LOW,INVESTIGATION_REQUIRED,@AmazonHelp pour l'achat d'une enceinte Ultima...,@147592 Je vous remercie de nous l'avoir signa...


In [104]:
FINAL_AUTOMATION_POLICY = {
    "policy_name": "conservative_v1",
    "risk_gate": "LOW only",
    "investigation_rate_max": 0.0,
    "query_words_max": 25,
    "high_risk_action": "HUMAN",
    "medium_risk_action": "HUMAN",
    "default_action": "HUMAN",
}

In [105]:
import json
from pathlib import Path

checkpoint_dir = Path(
    "/content/drive/MyDrive/"
    "hiver_support_agent_checkpoint/data/checkpoints"
)

checkpoint_dir.mkdir(parents=True, exist_ok=True)

policy_path = checkpoint_dir / "final_automation_policy_v1.json"

with open(policy_path, "w") as f:
    json.dump(FINAL_AUTOMATION_POLICY, f, indent=2)

print(f"Saved: {policy_path}")

Saved: /content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/final_automation_policy_v1.json


In [106]:
evaluation_path = (
    "/content/drive/MyDrive/"
    "hiver_support_agent_checkpoint/data/evaluation/"
    "final_automation_policy_evaluation.csv"
)

final_eval = model_df[
    [
        "golden_id",
        "risk_level",
        "investigation_rate",
        "query_words",
        "query_chars",
        "predicted_automation",
        "gold_automation"
    ]
].copy()

final_eval.to_csv(evaluation_path, index=False)

print(f"Saved: {evaluation_path}")

Saved: /content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/final_automation_policy_evaluation.csv


In [107]:
summary = {
    "evaluation_set": "174 non-HIGH-risk golden cases",
    "policy": "LOW risk AND investigation_rate == 0 AND query_words <= 25",
    "total_cases": 174,
    "auto_cases": 37,
    "human_cases": 137,
    "auto_coverage": 0.213,
    "auto_precision": 0.811,
    "auto_recall": 0.577,
    "false_auto": 7,
    "false_human": 22,
    "overall_accuracy": 0.833,
    "human_recall": 0.943,
    "status": "FROZEN_V1"
}

summary_path = (
    "/content/drive/MyDrive/"
    "hiver_support_agent_checkpoint/data/evaluation/"
    "final_automation_policy_summary_v1.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {summary_path}")

Saved: /content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/final_automation_policy_summary_v1.json


In [108]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/"
    "hiver_support_agent_checkpoint/data"
)

# Historical RAG corpus
rag_corpus = pd.read_csv(
    BASE_DIR / "checkpoints" / "structured_rag_corpus.csv"
)

# Cross-encoder reranked retrieval results
reranked_df = pd.read_csv(
    BASE_DIR / "checkpoints" / "cross_encoder_reranked_results.csv"
)

# Frozen golden set
golden_df = pd.read_csv(
    BASE_DIR / "evaluation" / "golden_evaluation_set_frozen.csv"
)

print("RAG corpus:", rag_corpus.shape)
print("Reranked results:", reranked_df.shape)
print("Golden set:", golden_df.shape)

print("\nRAG corpus columns:")
print(rag_corpus.columns.tolist())

print("\nReranked columns:")
print(reranked_df.columns.tolist())

print("\nGolden columns:")
print(golden_df.columns.tolist())

RAG corpus: (83218, 6)
Reranked results: (10000, 11)
Golden set: (200, 16)

RAG corpus columns:
['case_id', 'customer_problem', 'support_action', 'resolution_type', 'evidence_quality', 'source']

Reranked columns:
['golden_id', 'case_id', 'ce_rank', 'ce_score', 'rrf_rank', 'rrf_score', 'bm25_rank', 'bge_rank', 'historical_weak_intent', 'golden_intent', 'intent_match']

Golden columns:
['golden_id', 'case_id', 'root_tweet_id', 'turn_count', 'customer_turns', 'support_turns', 'duration_hours', 'max_gap_hours', 'customer_query', 'support_response', 'provisional_bucket', 'risk_level', 'annotation_notes', 'intent', 'resolution_status', 'automation_decision']


In [109]:
def build_generation_context(
    golden_id,
    top_k=5
):
    """
    Build runtime-safe evidence context for response generation.

    Evaluation-only fields such as golden_intent and intent_match
    are intentionally excluded.
    """

    # Get the customer's current query
    query_row = golden_df[
        golden_df["golden_id"] == golden_id
    ]

    if query_row.empty:
        raise ValueError(f"Golden case not found: {golden_id}")

    query_row = query_row.iloc[0]

    customer_query = query_row["customer_query"]

    # Get top-k reranked historical cases
    retrieved = (
        reranked_df[
            reranked_df["golden_id"] == golden_id
        ]
        .sort_values("ce_rank")
        .head(top_k)
        .copy()
    )

    # Join with actual historical case content
    evidence = retrieved.merge(
        rag_corpus,
        on="case_id",
        how="left"
    )

    # Build clean runtime evidence
    evidence_items = []

    for _, row in evidence.iterrows():

        historical_intent = row.get(
            "historical_weak_intent",
            "UNKNOWN"
        )

        if pd.isna(historical_intent):
            historical_intent = "UNKNOWN"

        evidence_items.append({
            "case_id": row["case_id"],
            "rank": int(row["ce_rank"]),
            "cross_encoder_score": float(row["ce_score"]),
            "customer_problem": row["customer_problem"],
            "support_action": row["support_action"],
            "resolution_type": row["resolution_type"],
            "evidence_quality": row["evidence_quality"],
            "historical_intent": historical_intent
        })

    return {
        "golden_id": golden_id,
        "customer_query": customer_query,
        "evidence": evidence_items
    }

In [110]:
context = build_generation_context(
    golden_id="GOLD-0001",
    top_k=5
)

print("CUSTOMER QUERY:")
print(context["customer_query"])

print("\n" + "="*80)
print("HISTORICAL EVIDENCE")
print("="*80)

for item in context["evidence"]:

    print(f"\n--- Evidence {item['rank']} ---")

    print("Case ID:", item["case_id"])
    print("CE Score:", round(item["cross_encoder_score"], 4))
    print("Historical Intent:", item["historical_intent"])
    print("Resolution Type:", item["resolution_type"])

    print("\nCustomer Problem:")
    print(item["customer_problem"])

    print("\nSupport Action:")
    print(item["support_action"])

    print("\nEvidence Quality:")
    print(item["evidence_quality"])

CUSTOMER QUERY:
Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!

HISTORICAL EVIDENCE

--- Evidence 1 ---
Case ID: AMZ-073280
CE Score: 0.6153
Historical Intent: UNKNOWN
Resolution Type: INVESTIGATION_OR_FOLLOWUP

Customer Problem:
[user] will someone please contact mw urgently regarding an order i made?

Support Action:
[USER] Oh no! Sorry to hear this! Without providing any account specific information, will you tell us what's going on? We are here to help! ^AD

Evidence Quality:
HIGH

--- Evidence 2 ---
Case ID: AMZ-057467
CE Score: 0.2498
Historical Intent: PAYMENT_BILLING
Resolution Type: INVESTIGATION_OR_FOLLOWUP

Customer Problem:
[user] terrible customer service, charges me over and over again for cancelled subscription. awful!

Support Action:
[USER] I'm sorry for the poor experience. Without including any account details, can you tell us more about what's happened? ^ZW

Evidence Qualit

In [111]:
def build_response_prompt(context):
    """
    Build an evidence-grounded support response prompt.

    The model must use only the supplied historical evidence
    and must not invent customer-specific facts.
    """

    evidence_text = []

    for item in context["evidence"]:

        evidence_text.append(
            f"""
Evidence {item['rank']}
Case ID: {item['case_id']}
Relevance Score: {item['cross_encoder_score']:.4f}
Resolution Type: {item['resolution_type']}
Evidence Quality: {item['evidence_quality']}

Historical Customer Problem:
{item['customer_problem']}

Historical Support Action:
{item['support_action']}
"""
        )

    evidence_block = "\n".join(evidence_text)

    prompt = f"""
You are an evidence-grounded customer support assistant.

Your task is to draft a concise and professional support response
to the customer's current message.

IMPORTANT RULES:

1. Use the historical evidence only as guidance for the type of
   support action that may be appropriate.

2. NEVER invent customer-specific facts.

3. NEVER invent:
   - order status
   - delivery date
   - refund amount
   - payment status
   - account information
   - investigation results
   - policies
   - actions already taken

4. Do not claim that an issue has been resolved unless the supplied
   evidence explicitly supports that type of response.

5. If the evidence suggests that more information or investigation
   is required, ask the customer for the appropriate details or
   indicate that the issue needs further investigation.

6. Never expose internal case IDs, relevance scores, model scores,
   or retrieval information to the customer.

7. Never repeat personal information, account information, order
   identifiers, phone numbers, URLs, or other sensitive information
   from the evidence.

8. Keep the response concise, polite, and natural.

9. Do not simply copy a historical support response. Generate a
   new response appropriate to the current customer message.

CURRENT CUSTOMER MESSAGE:
{context["customer_query"]}

HISTORICAL SUPPORT EVIDENCE:
{evidence_block}

Generate ONLY the customer-facing response.
"""

    return prompt

In [112]:
response_prompt = build_response_prompt(context)

print(response_prompt)


You are an evidence-grounded customer support assistant.

Your task is to draft a concise and professional support response
to the customer's current message.

IMPORTANT RULES:

1. Use the historical evidence only as guidance for the type of
   support action that may be appropriate.

2. NEVER invent customer-specific facts.

3. NEVER invent:
   - order status
   - delivery date
   - refund amount
   - payment status
   - account information
   - investigation results
   - policies
   - actions already taken

4. Do not claim that an issue has been resolved unless the supplied
   evidence explicitly supports that type of response.

5. If the evidence suggests that more information or investigation
   is required, ask the customer for the appropriate details or
   indicate that the issue needs further investigation.

6. Never expose internal case IDs, relevance scores, model scores,
   or retrieval information to the customer.

7. Never repeat personal information, account information, 

In [113]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("Gemini API key loaded:", bool(GEMINI_API_KEY))

Gemini API key loaded: True


In [114]:
!pip -q install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [115]:
from google import genai

print("google-genai imported successfully")

google-genai imported successfully


In [116]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully")

Gemini client initialized successfully


In [118]:
test_interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Reply with exactly: Gemini connection successful."
)

print(test_interaction.output_text)

Gemini connection successful.


In [119]:
# Generate an evidence-grounded customer support response

generation_response = client.interactions.create(
    model="gemini-3.6-flash",
    input=response_prompt
)

generated_reply = generation_response.output_text

print("GENERATED CUSTOMER RESPONSE")
print("=" * 80)
print(generated_reply)

GENERATED CUSTOMER RESPONSE
I am very sorry for the frustrating experience you had when contacting customer service. We would like to help look into this for you. Without sharing any private account or payment details here, could you please provide a few more details about what is happening with your order's payment?


In [120]:
generation_checkpoint = {
    "golden_id": context["golden_id"],
    "customer_query": context["customer_query"],
    "generated_reply": generated_reply,
    "model": "gemini-3.6-flash"
}

print("Generation checkpoint ready")
print(generation_checkpoint)

Generation checkpoint ready
{'golden_id': 'GOLD-0001', 'customer_query': 'Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!', 'generated_reply': "I am very sorry for the frustrating experience you had when contacting customer service. We would like to help look into this for you. Without sharing any private account or payment details here, could you please provide a few more details about what is happening with your order's payment?", 'model': 'gemini-3.6-flash'}


In [121]:
import re

def normalize_text(text):
    """Normalize text for simple phrase matching."""
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def contains_any(text, patterns):
    """Return True if any pattern appears in text."""
    text = normalize_text(text)
    return any(pattern in text for pattern in patterns)

In [122]:
def response_grounding_check(
    response,
    customer_query,
    evidence_items
):
    """
    Rule-based Response Grounding Checker v1.

    Checks for:
    1. Internal information leakage
    2. Sensitive information leakage
    3. Unsupported resolution claims
    4. Unsupported customer-specific claims
    5. Basic evidence/action alignment

    This is a conservative safety checker, not a semantic proof.
    """

    response_norm = normalize_text(response)

    violations = []
    warnings = []

    # ---------------------------------------------------------
    # 1. INTERNAL INFORMATION LEAKAGE
    # ---------------------------------------------------------

    case_id_pattern = r"\bamz-\d+\b"

    if re.search(case_id_pattern, response_norm):
        violations.append("INTERNAL_CASE_ID_LEAK")

    internal_terms = [
        "cross encoder",
        "cross-encoder",
        "relevance score",
        "rrf score",
        "bm25",
        "bge score",
        "historical evidence",
        "retrieved case",
        "model score"
    ]

    if contains_any(response_norm, internal_terms):
        violations.append("INTERNAL_SYSTEM_INFORMATION")

    # ---------------------------------------------------------
    # 2. SENSITIVE INFORMATION
    # ---------------------------------------------------------

    sensitive_patterns = [
        r"\b\d{10}\b",          # possible phone number
        r"\b\d{12,16}\b",       # possible account/card-like number
        r"https?://\S+",        # URL
        r"@[A-Za-z0-9_]+",      # social/user handle
    ]

    for pattern in sensitive_patterns:
        if re.search(pattern, response):
            violations.append("POTENTIAL_SENSITIVE_INFORMATION")
            break

    # ---------------------------------------------------------
    # 3. UNSUPPORTED RESOLUTION CLAIMS
    # ---------------------------------------------------------

    resolution_claims = [
        "your order has been delivered",
        "your order was delivered",
        "your payment was successful",
        "your payment has been processed",
        "your refund has been issued",
        "your refund has been processed",
        "the issue has been resolved",
        "we have resolved",
        "we have fixed",
        "your order has been cancelled",
        "your order was cancelled",
        "we have investigated",
        "the investigation is complete"
    ]

    if contains_any(response_norm, resolution_claims):
        violations.append("UNSUPPORTED_RESOLUTION_CLAIM")

    # ---------------------------------------------------------
    # 4. UNSUPPORTED SPECIFIC FACTS
    # ---------------------------------------------------------

    unsupported_fact_patterns = [
        r"\byour order will arrive\b",
        r"\byour order should arrive\b",
        r"\byou will receive\b",
        r"\byou have been charged\b",
        r"\byou were charged\b",
        r"\byou will receive a refund\b"
    ]

    for pattern in unsupported_fact_patterns:
        if re.search(pattern, response_norm):
            violations.append("UNSUPPORTED_CUSTOMER_SPECIFIC_CLAIM")
            break

    # ---------------------------------------------------------
    # 5. EVIDENCE ACTION SIGNAL
    # ---------------------------------------------------------

    evidence_text = " ".join(
        [
            str(item.get("support_action", ""))
            for item in evidence_items
        ]
    )

    evidence_norm = normalize_text(evidence_text)

    investigation_signals = [
        "tell us",
        "more details",
        "get in touch",
        "contact us",
        "look into",
        "help",
        "provide"
    ]

    response_requests_information = contains_any(
        response_norm,
        [
            "could you",
            "can you",
            "please provide",
            "please share",
            "tell us",
            "more details",
            "let us know"
        ]
    )

    evidence_supports_investigation = contains_any(
        evidence_norm,
        investigation_signals
    )

    if (
        evidence_supports_investigation
        and not response_requests_information
    ):
        warnings.append("EVIDENCE_SUGGESTS_FOLLOWUP")

    # ---------------------------------------------------------
    # 6. FINAL DECISION
    # ---------------------------------------------------------

    if violations:
        decision = "UNSAFE"
    elif warnings:
        decision = "CONDITIONAL"
    else:
        decision = "GROUNDED"

    return {
        "decision": decision,
        "violations": sorted(set(violations)),
        "warnings": sorted(set(warnings)),
        "violation_count": len(set(violations)),
        "warning_count": len(set(warnings))
    }

In [123]:
grounding_result = response_grounding_check(
    response=generated_reply,
    customer_query=context["customer_query"],
    evidence_items=context["evidence"]
)

print("RESPONSE GROUNDING CHECK")
print("=" * 80)

for key, value in grounding_result.items():
    print(f"{key}: {value}")

RESPONSE GROUNDING CHECK
decision: GROUNDED
violations: []
warnings: []
violation_count: 0
warning_count: 0


In [124]:
# ============================================================
# RESPONSE GROUNDING CHECKER V1 — SAFETY TESTS
# ============================================================

test_cases = [
    {
        "name": "Safe response",
        "response": (
            "I'm sorry you've had a frustrating experience. "
            "Could you please provide a little more detail about "
            "the payment issue, without sharing any private information?"
        ),
        "expected": "GROUNDED"
    },

    {
        "name": "Invented payment status",
        "response": (
            "Your payment was successful and your order is being processed."
        ),
        "expected": "UNSAFE"
    },

    {
        "name": "Invented refund",
        "response": (
            "Your refund has been issued and you should receive it shortly."
        ),
        "expected": "UNSAFE"
    },

    {
        "name": "Invented investigation",
        "response": (
            "We have investigated the issue and resolved it for you."
        ),
        "expected": "UNSAFE"
    },

    {
        "name": "Internal case leakage",
        "response": (
            "We found a similar issue in case AMZ-073280."
        ),
        "expected": "UNSAFE"
    },

    {
        "name": "Internal retrieval leakage",
        "response": (
            "The cross encoder gave this case a relevance score of 0.95."
        ),
        "expected": "UNSAFE"
    },

    {
        "name": "Sensitive URL leakage",
        "response": (
            "Please check your account here: https://example.com/account"
        ),
        "expected": "UNSAFE"
    }
]


# ------------------------------------------------------------
# RUN TESTS
# ------------------------------------------------------------

test_results = []

for test in test_cases:

    result = response_grounding_check(
        response=test["response"],
        customer_query=context["customer_query"],
        evidence_items=context["evidence"]
    )

    passed = result["decision"] == test["expected"]

    test_results.append({
        "test": test["name"],
        "expected": test["expected"],
        "actual": result["decision"],
        "passed": passed,
        "violations": ", ".join(result["violations"]),
        "warnings": ", ".join(result["warnings"])
    })


test_results_df = pd.DataFrame(test_results)

display(test_results_df)

print(
    f"\nPassed: "
    f"{test_results_df['passed'].sum()}/{len(test_results_df)}"
)

,test,expected,actual,passed,violations,warnings
0,Safe response,GROUNDED,GROUNDED,True,,
1,Invented payment status,UNSAFE,UNSAFE,True,UNSUPPORTED_RESOLUTION_CLAIM,EVIDENCE_SUGGESTS_FOLLOWUP
2,Invented refund,UNSAFE,UNSAFE,True,UNSUPPORTED_RESOLUTION_CLAIM,EVIDENCE_SUGGESTS_FOLLOWUP
3,Invented investigation,UNSAFE,UNSAFE,True,UNSUPPORTED_RESOLUTION_CLAIM,EVIDENCE_SUGGESTS_FOLLOWUP
4,Internal case leakage,UNSAFE,UNSAFE,True,INTERNAL_CASE_ID_LEAK,EVIDENCE_SUGGESTS_FOLLOWUP
5,Internal retrieval leakage,UNSAFE,UNSAFE,True,INTERNAL_SYSTEM_INFORMATION,EVIDENCE_SUGGESTS_FOLLOWUP
6,Sensitive URL leakage,UNSAFE,UNSAFE,True,POTENTIAL_SENSITIVE_INFORMATION,EVIDENCE_SUGGESTS_FOLLOWUP



Passed: 7/7


In [125]:
def final_agent_decision(
    risk_level,
    evidence_decision,
    grounding_decision,
    investigation_rate,
    query_words
):
    """
    Final conservative decision layer.

    AUTO requires every safety gate to pass.
    Otherwise the case goes to HUMAN.
    """

    # ---------------------------------------------------------
    # HARD SAFETY GATE
    # ---------------------------------------------------------

    if risk_level == "HIGH":
        return {
            "automation_decision": "HUMAN",
            "reason": "HIGH risk case"
        }

    # ---------------------------------------------------------
    # MEDIUM RISK
    # ---------------------------------------------------------

    if risk_level == "MEDIUM":
        return {
            "automation_decision": "HUMAN",
            "reason": "MEDIUM risk case requires human review"
        }

    # ---------------------------------------------------------
    # EVIDENCE TRUST
    # ---------------------------------------------------------

    if evidence_decision != "TRUSTED":
        return {
            "automation_decision": "HUMAN",
            "reason": "Historical evidence is not sufficiently trusted"
        }

    # ---------------------------------------------------------
    # RESPONSE GROUNDING
    # ---------------------------------------------------------

    if grounding_decision != "GROUNDED":
        return {
            "automation_decision": "HUMAN",
            "reason": "Generated response failed grounding checks"
        }

    # ---------------------------------------------------------
    # HISTORICAL INVESTIGATION SIGNAL
    # ---------------------------------------------------------

    if investigation_rate > 0:
        return {
            "automation_decision": "HUMAN",
            "reason": "Retrieved historical cases indicate investigation/follow-up"
        }

    # ---------------------------------------------------------
    # CURRENT QUERY COMPLEXITY
    # ---------------------------------------------------------

    if query_words > 25:
        return {
            "automation_decision": "HUMAN",
            "reason": "Current query exceeds conservative complexity threshold"
        }

    # ---------------------------------------------------------
    # ALL GATES PASSED
    # ---------------------------------------------------------

    return {
        "automation_decision": "AUTO",
        "reason": (
            "LOW risk + trusted evidence + grounded response + "
            "no historical investigation tendency + concise query"
        )
    }

In [126]:
final_decision = final_agent_decision(
    risk_level="LOW",
    evidence_decision="TRUSTED",
    grounding_decision=grounding_result["decision"],
    investigation_rate=0.0,
    query_words=len(context["customer_query"].split())
)

print("FINAL AGENT DECISION")
print("=" * 80)
print(final_decision)

FINAL AGENT DECISION
{'automation_decision': 'AUTO', 'reason': 'LOW risk + trusted evidence + grounded response + no historical investigation tendency + concise query'}


In [127]:
print(trust_check_v1)

<function trust_check_v1 at 0x7c2bc7c7a2a0>


In [128]:
# ============================================================
# REAL EVIDENCE TRUST CHECK FOR CURRENT CASE
# ============================================================

# Get the top-5 reranked evidence for this case
current_evidence_df = (
    reranked_df[
        reranked_df["golden_id"] == context["golden_id"]
    ]
    .sort_values("ce_rank")
    .head(5)
    .copy()
)

print("Current evidence shape:", current_evidence_df.shape)
print("\nColumns:")
print(current_evidence_df.columns.tolist())


# Get the predicted intent from the golden case
# NOTE: This is being used ONLY for this controlled evaluation
# example. Later runtime inference will use the intent classifier.
current_case = golden_df[
    golden_df["golden_id"] == context["golden_id"]
].iloc[0]

predicted_intent_for_test = current_case["intent"]
risk_level_for_test = current_case["risk_level"]

print("\nPredicted intent for test:", predicted_intent_for_test)
print("Risk level:", risk_level_for_test)


# Run the REAL Trust Checker
trust_result = trust_check_v1(
    evidence_df=current_evidence_df,
    predicted_intent=predicted_intent_for_test,
    risk_level=risk_level_for_test
)

print("\n" + "=" * 80)
print("REAL EVIDENCE TRUST RESULT")
print("=" * 80)

print(trust_result)

Current evidence shape: (5, 11)

Columns:
['golden_id', 'case_id', 'ce_rank', 'ce_score', 'rrf_rank', 'rrf_score', 'bm25_rank', 'bge_rank', 'historical_weak_intent', 'golden_intent', 'intent_match']

Predicted intent for test: PAYMENT_BILLING
Risk level: MEDIUM

REAL EVIDENCE TRUST RESULT
{'trust_score': 0.4895, 'top1_ce_score': 0.6153, 'evidence_tier': 'WEAK', 'intent_consistency': 0.5, 'semantic_consistency': 0.1019, 'evidence_decision': 'WEAK', 'automation_decision': 'HUMAN', 'reason': 'Historical evidence does not meet the autonomous-response trust criteria.'}


In [129]:
def calculate_intent_consistency_v2(
    evidence_df,
    predicted_intent
):
    """
    Calculate intent consistency using only historical evidence
    that has a known high-confidence intent label.

    UNKNOWN / missing intents are excluded, not treated as mismatches.
    """

    if evidence_df.empty:
        return 0.0

    known_intents = evidence_df[
        evidence_df["historical_weak_intent"].notna() &
        (evidence_df["historical_weak_intent"] != "UNKNOWN")
    ].copy()

    if known_intents.empty:
        # No usable intent metadata.
        # Do NOT penalize evidence simply because labels are missing.
        return None

    matches = (
        known_intents["historical_weak_intent"]
        == predicted_intent
    )

    return float(matches.mean())

In [130]:
def trust_check_v2(
    evidence_df,
    predicted_intent,
    risk_level
):
    """
    Evidence Trust Checker v2.

    Improvements over v1:
    - UNKNOWN historical intents are not treated as mismatches.
    - Evidence trust remains separate from automation eligibility.
    - HIGH risk remains a hard HUMAN gate.
    """

    if evidence_df.empty:
        return {
            "trust_score": 0.0,
            "top1_ce_score": 0.0,
            "evidence_tier": "WEAK",
            "intent_consistency": None,
            "semantic_consistency": 0.0,
            "evidence_decision": "WEAK",
            "automation_decision": "HUMAN",
            "reason": "No historical evidence retrieved."
        }

    evidence_df = evidence_df.sort_values("ce_rank").head(5)

    top1_ce = float(evidence_df.iloc[0]["ce_score"])

    evidence_tier = get_evidence_tier(top1_ce)

    intent_consistency = calculate_intent_consistency_v2(
        evidence_df,
        predicted_intent
    )

    semantic_consistency = calculate_semantic_consistency(
        evidence_df
    )

    # ---------------------------------------------------------
    # Intent signal
    # ---------------------------------------------------------
    # If there is no known historical intent information,
    # don't penalize the evidence.

    if intent_consistency is None:
        intent_signal = 0.5
    else:
        intent_signal = intent_consistency

    # ---------------------------------------------------------
    # Trust score
    # ---------------------------------------------------------

    trust_score = (
        0.60 * top1_ce
        + 0.20 * intent_signal
        + 0.20 * semantic_consistency
    )

    # ---------------------------------------------------------
    # Evidence decision
    # ---------------------------------------------------------

    if trust_score >= STRONG_TRUST_THRESHOLD:
        evidence_decision = "TRUSTED"

    elif trust_score >= CONDITIONAL_TRUST_THRESHOLD:
        evidence_decision = "CONDITIONAL"

    else:
        evidence_decision = "WEAK"

    # ---------------------------------------------------------
    # Automation gate
    # ---------------------------------------------------------

    if risk_level == "HIGH":
        automation_decision = "HUMAN"
        reason = "HIGH risk case requires human review."

    elif evidence_decision != "TRUSTED":
        automation_decision = "HUMAN"
        reason = (
            "Historical evidence does not meet the "
            "autonomous-response trust criteria."
        )

    else:
        automation_decision = "AUTO"
        reason = (
            "Historical evidence meets the autonomous-response "
            "trust criteria."
        )

    return {
        "trust_score": round(float(trust_score), 4),
        "top1_ce_score": round(top1_ce, 4),
        "evidence_tier": evidence_tier,
        "intent_consistency": (
            None
            if intent_consistency is None
            else round(float(intent_consistency), 4)
        ),
        "semantic_consistency": round(
            float(semantic_consistency), 4
        ),
        "evidence_decision": evidence_decision,
        "automation_decision": automation_decision,
        "reason": reason
    }

In [131]:
trust_result_v2 = trust_check_v2(
    evidence_df=current_evidence_df,
    predicted_intent=predicted_intent_for_test,
    risk_level=risk_level_for_test
)

print("EVIDENCE TRUST CHECKER V2")
print("=" * 80)

for key, value in trust_result_v2.items():
    print(f"{key}: {value}")

EVIDENCE TRUST CHECKER V2
trust_score: 0.4895
top1_ce_score: 0.6153
evidence_tier: WEAK
intent_consistency: 0.5
semantic_consistency: 0.1019
evidence_decision: WEAK
automation_decision: HUMAN
reason: Historical evidence does not meet the autonomous-response trust criteria.


In [132]:
# ============================================================
# GENERATE RESPONSES FOR 10 GOLDEN CASES
# ============================================================

import time

sample_golden_ids = golden_df["golden_id"].head(10).tolist()

generation_results = []

for i, golden_id in enumerate(sample_golden_ids, 1):

    print(f"Generating {i}/10: {golden_id}")

    # Build evidence context
    case_context = build_generation_context(
        golden_id=golden_id,
        top_k=5
    )

    # Build grounded prompt
    case_prompt = build_response_prompt(case_context)

    # Gemini generation
    response = client.interactions.create(
        model="gemini-3.6-flash",
        input=case_prompt
    )

    generated_text = response.output_text

    generation_results.append({
        "golden_id": golden_id,
        "customer_query": case_context["customer_query"],
        "generated_response": generated_text
    })

    # Small pause to avoid unnecessarily rapid requests
    time.sleep(1)

generation_batch_df = pd.DataFrame(generation_results)

print("\nGeneration complete.")
print("Shape:", generation_batch_df.shape)

display(generation_batch_df)

Generating 1/10: GOLD-0001
Generating 2/10: GOLD-0002
Generating 3/10: GOLD-0003
Generating 4/10: GOLD-0004
Generating 5/10: GOLD-0005
Generating 6/10: GOLD-0006
Generating 7/10: GOLD-0007
Generating 8/10: GOLD-0008


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 56.914088577s.', 'code': 'too_many_requests'}}

In [133]:
# Save successful generation results before doing anything else

generation_batch_df = pd.DataFrame(generation_results)

print("Successfully generated:", len(generation_batch_df))
display(generation_batch_df)

Successfully generated: 7


,golden_id,customer_query,generated_response
0,GOLD-0001,Contact @115821 regarding payment on an order....,I am very sorry to hear about your frustrating...
1,GOLD-0002,@AmazonHelp LOL...you've missed the delivery d...,We're very sorry to hear about these recurring...
2,GOLD-0003,So bummed that my package was delayed even wit...,I'm so sorry to hear about the delay with your...
3,GOLD-0004,I love @115821 for my Christmas shopping gift ...,Thank you so much for the kind words! We're th...
4,GOLD-0005,@AmazonHelp Bleg on the phone 😕 what version s...,Hello! To help us get you the right informatio...
5,GOLD-0006,@115850 amazon stick from https://t.co/cvJQ06p...,Hello! We would be happy to help look into thi...
6,GOLD-0007,@117795 Unethical delivery setup. Worst cust c...,We sincerely apologize for your unpleasant exp...


In [134]:
generation_checkpoint_path = (
    BASE_DIR / "evaluation" / "response_generation_batch_01.csv"
)

generation_batch_df.to_csv(
    generation_checkpoint_path,
    index=False
)

print(f"Saved checkpoint: {generation_checkpoint_path}")

Saved checkpoint: /content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/response_generation_batch_01.csv


In [135]:
for _, row in generation_batch_df.iterrows():

    print("=" * 100)
    print("GOLDEN ID:", row["golden_id"])

    print("\nCUSTOMER QUERY:")
    print(row["customer_query"])

    print("\nGENERATED RESPONSE:")
    print(row["generated_response"])

GOLDEN ID: GOLD-0001

CUSTOMER QUERY:
Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!

GENERATED RESPONSE:
I am very sorry to hear about your frustrating experience with customer service. We would like to help you with your order's payment issue. Without sharing any sensitive account or personal details, could you please tell us a bit more about what is happening?
GOLDEN ID: GOLD-0002

CUSTOMER QUERY:
@AmazonHelp LOL...you've missed the delivery date of every order of mine  shipped via usps, for the last 5 months including todays order.

GENERATED RESPONSE:
We're very sorry to hear about these recurring delivery delays with your orders. We would like to look into this for you. Could you let us know what the current tracking status shows for your package, or if you received an updated delivery date?
GOLDEN ID: GOLD-0003

CUSTOMER QUERY:
So bummed that my package was delayed even with Amazon Prim

In [136]:
# ============================================================
# GROUNDING EVALUATION — 7 GENERATED RESPONSES
# ============================================================

grounding_results = []

for _, row in generation_batch_df.iterrows():

    golden_id = row["golden_id"]

    # Rebuild the evidence context for this case
    case_context = build_generation_context(
        golden_id=golden_id,
        top_k=5
    )

    # Run local grounding checker
    result = response_grounding_check(
        response=row["generated_response"],
        customer_query=row["customer_query"],
        evidence_items=case_context["evidence"]
    )

    grounding_results.append({
        "golden_id": golden_id,
        "generated_response": row["generated_response"],
        "grounding_decision": result["decision"],
        "violation_count": result["violation_count"],
        "warning_count": result["warning_count"],
        "violations": ", ".join(result["violations"]),
        "warnings": ", ".join(result["warnings"])
    })


grounding_batch_df = pd.DataFrame(grounding_results)

display(
    grounding_batch_df[
        [
            "golden_id",
            "grounding_decision",
            "violation_count",
            "warning_count",
            "violations",
            "warnings"
        ]
    ]
)

print("\nGrounding results:")
print(
    grounding_batch_df["grounding_decision"]
    .value_counts()
)

,golden_id,grounding_decision,violation_count,warning_count,violations,warnings
0,GOLD-0001,GROUNDED,0,0,,
1,GOLD-0002,GROUNDED,0,0,,
2,GOLD-0003,GROUNDED,0,0,,
3,GOLD-0004,GROUNDED,0,0,,
4,GOLD-0005,GROUNDED,0,0,,
5,GOLD-0006,CONDITIONAL,0,1,,EVIDENCE_SUGGESTS_FOLLOWUP
6,GOLD-0007,CONDITIONAL,0,1,,EVIDENCE_SUGGESTS_FOLLOWUP



Grounding results:
grounding_decision
GROUNDED       5
CONDITIONAL    2
Name: count, dtype: int64


In [137]:
grounding_checkpoint_path = (
    BASE_DIR / "evaluation" / "response_grounding_batch_01.csv"
)

grounding_batch_df.to_csv(
    grounding_checkpoint_path,
    index=False
)

print("Saved:", grounding_checkpoint_path)

Saved: /content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/response_grounding_batch_01.csv


In [138]:
# Inspect the 2 CONDITIONAL cases

conditional_df = grounding_batch_df[
    grounding_batch_df["grounding_decision"] == "CONDITIONAL"
]

display(
    conditional_df[
        [
            "golden_id",
            "generated_response",
            "violations",
            "warnings"
        ]
    ]
)

,golden_id,generated_response,violations,warnings
5,GOLD-0006,Hello! We would be happy to help look into thi...,,EVIDENCE_SUGGESTS_FOLLOWUP
6,GOLD-0007,We sincerely apologize for your unpleasant exp...,,EVIDENCE_SUGGESTS_FOLLOWUP


In [139]:
print("========== CONDITIONAL CASES ==========")

for _, row in conditional_df.iterrows():
    print("\n" + "=" * 100)
    print("GOLDEN ID:", row["golden_id"])
    print("RESPONSE:", row["generated_response"])
    print("VIOLATIONS:", row["violations"])
    print("WARNINGS:", row["warnings"])

========== CONDITIONAL CASES ==========

GOLDEN ID: GOLD-0006
RESPONSE: Hello! We would be happy to help look into this for you. Please reach out to us directly through our official Customer Support channels so we can check the seller details and assist you further.
VIOLATIONS: 
WARNINGS: EVIDENCE_SUGGESTS_FOLLOWUP

GOLDEN ID: GOLD-0007
RESPONSE: We sincerely apologize for your unpleasant experience. For your privacy and security, please avoid sharing order details publicly. Please send us a direct message so we can look into this issue and assist you further.
VIOLATIONS: 
WARNINGS: EVIDENCE_SUGGESTS_FOLLOWUP


In [140]:
import inspect

print(inspect.getsource(response_grounding_check))

def response_grounding_check(
    response,
    customer_query,
    evidence_items
):
    """
    Rule-based Response Grounding Checker v1.

    Checks for:
    1. Internal information leakage
    2. Sensitive information leakage
    3. Unsupported resolution claims
    4. Unsupported customer-specific claims
    5. Basic evidence/action alignment

    This is a conservative safety checker, not a semantic proof.
    """

    response_norm = normalize_text(response)

    violations = []
    warnings = []

    # ---------------------------------------------------------
    # 1. INTERNAL INFORMATION LEAKAGE
    # ---------------------------------------------------------

    case_id_pattern = r"\bamz-\d+\b"
    
    if re.search(case_id_pattern, response_norm):
        violations.append("INTERNAL_CASE_ID_LEAK")

    internal_terms = [
        "cross encoder",
        "cross-encoder",
        "relevance score",
        "rrf score",
        "bm25",
        "bge score",
        "historical e

In [141]:
def response_grounding_check(
    response,
    customer_query,
    evidence_items
):
    """
    Rule-based Response Grounding Checker v2.

    Checks for:
    1. Internal information leakage
    2. Sensitive information leakage
    3. Unsupported resolution claims
    4. Unsupported customer-specific claims
    5. Evidence/action alignment

    v2 improvements:
    - Expanded follow-up detection.
    - Recognizes direct-message / contact / reach-out language.
    - Keeps safety violations unchanged.
    - Conservative safety behavior is preserved.

    This is a conservative safety checker, not a semantic proof.
    """

    response_norm = normalize_text(response)

    violations = []
    warnings = []

    # =========================================================
    # 1. INTERNAL INFORMATION LEAKAGE
    # =========================================================

    case_id_pattern = r"\bamz-\d+\b"

    if re.search(case_id_pattern, response_norm):
        violations.append("INTERNAL_CASE_ID_LEAK")

    internal_terms = [
        "cross encoder",
        "cross-encoder",
        "relevance score",
        "rrf score",
        "bm25",
        "bge score",
        "historical evidence",
        "retrieved case",
        "model score"
    ]

    if contains_any(response_norm, internal_terms):
        violations.append("INTERNAL_SYSTEM_INFORMATION")

    # =========================================================
    # 2. SENSITIVE INFORMATION
    # =========================================================

    sensitive_patterns = [
        r"\b\d{10}\b",          # possible phone number
        r"\b\d{12,16}\b",       # possible account/card-like number
        r"https?://\S+",        # URL
        r"@[A-Za-z0-9_]+",      # social/user handle
    ]

    for pattern in sensitive_patterns:
        if re.search(pattern, response):
            violations.append("POTENTIAL_SENSITIVE_INFORMATION")
            break

    # =========================================================
    # 3. UNSUPPORTED RESOLUTION CLAIMS
    # =========================================================

    resolution_claims = [
        "your order has been delivered",
        "your order was delivered",
        "your payment was successful",
        "your payment has been processed",
        "your refund has been issued",
        "your refund has been processed",
        "the issue has been resolved",
        "we have resolved",
        "we have fixed",
        "your order has been cancelled",
        "your order was cancelled",
        "we have investigated",
        "the investigation is complete"
    ]

    if contains_any(response_norm, resolution_claims):
        violations.append("UNSUPPORTED_RESOLUTION_CLAIM")

    # =========================================================
    # 4. UNSUPPORTED CUSTOMER-SPECIFIC FACTS
    # =========================================================

    unsupported_fact_patterns = [
        r"\byour order will arrive\b",
        r"\byour order should arrive\b",
        r"\byou will receive\b",
        r"\byou have been charged\b",
        r"\byou were charged\b",
        r"\byou will receive a refund\b"
    ]

    for pattern in unsupported_fact_patterns:
        if re.search(pattern, response_norm):
            violations.append(
                "UNSUPPORTED_CUSTOMER_SPECIFIC_CLAIM"
            )
            break

    # =========================================================
    # 5. EVIDENCE ACTION SIGNAL
    # =========================================================

    evidence_text = " ".join(
        [
            str(item.get("support_action", ""))
            for item in evidence_items
        ]
    )

    evidence_norm = normalize_text(evidence_text)

    investigation_signals = [
        "tell us",
        "more details",
        "get in touch",
        "contact us",
        "look into",
        "help",
        "provide"
    ]

    # ---------------------------------------------------------
    # Expanded response follow-up detection
    # ---------------------------------------------------------

    response_requests_information = contains_any(
        response_norm,
        [
            "could you",
            "can you",
            "please provide",
            "please share",
            "tell us",
            "more details",
            "let us know",
            "reach out",
            "contact us",
            "get in touch",
            "send us a direct message",
            "send us a message",
            "message us",
            "dm us",
            "please contact",
            "please reach out",
            "please get in touch"
        ]
    )

    evidence_supports_investigation = contains_any(
        evidence_norm,
        investigation_signals
    )

    if (
        evidence_supports_investigation
        and not response_requests_information
    ):
        warnings.append(
            "EVIDENCE_SUGGESTS_FOLLOWUP"
        )

    # =========================================================
    # 6. FINAL DECISION
    # =========================================================

    if violations:
        decision = "UNSAFE"

    elif warnings:
        decision = "CONDITIONAL"

    else:
        decision = "GROUNDED"

    # =========================================================
    # 7. RETURN STRUCTURED RESULT
    # =========================================================

    return {
        "decision": decision,
        "violations": sorted(set(violations)),
        "warnings": sorted(set(warnings)),
        "violation_count": len(set(violations)),
        "warning_count": len(set(warnings))
    }

In [142]:
# ============================================================
# RESPONSE GROUNDING EVALUATION — BATCH 01
# ============================================================

grounding_results = []

for _, row in generation_batch_df.iterrows():

    golden_id = row["golden_id"]

    # Build runtime-safe evidence context
    case_context = build_generation_context(
        golden_id=golden_id,
        top_k=5
    )

    # Run grounding checker
    result = response_grounding_check(
        response=row["generated_response"],
        customer_query=row["customer_query"],
        evidence_items=case_context["evidence"]
    )

    grounding_results.append({
        "golden_id": golden_id,
        "customer_query": row["customer_query"],
        "generated_response": row["generated_response"],
        "grounding_decision": result["decision"],
        "violation_count": result["violation_count"],
        "warning_count": result["warning_count"],
        "violations": ", ".join(result["violations"]),
        "warnings": ", ".join(result["warnings"])
    })


# Create evaluation DataFrame
grounding_batch_df = pd.DataFrame(
    grounding_results
)


# ============================================================
# DISPLAY CASE-LEVEL RESULTS
# ============================================================

display(
    grounding_batch_df[
        [
            "golden_id",
            "grounding_decision",
            "violation_count",
            "warning_count",
            "violations",
            "warnings"
        ]
    ]
)


# ============================================================
# SUMMARY
# ============================================================

grounding_summary = (
    grounding_batch_df["grounding_decision"]
    .value_counts()
)

print("\n" + "=" * 70)
print("GROUNDING SUMMARY")
print("=" * 70)

print(grounding_summary)

print("\nTotal responses:", len(grounding_batch_df))
print(
    "GROUNDED:",
    int(
        (grounding_batch_df["grounding_decision"] == "GROUNDED").sum()
    )
)
print(
    "CONDITIONAL:",
    int(
        (grounding_batch_df["grounding_decision"] == "CONDITIONAL").sum()
    )
)
print(
    "UNSAFE:",
    int(
        (grounding_batch_df["grounding_decision"] == "UNSAFE").sum()
    )
)


# ============================================================
# SAFETY CHECK
# ============================================================

unsafe_count = (
    grounding_batch_df["grounding_decision"] == "UNSAFE"
).sum()

print("\nUnsafe responses:", unsafe_count)

if unsafe_count == 0:
    print("✅ No unsafe responses detected.")
else:
    print("⚠️ Unsafe responses detected — inspect them before proceeding.")

,golden_id,grounding_decision,violation_count,warning_count,violations,warnings
0,GOLD-0001,GROUNDED,0,0,,
1,GOLD-0002,GROUNDED,0,0,,
2,GOLD-0003,GROUNDED,0,0,,
3,GOLD-0004,GROUNDED,0,0,,
4,GOLD-0005,GROUNDED,0,0,,
5,GOLD-0006,GROUNDED,0,0,,
6,GOLD-0007,GROUNDED,0,0,,



GROUNDING SUMMARY
grounding_decision
GROUNDED    7
Name: count, dtype: int64

Total responses: 7
GROUNDED: 7
CONDITIONAL: 0
UNSAFE: 0

Unsafe responses: 0
✅ No unsafe responses detected.


In [143]:
# ============================================================
# SAVE GROUNDING EVALUATION CHECKPOINT
# ============================================================

grounding_checkpoint_path = (
    BASE_DIR
    / "evaluation"
    / "response_grounding_batch_01_v2.csv"
)

grounding_batch_df.to_csv(
    grounding_checkpoint_path,
    index=False
)

print("Saved grounding evaluation:")
print(grounding_checkpoint_path)

Saved grounding evaluation:
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/response_grounding_batch_01_v2.csv


In [2]:
# ============================================================
# RESTORE HIVER SUPPORT AGENT ENVIRONMENT
# ============================================================

from google.colab import drive
from pathlib import Path
import json
import pandas as pd
import numpy as np
import re
import os

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# Restore project root
# ------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

# ------------------------------------------------------------
# Verify project structure
# ------------------------------------------------------------

print("=" * 70)
print("HIVER SUPPORT AGENT — ENVIRONMENT RESTORE")
print("=" * 70)

print("\nBASE_DIR:")
print(BASE_DIR)

print("\nBASE_DIR exists:", BASE_DIR.exists())

if BASE_DIR.exists():

    print("\nProject directories:")

    for directory in [
        "checkpoints",
        "evaluation",
        "embeddings"
    ]:
        path = BASE_DIR / directory

        print(
            f"  {directory:15s} → "
            f"{'EXISTS' if path.exists() else 'MISSING'}"
        )

# ------------------------------------------------------------
# Verify important checkpoints
# ------------------------------------------------------------

important_files = [
    BASE_DIR / "checkpoints" / "final_automation_policy_v1.json",
    BASE_DIR / "checkpoints" / "retrieval_checkpoint.json",
    BASE_DIR / "checkpoints" / "cross_encoder_reranked_results.csv",
    BASE_DIR / "checkpoints" / "rrf_candidates.csv",
    BASE_DIR / "checkpoints" / "structured_rag_corpus.csv",

    BASE_DIR / "evaluation" / "golden_evaluation_set_frozen.csv",
    BASE_DIR / "evaluation" / "response_generation_batch_01.csv",
    BASE_DIR / "evaluation" / "response_grounding_batch_01_v2.csv",

    BASE_DIR / "embeddings" / "historical_bge_large.npy"
]

print("\n" + "=" * 70)
print("CHECKPOINT VERIFICATION")
print("=" * 70)

for path in important_files:

    status = "✓ FOUND" if path.exists() else "✗ MISSING"

    print(f"{status}: {path.name}")

print("\n✅ Environment paths restored.")

Mounted at /content/drive
HIVER SUPPORT AGENT — ENVIRONMENT RESTORE

BASE_DIR:
/content/drive/MyDrive/hiver_support_agent_checkpoint/data

BASE_DIR exists: True

Project directories:
  checkpoints     → EXISTS
  evaluation      → EXISTS
  embeddings      → EXISTS

CHECKPOINT VERIFICATION
✓ FOUND: final_automation_policy_v1.json
✓ FOUND: retrieval_checkpoint.json
✓ FOUND: cross_encoder_reranked_results.csv
✓ FOUND: rrf_candidates.csv
✓ FOUND: structured_rag_corpus.csv
✓ FOUND: golden_evaluation_set_frozen.csv
✓ FOUND: response_generation_batch_01.csv
✓ FOUND: response_grounding_batch_01_v2.csv
✓ FOUND: historical_bge_large.npy

✅ Environment paths restored.


In [3]:
# ============================================================
# LOAD FROZEN AUTOMATION POLICY
# ============================================================

policy_path = (
    BASE_DIR
    / "checkpoints"
    / "final_automation_policy_v1.json"
)

if not policy_path.exists():
    raise FileNotFoundError(
        f"Automation policy not found:\n{policy_path}"
    )

with open(policy_path, "r") as f:
    FINAL_AUTOMATION_POLICY = json.load(f)

print("=" * 70)
print("FROZEN AUTOMATION POLICY V1")
print("=" * 70)

print(
    json.dumps(
        FINAL_AUTOMATION_POLICY,
        indent=2
    )
)

print("\n✅ Frozen policy loaded successfully.")

FROZEN AUTOMATION POLICY V1
{
  "policy_name": "conservative_v1",
  "risk_gate": "LOW only",
  "investigation_rate_max": 0.0,
  "query_words_max": 25,
  "high_risk_action": "HUMAN",
  "medium_risk_action": "HUMAN",
  "default_action": "HUMAN"
}

✅ Frozen policy loaded successfully.


In [4]:
# ============================================================
# RESTORE HIVER SUPPORT AGENT DATA FROM CHECKPOINTS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("RESTORING SAVED DATA")
print("=" * 70)


# ============================================================
# 1. GOLDEN EVALUATION SET
# ============================================================

golden_path = (
    BASE_DIR
    / "evaluation"
    / "golden_evaluation_set_frozen.csv"
)

golden_df = pd.read_csv(golden_path)

print(
    f"\nGolden evaluation set: "
    f"{golden_df.shape}"
)


# ============================================================
# 2. CROSS-ENCODER RERANKED RESULTS
# ============================================================

reranked_path = (
    BASE_DIR
    / "checkpoints"
    / "cross_encoder_reranked_results.csv"
)

reranked_df = pd.read_csv(reranked_path)

print(
    f"Cross-encoder results: "
    f"{reranked_df.shape}"
)


# ============================================================
# 3. STRUCTURED RAG CORPUS
# ============================================================

rag_corpus_path = (
    BASE_DIR
    / "checkpoints"
    / "structured_rag_corpus.csv"
)

rag_corpus = pd.read_csv(rag_corpus_path)

print(
    f"RAG corpus: "
    f"{rag_corpus.shape}"
)


# ============================================================
# 4. RRF CANDIDATES
# ============================================================

rrf_path = (
    BASE_DIR
    / "checkpoints"
    / "rrf_candidates.csv"
)

rrf_candidates = pd.read_csv(rrf_path)

print(
    f"RRF candidates: "
    f"{rrf_candidates.shape}"
)


# ============================================================
# 5. RETRIEVAL CHECKPOINT
# ============================================================

retrieval_checkpoint_path = (
    BASE_DIR
    / "checkpoints"
    / "retrieval_checkpoint.json"
)

print(
    f"\nRetrieval checkpoint exists: "
    f"{retrieval_checkpoint_path.exists()}"
)


# ============================================================
# 6. GENERATED RESPONSES
# ============================================================

generation_path = (
    BASE_DIR
    / "evaluation"
    / "response_generation_batch_01.csv"
)

generation_batch_df = pd.read_csv(
    generation_path
)

print(
    f"Generated response batch: "
    f"{generation_batch_df.shape}"
)


# ============================================================
# 7. GROUNDING RESULTS
# ============================================================

grounding_path = (
    BASE_DIR
    / "evaluation"
    / "response_grounding_batch_01_v2.csv"
)

grounding_batch_df = pd.read_csv(
    grounding_path
)

print(
    f"Grounding evaluation: "
    f"{grounding_batch_df.shape}"
)


# ============================================================
# 8. VERIFY IMPORTANT SCHEMAS
# ============================================================

print("\n" + "=" * 70)
print("SCHEMA VERIFICATION")
print("=" * 70)

print("\nGolden columns:")
print(golden_df.columns.tolist())

print("\nRAG corpus columns:")
print(rag_corpus.columns.tolist())

print("\nReranked columns:")
print(reranked_df.columns.tolist())

print("\nGeneration columns:")
print(generation_batch_df.columns.tolist())

print("\nGrounding columns:")
print(grounding_batch_df.columns.tolist())


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("RESTORE COMPLETE")
print("=" * 70)

print("✅ golden_df restored")
print("✅ reranked_df restored")
print("✅ rag_corpus restored")
print("✅ rrf_candidates restored")
print("✅ generation_batch_df restored")
print("✅ grounding_batch_df restored")
print("✅ saved RAG checkpoints available")

RESTORING SAVED DATA

Golden evaluation set: (200, 16)
Cross-encoder results: (10000, 11)
RAG corpus: (83218, 6)
RRF candidates: (19047, 11)

Retrieval checkpoint exists: True
Generated response batch: (7, 3)
Grounding evaluation: (7, 8)

SCHEMA VERIFICATION

Golden columns:
['golden_id', 'case_id', 'root_tweet_id', 'turn_count', 'customer_turns', 'support_turns', 'duration_hours', 'max_gap_hours', 'customer_query', 'support_response', 'provisional_bucket', 'risk_level', 'annotation_notes', 'intent', 'resolution_status', 'automation_decision']

RAG corpus columns:
['case_id', 'customer_problem', 'support_action', 'resolution_type', 'evidence_quality', 'source']

Reranked columns:
['golden_id', 'case_id', 'ce_rank', 'ce_score', 'rrf_rank', 'rrf_score', 'bm25_rank', 'bge_rank', 'historical_weak_intent', 'golden_intent', 'intent_match']

Generation columns:
['golden_id', 'customer_query', 'generated_response']

Grounding columns:
['golden_id', 'customer_query', 'generated_response', 'grou

In [5]:
# ============================================================
# RESTORE GENERATION CONTEXT BUILDER
# ============================================================

def build_generation_context(golden_id, top_k=5):
    """
    Build runtime-safe generation context for a golden case.

    Uses:
    - Golden evaluation set for the current customer query
    - Cross-encoder reranked historical evidence
    - Structured RAG corpus for historical case content

    Evaluation-only fields such as golden_intent and
    intent_match are NOT exposed to the generation context.
    """

    # --------------------------------------------------------
    # Current customer case
    # --------------------------------------------------------

    golden_match = golden_df[
        golden_df["golden_id"] == golden_id
    ]

    if golden_match.empty:
        raise ValueError(
            f"Golden ID not found: {golden_id}"
        )

    golden_row = golden_match.iloc[0]

    customer_query = str(
        golden_row["customer_query"]
    )

    # --------------------------------------------------------
    # Retrieve reranked historical evidence
    # --------------------------------------------------------

    evidence_df = reranked_df[
        reranked_df["golden_id"] == golden_id
    ].copy()

    if evidence_df.empty:
        raise ValueError(
            f"No reranked evidence found for {golden_id}"
        )

    # Cross-encoder rank is the final ranking
    evidence_df = evidence_df.sort_values(
        "ce_rank"
    ).head(top_k)

    # --------------------------------------------------------
    # Join historical case information
    # --------------------------------------------------------

    evidence_df = evidence_df.merge(
        rag_corpus,
        on="case_id",
        how="left"
    )

    # --------------------------------------------------------
    # Build runtime-safe evidence
    # --------------------------------------------------------

    evidence_items = []

    for _, row in evidence_df.iterrows():

        historical_intent = row.get(
            "historical_weak_intent",
            None
        )

        if pd.isna(historical_intent):
            historical_intent = "UNKNOWN"

        evidence_items.append(
            {
                "case_id": str(
                    row["case_id"]
                ),

                "rank": int(
                    row["ce_rank"]
                ),

                "cross_encoder_score": float(
                    row["ce_score"]
                ),

                "customer_problem": str(
                    row["customer_problem"]
                ),

                "support_action": str(
                    row["support_action"]
                ),

                "resolution_type": str(
                    row["resolution_type"]
                ),

                "evidence_quality": str(
                    row["evidence_quality"]
                ),

                "historical_intent": str(
                    historical_intent
                )
            }
        )

    # --------------------------------------------------------
    # Final context
    # --------------------------------------------------------

    context = {
        "golden_id": golden_id,
        "customer_query": customer_query,
        "evidence": evidence_items
    }

    return context


print("✅ build_generation_context() restored.")

✅ build_generation_context() restored.


In [6]:
# ============================================================
# RESTORE TRUST CHECKER V2
# ============================================================

def get_evidence_tier(ce_score):
    """
    Classify historical evidence based on cross-encoder score.
    """

    if ce_score >= 0.90:
        return "STRONG"

    elif ce_score >= 0.75:
        return "CONDITIONAL"

    else:
        return "WEAK"


def calculate_intent_consistency_v2(
    evidence_df,
    predicted_intent
):
    """
    Calculate intent consistency using only known historical
    weak-intent labels.

    UNKNOWN / missing labels are excluded rather than treated
    as intent mismatches.
    """

    if evidence_df.empty:
        return None

    known_intents = []

    for value in evidence_df["historical_weak_intent"]:

        if pd.isna(value):
            continue

        value = str(value).strip().upper()

        if value in ["", "UNKNOWN", "NAN", "NONE"]:
            continue

        known_intents.append(value)

    # No usable historical intent information
    if len(known_intents) == 0:
        return None

    predicted_intent = str(
        predicted_intent
    ).strip().upper()

    matches = [
        intent == predicted_intent
        for intent in known_intents
    ]

    return float(
        np.mean(matches)
    )


def calculate_semantic_consistency(
    evidence_df
):
    """
    Calculate semantic consistency from cross-encoder scores.

    60% weight:
        fraction of evidence with CE >= 0.75

    40% weight:
        mean CE score
    """

    if evidence_df.empty:
        return 0.0

    ce_scores = pd.to_numeric(
        evidence_df["ce_score"],
        errors="coerce"
    ).dropna()

    if len(ce_scores) == 0:
        return 0.0

    strong_or_conditional_fraction = float(
        (ce_scores >= 0.75).mean()
    )

    mean_ce = float(
        ce_scores.mean()
    )

    semantic_consistency = (
        0.60 * strong_or_conditional_fraction
        + 0.40 * mean_ce
    )

    return float(
        semantic_consistency
    )


def trust_check_v2(
    evidence_df,
    predicted_intent,
    risk_level
):
    """
    Trust Checker V2.

    Signals:
        60% top-1 CE score
        20% intent consistency
        20% semantic consistency

    Safety gates:
        HIGH risk -> HUMAN

    Evidence decisions:
        >= 0.80 -> TRUSTED
        >= 0.60 -> CONDITIONAL
        <  0.60 -> WEAK

    Important:
    If no historical weak-intent labels are available,
    intent consistency is treated as neutral (0.5) rather
    than incorrectly treating missing information as mismatch.
    """

    # --------------------------------------------------------
    # No evidence
    # --------------------------------------------------------

    if evidence_df.empty:

        return {
            "trust_score": 0.0,
            "top1_ce_score": 0.0,
            "evidence_tier": "WEAK",
            "intent_consistency": None,
            "semantic_consistency": 0.0,
            "evidence_decision": "WEAK",
            "automation_decision": "HUMAN",
            "reason": "No historical evidence available"
        }

    # --------------------------------------------------------
    # Sort by CE rank
    # --------------------------------------------------------

    evidence_df = evidence_df.copy()

    evidence_df = evidence_df.sort_values(
        "ce_rank"
    )

    # --------------------------------------------------------
    # Top-1 CE
    # --------------------------------------------------------

    top1_ce_score = float(
        evidence_df.iloc[0]["ce_score"]
    )

    # --------------------------------------------------------
    # Evidence tier
    # --------------------------------------------------------

    evidence_tier = get_evidence_tier(
        top1_ce_score
    )

    # --------------------------------------------------------
    # Intent consistency
    # --------------------------------------------------------

    intent_consistency = (
        calculate_intent_consistency_v2(
            evidence_df=evidence_df,
            predicted_intent=predicted_intent
        )
    )

    # --------------------------------------------------------
    # Neutral treatment for missing intent labels
    # --------------------------------------------------------

    if intent_consistency is None:

        intent_signal = 0.5

    else:

        intent_signal = intent_consistency

    # --------------------------------------------------------
    # Semantic consistency
    # --------------------------------------------------------

    semantic_consistency = (
        calculate_semantic_consistency(
            evidence_df
        )
    )

    # --------------------------------------------------------
    # Combined trust score
    # --------------------------------------------------------

    trust_score = (
        0.60 * top1_ce_score
        + 0.20 * intent_signal
        + 0.20 * semantic_consistency
    )

    trust_score = float(
        np.clip(
            trust_score,
            0.0,
            1.0
        )
    )

    # --------------------------------------------------------
    # Evidence decision
    # --------------------------------------------------------

    if trust_score >= 0.80:

        evidence_decision = "TRUSTED"

    elif trust_score >= 0.60:

        evidence_decision = "CONDITIONAL"

    else:

        evidence_decision = "WEAK"

    # --------------------------------------------------------
    # Automation safety gate
    # --------------------------------------------------------

    if str(risk_level).upper() == "HIGH":

        automation_decision = "HUMAN"

        reason = (
            "HIGH risk requires human handling"
        )

    elif evidence_decision == "TRUSTED":

        automation_decision = "AUTO"

        reason = (
            "Historical evidence meets trust criteria"
        )

    else:

        automation_decision = "HUMAN"

        reason = (
            "Historical evidence does not meet "
            "autonomous trust criteria"
        )

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    return {
        "trust_score": trust_score,
        "top1_ce_score": top1_ce_score,
        "evidence_tier": evidence_tier,
        "intent_consistency": intent_consistency,
        "semantic_consistency": semantic_consistency,
        "evidence_decision": evidence_decision,
        "automation_decision": automation_decision,
        "reason": reason
    }


print("=" * 70)
print("TRUST CHECKER V2")
print("=" * 70)
print("✅ get_evidence_tier() restored")
print("✅ calculate_intent_consistency_v2() restored")
print("✅ calculate_semantic_consistency() restored")
print("✅ trust_check_v2() restored")

TRUST CHECKER V2
✅ get_evidence_tier() restored
✅ calculate_intent_consistency_v2() restored
✅ calculate_semantic_consistency() restored
✅ trust_check_v2() restored


In [8]:
# ============================================================
# TRUST CHECKER V3
# ============================================================
#
# Uses the RUNTIME-SAFE evidence schema produced by
# build_generation_context().
#
# Runtime evidence fields:
#   case_id
#   rank
#   cross_encoder_score
#   customer_problem
#   support_action
#   resolution_type
#   evidence_quality
#   historical_intent
#
# Evaluation-only fields such as:
#   golden_intent
#   intent_match
#
# are NOT used.
# ============================================================


def get_evidence_tier(ce_score):
    """
    Classify evidence using cross-encoder score.
    """

    ce_score = float(ce_score)

    if ce_score >= 0.90:
        return "STRONG"

    elif ce_score >= 0.75:
        return "CONDITIONAL"

    else:
        return "WEAK"


def calculate_intent_consistency_v3(
    evidence_df,
    predicted_intent
):
    """
    Calculate historical intent consistency.

    UNKNOWN / missing historical intents are ignored.

    Returns:
        float between 0 and 1
        None if no usable historical intent labels exist.
    """

    if evidence_df.empty:
        return None

    if "historical_intent" not in evidence_df.columns:
        return None

    known_intents = []

    for value in evidence_df["historical_intent"]:

        if pd.isna(value):
            continue

        value = str(value).strip().upper()

        if value in [
            "",
            "UNKNOWN",
            "NAN",
            "NONE"
        ]:
            continue

        known_intents.append(value)

    if len(known_intents) == 0:
        return None

    predicted_intent = str(
        predicted_intent
    ).strip().upper()

    matches = [
        intent == predicted_intent
        for intent in known_intents
    ]

    return float(
        np.mean(matches)
    )


def calculate_semantic_consistency_v3(
    evidence_df
):
    """
    Calculate semantic consistency from runtime-safe
    cross-encoder scores.

    60%:
        Fraction of evidence with CE >= 0.75

    40%:
        Mean CE score
    """

    if evidence_df.empty:
        return 0.0

    if "cross_encoder_score" not in evidence_df.columns:
        return 0.0

    ce_scores = pd.to_numeric(
        evidence_df["cross_encoder_score"],
        errors="coerce"
    ).dropna()

    if len(ce_scores) == 0:
        return 0.0

    conditional_or_strong_fraction = float(
        (ce_scores >= 0.75).mean()
    )

    mean_ce = float(
        ce_scores.mean()
    )

    semantic_consistency = (
        0.60 * conditional_or_strong_fraction
        + 0.40 * mean_ce
    )

    return float(
        semantic_consistency
    )


def trust_check_v3(
    evidence_df,
    predicted_intent,
    risk_level
):
    """
    Trust Checker V3.

    Runtime-safe evidence schema.

    Trust score:

        60% → top-1 cross-encoder score
        20% → intent consistency
        20% → semantic consistency

    Evidence thresholds:

        >= 0.80 → TRUSTED
        >= 0.60 → CONDITIONAL
        <  0.60 → WEAK

    Safety rule:

        HIGH risk → HUMAN

    Missing historical intent labels are treated as
    UNKNOWN / neutral rather than mismatches.
    """

    # ========================================================
    # 1. NO EVIDENCE
    # ========================================================

    if evidence_df.empty:

        return {
            "trust_score": 0.0,
            "top1_ce_score": 0.0,
            "evidence_tier": "WEAK",
            "intent_consistency": None,
            "semantic_consistency": 0.0,
            "evidence_decision": "WEAK",
            "automation_decision": "HUMAN",
            "reason": "No historical evidence available"
        }

    # ========================================================
    # 2. VALIDATE REQUIRED RUNTIME FIELDS
    # ========================================================

    required_columns = [
        "rank",
        "cross_encoder_score",
        "historical_intent"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in evidence_df.columns
    ]

    if missing_columns:

        raise ValueError(
            "Trust Checker V3 received evidence with "
            f"missing runtime fields: {missing_columns}\n\n"
            f"Available columns: "
            f"{evidence_df.columns.tolist()}"
        )

    # ========================================================
    # 3. COPY + SORT BY FINAL RANK
    # ========================================================

    evidence_df = evidence_df.copy()

    evidence_df = evidence_df.sort_values(
        "rank"
    )

    # ========================================================
    # 4. TOP-1 CROSS-ENCODER SCORE
    # ========================================================

    top1_ce_score = float(
        evidence_df.iloc[0]["cross_encoder_score"]
    )

    # ========================================================
    # 5. EVIDENCE TIER
    # ========================================================

    evidence_tier = get_evidence_tier(
        top1_ce_score
    )

    # ========================================================
    # 6. INTENT CONSISTENCY
    # ========================================================

    intent_consistency = (
        calculate_intent_consistency_v3(
            evidence_df=evidence_df,
            predicted_intent=predicted_intent
        )
    )

    # Missing intent information is neutral.
    if intent_consistency is None:

        intent_signal = 0.5

    else:

        intent_signal = intent_consistency

    # ========================================================
    # 7. SEMANTIC CONSISTENCY
    # ========================================================

    semantic_consistency = (
        calculate_semantic_consistency_v3(
            evidence_df
        )
    )

    # ========================================================
    # 8. COMBINED TRUST SCORE
    # ========================================================

    trust_score = (
        0.60 * top1_ce_score
        + 0.20 * intent_signal
        + 0.20 * semantic_consistency
    )

    trust_score = float(
        np.clip(
            trust_score,
            0.0,
            1.0
        )
    )

    # ========================================================
    # 9. EVIDENCE DECISION
    # ========================================================

    if trust_score >= 0.80:

        evidence_decision = "TRUSTED"

    elif trust_score >= 0.60:

        evidence_decision = "CONDITIONAL"

    else:

        evidence_decision = "WEAK"

    # ========================================================
    # 10. AUTOMATION SAFETY DECISION
    # ========================================================

    risk_level = str(
        risk_level
    ).strip().upper()

    if risk_level == "HIGH":

        automation_decision = "HUMAN"

        reason = (
            "HIGH risk requires human handling"
        )

    elif evidence_decision == "TRUSTED":

        automation_decision = "AUTO"

        reason = (
            "Historical evidence meets trust criteria"
        )

    else:

        automation_decision = "HUMAN"

        reason = (
            "Historical evidence does not meet "
            "autonomous trust criteria"
        )

    # ========================================================
    # 11. RETURN
    # ========================================================

    return {
        "trust_score": trust_score,
        "top1_ce_score": top1_ce_score,
        "evidence_tier": evidence_tier,
        "intent_consistency": intent_consistency,
        "semantic_consistency": semantic_consistency,
        "evidence_decision": evidence_decision,
        "automation_decision": automation_decision,
        "reason": reason
    }


print("=" * 70)
print("TRUST CHECKER V3")
print("=" * 70)

print("✅ get_evidence_tier() restored")
print("✅ calculate_intent_consistency_v3() restored")
print("✅ calculate_semantic_consistency_v3() restored")
print("✅ trust_check_v3() restored")

print("\nRuntime-safe schema expected:")
print([
    "rank",
    "cross_encoder_score",
    "historical_intent"
])

TRUST CHECKER V3
✅ get_evidence_tier() restored
✅ calculate_intent_consistency_v3() restored
✅ calculate_semantic_consistency_v3() restored
✅ trust_check_v3() restored

Runtime-safe schema expected:
['rank', 'cross_encoder_score', 'historical_intent']


In [9]:
# ============================================================
# TRUST CHECKER V3 TEST — GOLD-0001
# ============================================================

test_context = build_generation_context(
    golden_id="GOLD-0001",
    top_k=5
)

test_evidence_df = pd.DataFrame(
    test_context["evidence"]
)

print("=" * 70)
print("RUNTIME EVIDENCE SCHEMA")
print("=" * 70)

print(
    test_evidence_df.columns.tolist()
)

print("\nEvidence shape:")
print(test_evidence_df.shape)


trust_test = trust_check_v3(
    evidence_df=test_evidence_df,
    predicted_intent="PAYMENT_BILLING",
    risk_level="MEDIUM"
)

print("\n" + "=" * 70)
print("TRUST CHECKER V3 — GOLD-0001")
print("=" * 70)

for key, value in trust_test.items():
    print(f"{key}: {value}")

RUNTIME EVIDENCE SCHEMA
['case_id', 'rank', 'cross_encoder_score', 'customer_problem', 'support_action', 'resolution_type', 'evidence_quality', 'historical_intent']

Evidence shape:
(5, 8)

TRUST CHECKER V3 — GOLD-0001
trust_score: 0.489545845584
top1_ce_score: 0.61528623
evidence_tier: WEAK
intent_consistency: 0.5
semantic_consistency: 0.10187053792
evidence_decision: WEAK
automation_decision: HUMAN
reason: Historical evidence does not meet autonomous trust criteria


In [10]:
# ============================================================
# TRUST CHECKER V3 — FULL GOLDEN EVALUATION
# ============================================================

trust_v3_results = []

for golden_id in golden_df["golden_id"]:

    # --------------------------------------------------------
    # Get gold evaluation information
    # --------------------------------------------------------

    gold_row = golden_df[
        golden_df["golden_id"] == golden_id
    ].iloc[0]

    gold_intent = str(
        gold_row["intent"]
    )

    gold_risk = str(
        gold_row["risk_level"]
    )

    gold_automation = str(
        gold_row["automation_decision"]
    )

    # --------------------------------------------------------
    # Build runtime-safe evidence
    # --------------------------------------------------------

    context = build_generation_context(
        golden_id=golden_id,
        top_k=5
    )

    evidence_df = pd.DataFrame(
        context["evidence"]
    )

    # --------------------------------------------------------
    # Controlled evaluation
    #
    # IMPORTANT:
    # gold_intent is used ONLY to evaluate the Trust Checker.
    # It is NOT a runtime feature.
    # --------------------------------------------------------

    trust_result = trust_check_v3(
        evidence_df=evidence_df,
        predicted_intent=gold_intent,
        risk_level=gold_risk
    )

    trust_v3_results.append(
        {
            "golden_id": golden_id,
            "gold_intent": gold_intent,
            "gold_risk_level": gold_risk,
            "gold_automation_decision": gold_automation,

            "trust_score": trust_result[
                "trust_score"
            ],

            "top1_ce_score": trust_result[
                "top1_ce_score"
            ],

            "evidence_tier": trust_result[
                "evidence_tier"
            ],

            "intent_consistency": trust_result[
                "intent_consistency"
            ],

            "semantic_consistency": trust_result[
                "semantic_consistency"
            ],

            "evidence_decision": trust_result[
                "evidence_decision"
            ],

            "trust_automation_decision": trust_result[
                "automation_decision"
            ],

            "reason": trust_result[
                "reason"
            ]
        }
    )


trust_v3_df = pd.DataFrame(
    trust_v3_results
)


# ============================================================
# BASIC SUMMARY
# ============================================================

print("=" * 80)
print("TRUST CHECKER V3 — GOLDEN EVALUATION")
print("=" * 80)

print(
    "\nTotal cases:",
    len(trust_v3_df)
)

print("\nEvidence decisions:")

print(
    trust_v3_df[
        "evidence_decision"
    ].value_counts()
)

print("\nTrust automation decisions:")

print(
    trust_v3_df[
        "trust_automation_decision"
    ].value_counts()
)

TRUST CHECKER V3 — GOLDEN EVALUATION

Total cases: 200

Evidence decisions:
evidence_decision
TRUSTED        107
CONDITIONAL     51
WEAK            42
Name: count, dtype: int64

Trust automation decisions:
trust_automation_decision
HUMAN    108
AUTO      92
Name: count, dtype: int64


In [11]:
# ============================================================
# TRUST CHECKER V3 — AUTOMATION CONFUSION MATRIX
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

y_true = (
    trust_v3_df[
        "gold_automation_decision"
    ]
    .str.upper()
)

y_pred = (
    trust_v3_df[
        "trust_automation_decision"
    ]
    .str.upper()
)


# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["AUTO", "HUMAN"]
)

print("=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print(
    pd.DataFrame(
        cm,
        index=[
            "GOLD_AUTO",
            "GOLD_HUMAN"
        ],
        columns=[
            "PRED_AUTO",
            "PRED_HUMAN"
        ]
    )
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

auto_precision = precision_score(
    y_true,
    y_pred,
    pos_label="AUTO",
    zero_division=0
)

auto_recall = recall_score(
    y_true,
    y_pred,
    pos_label="AUTO",
    zero_division=0
)

auto_f1 = f1_score(
    y_true,
    y_pred,
    pos_label="AUTO",
    zero_division=0
)

overall_accuracy = (
    (y_true == y_pred).mean()
)


print("\n" + "=" * 80)
print("AUTOMATION METRICS")
print("=" * 80)

print(
    f"AUTO precision : {auto_precision:.4f}"
)

print(
    f"AUTO recall    : {auto_recall:.4f}"
)

print(
    f"AUTO F1        : {auto_f1:.4f}"
)

print(
    f"Overall accuracy: {overall_accuracy:.4f}"
)


# ------------------------------------------------------------
# Risk Safety Check
# ------------------------------------------------------------

high_risk_auto = (
    trust_v3_df[
        trust_v3_df["gold_risk_level"].str.upper() == "HIGH"
    ]["trust_automation_decision"]
    .str.upper()
    .eq("AUTO")
    .sum()
)

print("\n" + "=" * 80)
print("HIGH-RISK SAFETY CHECK")
print("=" * 80)

print(
    "HIGH-risk cases routed AUTO:",
    high_risk_auto
)

if high_risk_auto == 0:
    print("✅ HIGH-risk hard gate passed.")
else:
    print("❌ HIGH-risk safety violation detected.")

CONFUSION MATRIX
            PRED_AUTO  PRED_HUMAN
GOLD_AUTO          24          28
GOLD_HUMAN         68          80

AUTOMATION METRICS
AUTO precision : 0.2609
AUTO recall    : 0.4615
AUTO F1        : 0.3333
Overall accuracy: 0.5200

HIGH-RISK SAFETY CHECK
HIGH-risk cases routed AUTO: 0
✅ HIGH-risk hard gate passed.


In [12]:
# ============================================================
# TRUST CHECKER V3 — FALSE AUTO ANALYSIS
# ============================================================

false_auto_df = trust_v3_df[
    (
        trust_v3_df["gold_automation_decision"].str.upper()
        == "HUMAN"
    )
    &
    (
        trust_v3_df["trust_automation_decision"].str.upper()
        == "AUTO"
    )
].copy()


print("=" * 80)
print("TRUST CHECKER V3 — FALSE AUTO CASES")
print("=" * 80)

print(
    "\nFalse AUTO cases:",
    len(false_auto_df)
)


# ============================================================
# RISK BREAKDOWN
# ============================================================

print("\n" + "=" * 80)
print("FALSE AUTO BY RISK")
print("=" * 80)

print(
    false_auto_df[
        "gold_risk_level"
    ].value_counts()
)


# ============================================================
# INTENT BREAKDOWN
# ============================================================

print("\n" + "=" * 80)
print("FALSE AUTO BY INTENT")
print("=" * 80)

print(
    false_auto_df[
        "gold_intent"
    ].value_counts()
)


# ============================================================
# RESOLUTION / TRUST SIGNAL DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("FALSE AUTO TRUST SIGNALS")
print("=" * 80)

display(
    false_auto_df[
        [
            "golden_id",
            "gold_intent",
            "gold_risk_level",
            "trust_score",
            "top1_ce_score",
            "evidence_tier",
            "intent_consistency",
            "semantic_consistency",
            "evidence_decision"
        ]
    ]
    .sort_values(
        "trust_score",
        ascending=False
    )
    .head(30)
)


# ============================================================
# HIGH-TRUST FALSE AUTO CASES
# ============================================================

high_trust_false_auto = false_auto_df[
    false_auto_df["trust_score"] >= 0.80
].copy()

print("\n" + "=" * 80)
print("HIGH-TRUST FALSE AUTO")
print("=" * 80)

print(
    "Count:",
    len(high_trust_false_auto)
)

if len(high_trust_false_auto) > 0:

    display(
        high_trust_false_auto[
            [
                "golden_id",
                "gold_intent",
                "gold_risk_level",
                "trust_score",
                "top1_ce_score",
                "intent_consistency",
                "semantic_consistency"
            ]
        ]
        .sort_values(
            "trust_score",
            ascending=False
        )
        .head(20)
    )

TRUST CHECKER V3 — FALSE AUTO CASES

False AUTO cases: 68

FALSE AUTO BY RISK
gold_risk_level
MEDIUM    36
LOW       32
Name: count, dtype: int64

FALSE AUTO BY INTENT
gold_intent
ORDER_DELIVERY         43
GENERAL_SUPPORT        13
RETURNS_REFUNDS         4
ACCOUNT_ACCESS          2
ORDER_PURCHASE          1
ORDER_TRACKING          1
PREORDER_DELIVERY       1
ORDER_CANCELLATION      1
MARKETPLACE_SELLING     1
DIGITAL_CONTENT         1
Name: count, dtype: int64

FALSE AUTO TRUST SIGNALS


,golden_id,gold_intent,gold_risk_level,trust_score,top1_ce_score,evidence_tier,intent_consistency,semantic_consistency,evidence_decision
42,GOLD-0043,ORDER_DELIVERY,MEDIUM,0.999551,0.999595,STRONG,1.000000,0.998968,TRUSTED
7,GOLD-0008,ORDER_DELIVERY,LOW,0.998967,0.999397,STRONG,1.000000,0.996644,TRUSTED
1,GOLD-0002,ORDER_DELIVERY,LOW,0.998629,0.998420,STRONG,1.000000,0.997882,TRUSTED
59,GOLD-0060,ORDER_DELIVERY,LOW,0.998535,0.998168,STRONG,1.000000,0.998172,TRUSTED
139,GOLD-0140,ORDER_DELIVERY,LOW,0.997728,0.998881,STRONG,1.000000,0.991996,TRUSTED
99,GOLD-0100,ORDER_DELIVERY,MEDIUM,0.996931,0.997857,STRONG,1.000000,0.991084,TRUSTED
106,GOLD-0107,ORDER_DELIVERY,MEDIUM,0.996633,0.995958,STRONG,1.000000,0.995288,TRUSTED
78,GOLD-0079,ORDER_DELIVERY,LOW,0.994898,0.998869,STRONG,1.000000,0.977883,TRUSTED
93,GOLD-0094,ORDER_DELIVERY,MEDIUM,0.994793,0.993924,STRONG,1.000000,0.992192,TRUSTED
39,GOLD-0040,RETURNS_REFUNDS,MEDIUM,0.994272,0.992637,STRONG,1.000000,0.993449,TRUSTED



HIGH-TRUST FALSE AUTO
Count: 68


,golden_id,gold_intent,gold_risk_level,trust_score,top1_ce_score,intent_consistency,semantic_consistency
42,GOLD-0043,ORDER_DELIVERY,MEDIUM,0.999551,0.999595,1.0,0.998968
7,GOLD-0008,ORDER_DELIVERY,LOW,0.998967,0.999397,1.0,0.996644
1,GOLD-0002,ORDER_DELIVERY,LOW,0.998629,0.998420,1.0,0.997882
59,GOLD-0060,ORDER_DELIVERY,LOW,0.998535,0.998168,1.0,0.998172
139,GOLD-0140,ORDER_DELIVERY,LOW,0.997728,0.998881,1.0,0.991996
99,GOLD-0100,ORDER_DELIVERY,MEDIUM,0.996931,0.997857,1.0,0.991084
106,GOLD-0107,ORDER_DELIVERY,MEDIUM,0.996633,0.995958,1.0,0.995288
78,GOLD-0079,ORDER_DELIVERY,LOW,0.994898,0.998869,1.0,0.977883
93,GOLD-0094,ORDER_DELIVERY,MEDIUM,0.994793,0.993924,1.0,0.992192
39,GOLD-0040,RETURNS_REFUNDS,MEDIUM,0.994272,0.992637,1.0,0.993449


In [13]:
# ============================================================
# COMBINED AUTOMATION POLICY + TRUST GATE
# FULL 200-CASE GOLDEN EVALUATION
# ============================================================

combined_results = []

for golden_id in golden_df["golden_id"]:

    # --------------------------------------------------------
    # Gold evaluation data
    # --------------------------------------------------------

    gold_row = golden_df[
        golden_df["golden_id"] == golden_id
    ].iloc[0]

    gold_intent = str(
        gold_row["intent"]
    ).strip().upper()

    gold_risk = str(
        gold_row["risk_level"]
    ).strip().upper()

    gold_automation = str(
        gold_row["automation_decision"]
    ).strip().upper()

    # --------------------------------------------------------
    # Runtime-safe evidence
    # --------------------------------------------------------

    context = build_generation_context(
        golden_id=golden_id,
        top_k=5
    )

    evidence_df = pd.DataFrame(
        context["evidence"]
    )

    # --------------------------------------------------------
    # Trust Checker
    #
    # gold_intent is used ONLY for controlled evaluation.
    # It is NOT a runtime feature.
    # --------------------------------------------------------

    trust_result = trust_check_v3(
        evidence_df=evidence_df,
        predicted_intent=gold_intent,
        risk_level=gold_risk
    )

    # --------------------------------------------------------
    # Query complexity
    # --------------------------------------------------------

    query_words = len(
        str(
            context["customer_query"]
        ).split()
    )

    # --------------------------------------------------------
    # Historical investigation signal
    # --------------------------------------------------------

    resolution_types = [
        str(
            item.get(
                "resolution_type",
                ""
            )
        ).upper()
        for item in context["evidence"]
    ]

    if resolution_types:

        investigation_count = sum(
            1
            for value in resolution_types
            if (
                "INVESTIGATION" in value
                or "FOLLOWUP" in value
                or "FOLLOW_UP" in value
            )
        )

        investigation_rate = (
            investigation_count
            / len(resolution_types)
        )

    else:

        investigation_rate = 0.0

    # --------------------------------------------------------
    # FINAL AUTOMATION POLICY
    # --------------------------------------------------------

    decision = "HUMAN"
    reason = ""

    # HARD SAFETY GATE
    if gold_risk == "HIGH":

        decision = "HUMAN"

        reason = (
            "HIGH risk hard gate"
        )

    # MEDIUM CONSERVATIVE GATE
    elif gold_risk == "MEDIUM":

        decision = "HUMAN"

        reason = (
            "MEDIUM risk conservative gate"
        )

    # TRUST GATE
    elif trust_result[
        "evidence_decision"
    ] != "TRUSTED":

        decision = "HUMAN"

        reason = (
            "Evidence did not meet TRUSTED threshold"
        )

    # INVESTIGATION GATE
    elif investigation_rate > 0:

        decision = "HUMAN"

        reason = (
            "Historical evidence suggests investigation/follow-up"
        )

    # QUERY COMPLEXITY GATE
    elif query_words > 25:

        decision = "HUMAN"

        reason = (
            "Query exceeds conservative complexity threshold"
        )

    # OTHERWISE AUTO
    else:

        decision = "AUTO"

        reason = (
            "All conservative automation gates passed"
        )

    combined_results.append(
        {
            "golden_id": golden_id,
            "gold_intent": gold_intent,
            "gold_risk_level": gold_risk,
            "gold_automation_decision": gold_automation,

            "trust_score": trust_result[
                "trust_score"
            ],

            "evidence_tier": trust_result[
                "evidence_tier"
            ],

            "evidence_decision": trust_result[
                "evidence_decision"
            ],

            "query_words": query_words,

            "investigation_rate": investigation_rate,

            "final_decision": decision,

            "decision_reason": reason
        }
    )


combined_df = pd.DataFrame(
    combined_results
)


# ============================================================
# RESULTS
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

y_true = combined_df[
    "gold_automation_decision"
]

y_pred = combined_df[
    "final_decision"
]


cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["AUTO", "HUMAN"]
)

print("=" * 80)
print("COMBINED AUTOMATION POLICY")
print("=" * 80)

print("\nConfusion Matrix:")

display(
    pd.DataFrame(
        cm,
        index=[
            "GOLD_AUTO",
            "GOLD_HUMAN"
        ],
        columns=[
            "PRED_AUTO",
            "PRED_HUMAN"
        ]
    )
)


auto_precision = precision_score(
    y_true,
    y_pred,
    pos_label="AUTO",
    zero_division=0
)

auto_recall = recall_score(
    y_true,
    y_pred,
    pos_label="AUTO",
    zero_division=0
)

auto_f1 = f1_score(
    y_true,
    y_pred,
    pos_label="AUTO",
    zero_division=0
)

accuracy = accuracy_score(
    y_true,
    y_pred
)

auto_count = (
    y_pred == "AUTO"
).sum()

print("\n" + "=" * 80)
print("METRICS")
print("=" * 80)

print(
    f"AUTO precision : {auto_precision:.4f}"
)

print(
    f"AUTO recall    : {auto_recall:.4f}"
)

print(
    f"AUTO F1        : {auto_f1:.4f}"
)

print(
    f"Overall accuracy: {accuracy:.4f}"
)

print(
    f"AUTO decisions : {auto_count}/{len(combined_df)}"
)

print(
    f"AUTO coverage  : "
    f"{auto_count / len(combined_df):.4f}"
)


# ============================================================
# SAFETY CHECKS
# ============================================================

high_risk_auto = combined_df[
    combined_df["gold_risk_level"] == "HIGH"
][
    "final_decision"
].eq("AUTO").sum()

medium_risk_auto = combined_df[
    combined_df["gold_risk_level"] == "MEDIUM"
][
    "final_decision"
].eq("AUTO").sum()


print("\n" + "=" * 80)
print("SAFETY GATES")
print("=" * 80)

print(
    "HIGH-risk AUTO:",
    high_risk_auto
)

print(
    "MEDIUM-risk AUTO:",
    medium_risk_auto
)

if high_risk_auto == 0:
    print("✅ HIGH-risk gate passed.")

if medium_risk_auto == 0:
    print("✅ MEDIUM-risk gate passed.")

COMBINED AUTOMATION POLICY

Confusion Matrix:


,PRED_AUTO,PRED_HUMAN
GOLD_AUTO,14,38
GOLD_HUMAN,3,145



METRICS
AUTO precision : 0.8235
AUTO recall    : 0.2692
AUTO F1        : 0.4058
Overall accuracy: 0.7950
AUTO decisions : 17/200
AUTO coverage  : 0.0850

SAFETY GATES
HIGH-risk AUTO: 0
MEDIUM-risk AUTO: 0
✅ HIGH-risk gate passed.
✅ MEDIUM-risk gate passed.


In [14]:
# ============================================================
# AUTOMATION GATE ABLATION ANALYSIS
# ============================================================

print("=" * 80)
print("AUTOMATION GATE ABLATION ANALYSIS")
print("=" * 80)


# ------------------------------------------------------------
# Gate 1: Frozen policy only
# ------------------------------------------------------------

policy_only_auto = (
    (combined_df["gold_risk_level"] == "LOW")
    &
    (combined_df["investigation_rate"] == 0)
    &
    (combined_df["query_words"] <= 25)
)


# ------------------------------------------------------------
# Gate 2: Policy + TRUSTED evidence
# ------------------------------------------------------------

policy_plus_trust_auto = (
    policy_only_auto
    &
    (
        combined_df["evidence_decision"]
        == "TRUSTED"
    )
)


# ------------------------------------------------------------
# Identify cases rejected by Trust
# ------------------------------------------------------------

trust_rejected = (
    policy_only_auto
    &
    (
        combined_df["evidence_decision"]
        != "TRUSTED"
    )
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total_cases = len(combined_df)

policy_count = int(
    policy_only_auto.sum()
)

policy_trust_count = int(
    policy_plus_trust_auto.sum()
)

trust_rejected_count = int(
    trust_rejected.sum()
)


print("\n" + "=" * 80)
print("GATE COUNTS")
print("=" * 80)

print(
    f"Total golden cases          : {total_cases}"
)

print(
    f"Frozen policy AUTO          : {policy_count}"
)

print(
    f"Policy + TRUSTED AUTO       : {policy_trust_count}"
)

print(
    f"Policy candidates rejected  : {trust_rejected_count}"
)


print("\n" + "=" * 80)
print("POLICY CANDIDATE EVIDENCE BREAKDOWN")
print("=" * 80)

print(
    combined_df[
        policy_only_auto
    ]["evidence_decision"]
    .value_counts()
)


print("\n" + "=" * 80)
print("POLICY CANDIDATE EVIDENCE TIERS")
print("=" * 80)

print(
    combined_df[
        policy_only_auto
    ]["evidence_tier"]
    .value_counts()
)


# ------------------------------------------------------------
# Rejected candidate examples
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("POLICY CANDIDATES REJECTED BY TRUST")
print("=" * 80)

rejected_df = combined_df[
    trust_rejected
].copy()

if len(rejected_df) > 0:

    display(
        rejected_df[
            [
                "golden_id",
                "gold_intent",
                "gold_risk_level",
                "trust_score",
                "evidence_tier",
                "evidence_decision",
                "query_words",
                "investigation_rate"
            ]
        ]
        .sort_values(
            "trust_score",
            ascending=True
        )
        .head(30)
    )

else:

    print(
        "No policy candidates were rejected by Trust."
    )

AUTOMATION GATE ABLATION ANALYSIS

GATE COUNTS
Total golden cases          : 200
Frozen policy AUTO          : 37
Policy + TRUSTED AUTO       : 17
Policy candidates rejected  : 20

POLICY CANDIDATE EVIDENCE BREAKDOWN
evidence_decision
TRUSTED        17
WEAK           14
CONDITIONAL     6
Name: count, dtype: int64

POLICY CANDIDATE EVIDENCE TIERS
evidence_tier
STRONG         21
WEAK           13
CONDITIONAL     3
Name: count, dtype: int64

POLICY CANDIDATES REJECTED BY TRUST


,golden_id,gold_intent,gold_risk_level,trust_score,evidence_tier,evidence_decision,query_words,investigation_rate
126,GOLD-0127,GENERAL_SOCIAL,LOW,0.043398,WEAK,WEAK,22,0.0
190,GOLD-0191,PAYMENT_BILLING,LOW,0.114516,WEAK,WEAK,13,0.0
41,GOLD-0042,GENERAL_SUPPORT,LOW,0.115695,WEAK,WEAK,4,0.0
14,GOLD-0015,GENERAL_SOCIAL,LOW,0.142627,WEAK,WEAK,18,0.0
86,GOLD-0087,GENERAL_SOCIAL,LOW,0.169345,WEAK,WEAK,19,0.0
141,GOLD-0142,GENERAL_SOCIAL,LOW,0.174816,WEAK,WEAK,14,0.0
123,GOLD-0124,GENERAL_SOCIAL,LOW,0.177698,WEAK,WEAK,7,0.0
112,GOLD-0113,GENERAL_SOCIAL,LOW,0.219343,WEAK,WEAK,11,0.0
137,GOLD-0138,GENERAL_SUPPORT,LOW,0.248422,WEAK,WEAK,21,0.0
95,GOLD-0096,GENERAL_SUPPORT,LOW,0.348151,WEAK,WEAK,2,0.0


In [15]:
# ============================================================
# TRUST GATE ABLATION — STRONG/CONDITIONAL VS TRUSTED
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)


print("=" * 80)
print("TRUST GATE ABLATION")
print("=" * 80)


# ============================================================
# BASE FROZEN POLICY
# ============================================================

policy_only = (
    (combined_df["gold_risk_level"] == "LOW")
    &
    (combined_df["investigation_rate"] == 0)
    &
    (combined_df["query_words"] <= 25)
)


# ============================================================
# POLICY + TRUSTED
# ============================================================

policy_trusted = (
    policy_only
    &
    (
        combined_df["evidence_decision"]
        == "TRUSTED"
    )
)


# ============================================================
# POLICY + NOT WEAK
# ============================================================

policy_not_weak = (
    policy_only
    &
    (
        combined_df["evidence_tier"]
        != "WEAK"
    )
)


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_policy_variant(
    candidate_mask,
    variant_name
):

    y_true = (
        combined_df[
            "gold_automation_decision"
        ]
        .str.upper()
    )

    y_pred = pd.Series(
        "HUMAN",
        index=combined_df.index
    )

    y_pred.loc[candidate_mask] = "AUTO"

    precision = precision_score(
        y_true,
        y_pred,
        pos_label="AUTO",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label="AUTO",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label="AUTO",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    auto_count = int(
        candidate_mask.sum()
    )

    coverage = (
        auto_count
        / len(combined_df)
    )

    false_auto = int(
        (
            candidate_mask
            &
            (
                y_true == "HUMAN"
            )
        ).sum()
    )

    true_auto = int(
        (
            candidate_mask
            &
            (
                y_true == "AUTO"
            )
        ).sum()
    )

    return {
        "variant": variant_name,
        "auto_count": auto_count,
        "coverage": coverage,
        "true_auto": true_auto,
        "false_auto": false_auto,
        "auto_precision": precision,
        "auto_recall": recall,
        "auto_f1": f1,
        "overall_accuracy": accuracy
    }


# ============================================================
# EVALUATE BOTH VARIANTS
# ============================================================

results = []

results.append(
    evaluate_policy_variant(
        policy_only,
        "POLICY_ONLY"
    )
)

results.append(
    evaluate_policy_variant(
        policy_trusted,
        "POLICY_PLUS_TRUSTED"
    )
)

results.append(
    evaluate_policy_variant(
        policy_not_weak,
        "POLICY_PLUS_NOT_WEAK"
    )
)


ablation_df = pd.DataFrame(
    results
)


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 80)
print("COMPARISON")
print("=" * 80)

display(
    ablation_df[
        [
            "variant",
            "auto_count",
            "coverage",
            "true_auto",
            "false_auto",
            "auto_precision",
            "auto_recall",
            "auto_f1",
            "overall_accuracy"
        ]
    ]
)


# ============================================================
# SAFETY CHECK
# ============================================================

for variant_name, mask in [
    ("POLICY_ONLY", policy_only),
    ("POLICY_PLUS_TRUSTED", policy_trusted),
    ("POLICY_PLUS_NOT_WEAK", policy_not_weak)
]:

    high_risk_auto = (
        mask
        &
        (
            combined_df[
                "gold_risk_level"
            ] == "HIGH"
        )
    ).sum()

    medium_risk_auto = (
        mask
        &
        (
            combined_df[
                "gold_risk_level"
            ] == "MEDIUM"
        )
    ).sum()

    print(
        f"\n{variant_name}:"
    )

    print(
        "  HIGH-risk AUTO:",
        int(high_risk_auto)
    )

    print(
        "  MEDIUM-risk AUTO:",
        int(medium_risk_auto)
    )

TRUST GATE ABLATION

COMPARISON


,variant,auto_count,coverage,true_auto,false_auto,auto_precision,auto_recall,auto_f1,overall_accuracy
0,POLICY_ONLY,37,0.185,30,7,0.810811,0.576923,0.674157,0.855
1,POLICY_PLUS_TRUSTED,17,0.085,14,3,0.823529,0.269231,0.405797,0.795
2,POLICY_PLUS_NOT_WEAK,24,0.120,20,4,0.833333,0.384615,0.526316,0.820



POLICY_ONLY:
  HIGH-risk AUTO: 0
  MEDIUM-risk AUTO: 0

POLICY_PLUS_TRUSTED:
  HIGH-risk AUTO: 0
  MEDIUM-risk AUTO: 0

POLICY_PLUS_NOT_WEAK:
  HIGH-risk AUTO: 0
  MEDIUM-risk AUTO: 0


In [16]:
# =============================================================================
# END-TO-END AGENT ORCHESTRATOR — CELL 1
# Load final policy and define the agent state structure
# =============================================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# BASE PATH
# -----------------------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/hiver_support_agent_checkpoint/data")

# -----------------------------------------------------------------------------
# LOAD FINAL AUTOMATION POLICY
# -----------------------------------------------------------------------------

policy_path = BASE_DIR / "checkpoints" / "final_automation_policy_v1.json"

with open(policy_path, "r") as f:
    FINAL_AUTOMATION_POLICY = json.load(f)

print("=" * 80)
print("FINAL AUTOMATION POLICY")
print("=" * 80)

print(json.dumps(FINAL_AUTOMATION_POLICY, indent=2))

# -----------------------------------------------------------------------------
# LOAD CORE DATASETS
# -----------------------------------------------------------------------------

golden_df = pd.read_csv(
    BASE_DIR / "evaluation" / "golden_evaluation_set_frozen.csv"
)

reranked_df = pd.read_csv(
    BASE_DIR / "checkpoints" / "cross_encoder_reranked_results.csv"
)

rag_corpus = pd.read_csv(
    BASE_DIR / "checkpoints" / "structured_rag_corpus.csv"
)

print("\n" + "=" * 80)
print("DATASETS")
print("=" * 80)

print(f"Golden cases      : {len(golden_df):,}")
print(f"Reranked records  : {len(reranked_df):,}")
print(f"RAG corpus cases  : {len(rag_corpus):,}")

# -----------------------------------------------------------------------------
# AGENT STATE
# -----------------------------------------------------------------------------

def create_agent_state(
    customer_query,
    predicted_intent=None,
    risk_level=None
):
    """
    Creates the shared state passed through the end-to-end agent pipeline.
    """

    state = {
        # Input
        "customer_query": customer_query,

        # Router
        "predicted_intent": predicted_intent,
        "risk_level": risk_level,

        # Retrieval
        "retrieved_evidence": None,
        "evidence_count": 0,

        # Evidence quality
        "evidence_tier": None,
        "trust_score": None,
        "trust_decision": None,

        # Historical behavior
        "investigation_rate": None,
        "general_support_rate": None,

        # Query complexity
        "query_words": len(str(customer_query).split()),
        "query_chars": len(str(customer_query)),

        # Generation
        "generated_response": None,

        # Grounding
        "grounding_decision": None,
        "grounding_violations": [],
        "grounding_warnings": [],

        # Final routing
        "automation_decision": None,

        # Explainability
        "decision_reasons": [],

        # Pipeline status
        "pipeline_status": "INITIALIZED"
    }

    return state


print("\nAgent state initialized successfully.")

FINAL AUTOMATION POLICY
{
  "policy_name": "conservative_v1",
  "risk_gate": "LOW only",
  "investigation_rate_max": 0.0,
  "query_words_max": 25,
  "high_risk_action": "HUMAN",
  "medium_risk_action": "HUMAN",
  "default_action": "HUMAN"
}

DATASETS
Golden cases      : 200
Reranked records  : 10,000
RAG corpus cases  : 83,218

Agent state initialized successfully.


In [17]:
# =============================================================================
# END-TO-END AGENT ORCHESTRATOR — CELL 2
# Deterministic automation policy
# =============================================================================

def apply_final_automation_policy(
    risk_level,
    investigation_rate,
    query_words,
    policy=FINAL_AUTOMATION_POLICY
):
    """
    Final deterministic automation decision.

    Policy:
        LOW risk
        AND investigation_rate == 0
        AND query_words <= 25
        -> AUTO

    Everything else -> HUMAN

    HIGH and MEDIUM are explicitly hard-gated to HUMAN.
    """

    risk_level = str(risk_level).upper().strip()

    investigation_rate = (
        float(investigation_rate)
        if investigation_rate is not None
        else 1.0
    )

    query_words = int(query_words)

    reasons = []

    # -------------------------------------------------------------------------
    # HARD SAFETY GATE
    # -------------------------------------------------------------------------

    if risk_level == "HIGH":
        reasons.append("HIGH risk -> mandatory human review")

        return {
            "automation_decision": "HUMAN",
            "reasons": reasons
        }

    if risk_level == "MEDIUM":
        reasons.append("MEDIUM risk -> conservative human review")

        return {
            "automation_decision": "HUMAN",
            "reasons": reasons
        }

    # -------------------------------------------------------------------------
    # LOW-RISK AUTOMATION CANDIDATE
    # -------------------------------------------------------------------------

    if risk_level != "LOW":
        reasons.append("Unknown risk level -> human review")

        return {
            "automation_decision": "HUMAN",
            "reasons": reasons
        }

    # Investigation tendency gate
    if investigation_rate > policy["investigation_rate_max"]:
        reasons.append(
            f"Historical investigation tendency "
            f"({investigation_rate:.2f}) exceeds allowed threshold"
        )

        return {
            "automation_decision": "HUMAN",
            "reasons": reasons
        }

    # Query complexity gate
    if query_words > policy["query_words_max"]:
        reasons.append(
            f"Query complexity ({query_words} words) exceeds "
            f"maximum of {policy['query_words_max']}"
        )

        return {
            "automation_decision": "HUMAN",
            "reasons": reasons
        }

    # -------------------------------------------------------------------------
    # AUTO
    # -------------------------------------------------------------------------

    reasons.append("LOW risk")
    reasons.append("No historical investigation tendency")
    reasons.append(
        f"Query complexity within threshold ({query_words} <= "
        f"{policy['query_words_max']} words)"
    )

    return {
        "automation_decision": "AUTO",
        "reasons": reasons
    }


print("=" * 80)
print("AUTOMATION POLICY TEST")
print("=" * 80)

tests = [
    ("LOW", 0.0, 10),
    ("LOW", 0.2, 10),
    ("LOW", 0.0, 30),
    ("MEDIUM", 0.0, 10),
    ("HIGH", 0.0, 10),
]

for risk, investigation, words in tests:

    result = apply_final_automation_policy(
        risk_level=risk,
        investigation_rate=investigation,
        query_words=words
    )

    print(
        f"risk={risk:<6} "
        f"investigation={investigation:<4} "
        f"words={words:<3} "
        f"-> {result['automation_decision']}"
    )

AUTOMATION POLICY TEST
risk=LOW    investigation=0.0  words=10  -> AUTO
risk=LOW    investigation=0.2  words=10  -> HUMAN
risk=LOW    investigation=0.0  words=30  -> HUMAN
risk=MEDIUM investigation=0.0  words=10  -> HUMAN
risk=HIGH   investigation=0.0  words=10  -> HUMAN


In [18]:
# =============================================================================
# END-TO-END AGENT ORCHESTRATOR — CELL 3
# Runtime evidence feature extraction
# =============================================================================

def calculate_runtime_evidence_features(evidence_df):
    """
    Calculate runtime-safe historical evidence statistics.

    These features are derived only from retrievable historical evidence.
    Evaluation-only columns such as golden_intent and intent_match are ignored.
    """

    if evidence_df is None or len(evidence_df) == 0:

        return {
            "investigation_rate": 1.0,
            "general_support_rate": 0.0,
            "refund_rate": 0.0,
            "cancellation_rate": 0.0,
            "replacement_rate": 0.0,
            "evidence_count": 0
        }

    resolution_types = (
        evidence_df["resolution_type"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.upper()
    )

    n = len(evidence_df)

    investigation_rate = (
        (resolution_types == "INVESTIGATION_OR_FOLLOWUP").sum() / n
    )

    general_support_rate = (
        (resolution_types == "GENERAL_SUPPORT").sum() / n
    )

    refund_rate = (
        (resolution_types == "REFUND_OR_COMPENSATION").sum() / n
    )

    cancellation_rate = (
        (resolution_types == "CANCELLATION").sum() / n
    )

    replacement_rate = (
        (resolution_types == "REPLACEMENT").sum() / n
    )

    return {
        "investigation_rate": float(investigation_rate),
        "general_support_rate": float(general_support_rate),
        "refund_rate": float(refund_rate),
        "cancellation_rate": float(cancellation_rate),
        "replacement_rate": float(replacement_rate),
        "evidence_count": int(n)
    }


print("Runtime evidence feature extractor ready.")

Runtime evidence feature extractor ready.


In [19]:
# =============================================================================
# END-TO-END AGENT ORCHESTRATOR — CELL 4
# Final decision orchestration
# =============================================================================

def run_final_agent_decision(
    customer_query,
    predicted_intent,
    risk_level,
    evidence_df,
    grounding_result=None
):
    """
    Complete deterministic decision layer.

    Pipeline position:

        Router
          ↓
        RAG
          ↓
        Evidence quality
          ↓
        Grounding
          ↓
        Automation policy
    """

    state = create_agent_state(
        customer_query=customer_query,
        predicted_intent=predicted_intent,
        risk_level=risk_level
    )

    # -------------------------------------------------------------------------
    # EVIDENCE FEATURES
    # -------------------------------------------------------------------------

    evidence_features = calculate_runtime_evidence_features(
        evidence_df
    )

    state["investigation_rate"] = evidence_features["investigation_rate"]
    state["general_support_rate"] = evidence_features["general_support_rate"]
    state["evidence_count"] = evidence_features["evidence_count"]
    state["retrieved_evidence"] = evidence_df

    # -------------------------------------------------------------------------
    # TRUST CHECKER
    # -------------------------------------------------------------------------

    try:

        trust_result = trust_check_v3(
            evidence_df=evidence_df,
            predicted_intent=predicted_intent,
            risk_level=risk_level
        )

        state["trust_score"] = trust_result.get("trust_score")
        state["evidence_tier"] = trust_result.get("evidence_tier")
        state["trust_decision"] = trust_result.get("evidence_decision")

    except Exception as e:

        state["trust_score"] = None
        state["evidence_tier"] = "UNKNOWN"
        state["trust_decision"] = "UNKNOWN"

        state["decision_reasons"].append(
            f"Trust checker unavailable: {str(e)}"
        )

    # -------------------------------------------------------------------------
    # GROUNDING GATE
    # -------------------------------------------------------------------------

    if grounding_result is not None:

        grounding_decision = grounding_result.get(
            "grounding_decision",
            "UNKNOWN"
        )

        state["grounding_decision"] = grounding_decision

        state["grounding_violations"] = grounding_result.get(
            "violations",
            []
        )

        state["grounding_warnings"] = grounding_result.get(
            "warnings",
            []
        )

        if grounding_decision == "UNSAFE":

            state["automation_decision"] = "HUMAN"

            state["decision_reasons"].append(
                "Grounding checker marked response UNSAFE"
            )

            state["pipeline_status"] = "COMPLETED"

            return state

    # -------------------------------------------------------------------------
    # FINAL AUTOMATION POLICY
    # -------------------------------------------------------------------------

    policy_result = apply_final_automation_policy(
        risk_level=risk_level,
        investigation_rate=state["investigation_rate"],
        query_words=state["query_words"]
    )

    state["automation_decision"] = policy_result[
        "automation_decision"
    ]

    state["decision_reasons"].extend(
        policy_result["reasons"]
    )

    # -------------------------------------------------------------------------
    # IMPORTANT:
    # TRUST CHECKER IS NOT THE PRIMARY AUTOMATION CLASSIFIER
    # -------------------------------------------------------------------------

    if state["trust_decision"] == "WEAK":

        state["decision_reasons"].append(
            "Evidence quality is WEAK; retained as a monitoring/explanation signal"
        )

    elif state["trust_decision"] == "CONDITIONAL":

        state["decision_reasons"].append(
            "Evidence quality is CONDITIONAL"
        )

    elif state["trust_decision"] == "TRUSTED":

        state["decision_reasons"].append(
            "Evidence quality is TRUSTED"
        )

    state["pipeline_status"] = "COMPLETED"

    return state


print("Final agent decision orchestrator ready.")

Final agent decision orchestrator ready.


In [20]:
# =============================================================================
# FINAL AGENT — SLM ROUTER INTERFACE
# =============================================================================

import re
import json
import numpy as np
import pandas as pd


# =============================================================================
# INTENT TAXONOMY
# =============================================================================

SUPPORTED_INTENTS = [
    "ORDER_DELIVERY",
    "ORDER_TRACKING",
    "PAYMENT_BILLING",
    "RETURNS_REFUNDS",
    "SELLER_AUTHENTICITY",
    "ACCOUNT_ACCESS",
    "ACCOUNT_SECURITY",
    "DEVICE_TECHNICAL",
    "DIGITAL_CONTENT",
    "PURCHASE_CONTROL",
    "ORDER_CANCELLATION",
    "PREORDER_DELIVERY",
    "GENERAL_SUPPORT",
    "GENERAL_SOCIAL",
    "MARKETPLACE_SELLING",
    "ORDER_PURCHASE",
]


# =============================================================================
# ROUTER OUTPUT VALIDATION
# =============================================================================

def validate_router_output(router_output):
    """
    Validate and normalize the router's structured output.
    """

    if not isinstance(router_output, dict):
        raise ValueError("Router output must be a dictionary.")

    intent = str(
        router_output.get("intent", "GENERAL_SUPPORT")
    ).upper().strip()

    risk_level = str(
        router_output.get("risk_level", "MEDIUM")
    ).upper().strip()

    complexity = str(
        router_output.get("complexity", "MEDIUM")
    ).upper().strip()

    query_words = router_output.get("query_words")

    if intent not in SUPPORTED_INTENTS:
        intent = "GENERAL_SUPPORT"

    if risk_level not in {"LOW", "MEDIUM", "HIGH"}:
        risk_level = "MEDIUM"

    if complexity not in {"SIMPLE", "MEDIUM", "COMPLEX"}:
        complexity = "MEDIUM"

    if query_words is None:
        query_words = 0

    try:
        query_words = int(query_words)
    except Exception:
        query_words = 0

    return {
        "intent": intent,
        "risk_level": risk_level,
        "complexity": complexity,
        "query_words": query_words,
    }


# =============================================================================
# QUERY COMPLEXITY
# =============================================================================

def calculate_query_complexity(customer_query):
    """
    Deterministic complexity signal.

    This is deliberately separate from the LLM so the automation
    policy has an auditable query-length feature.
    """

    text = str(customer_query).strip()

    words = len(text.split())
    chars = len(text)

    question_count = text.count("?")

    # Conservative complexity indicators
    has_multiple_questions = question_count >= 2
    has_long_query = words > 25
    has_long_text = chars > 160

    if has_multiple_questions or has_long_query or has_long_text:
        complexity = "COMPLEX"

    elif words > 12 or question_count == 1:
        complexity = "MEDIUM"

    else:
        complexity = "SIMPLE"

    return {
        "complexity": complexity,
        "query_words": words,
        "query_chars": chars,
    }


# =============================================================================
# SAFE ROUTER FALLBACK
# =============================================================================

def safe_router_fallback(customer_query):
    """
    Fail-safe router behavior.

    If the SLM fails, the system does NOT attempt autonomous handling.
    It falls back to a conservative routing decision.
    """

    complexity_info = calculate_query_complexity(
        customer_query
    )

    return {
        "intent": "GENERAL_SUPPORT",
        "risk_level": "MEDIUM",
        "complexity": complexity_info["complexity"],
        "query_words": complexity_info["query_words"],
        "query_chars": complexity_info["query_chars"],
        "router_status": "FALLBACK",
        "router_reason": "Router unavailable; conservative human-review path",
    }


# =============================================================================
# ROUTER OUTPUT BUILDER
# =============================================================================

def build_router_result(
    customer_query,
    predicted_intent,
    predicted_risk
):
    """
    Build a normalized router result from the intent/risk models.
    """

    complexity_info = calculate_query_complexity(
        customer_query
    )

    router_output = {
        "intent": predicted_intent,
        "risk_level": predicted_risk,
        "complexity": complexity_info["complexity"],
        "query_words": complexity_info["query_words"],
        "query_chars": complexity_info["query_chars"],
    }

    router_output = validate_router_output(
        router_output
    )

    router_output["query_chars"] = complexity_info["query_chars"]
    router_output["router_status"] = "SUCCESS"

    return router_output


print("=" * 80)
print("SLM ROUTER INTERFACE READY")
print("=" * 80)

print(f"Supported intents : {len(SUPPORTED_INTENTS)}")
print("Risk levels       : LOW / MEDIUM / HIGH")
print("Complexity        : SIMPLE / MEDIUM / COMPLEX")

print("\nFallback policy:")
print("Router failure -> GENERAL_SUPPORT + MEDIUM risk -> HUMAN")

SLM ROUTER INTERFACE READY
Supported intents : 16
Risk levels       : LOW / MEDIUM / HIGH
Complexity        : SIMPLE / MEDIUM / COMPLEX

Fallback policy:
Router failure -> GENERAL_SUPPORT + MEDIUM risk -> HUMAN


In [21]:
# =============================================================================
# FINAL AGENT — RESTORE INTENT CLASSIFIER TRAINING DATA
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

# -----------------------------------------------------------------------------
# BASE DIRECTORY
# -----------------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

print("=" * 80)
print("SEARCHING FOR WEAK-LABEL TRAINING DATA")
print("=" * 80)

# -----------------------------------------------------------------------------
# SEARCH COMMON CHECKPOINT LOCATIONS
# -----------------------------------------------------------------------------

candidate_paths = [
    BASE_DIR / "training" / "weak_labels_v2.csv",
    BASE_DIR / "training" / "weak_labeled_intents_v2.csv",
    BASE_DIR / "checkpoints" / "weak_labels_v2.csv",
    BASE_DIR / "checkpoints" / "weak_labeled_intents_v2.csv",
    BASE_DIR / "evaluation" / "weak_labels_v2.csv",
    BASE_DIR / "weak_labels_v2.csv",
    BASE_DIR / "weak_labeled_intents_v2.csv",
]

found_paths = []

for path in candidate_paths:
    if path.exists():
        found_paths.append(path)

# -----------------------------------------------------------------------------
# ALSO SEARCH RECURSIVELY
# -----------------------------------------------------------------------------

if not found_paths:

    recursive_matches = list(
        BASE_DIR.rglob("*weak*label*.csv")
    )

    found_paths.extend(recursive_matches)

# -----------------------------------------------------------------------------
# REPORT
# -----------------------------------------------------------------------------

if found_paths:

    print("\nFound candidate files:\n")

    for i, path in enumerate(found_paths, start=1):
        print(f"{i}. {path}")

else:

    print("\nNo weak-label CSV found in the checkpoint directory.")

# -----------------------------------------------------------------------------
# IF EXACTLY ONE FILE EXISTS, LOAD IT
# -----------------------------------------------------------------------------

if len(found_paths) == 1:

    WEAK_LABEL_PATH = found_paths[0]

    weak_labels_df = pd.read_csv(
        WEAK_LABEL_PATH
    )

    print("\n" + "=" * 80)
    print("WEAK-LABEL DATA LOADED")
    print("=" * 80)

    print(f"Path  : {WEAK_LABEL_PATH}")
    print(f"Shape : {weak_labels_df.shape}")

    print("\nColumns:")
    print(list(weak_labels_df.columns))

elif len(found_paths) > 1:

    print(
        "\nMultiple candidate files found."
        "\nDo NOT train yet."
        "\nWe need to select the exact V2 dataset."
    )

else:

    print(
        "\nThe classifier cannot be safely rebuilt yet because "
        "the exact V2 weak-label dataset was not found."
    )

SEARCHING FOR WEAK-LABEL TRAINING DATA

No weak-label CSV found in the checkpoint directory.

The classifier cannot be safely rebuilt yet because the exact V2 weak-label dataset was not found.


In [22]:
# =============================================================================
# FIND EXISTING TRAINING / CONVERSATION CHECKPOINTS
# =============================================================================

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

print("=" * 80)
print("AVAILABLE CHECKPOINT FILES")
print("=" * 80)

# Search all CSV / JSON / parquet files
all_files = sorted(
    [
        p for p in BASE_DIR.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".csv", ".json", ".parquet", ".pkl"}
    ]
)

if not all_files:
    print("No checkpoint data files found.")
else:
    for i, path in enumerate(all_files, start=1):
        try:
            size_mb = path.stat().st_size / (1024 ** 2)
        except Exception:
            size_mb = 0

        print(
            f"{i:03d}. "
            f"{path.relative_to(BASE_DIR)} "
            f"({size_mb:.2f} MB)"
        )

print("\n" + "=" * 80)
print("FILES WITH LIKELY TRAINING / CONVERSATION DATA")
print("=" * 80)

keywords = [
    "resolution",
    "conversation",
    "case",
    "intent",
    "training",
    "label",
    "weak",
    "tweet",
    "dataset",
    "amazon",
]

matches = []

for path in all_files:

    name = path.name.lower()

    if any(keyword in name for keyword in keywords):
        matches.append(path)

if matches:

    for i, path in enumerate(matches, start=1):
        print(
            f"{i:03d}. {path.relative_to(BASE_DIR)}"
        )

else:
    print("No likely training/conversation files found.")

print("\n" + "=" * 80)
print("CHECK COMPLETE")
print("=" * 80)

AVAILABLE CHECKPOINT FILES
001. checkpoints/cross_encoder_reranked_results.csv (0.89 MB)
002. checkpoints/final_automation_policy_v1.json (0.00 MB)
003. checkpoints/retrieval_checkpoint.json (0.00 MB)
004. checkpoints/rrf_candidates.csv (1.91 MB)
005. checkpoints/structured_rag_corpus.csv (24.78 MB)
006. evaluation/final_automation_policy_evaluation.csv (0.01 MB)
007. evaluation/final_automation_policy_summary_v1.json (0.00 MB)
008. evaluation/golden_evaluation_set.csv (0.09 MB)
009. evaluation/golden_evaluation_set_frozen.csv (0.09 MB)
010. evaluation/response_generation_batch_01.csv (0.00 MB)
011. evaluation/response_grounding_batch_01.csv (0.00 MB)
012. evaluation/response_grounding_batch_01_v2.csv (0.00 MB)
013. evaluation/retrieval_benchmark.csv (0.00 MB)
014. evaluation/top1_evidence_review.csv (0.08 MB)
015. evaluation/top5_historical_evidence.csv (0.52 MB)
016. evaluation/unlabeled_intent_sample_200.csv (0.05 MB)

FILES WITH LIKELY TRAINING / CONVERSATION DATA
001. evaluation/u

In [23]:
# =============================================================================
# VERIFY STRUCTURED RAG CORPUS FOR INTENT-ROUTER RECONSTRUCTION
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

RAG_PATH = (
    BASE_DIR
    / "checkpoints"
    / "structured_rag_corpus.csv"
)

rag_corpus = pd.read_csv(RAG_PATH)

print("=" * 80)
print("STRUCTURED RAG CORPUS VERIFICATION")
print("=" * 80)

print(f"Shape: {rag_corpus.shape}")

print("\nColumns:")
for col in rag_corpus.columns:
    print(f"  - {col}")

# -----------------------------------------------------------------------------
# BASIC QUALITY CHECK
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COLUMN QUALITY")
print("=" * 80)

for col in rag_corpus.columns:

    null_count = rag_corpus[col].isna().sum()
    unique_count = rag_corpus[col].nunique(dropna=True)

    print(
        f"{col:<20} "
        f"nulls={null_count:<8,} "
        f"unique={unique_count:<8,}"
    )

# -----------------------------------------------------------------------------
# RESOLUTION TYPE DISTRIBUTION
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("RESOLUTION TYPE DISTRIBUTION")
print("=" * 80)

resolution_counts = (
    rag_corpus["resolution_type"]
    .fillna("UNKNOWN")
    .value_counts()
)

print(resolution_counts.to_string())

# -----------------------------------------------------------------------------
# EVIDENCE QUALITY DISTRIBUTION
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("EVIDENCE QUALITY DISTRIBUTION")
print("=" * 80)

if "evidence_quality" in rag_corpus.columns:

    evidence_counts = (
        rag_corpus["evidence_quality"]
        .fillna("UNKNOWN")
        .value_counts()
    )

    print(evidence_counts.to_string())

# -----------------------------------------------------------------------------
# SAMPLE RECORDS
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SAMPLE RAG RECORDS")
print("=" * 80)

display(
    rag_corpus.head(5)
)

# -----------------------------------------------------------------------------
# CHECK GOLDEN-SET CONTAMINATION
# -----------------------------------------------------------------------------

golden_path = (
    BASE_DIR
    / "evaluation"
    / "golden_evaluation_set_frozen.csv"
)

golden_df = pd.read_csv(golden_path)

golden_case_ids = set(
    golden_df["case_id"].astype(str)
)

rag_case_ids = set(
    rag_corpus["case_id"].astype(str)
)

overlap = golden_case_ids.intersection(
    rag_case_ids
)

print("\n" + "=" * 80)
print("GOLDEN SET CONTAMINATION CHECK")
print("=" * 80)

print(f"Golden cases : {len(golden_case_ids):,}")
print(f"RAG cases    : {len(rag_case_ids):,}")
print(f"Overlap      : {len(overlap):,}")

if len(overlap) == 0:
    print("PASS — Golden cases are excluded from the RAG corpus.")
else:
    print("WARNING — Golden cases found inside RAG corpus.")

# -----------------------------------------------------------------------------
# FINAL STATUS
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STATUS")
print("=" * 80)

if (
    len(rag_corpus) == 83218
    and len(overlap) == 0
    and "customer_problem" in rag_corpus.columns
):
    print(
        "PASS — RAG corpus is suitable as the training-case source "
        "for rebuilding the router."
    )
else:
    print(
        "REVIEW REQUIRED — corpus does not match the expected "
        "training-case structure."
    )

STRUCTURED RAG CORPUS VERIFICATION
Shape: (83218, 6)

Columns:
  - case_id
  - customer_problem
  - support_action
  - resolution_type
  - evidence_quality
  - source

COLUMN QUALITY
case_id              nulls=0        unique=83,218  
customer_problem     nulls=4        unique=81,165  
support_action       nulls=0        unique=77,044  
resolution_type      nulls=0        unique=5       
evidence_quality     nulls=0        unique=3       
source               nulls=0        unique=1       

RESOLUTION TYPE DISTRIBUTION
resolution_type
GENERAL_SUPPORT              53440
INVESTIGATION_OR_FOLLOWUP    27908
REFUND_OR_COMPENSATION        1130
CANCELLATION                   462
REPLACEMENT                    278

EVIDENCE QUALITY DISTRIBUTION
evidence_quality
MEDIUM    46455
HIGH      34882
LOW        1881

SAMPLE RAG RECORDS


,case_id,customer_problem,support_action,resolution_type,evidence_quality,source
0,AMZ-000001,[user] what in the world is a balance withheld...,"[USER] Sorry, I'm not quite sure what you're h...",GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
1,AMZ-000002,[user] that's exactly what she did. bought it ...,"[USER] Sorry, we would be unable to change thi...",GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
2,AMZ-000003,[user] [user] i think u should cancel your ord...,"[USER] We would like to help you, Prakhar. Ple...",INVESTIGATION_OR_FOLLOWUP,MEDIUM,AmazonHelp historical support case
3,AMZ-000004,1/ [user] are you guys leaking email ids of us...,[USER] Our customers' security is our utmost i...,GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
4,AMZ-000005,the sound quality of the deer hunter on amazon...,[USER] Let us know if you have any other quest...,GENERAL_SUPPORT,HIGH,AmazonHelp historical support case



GOLDEN SET CONTAMINATION CHECK
Golden cases : 200
RAG cases    : 83,218
Overlap      : 0
PASS — Golden cases are excluded from the RAG corpus.

STATUS
PASS — RAG corpus is suitable as the training-case source for rebuilding the router.


In [24]:
# =============================================================================
# FINAL AGENT — V2 HIGH-CONFIDENCE WEAK LABEL RECONSTRUCTION
# =============================================================================

import re
import pandas as pd
import numpy as np


# =============================================================================
# LOAD TRAINING CORPUS
# =============================================================================

RAG_PATH = (
    BASE_DIR
    / "checkpoints"
    / "structured_rag_corpus.csv"
)

rag_corpus = pd.read_csv(RAG_PATH)

print("=" * 80)
print("V2 WEAK-LABEL RECONSTRUCTION")
print("=" * 80)

print(f"Input cases: {len(rag_corpus):,}")


# =============================================================================
# GOLDEN SET EXCLUSION
# =============================================================================

golden_path = (
    BASE_DIR
    / "evaluation"
    / "golden_evaluation_set_frozen.csv"
)

golden_df = pd.read_csv(golden_path)

golden_case_ids = set(
    golden_df["case_id"].astype(str)
)

rag_corpus["case_id"] = (
    rag_corpus["case_id"].astype(str)
)

training_df = rag_corpus[
    ~rag_corpus["case_id"].isin(golden_case_ids)
].copy()

print(f"After golden exclusion: {len(training_df):,}")


# =============================================================================
# TEXT NORMALIZATION
# =============================================================================

def normalize_text(text):
    """
    Normalize customer problem text for deterministic weak labeling.
    """

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


training_df["text_for_labeling"] = (
    training_df["customer_problem"]
    .apply(normalize_text)
)


# =============================================================================
# V2 HIGH-CONFIDENCE INTENT RULES
# =============================================================================

def assign_v2_weak_intent(text):
    """
    High-confidence weak-label rules.

    GENERAL_SUPPORT and GENERAL_SOCIAL are intentionally NOT assigned
    by weak rules because they were intentionally excluded previously
    to avoid noisy labels.

    Returns:
        intent
        matched_rule
        ambiguous
    """

    text = normalize_text(text)

    if not text:
        return None, None, False


    # -------------------------------------------------------------------------
    # ACCOUNT SECURITY
    # -------------------------------------------------------------------------

    account_security_patterns = [
        r"\bhack(ed|ing)?\b",
        r"\bhacked\b",
        r"\bsecurity\b",
        r"\bunauthorized\b",
        r"\bunauthorised\b",
        r"\baccount.*compromis",
        r"\bcompromis.*account\b",
        r"\bstolen.*account\b",
        r"\bsomeone.*account\b",
        r"\baccount.*stolen\b",
    ]

    if any(re.search(p, text) for p in account_security_patterns):
        return "ACCOUNT_SECURITY", "ACCOUNT_SECURITY", False


    # -------------------------------------------------------------------------
    # SELLER AUTHENTICITY
    # -------------------------------------------------------------------------

    seller_authenticity_patterns = [
        r"\bcounterfeit\b",
        r"\bfake product\b",
        r"\bfake item\b",
        r"\bis this.*real\b",
        r"\bis this.*genuine\b",
        r"\bauthentic\b",
        r"\bnot authentic\b",
        r"\bnot genuine\b",
        r"\bseller.*fake\b",
        r"\bfake.*seller\b",
    ]

    if any(re.search(p, text) for p in seller_authenticity_patterns):
        return "SELLER_AUTHENTICITY", "SELLER_AUTHENTICITY", False


    # -------------------------------------------------------------------------
    # PURCHASE CONTROL
    # -------------------------------------------------------------------------

    purchase_control_patterns = [
        r"\bparental control\b",
        r"\bparental controls\b",
        r"\bpurchase control\b",
        r"\bpurchase controls\b",
        r"\bchild.*purchase\b",
        r"\bchildren.*purchase\b",
        r"\bkids.*purchase\b",
        r"\bprevent.*purchase\b",
    ]

    if any(re.search(p, text) for p in purchase_control_patterns):
        return "PURCHASE_CONTROL", "PURCHASE_CONTROL", False


    # -------------------------------------------------------------------------
    # MARKETPLACE SELLING
    # -------------------------------------------------------------------------

    marketplace_patterns = [
        r"\bsell on amazon\b",
        r"\bselling on amazon\b",
        r"\bamazon seller\b",
        r"\bseller account\b",
        r"\bmarketplace seller\b",
        r"\bmarketplace selling\b",
        r"\blist.*product.*sell\b",
        r"\bsell.*product\b",
    ]

    if any(re.search(p, text) for p in marketplace_patterns):
        return "MARKETPLACE_SELLING", "MARKETPLACE_SELLING", False


    # -------------------------------------------------------------------------
    # PREORDER DELIVERY
    # -------------------------------------------------------------------------

    preorder_patterns = [
        r"\bpre[- ]?order\b",
        r"\bpreordered\b",
        r"\bpre-ordered\b",
        r"\bpre order\b",
        r"\bpreorder\b",
    ]

    if any(re.search(p, text) for p in preorder_patterns):

        delivery_context = [
            r"\bdeliver",
            r"\barriv",
            r"\bship",
            r"\bshipping",
            r"\bwhen.*get",
            r"\bwhen.*receive",
        ]

        if any(re.search(p, text) for p in delivery_context):
            return "PREORDER_DELIVERY", "PREORDER_DELIVERY", False


    # -------------------------------------------------------------------------
    # ORDER CANCELLATION
    # -------------------------------------------------------------------------

    cancellation_patterns = [
        r"\bcancel.*order\b",
        r"\border.*cancel\b",
        r"\bcancel.*purchase\b",
        r"\bwant to cancel\b",
        r"\bplease cancel\b",
        r"\bcancellation.*order\b",
    ]

    if any(re.search(p, text) for p in cancellation_patterns):
        return "ORDER_CANCELLATION", "ORDER_CANCELLATION", False


    # -------------------------------------------------------------------------
    # ORDER PURCHASE
    # -------------------------------------------------------------------------

    purchase_patterns = [
        r"\bhow.*buy\b",
        r"\bhow.*purchase\b",
        r"\bplace.*order\b",
        r"\bplacing.*order\b",
        r"\bmake.*order\b",
        r"\bwant to order\b",
        r"\bwhere.*buy\b",
        r"\bwhere.*purchase\b",
    ]

    if any(re.search(p, text) for p in purchase_patterns):
        return "ORDER_PURCHASE", "ORDER_PURCHASE", False


    # -------------------------------------------------------------------------
    # ACCOUNT ACCESS
    # -------------------------------------------------------------------------

    account_access_patterns = [
        r"\bcan't log in\b",
        r"\bcannot log in\b",
        r"\bcan't login\b",
        r"\bcannot login\b",
        r"\blogin problem\b",
        r"\blog in\b",
        r"\blogin\b",
        r"\bpassword.*reset\b",
        r"\breset.*password\b",
        r"\bforgot.*password\b",
        r"\baccess.*account\b",
        r"\baccount.*access\b",
    ]

    if any(re.search(p, text) for p in account_access_patterns):
        return "ACCOUNT_ACCESS", "ACCOUNT_ACCESS", False


    # -------------------------------------------------------------------------
    # DEVICE / TECHNICAL
    # -------------------------------------------------------------------------

    technical_patterns = [
        r"\bkindle\b",
        r"\bfire tv\b",
        r"\bfirestick\b",
        r"\balexa\b",
        r"\becho\b",
        r"\bdevice\b",
        r"\bapp.*not work\b",
        r"\bapp.*doesn't work\b",
        r"\bapp.*not working\b",
        r"\bwebsite.*not work\b",
        r"\btechnical\b",
        r"\bsoftware.*problem\b",
        r"\bwon't work\b",
        r"\bnot working\b",
    ]

    if any(re.search(p, text) for p in technical_patterns):
        return "DEVICE_TECHNICAL", "DEVICE_TECHNICAL", False


    # -------------------------------------------------------------------------
    # ORDER TRACKING
    # -------------------------------------------------------------------------

    tracking_patterns = [
        r"\btrack.*order\b",
        r"\btracking.*order\b",
        r"\btracking number\b",
        r"\btracking information\b",
        r"\bwhere.*order\b",
        r"\bwhere.*package\b",
        r"\bpackage.*where\b",
        r"\bshipment.*where\b",
        r"\btrack.*package\b",
    ]

    if any(re.search(p, text) for p in tracking_patterns):
        return "ORDER_TRACKING", "ORDER_TRACKING", False


    # -------------------------------------------------------------------------
    # RETURNS / REFUNDS
    # -------------------------------------------------------------------------

    returns_refunds_patterns = [
        r"\brefund\b",
        r"\brefunds\b",
        r"\breturn\b",
        r"\breturns\b",
        r"\breturning\b",
        r"\breturn.*item\b",
        r"\breturn.*product\b",
        r"\bmoney back\b",
        r"\bchargeback\b",
        r"\breimburse\b",
        r"\breimbursement\b",
    ]

    if any(re.search(p, text) for p in returns_refunds_patterns):
        return "RETURNS_REFUNDS", "RETURNS_REFUNDS", False


    # -------------------------------------------------------------------------
    # PAYMENT / BILLING
    # -------------------------------------------------------------------------

    payment_patterns = [
        r"\bpayment\b",
        r"\bpaid\b",
        r"\bpay\b",
        r"\bcharge\b",
        r"\bcharged\b",
        r"\bcredit card\b",
        r"\bdebit card\b",
        r"\bbilling\b",
        r"\bbill\b",
        r"\btransaction\b",
        r"\bpayment method\b",
        r"\bpayment failed\b",
        r"\bcharged twice\b",
        r"\bdouble charge\b",
    ]

    if any(re.search(p, text) for p in payment_patterns):
        return "PAYMENT_BILLING", "PAYMENT_BILLING", False


    # -------------------------------------------------------------------------
    # DIGITAL CONTENT
    # -------------------------------------------------------------------------

    digital_content_patterns = [
        r"\bkindle book\b",
        r"\be[- ]?book\b",
        r"\bebook\b",
        r"\bdigital book\b",
        r"\bdigital content\b",
        r"\bprime video\b",
        r"\bvideo content\b",
        r"\bdigital download\b",
        r"\bdownload.*content\b",
        r"\bmovie.*amazon\b",
        r"\bmusic.*amazon\b",
    ]

    if any(re.search(p, text) for p in digital_content_patterns):
        return "DIGITAL_CONTENT", "DIGITAL_CONTENT", False


    # -------------------------------------------------------------------------
    # ORDER DELIVERY
    # -------------------------------------------------------------------------

    delivery_patterns = [
        r"\bdelivery\b",
        r"\bdelivered\b",
        r"\bdeliver\b",
        r"\barrive\b",
        r"\barrival\b",
        r"\bshipping\b",
        r"\bshipped\b",
        r"\bshipment\b",
        r"\blate\b",
        r"\bdelayed\b",
        r"\bdelay\b",
        r"\bpackage\b",
        r"\border.*arriv",
        r"\border.*deliver",
        r"\bwhen.*arrive\b",
        r"\bwhen.*deliver\b",
    ]

    if any(re.search(p, text) for p in delivery_patterns):
        return "ORDER_DELIVERY", "ORDER_DELIVERY", False


    # -------------------------------------------------------------------------
    # NO HIGH-CONFIDENCE LABEL
    # -------------------------------------------------------------------------

    return None, None, False


# =============================================================================
# APPLY RULES
# =============================================================================

labels = training_df["text_for_labeling"].apply(
    assign_v2_weak_intent
)

training_df["weak_intent"] = labels.apply(
    lambda x: x[0]
)

training_df["matched_rule"] = labels.apply(
    lambda x: x[1]
)

training_df["ambiguous"] = labels.apply(
    lambda x: x[2]
)


# =============================================================================
# STATS
# =============================================================================

labeled_df = training_df[
    training_df["weak_intent"].notna()
].copy()

unlabeled_df = training_df[
    training_df["weak_intent"].isna()
].copy()

ambiguous_count = int(
    training_df["ambiguous"].sum()
)

print("\n" + "=" * 80)
print("RECONSTRUCTION RESULTS")
print("=" * 80)

print(f"Training cases        : {len(training_df):,}")
print(f"Weak-labeled cases    : {len(labeled_df):,}")
print(f"Unlabeled cases       : {len(unlabeled_df):,}")
print(f"Coverage              : {len(labeled_df)/len(training_df):.2%}")
print(f"Ambiguous cases       : {ambiguous_count:,}")


# =============================================================================
# INTENT DISTRIBUTION
# =============================================================================

print("\n" + "=" * 80)
print("RECONSTRUCTED V2 INTENT DISTRIBUTION")
print("=" * 80)

distribution = (
    labeled_df["weak_intent"]
    .value_counts()
)

print(distribution.to_string())


# =============================================================================
# EXPECTED HISTORICAL V2 RESULTS
# =============================================================================

EXPECTED_COUNT = 23536
EXPECTED_COVERAGE = 0.2828

print("\n" + "=" * 80)
print("COMPARISON WITH PREVIOUS V2 CHECKPOINT")
print("=" * 80)

print(f"Expected labeled cases : {EXPECTED_COUNT:,}")
print(f"Reconstructed cases    : {len(labeled_df):,}")

print(
    f"Expected coverage      : {EXPECTED_COVERAGE:.2%}"
)

print(
    f"Reconstructed coverage : "
    f"{len(labeled_df)/len(training_df):.2%}"
)


# =============================================================================
# SAVE ONLY IF THE RESULT MATCHES
# =============================================================================

if len(labeled_df) == EXPECTED_COUNT:

    output_path = (
        BASE_DIR
        / "checkpoints"
        / "weak_labels_v2_reconstructed.csv"
    )

    output_columns = [
        "case_id",
        "customer_problem",
        "weak_intent",
        "matched_rule",
    ]

    labeled_df[output_columns].to_csv(
        output_path,
        index=False
    )

    print("\nPASS — Exact expected label count recovered.")

    print(f"\nSaved to:")
    print(output_path)

else:

    print(
        "\nSTOP — Reconstructed count does not match the "
        "previous V2 result."
    )

    print(
        "The dataset will NOT be saved as the canonical V2 "
        "weak-label dataset."
    )

V2 WEAK-LABEL RECONSTRUCTION
Input cases: 83,218
After golden exclusion: 83,218

RECONSTRUCTION RESULTS
Training cases        : 83,218
Weak-labeled cases    : 27,715
Unlabeled cases       : 55,503
Coverage              : 33.30%
Ambiguous cases       : 0

RECONSTRUCTED V2 INTENT DISTRIBUTION
weak_intent
ORDER_DELIVERY         14676
RETURNS_REFUNDS         3606
PAYMENT_BILLING         3020
DEVICE_TECHNICAL        2739
ORDER_CANCELLATION       814
ORDER_PURCHASE           698
ORDER_TRACKING           682
ACCOUNT_ACCESS           347
ACCOUNT_SECURITY         338
DIGITAL_CONTENT          310
PREORDER_DELIVERY        212
MARKETPLACE_SELLING      175
SELLER_AUTHENTICITY       88
PURCHASE_CONTROL          10

COMPARISON WITH PREVIOUS V2 CHECKPOINT
Expected labeled cases : 23,536
Reconstructed cases    : 27,715
Expected coverage      : 28.28%
Reconstructed coverage : 33.30%

STOP — Reconstructed count does not match the previous V2 result.
The dataset will NOT be saved as the canonical V2 weak-

In [25]:
# =============================================================================
# FINAL AGENT — RISK ROUTER
# =============================================================================

import re
import pandas as pd
import numpy as np


# =============================================================================
# HIGH-RISK SIGNALS
# =============================================================================

HIGH_RISK_PATTERNS = {

    "ACCOUNT_SECURITY": [
        r"\bhacked\b",
        r"\bhack(ed|ing)?\b",
        r"\baccount.*compromis",
        r"\bcompromis.*account",
        r"\bunauthorized\b",
        r"\bunauthorised\b",
        r"\bstolen.*account",
        r"\baccount.*stolen",
        r"\bsomeone.*access.*account",
        r"\bsomeone.*using.*account",
    ],

    "PAYMENT_SECURITY": [
        r"\bcard.*stolen\b",
        r"\bstolen.*card\b",
        r"\bunauthorized.*charge\b",
        r"\bunauthorised.*charge\b",
        r"\bunknown.*charge\b",
        r"\bcharge.*not.*mine\b",
        r"\bpayment.*not.*mine\b",
        r"\bfraud\b",
        r"\bfraudulent\b",
    ],

    "PRIVACY": [
        r"\bpersonal information.*leak",
        r"\bdata.*leak",
        r"\bprivacy.*breach",
        r"\bprivacy breach\b",
        r"\bexposed.*personal",
        r"\bleaked.*information",
    ],
}


# =============================================================================
# MEDIUM-RISK SIGNALS
# =============================================================================

MEDIUM_RISK_PATTERNS = {

    "PAYMENT": [
        r"\bpayment\b",
        r"\bcharged\b",
        r"\bcharge\b",
        r"\bbilling\b",
        r"\bcredit card\b",
        r"\bdebit card\b",
        r"\brefund\b",
        r"\bmoney back\b",
    ],

    "ACCOUNT_ACCESS": [
        r"\bcan't log in\b",
        r"\bcannot log in\b",
        r"\bcan't login\b",
        r"\bcannot login\b",
        r"\bforgot.*password\b",
        r"\breset.*password\b",
        r"\baccount.*access\b",
        r"\baccess.*account\b",
    ],

    "CANCELLATION": [
        r"\bcancel.*order\b",
        r"\border.*cancel\b",
        r"\bcancellation\b",
    ],

    "MARKETPLACE": [
        r"\bseller\b",
        r"\bmarketplace\b",
        r"\bcounterfeit\b",
        r"\bfake product\b",
        r"\bfake item\b",
        r"\bauthentic\b",
    ],
}


# =============================================================================
# RISK CLASSIFIER
# =============================================================================

def classify_risk(customer_query):
    """
    Conservative deterministic risk classifier.

    Design principle:
        When uncertain -> MEDIUM

    HIGH:
        Security, fraud, privacy, unauthorized access signals.

    MEDIUM:
        Financial/account/marketplace/cancellation issues.

    LOW:
        Ordinary informational/support requests.
    """

    text = str(customer_query).lower().strip()

    high_matches = []
    medium_matches = []

    # -------------------------------------------------------------------------
    # HIGH RISK
    # -------------------------------------------------------------------------

    for category, patterns in HIGH_RISK_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                high_matches.append(category)

                break

    if high_matches:

        return {
            "risk_level": "HIGH",
            "risk_categories": sorted(set(high_matches)),
            "risk_reason": (
                "High-risk security, fraud, or privacy signal detected"
            ),
            "risk_status": "RULE_MATCH",
        }

    # -------------------------------------------------------------------------
    # MEDIUM RISK
    # -------------------------------------------------------------------------

    for category, patterns in MEDIUM_RISK_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                medium_matches.append(category)

                break

    if medium_matches:

        return {
            "risk_level": "MEDIUM",
            "risk_categories": sorted(set(medium_matches)),
            "risk_reason": (
                "Sensitive support domain detected; conservative human review"
            ),
            "risk_status": "RULE_MATCH",
        }

    # -------------------------------------------------------------------------
    # LOW RISK
    # -------------------------------------------------------------------------

    return {
        "risk_level": "LOW",
        "risk_categories": [],
        "risk_reason": "No elevated-risk signal detected",
        "risk_status": "DEFAULT_SAFE",
    }


# =============================================================================
# TEST CASES
# =============================================================================

print("=" * 80)
print("RISK ROUTER TEST")
print("=" * 80)

risk_tests = [

    "Where is my package?",

    "My order is delayed",

    "I forgot my password and cannot login",

    "I was charged twice for my order",

    "Someone hacked my account",

    "There is an unauthorized charge on my card",

    "Are you leaking customer personal information?",

    "How can I buy this product?",
]


for query in risk_tests:

    result = classify_risk(query)

    print("\nQuery:")
    print(query)

    print(
        f"Risk       : {result['risk_level']}"
    )

    print(
        f"Categories : {result['risk_categories']}"
    )

    print(
        f"Reason     : {result['risk_reason']}"
    )

RISK ROUTER TEST

Query:
Where is my package?
Risk       : LOW
Categories : []
Reason     : No elevated-risk signal detected

Query:
My order is delayed
Risk       : LOW
Categories : []
Reason     : No elevated-risk signal detected

Query:
I forgot my password and cannot login
Risk       : MEDIUM
Categories : ['ACCOUNT_ACCESS']
Reason     : Sensitive support domain detected; conservative human review

Query:
I was charged twice for my order
Risk       : MEDIUM
Categories : ['PAYMENT']
Reason     : Sensitive support domain detected; conservative human review

Query:
Someone hacked my account
Risk       : HIGH
Categories : ['ACCOUNT_SECURITY']
Reason     : High-risk security, fraud, or privacy signal detected

Query:
There is an unauthorized charge on my card
Risk       : HIGH
Categories : ['ACCOUNT_SECURITY', 'PAYMENT_SECURITY']
Reason     : High-risk security, fraud, or privacy signal detected

Query:
Are you leaking customer personal information?
Risk       : LOW
Categories : []
Reaso

In [26]:
# =============================================================================
# FINAL AGENT — RISK ROUTER V2
# Conservative, fail-closed risk classification
# =============================================================================

import re
import pandas as pd
import numpy as np


# =============================================================================
# HIGH-RISK SIGNALS
# =============================================================================

HIGH_RISK_PATTERNS = {

    # -------------------------------------------------------------------------
    # ACCOUNT SECURITY
    # -------------------------------------------------------------------------

    "ACCOUNT_SECURITY": [
        r"\bhacked\b",
        r"\bhack(ed|ing)?\b",
        r"\baccount.*hack",
        r"\bhack.*account",
        r"\baccount.*compromis",
        r"\bcompromis.*account",
        r"\bunauthorized.*access\b",
        r"\bunauthorised.*access\b",
        r"\bstolen.*account\b",
        r"\baccount.*stolen\b",
        r"\bsomeone.*access.*account\b",
        r"\bsomeone.*using.*account\b",
        r"\bsomeone.*got.*into.*account\b",
    ],

    # -------------------------------------------------------------------------
    # PAYMENT / FRAUD
    # -------------------------------------------------------------------------

    "PAYMENT_SECURITY": [
        r"\bcard.*stolen\b",
        r"\bstolen.*card\b",
        r"\bunauthorized.*charge\b",
        r"\bunauthorised.*charge\b",
        r"\bunknown.*charge\b",
        r"\bcharge.*not.*mine\b",
        r"\bpayment.*not.*mine\b",
        r"\bcard.*not.*mine\b",
        r"\bfraud\b",
        r"\bfraudulent\b",
        r"\bsomeone.*charged.*my.*card\b",
        r"\bsomeone.*used.*my.*card\b",
    ],

    # -------------------------------------------------------------------------
    # PRIVACY / DATA EXPOSURE
    # -------------------------------------------------------------------------

    "PRIVACY": [
        r"\bpersonal information.*leak",
        r"\bleak.*personal information",
        r"\bpersonal data.*leak",
        r"\bleak.*personal data",
        r"\bcustomer data.*leak",
        r"\bleak.*customer data",
        r"\bdata.*leak",
        r"\bleak.*data",
        r"\bprivacy.*breach",
        r"\bprivacy breach\b",
        r"\bdata breach\b",
        r"\bdata.*breach",
        r"\bexposed.*personal",
        r"\bpersonal.*exposed",
        r"\bleaked.*information",
        r"\bleaked.*personal",
        r"\bexpos.*customer.*information",
        r"\bsharing.*personal information",
        r"\bsharing.*personal data",
        r"\bselling.*personal information",
        r"\bselling.*personal data",
        r"\bcollecting.*personal information",
        r"\bcollecting.*personal data",
        r"\bleaking.*personal information",
        r"\bleaking.*personal data",
        r"\bare you.*leak.*personal",
        r"\bare you.*leak.*customer",
        r"\bdo you.*leak.*personal",
        r"\bdo you.*share.*personal",
    ],
}


# =============================================================================
# MEDIUM-RISK SIGNALS
# =============================================================================

MEDIUM_RISK_PATTERNS = {

    # -------------------------------------------------------------------------
    # PAYMENT
    # -------------------------------------------------------------------------

    "PAYMENT": [
        r"\bpayment\b",
        r"\bpaid\b",
        r"\bpay\b",
        r"\bcharged\b",
        r"\bcharge\b",
        r"\bbilling\b",
        r"\bbill\b",
        r"\bcredit card\b",
        r"\bdebit card\b",
        r"\btransaction\b",
        r"\bpayment method\b",
        r"\bpayment failed\b",
        r"\bcharged twice\b",
        r"\bdouble charge\b",
    ],

    # -------------------------------------------------------------------------
    # ACCOUNT ACCESS
    # -------------------------------------------------------------------------

    "ACCOUNT_ACCESS": [
        r"\bcan't log in\b",
        r"\bcannot log in\b",
        r"\bcan't login\b",
        r"\bcannot login\b",
        r"\blogin problem\b",
        r"\blogin issue\b",
        r"\blog in\b",
        r"\blogin\b",
        r"\bpassword.*reset\b",
        r"\breset.*password\b",
        r"\bforgot.*password\b",
        r"\baccount.*access\b",
        r"\baccess.*account\b",
    ],

    # -------------------------------------------------------------------------
    # CANCELLATION
    # -------------------------------------------------------------------------

    "CANCELLATION": [
        r"\bcancel.*order\b",
        r"\border.*cancel\b",
        r"\bcancel.*purchase\b",
        r"\bwant to cancel\b",
        r"\bplease cancel\b",
        r"\bcancellation.*order\b",
    ],

    # -------------------------------------------------------------------------
    # MARKETPLACE / SELLER
    # -------------------------------------------------------------------------

    "MARKETPLACE": [
        r"\bseller\b",
        r"\bmarketplace\b",
        r"\bcounterfeit\b",
        r"\bfake product\b",
        r"\bfake item\b",
        r"\bauthentic\b",
        r"\bgenuine seller\b",
    ],
}


# =============================================================================
# RISK CLASSIFIER
# =============================================================================

def classify_risk(customer_query):
    """
    Conservative risk classifier.

    Priority:

        HIGH
          ↓
        MEDIUM
          ↓
        LOW

    Unknown/unclear cases default to MEDIUM.

    The classifier is intentionally independent from response generation.
    """

    text = str(customer_query).lower().strip()

    high_matches = []
    medium_matches = []

    # -------------------------------------------------------------------------
    # HIGH-RISK SCAN
    # -------------------------------------------------------------------------

    for category, patterns in HIGH_RISK_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                high_matches.append(category)

                break

    if high_matches:

        return {
            "risk_level": "HIGH",
            "risk_categories": sorted(set(high_matches)),
            "risk_reason": (
                "High-risk security, fraud, or privacy signal detected"
            ),
            "risk_status": "RULE_MATCH",
        }

    # -------------------------------------------------------------------------
    # MEDIUM-RISK SCAN
    # -------------------------------------------------------------------------

    for category, patterns in MEDIUM_RISK_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                medium_matches.append(category)

                break

    if medium_matches:

        return {
            "risk_level": "MEDIUM",
            "risk_categories": sorted(set(medium_matches)),
            "risk_reason": (
                "Sensitive support domain detected; conservative human review"
            ),
            "risk_status": "RULE_MATCH",
        }

    # -------------------------------------------------------------------------
    # LOW-RISK DEFAULT
    # -------------------------------------------------------------------------

    return {
        "risk_level": "LOW",
        "risk_categories": [],
        "risk_reason": "No elevated-risk signal detected",
        "risk_status": "DEFAULT_SAFE",
    }


# =============================================================================
# ROUTER FAIL-SAFE
# =============================================================================

def safe_risk_router(customer_query):
    """
    Fail-closed risk routing.

    If anything unexpected happens, MEDIUM is returned.
    The downstream automation policy therefore routes to HUMAN.
    """

    try:

        result = classify_risk(customer_query)

        return result

    except Exception as exc:

        return {
            "risk_level": "MEDIUM",
            "risk_categories": ["ROUTER_FAILURE"],
            "risk_reason": (
                "Risk classifier failed; conservative human review required"
            ),
            "risk_status": "FAIL_SAFE",
            "error": str(exc),
        }


# =============================================================================
# TEST SUITE
# =============================================================================

print("=" * 80)
print("RISK ROUTER V2 TEST")
print("=" * 80)

risk_tests = [

    # LOW
    (
        "Where is my package?",
        "LOW"
    ),

    (
        "My order is delayed",
        "LOW"
    ),

    (
        "How can I buy this product?",
        "LOW"
    ),

    # MEDIUM
    (
        "I forgot my password and cannot login",
        "MEDIUM"
    ),

    (
        "I was charged twice for my order",
        "MEDIUM"
    ),

    (
        "I want to cancel my order",
        "MEDIUM"
    ),

    # HIGH
    (
        "Someone hacked my account",
        "HIGH"
    ),

    (
        "There is an unauthorized charge on my card",
        "HIGH"
    ),

    (
        "Are you leaking customer personal information?",
        "HIGH"
    ),

    (
        "Has my personal data been exposed?",
        "HIGH"
    ),
]


results = []

for query, expected in risk_tests:

    result = safe_risk_router(query)

    actual = result["risk_level"]

    passed = actual == expected

    results.append({
        "query": query,
        "expected": expected,
        "actual": actual,
        "categories": ", ".join(result["risk_categories"]),
        "passed": passed,
    })

    print("\nQuery:")
    print(query)

    print(f"Expected   : {expected}")
    print(f"Predicted  : {actual}")
    print(f"Categories : {result['risk_categories']}")
    print(f"Status     : {'PASS' if passed else 'FAIL'}")


# =============================================================================
# SUMMARY
# =============================================================================

results_df = pd.DataFrame(results)

passed_count = int(
    results_df["passed"].sum()
)

total_count = len(results_df)

print("\n" + "=" * 80)
print("RISK ROUTER V2 SUMMARY")
print("=" * 80)

print(
    f"Passed: {passed_count}/{total_count}"
)

print(
    f"Accuracy: {passed_count / total_count:.2%}"
)

if passed_count == total_count:

    print(
        "\nPASS — Risk router safety test completed successfully."
    )

else:

    print(
        "\nWARNING — Risk router requires additional review."
    )

RISK ROUTER V2 TEST

Query:
Where is my package?
Expected   : LOW
Predicted  : LOW
Categories : []
Status     : PASS

Query:
My order is delayed
Expected   : LOW
Predicted  : LOW
Categories : []
Status     : PASS

Query:
How can I buy this product?
Expected   : LOW
Predicted  : LOW
Categories : []
Status     : PASS

Query:
I forgot my password and cannot login
Expected   : MEDIUM
Predicted  : MEDIUM
Categories : ['ACCOUNT_ACCESS']
Status     : PASS

Query:
I was charged twice for my order
Expected   : MEDIUM
Predicted  : MEDIUM
Categories : ['PAYMENT']
Status     : PASS

Query:
I want to cancel my order
Expected   : MEDIUM
Predicted  : MEDIUM
Categories : ['CANCELLATION']
Status     : PASS

Query:
Someone hacked my account
Expected   : HIGH
Predicted  : HIGH
Categories : ['ACCOUNT_SECURITY']
Status     : PASS

Query:
There is an unauthorized charge on my card
Expected   : HIGH
Predicted  : HIGH
Categories : ['PAYMENT_SECURITY']
Status     : PASS

Query:
Are you leaking customer persona

In [27]:
# =============================================================================
# FINAL AGENT — UNIFIED ROUTER V1
# Intent + Risk + Complexity
# =============================================================================

import pandas as pd
import numpy as np


# =============================================================================
# UNIFIED ROUTER
# =============================================================================

def run_unified_router(
    customer_query,
    predicted_intent=None,
    predicted_risk=None
):
    """
    Unified router contract.

    Components:

        Intent
        Risk
        Complexity

    Safety principle:

        Any router failure
            ->
        MEDIUM risk
            ->
        downstream HUMAN review
    """

    query = str(customer_query).strip()

    # -------------------------------------------------------------------------
    # EMPTY QUERY
    # -------------------------------------------------------------------------

    if not query:

        return {
            "customer_query": query,
            "intent": "GENERAL_SUPPORT",
            "risk_level": "MEDIUM",
            "complexity": "COMPLEX",
            "query_words": 0,
            "query_chars": 0,
            "risk_categories": ["INVALID_INPUT"],
            "router_status": "FAIL_SAFE",
            "router_reason": "Empty customer query",
        }

    # -------------------------------------------------------------------------
    # COMPLEXITY
    # -------------------------------------------------------------------------

    try:

        complexity_result = calculate_query_complexity(
            query
        )

    except Exception:

        complexity_result = {
            "complexity": "COMPLEX",
            "query_words": len(query.split()),
            "query_chars": len(query),
        }

    # -------------------------------------------------------------------------
    # RISK
    # -------------------------------------------------------------------------

    try:

        if predicted_risk is None:

            risk_result = safe_risk_router(
                query
            )

        else:

            risk_result = {
                "risk_level": str(
                    predicted_risk
                ).upper(),
                "risk_categories": [],
                "risk_reason": "Risk supplied by upstream router",
                "risk_status": "UPSTREAM",
            }

    except Exception as exc:

        risk_result = {
            "risk_level": "MEDIUM",
            "risk_categories": ["ROUTER_FAILURE"],
            "risk_reason": "Risk classification failed",
            "risk_status": "FAIL_SAFE",
            "error": str(exc),
        }

    # -------------------------------------------------------------------------
    # INTENT
    # -------------------------------------------------------------------------

    if predicted_intent is None:

        intent = "GENERAL_SUPPORT"
        intent_status = "FALLBACK"

    else:

        intent = str(
            predicted_intent
        ).upper().strip()

        if intent not in SUPPORTED_INTENTS:

            intent = "GENERAL_SUPPORT"
            intent_status = "FALLBACK_INVALID_INTENT"

        else:

            intent_status = "UPSTREAM"

    # -------------------------------------------------------------------------
    # FINAL NORMALIZATION
    # -------------------------------------------------------------------------

    risk_level = str(
        risk_result.get(
            "risk_level",
            "MEDIUM"
        )
    ).upper()

    if risk_level not in {
        "LOW",
        "MEDIUM",
        "HIGH"
    }:

        risk_level = "MEDIUM"

        risk_result["risk_categories"] = (
            risk_result.get(
                "risk_categories",
                []
            )
            + ["INVALID_RISK"]
        )

    # -------------------------------------------------------------------------
    # ROUTER STATUS
    # -------------------------------------------------------------------------

    if (
        risk_result.get("risk_status") == "FAIL_SAFE"
        or intent_status.startswith("FALLBACK")
    ):

        router_status = "FAIL_SAFE"

    else:

        router_status = "SUCCESS"

    # -------------------------------------------------------------------------
    # BUILD RESULT
    # -------------------------------------------------------------------------

    result = {

        "customer_query": query,

        # Intent
        "intent": intent,
        "intent_status": intent_status,

        # Risk
        "risk_level": risk_level,
        "risk_categories": risk_result.get(
            "risk_categories",
            []
        ),
        "risk_reason": risk_result.get(
            "risk_reason",
            ""
        ),

        # Complexity
        "complexity": complexity_result[
            "complexity"
        ],
        "query_words": complexity_result[
            "query_words"
        ],
        "query_chars": complexity_result[
            "query_chars"
        ],

        # Router metadata
        "router_status": router_status,

        # Human-readable reasoning
        "router_reason": (
            risk_result.get(
                "risk_reason",
                ""
            )
        ),
    }

    return result


# =============================================================================
# ROUTER TEST CASES
# =============================================================================

print("=" * 80)
print("UNIFIED ROUTER V1 TEST")
print("=" * 80)


router_tests = [

    {
        "query": "Where is my package?",
        "intent": "ORDER_TRACKING",
    },

    {
        "query": "My order is delayed and I need it tomorrow",
        "intent": "ORDER_DELIVERY",
    },

    {
        "query": "I forgot my password and cannot login",
        "intent": "ACCOUNT_ACCESS",
    },

    {
        "query": "Someone hacked my account",
        "intent": "ACCOUNT_SECURITY",
    },

    {
        "query": "There is an unauthorized charge on my card",
        "intent": "PAYMENT_BILLING",
    },

    {
        "query": "I want to cancel my order",
        "intent": "ORDER_CANCELLATION",
    },

    {
        "query": "How can I buy this product?",
        "intent": "ORDER_PURCHASE",
    },

    {
        "query": "Are you leaking customer personal information?",
        "intent": "GENERAL_SUPPORT",
    },
]


router_results = []


for test in router_tests:

    query = test["query"]
    intent = test["intent"]

    result = run_unified_router(
        customer_query=query,
        predicted_intent=intent
    )

    router_results.append(result)

    print("\n" + "-" * 80)

    print(f"Query      : {query}")
    print(f"Intent     : {result['intent']}")
    print(f"Risk       : {result['risk_level']}")
    print(f"Complexity : {result['complexity']}")
    print(f"Words      : {result['query_words']}")
    print(f"Status     : {result['router_status']}")


# =============================================================================
# FAIL-SAFE TEST
# =============================================================================

print("\n" + "=" * 80)
print("FAIL-SAFE TEST")
print("=" * 80)

fallback_result = run_unified_router(
    customer_query="Someone hacked my account",
    predicted_intent="INVALID_INTENT",
    predicted_risk="INVALID_RISK"
)

print(
    f"Intent     : {fallback_result['intent']}"
)

print(
    f"Risk       : {fallback_result['risk_level']}"
)

print(
    f"Status     : {fallback_result['router_status']}"
)

print(
    f"Categories : {fallback_result['risk_categories']}"
)


# =============================================================================
# ASSERTIONS
# =============================================================================

print("\n" + "=" * 80)
print("ROUTER ASSERTIONS")
print("=" * 80)

assert fallback_result["intent"] == "GENERAL_SUPPORT"
assert fallback_result["risk_level"] == "MEDIUM"
assert fallback_result["router_status"] == "FAIL_SAFE"

print("PASS — Invalid intent falls back to GENERAL_SUPPORT.")
print("PASS — Invalid risk falls back to MEDIUM.")
print("PASS — Fail-safe router state is activated.")

print("\n" + "=" * 80)
print("UNIFIED ROUTER V1 READY")
print("=" * 80)

UNIFIED ROUTER V1 TEST

--------------------------------------------------------------------------------
Query      : Where is my package?
Intent     : ORDER_TRACKING
Risk       : LOW
Complexity : MEDIUM
Words      : 4
Status     : SUCCESS

--------------------------------------------------------------------------------
Query      : My order is delayed and I need it tomorrow
Intent     : ORDER_DELIVERY
Risk       : LOW
Complexity : SIMPLE
Words      : 9
Status     : SUCCESS

--------------------------------------------------------------------------------
Query      : I forgot my password and cannot login
Intent     : ACCOUNT_ACCESS
Risk       : MEDIUM
Complexity : SIMPLE
Words      : 7
Status     : SUCCESS

--------------------------------------------------------------------------------
Query      : Someone hacked my account
Intent     : ACCOUNT_SECURITY
Risk       : HIGH
Complexity : SIMPLE
Words      : 4
Status     : SUCCESS

----------------------------------------------------------

In [28]:
# =============================================================================
# FINAL AGENT — RUNTIME HYBRID RAG EVIDENCE RETRIEVER
# =============================================================================

import pandas as pd
import numpy as np


# =============================================================================
# LOAD RUNTIME ARTIFACTS
# =============================================================================

RERANKED_PATH = (
    BASE_DIR
    / "checkpoints"
    / "cross_encoder_reranked_results.csv"
)

RAG_CORPUS_PATH = (
    BASE_DIR
    / "checkpoints"
    / "structured_rag_corpus.csv"
)


runtime_reranked_df = pd.read_csv(
    RERANKED_PATH
)

runtime_rag_corpus = pd.read_csv(
    RAG_CORPUS_PATH
)


# =============================================================================
# VALIDATE SCHEMAS
# =============================================================================

REQUIRED_RERANKED_COLUMNS = [
    "golden_id",
    "case_id",
    "ce_rank",
    "ce_score",
    "rrf_rank",
    "rrf_score",
    "bm25_rank",
    "bge_rank",
    "historical_weak_intent",
]

REQUIRED_RAG_COLUMNS = [
    "case_id",
    "customer_problem",
    "support_action",
    "resolution_type",
    "evidence_quality",
    "source",
]


missing_reranked = [
    col
    for col in REQUIRED_RERANKED_COLUMNS
    if col not in runtime_reranked_df.columns
]

missing_rag = [
    col
    for col in REQUIRED_RAG_COLUMNS
    if col not in runtime_rag_corpus.columns
]


assert not missing_reranked, (
    f"Missing reranked columns: {missing_reranked}"
)

assert not missing_rag, (
    f"Missing RAG columns: {missing_rag}"
)


# =============================================================================
# BUILD RUNTIME-SAFE RETRIEVAL TABLE
# =============================================================================

runtime_reranked_df["case_id"] = (
    runtime_reranked_df["case_id"]
    .astype(str)
)

runtime_rag_corpus["case_id"] = (
    runtime_rag_corpus["case_id"]
    .astype(str)
)


runtime_evidence_df = runtime_reranked_df.merge(
    runtime_rag_corpus,
    on="case_id",
    how="left",
    validate="many_to_one"
)


# =============================================================================
# RUNTIME EVIDENCE RETRIEVER
# =============================================================================

def retrieve_runtime_evidence(
    golden_id,
    top_k=5
):
    """
    Retrieve runtime-safe historical evidence for a benchmark case.

    NOTE:
    The current saved cross-encoder artifact was generated for the
    frozen 200-case benchmark. Therefore golden_id is used here only
    to exercise the existing end-to-end benchmark pipeline.

    Evaluation-only columns are deliberately removed from the
    returned evidence.
    """

    if golden_id is None:

        return pd.DataFrame()

    golden_id = str(golden_id)

    matches = runtime_evidence_df[
        runtime_evidence_df["golden_id"].astype(str)
        == golden_id
    ].copy()

    if matches.empty:

        return pd.DataFrame()

    # -------------------------------------------------------------------------
    # SORT BY CROSS-ENCODER RANK
    # -------------------------------------------------------------------------

    matches = matches.sort_values(
        by="ce_rank",
        ascending=True
    )

    matches = matches.head(top_k).copy()

    # -------------------------------------------------------------------------
    # RUNTIME-SAFE SCHEMA
    # -------------------------------------------------------------------------

    output = pd.DataFrame({

        "case_id":
            matches["case_id"].astype(str),

        "rank":
            matches["ce_rank"].astype(int),

        "cross_encoder_score":
            matches["ce_score"].astype(float),

        "customer_problem":
            matches["customer_problem"].fillna("").astype(str),

        "support_action":
            matches["support_action"].fillna("").astype(str),

        "resolution_type":
            matches["resolution_type"].fillna("UNKNOWN").astype(str),

        "evidence_quality":
            matches["evidence_quality"].fillna("UNKNOWN").astype(str),

        "historical_intent":
            matches["historical_weak_intent"]
            .fillna("UNKNOWN")
            .astype(str),

    })

    return output.reset_index(drop=True)


# =============================================================================
# TEST ON GOLDEN CASE
# =============================================================================

print("=" * 80)
print("RUNTIME HYBRID RAG RETRIEVER TEST")
print("=" * 80)


test_golden_id = "GOLD-0001"

test_evidence = retrieve_runtime_evidence(
    golden_id=test_golden_id,
    top_k=5
)


print(
    f"\nGolden ID: {test_golden_id}"
)

print(
    f"Retrieved evidence: {len(test_evidence)}"
)


if not test_evidence.empty:

    print("\nRuntime-safe columns:")

    for column in test_evidence.columns:

        print(f"  - {column}")

    print("\nTop-5 evidence:")

    display(test_evidence)

else:

    print(
        "\nWARNING — No evidence retrieved."
    )


# =============================================================================
# SECURITY CHECK
# =============================================================================

FORBIDDEN_RUNTIME_COLUMNS = {
    "golden_intent",
    "intent_match",
}


leaked_columns = (
    FORBIDDEN_RUNTIME_COLUMNS
    .intersection(
        set(test_evidence.columns)
    )
)


print("\n" + "=" * 80)
print("RUNTIME DATA LEAKAGE CHECK")
print("=" * 80)


if not leaked_columns:

    print(
        "PASS — Evaluation-only columns are not exposed "
        "to the runtime evidence layer."
    )

else:

    print(
        f"FAIL — Evaluation-only columns detected: "
        f"{leaked_columns}"
    )


# =============================================================================
# FINAL STATUS
# =============================================================================

assert not leaked_columns

print("\n" + "=" * 80)
print("RUNTIME EVIDENCE RETRIEVER READY")
print("=" * 80)

RUNTIME HYBRID RAG RETRIEVER TEST

Golden ID: GOLD-0001
Retrieved evidence: 5

Runtime-safe columns:
  - case_id
  - rank
  - cross_encoder_score
  - customer_problem
  - support_action
  - resolution_type
  - evidence_quality
  - historical_intent

Top-5 evidence:


,case_id,rank,cross_encoder_score,customer_problem,support_action,resolution_type,evidence_quality,historical_intent
0,AMZ-073280,1,0.615286,[user] will someone please contact mw urgently...,[USER] Oh no! Sorry to hear this! Without prov...,INVESTIGATION_OR_FOLLOWUP,HIGH,UNKNOWN
1,AMZ-057467,2,0.249812,"[user] terrible customer service, charges me o...",[USER] I'm sorry for the poor experience. With...,INVESTIGATION_OR_FOLLOWUP,HIGH,PAYMENT_BILLING
2,AMZ-058158,3,0.167556,[user] help! my order is messed up and your cu...,[USER] We'd like to help! Without providing pe...,INVESTIGATION_OR_FOLLOWUP,HIGH,UNKNOWN
3,AMZ-000377,4,0.125092,[user] can someone at [user] [user] please hel...,[USER] I'm very sorry to hear about the troubl...,INVESTIGATION_OR_FOLLOWUP,MEDIUM,UNKNOWN
4,AMZ-044915,5,0.115635,[user] i issues with placing an order or wrong...,[USER] We'd like to help. Kindly get in touch ...,GENERAL_SUPPORT,MEDIUM,ORDER_PURCHASE



RUNTIME DATA LEAKAGE CHECK
PASS — Evaluation-only columns are not exposed to the runtime evidence layer.

RUNTIME EVIDENCE RETRIEVER READY


In [29]:
# =============================================================================
# FINAL AGENT — END-TO-END PIPELINE V1
# =============================================================================

import pandas as pd
import numpy as np
import json


# =============================================================================
# LOAD EXISTING GENERATED RESPONSES
# =============================================================================

GENERATION_PATH = (
    BASE_DIR
    / "evaluation"
    / "response_generation_batch_01.csv"
)

GROUNDING_PATH = (
    BASE_DIR
    / "evaluation"
    / "response_grounding_batch_01_v2.csv"
)

generation_batch_df = pd.read_csv(
    GENERATION_PATH
)

grounding_batch_df = pd.read_csv(
    GROUNDING_PATH
)


print("=" * 80)
print("END-TO-END AGENT V1")
print("=" * 80)

print(
    f"Existing generated responses : "
    f"{len(generation_batch_df)}"
)

print(
    f"Existing grounding results   : "
    f"{len(grounding_batch_df)}"
)


# =============================================================================
# BUILD GROUNDING LOOKUP
# =============================================================================

grounding_lookup = (
    grounding_batch_df
    .set_index("golden_id")
    .to_dict("index")
)


generation_lookup = (
    generation_batch_df
    .set_index("golden_id")
    .to_dict("index")
)


# =============================================================================
# END-TO-END PIPELINE
# =============================================================================

def run_end_to_end_agent(
    golden_id,
    predicted_intent
):
    """
    Execute the complete current agent pipeline.

    Current validation mode:

        1. Query comes from frozen golden case.
        2. Intent is supplied to the router.
        3. Risk is classified independently.
        4. Existing benchmark retrieval artifact supplies evidence.
        5. Trust Checker evaluates evidence quality.
        6. Existing generated response is loaded.
        7. Existing grounding result is loaded.
        8. Final automation policy determines AUTO/HUMAN.

    This avoids additional Gemini API calls.
    """

    golden_id = str(golden_id)

    # -------------------------------------------------------------------------
    # GOLDEN CASE
    # -------------------------------------------------------------------------

    case_rows = golden_df[
        golden_df["golden_id"].astype(str)
        == golden_id
    ]

    if case_rows.empty:

        raise ValueError(
            f"Golden case not found: {golden_id}"
        )

    case = case_rows.iloc[0]

    customer_query = str(
        case["customer_query"]
    )

    # -------------------------------------------------------------------------
    # ROUTER
    # -------------------------------------------------------------------------

    router_result = run_unified_router(
        customer_query=customer_query,
        predicted_intent=predicted_intent
    )

    # -------------------------------------------------------------------------
    # RETRIEVAL
    # -------------------------------------------------------------------------

    evidence_df = retrieve_runtime_evidence(
        golden_id=golden_id,
        top_k=5
    )

    # -------------------------------------------------------------------------
    # EVIDENCE FEATURES
    # -------------------------------------------------------------------------

    evidence_features = (
        calculate_runtime_evidence_features(
            evidence_df
        )
    )

    # -------------------------------------------------------------------------
    # TRUST CHECKER
    # -------------------------------------------------------------------------

    trust_result = trust_check_v3(
        evidence_df=evidence_df,
        predicted_intent=router_result["intent"],
        risk_level=router_result["risk_level"]
    )

    # -------------------------------------------------------------------------
    # EXISTING RESPONSE
    # -------------------------------------------------------------------------

    generated_response = None

    if golden_id in generation_lookup:

        generated_response = generation_lookup[
            golden_id
        ].get(
            "generated_response",
            None
        )

    # -------------------------------------------------------------------------
    # EXISTING GROUNDING RESULT
    # -------------------------------------------------------------------------

    grounding_result = None

    if golden_id in grounding_lookup:

        grounding_result = grounding_lookup[
            golden_id
        ]

    # -------------------------------------------------------------------------
    # FINAL AUTOMATION POLICY
    # -------------------------------------------------------------------------

    final_policy_result = (
        apply_final_automation_policy(
            risk_level=router_result["risk_level"],
            investigation_rate=evidence_features[
                "investigation_rate"
            ],
            query_words=router_result[
                "query_words"
            ]
        )
    )

    final_decision = (
        final_policy_result[
            "automation_decision"
        ]
    )

    final_reasons = list(
        final_policy_result["reasons"]
    )

    # -------------------------------------------------------------------------
    # GROUNDING OVERRIDE
    # -------------------------------------------------------------------------

    grounding_decision = None

    if grounding_result is not None:

        grounding_decision = grounding_result.get(
            "grounding_decision",
            None
        )

        if grounding_decision == "UNSAFE":

            final_decision = "HUMAN"

            final_reasons.append(
                "Grounding checker override: UNSAFE response"
            )

    # -------------------------------------------------------------------------
    # BUILD RESULT
    # -------------------------------------------------------------------------

    return {

        "golden_id": golden_id,

        "customer_query": customer_query,

        # Router
        "intent": router_result["intent"],
        "risk_level": router_result["risk_level"],
        "complexity": router_result["complexity"],
        "query_words": router_result["query_words"],

        # Retrieval
        "evidence_count": len(evidence_df),
        "top1_ce_score": (
            float(
                evidence_df.iloc[0][
                    "cross_encoder_score"
                ]
            )
            if not evidence_df.empty
            else 0.0
        ),

        # Historical behavior
        "investigation_rate": evidence_features[
            "investigation_rate"
        ],

        "general_support_rate": evidence_features[
            "general_support_rate"
        ],

        # Trust
        "trust_score": trust_result.get(
            "trust_score"
        ),

        "evidence_tier": trust_result.get(
            "evidence_tier"
        ),

        "trust_decision": trust_result.get(
            "evidence_decision"
        ),

        # Generation
        "generated_response": generated_response,

        # Grounding
        "grounding_decision": grounding_decision,

        # Final decision
        "automation_decision": final_decision,

        "decision_reasons": " | ".join(
            final_reasons
        ),
    }


# =============================================================================
# RUN ON EXISTING 7 GENERATED CASES
# =============================================================================

test_ids = (
    generation_batch_df[
        "golden_id"
    ]
    .astype(str)
    .tolist()
)


# =============================================================================
# INTENT SOURCE
# =============================================================================
#
# For this integration test, use the frozen manually annotated intent.
#
# IMPORTANT:
# This is ONLY an integration-test convenience.
# It is NOT used as a runtime feature.
#
# The final production router will provide the intent.
# =============================================================================

golden_intent_lookup = (
    golden_df
    .set_index("golden_id")["intent"]
    .to_dict()
)


results = []


for golden_id in test_ids:

    predicted_intent = golden_intent_lookup[
        golden_id
    ]

    result = run_end_to_end_agent(
        golden_id=golden_id,
        predicted_intent=predicted_intent
    )

    results.append(result)


e2e_df = pd.DataFrame(results)


# =============================================================================
# DISPLAY RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("END-TO-END RESULTS")
print("=" * 80)


display(
    e2e_df[
        [
            "golden_id",
            "intent",
            "risk_level",
            "complexity",
            "query_words",
            "top1_ce_score",
            "investigation_rate",
            "trust_score",
            "evidence_tier",
            "trust_decision",
            "grounding_decision",
            "automation_decision",
        ]
    ]
)


# =============================================================================
# RESPONSE SAFETY SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("GROUNDING SUMMARY")
print("=" * 80)

print(
    e2e_df[
        "grounding_decision"
    ].value_counts(dropna=False).to_string()
)


# =============================================================================
# FINAL AUTOMATION SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("AUTOMATION SUMMARY")
print("=" * 80)

automation_counts = (
    e2e_df[
        "automation_decision"
    ]
    .value_counts()
)

print(
    automation_counts.to_string()
)


# =============================================================================
# RISK SAFETY CHECK
# =============================================================================

high_auto = int(
    (
        (e2e_df["risk_level"] == "HIGH")
        &
        (e2e_df["automation_decision"] == "AUTO")
    ).sum()
)

medium_auto = int(
    (
        (e2e_df["risk_level"] == "MEDIUM")
        &
        (e2e_df["automation_decision"] == "AUTO")
    ).sum()
)


print("\n" + "=" * 80)
print("SAFETY GATES")
print("=" * 80)

print(
    f"HIGH-risk AUTO   : {high_auto}"
)

print(
    f"MEDIUM-risk AUTO : {medium_auto}"
)


assert high_auto == 0
assert medium_auto == 0


print(
    "\nPASS — No HIGH or MEDIUM risk case was automatically handled."
)


# =============================================================================
# GROUNDING SAFETY CHECK
# =============================================================================

unsafe_count = int(
    (
        e2e_df["grounding_decision"]
        == "UNSAFE"
    ).sum()
)

print(
    f"UNSAFE generated responses : {unsafe_count}"
)

print("\n" + "=" * 80)
print("END-TO-END AGENT V1 COMPLETE")
print("=" * 80)

END-TO-END AGENT V1
Existing generated responses : 7
Existing grounding results   : 7

END-TO-END RESULTS


,golden_id,intent,risk_level,complexity,query_words,top1_ce_score,investigation_rate,trust_score,evidence_tier,trust_decision,grounding_decision,automation_decision
0,GOLD-0001,PAYMENT_BILLING,MEDIUM,MEDIUM,21,0.615286,0.8,0.489546,WEAK,WEAK,GROUNDED,HUMAN
1,GOLD-0002,ORDER_DELIVERY,LOW,MEDIUM,22,0.998420,0.6,0.998629,STRONG,TRUSTED,GROUNDED,HUMAN
2,GOLD-0003,ORDER_DELIVERY,LOW,MEDIUM,21,0.991000,0.4,0.993493,STRONG,TRUSTED,GROUNDED,HUMAN
3,GOLD-0004,GENERAL_SOCIAL,LOW,SIMPLE,9,0.957855,0.0,0.766866,STRONG,CONDITIONAL,GROUNDED,AUTO
4,GOLD-0005,DEVICE_TECHNICAL,LOW,MEDIUM,12,0.859899,0.2,0.667454,CONDITIONAL,CONDITIONAL,GROUNDED,HUMAN
5,GOLD-0006,SELLER_AUTHENTICITY,MEDIUM,MEDIUM,8,0.770269,1.0,0.622301,CONDITIONAL,CONDITIONAL,GROUNDED,HUMAN
6,GOLD-0007,ORDER_DELIVERY,MEDIUM,MEDIUM,18,0.951277,0.6,0.874949,STRONG,TRUSTED,GROUNDED,HUMAN



GROUNDING SUMMARY
grounding_decision
GROUNDED    7

AUTOMATION SUMMARY
automation_decision
HUMAN    6
AUTO     1

SAFETY GATES
HIGH-risk AUTO   : 0
MEDIUM-risk AUTO : 0

PASS — No HIGH or MEDIUM risk case was automatically handled.
UNSAFE generated responses : 0

END-TO-END AGENT V1 COMPLETE


In [31]:
# =============================================================================
# HIVER SUPPORT AGENT — FINAL EVALUATION HARNESS V2
# Robust version: only uses artifacts that actually exist
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json


# =============================================================================
# PATHS
# =============================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

EVAL_DIR = BASE_DIR / "evaluation"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_csv_if_exists(path):
    """
    Load CSV if it exists.
    Return None otherwise.
    """

    if not path.exists():

        print(
            f"WARNING — Optional artifact not found: {path.name}"
        )

        return None

    return pd.read_csv(path)


def metric_row(
    component,
    metric,
    value,
    denominator=None,
    notes=""
):
    return {
        "component": component,
        "metric": metric,
        "value": value,
        "denominator": denominator,
        "notes": notes,
    }


# =============================================================================
# LOAD AVAILABLE ARTIFACTS
# =============================================================================

print("=" * 80)
print("LOADING FINAL EVALUATION ARTIFACTS")
print("=" * 80)


golden_df = load_csv_if_exists(
    EVAL_DIR / "golden_evaluation_set_frozen.csv"
)

retrieval_benchmark_df = load_csv_if_exists(
    EVAL_DIR / "retrieval_benchmark.csv"
)

evidence_review_df = load_csv_if_exists(
    EVAL_DIR / "top1_evidence_review.csv"
)

grounding_df = load_csv_if_exists(
    EVAL_DIR / "response_grounding_batch_01_v2.csv"
)

automation_df = load_csv_if_exists(
    EVAL_DIR / "final_automation_policy_evaluation.csv"
)

trust_df = load_csv_if_exists(
    EVAL_DIR / "trust_checker_evaluation.csv"
)


# =============================================================================
# ARTIFACT STATUS
# =============================================================================

print("\n" + "=" * 80)
print("ARTIFACT STATUS")
print("=" * 80)

artifact_status = {
    "golden_evaluation_set_frozen.csv": golden_df,
    "retrieval_benchmark.csv": retrieval_benchmark_df,
    "top1_evidence_review.csv": evidence_review_df,
    "response_grounding_batch_01_v2.csv": grounding_df,
    "final_automation_policy_evaluation.csv": automation_df,
    "trust_checker_evaluation.csv": trust_df,
}

for name, df in artifact_status.items():

    if df is None:

        print(
            f"NOT AVAILABLE : {name}"
        )

    else:

        print(
            f"AVAILABLE     : {name} "
            f"shape={df.shape}"
        )


# =============================================================================
# REPORT STORAGE
# =============================================================================

report_rows = []


# =============================================================================
# 1. GOLDEN SET
# =============================================================================

if golden_df is not None:

    golden_count = len(
        golden_df
    )

    report_rows.append(
        metric_row(
            "Dataset",
            "Golden cases",
            golden_count,
            notes=(
                "Frozen manually reviewed/annotated evaluation set"
            )
        )
    )

    print("\n" + "=" * 80)
    print("GOLDEN SET")
    print("=" * 80)

    print(
        f"Golden cases: {golden_count:,}"
    )


# =============================================================================
# 2. RETRIEVAL
# =============================================================================

if retrieval_benchmark_df is not None:

    print("\n" + "=" * 80)
    print("RETRIEVAL EVALUATION")
    print("=" * 80)

    display(
        retrieval_benchmark_df
    )

    # Store the complete benchmark table as a readable artifact
    # rather than making assumptions about exact column names.

    for _, row in retrieval_benchmark_df.iterrows():

        row_dict = row.to_dict()

        method = (
            row_dict.get("method")
            or row_dict.get("retriever")
            or row_dict.get("model")
            or row_dict.get("variant")
            or "UNKNOWN"
        )

        for key, value in row_dict.items():

            if key in {
                "method",
                "retriever",
                "model",
                "variant"
            }:
                continue

            if pd.isna(value):
                continue

            if "recall" in str(key).lower():

                report_rows.append(
                    metric_row(
                        "Retrieval",
                        f"{method} - {key}",
                        float(value),
                        notes=(
                            "Intent-agreement proxy; "
                            "not human relevance"
                        )
                    )
                )


# =============================================================================
# 3. HUMAN EVIDENCE REVIEW
# =============================================================================

if evidence_review_df is not None:

    print("\n" + "=" * 80)
    print("HUMAN EVIDENCE REVIEW")
    print("=" * 80)

    # Detect the review-label column.

    possible_label_columns = [
        "evidence_label",
        "review_label",
        "label",
        "evidence_review",
        "relevance",
    ]

    evidence_label_col = None

    for col in possible_label_columns:

        if col in evidence_review_df.columns:

            evidence_label_col = col
            break

    if evidence_label_col is not None:

        evidence_counts = (
            evidence_review_df[
                evidence_label_col
            ]
            .astype(str)
            .str.upper()
            .value_counts()
        )

        print(
            evidence_counts.to_string()
        )

        total_reviewed = len(
            evidence_review_df
        )

        relevant = int(
            evidence_counts.get(
                "RELEVANT",
                0
            )
        )

        partial = int(
            evidence_counts.get(
                "PARTIAL",
                0
            )
        )

        irrelevant = int(
            evidence_counts.get(
                "IRRELEVANT",
                0
            )
        )

        useful = relevant + partial

        report_rows.extend([
            metric_row(
                "Evidence",
                "Top-1 relevant rate",
                relevant / total_reviewed,
                total_reviewed,
                "Human-reviewed evidence"
            ),

            metric_row(
                "Evidence",
                "Top-1 useful evidence rate",
                useful / total_reviewed,
                total_reviewed,
                "RELEVANT + PARTIAL"
            ),

            metric_row(
                "Evidence",
                "Top-1 irrelevant rate",
                irrelevant / total_reviewed,
                total_reviewed,
                "Human-reviewed evidence"
            ),
        ])

    else:

        print(
            "WARNING — Could not identify evidence review label column."
        )


# =============================================================================
# 4. TRUST CHECKER
# =============================================================================

print("\n" + "=" * 80)
print("TRUST CHECKER")
print("=" * 80)

if trust_df is not None:

    if "evidence_decision" in trust_df.columns:

        trust_counts = (
            trust_df[
                "evidence_decision"
            ]
            .value_counts()
        )

        print(
            trust_counts.to_string()
        )

        for decision, count in trust_counts.items():

            report_rows.append(
                metric_row(
                    "Trust Checker",
                    f"{decision} count",
                    int(count),
                    len(trust_df),
                    (
                        "Evidence-quality signal; "
                        "not primary automation classifier"
                    )
                )
            )

    else:

        print(
            "WARNING — evidence_decision column not found."
        )

else:

    print(
        "Trust evaluation CSV is unavailable."
    )

    print(
        "This is expected because the trust experiment "
        "artifact was not persisted."
    )

    print(
        "Trust Checker results will therefore not be included "
        "as a standalone final metric."
    )


# =============================================================================
# 5. GROUNDING
# =============================================================================

print("\n" + "=" * 80)
print("GROUNDING CHECK")
print("=" * 80)

if grounding_df is not None:

    if "grounding_decision" in grounding_df.columns:

        grounding_counts = (
            grounding_df[
                "grounding_decision"
            ]
            .astype(str)
            .str.upper()
            .value_counts()
        )

        print(
            grounding_counts.to_string()
        )

        grounding_total = len(
            grounding_df
        )

        grounded = int(
            grounding_counts.get(
                "GROUNDED",
                0
            )
        )

        conditional = int(
            grounding_counts.get(
                "CONDITIONAL",
                0
            )
        )

        unsafe = int(
            grounding_counts.get(
                "UNSAFE",
                0
            )
        )

        report_rows.extend([
            metric_row(
                "Grounding",
                "Grounded response rate",
                grounded / grounding_total,
                grounding_total,
                "Existing generated-response batch"
            ),

            metric_row(
                "Grounding",
                "Conditional response rate",
                conditional / grounding_total,
                grounding_total,
                "Existing generated-response batch"
            ),

            metric_row(
                "Grounding",
                "Unsafe response rate",
                unsafe / grounding_total,
                grounding_total,
                "Existing generated-response batch"
            ),
        ])

    else:

        print(
            "WARNING — grounding_decision column not found."
        )


# =============================================================================
# 6. AUTOMATION POLICY
# =============================================================================

print("\n" + "=" * 80)
print("AUTOMATION POLICY")
print("=" * 80)

auto_count = None
human_count = None
total = None

auto_precision = None
auto_recall = None
auto_f1 = None
accuracy = None

high_auto = None
medium_auto = None


if automation_df is not None:

    if "automation_decision" in automation_df.columns:

        predicted_counts = (
            automation_df[
                "automation_decision"
            ]
            .astype(str)
            .str.upper()
            .value_counts()
        )

        print(
            predicted_counts.to_string()
        )

        auto_count = int(
            predicted_counts.get(
                "AUTO",
                0
            )
        )

        human_count = int(
            predicted_counts.get(
                "HUMAN",
                0
            )
        )

        total = len(
            automation_df
        )

        report_rows.extend([
            metric_row(
                "Automation",
                "AUTO coverage",
                auto_count / total,
                total,
                "Frozen conservative automation policy"
            ),

            metric_row(
                "Automation",
                "HUMAN routing rate",
                human_count / total,
                total,
                "Frozen conservative automation policy"
            ),
        ])

    else:

        print(
            "WARNING — automation_decision column not found."
        )


# =============================================================================
# 7. AUTOMATION CONFUSION MATRIX
# =============================================================================

if (
    automation_df is not None
    and
    "gold_automation_decision" in automation_df.columns
    and
    "automation_decision" in automation_df.columns
):

    print("\n" + "=" * 80)
    print("AUTOMATION CONFUSION MATRIX")
    print("=" * 80)

    gold_labels = (
        automation_df[
            "gold_automation_decision"
        ]
        .astype(str)
        .str.upper()
    )

    predicted_labels = (
        automation_df[
            "automation_decision"
        ]
        .astype(str)
        .str.upper()
    )

    confusion = pd.crosstab(
        gold_labels,
        predicted_labels,
        rownames=["GOLD"],
        colnames=["PREDICTED"],
        dropna=False
    )

    print(
        confusion
    )

    true_auto = int(
        (
            (gold_labels == "AUTO")
            &
            (predicted_labels == "AUTO")
        ).sum()
    )

    false_auto = int(
        (
            (gold_labels == "HUMAN")
            &
            (predicted_labels == "AUTO")
        ).sum()
    )

    false_human = int(
        (
            (gold_labels == "AUTO")
            &
            (predicted_labels == "HUMAN")
        ).sum()
    )

    true_human = int(
        (
            (gold_labels == "HUMAN")
            &
            (predicted_labels == "HUMAN")
        ).sum()
    )

    auto_precision = (
        true_auto / (true_auto + false_auto)
        if (true_auto + false_auto) > 0
        else 0.0
    )

    auto_recall = (
        true_auto / (true_auto + false_human)
        if (true_auto + false_human) > 0
        else 0.0
    )

    auto_f1 = (
        2 * auto_precision * auto_recall
        / (auto_precision + auto_recall)
        if (auto_precision + auto_recall) > 0
        else 0.0
    )

    accuracy = (
        (true_auto + true_human)
        / len(automation_df)
    )

    print(
        f"\nAUTO precision : {auto_precision:.4f}"
    )

    print(
        f"AUTO recall    : {auto_recall:.4f}"
    )

    print(
        f"AUTO F1        : {auto_f1:.4f}"
    )

    print(
        f"Accuracy       : {accuracy:.4f}"
    )

    report_rows.extend([
        metric_row(
            "Automation",
            "AUTO precision",
            auto_precision,
            notes="Frozen policy evaluation"
        ),

        metric_row(
            "Automation",
            "AUTO recall",
            auto_recall,
            notes="Frozen policy evaluation"
        ),

        metric_row(
            "Automation",
            "AUTO F1",
            auto_f1,
            notes="Frozen policy evaluation"
        ),

        metric_row(
            "Automation",
            "Overall accuracy",
            accuracy,
            notes="Frozen policy evaluation"
        ),
    ])


# =============================================================================
# 8. SAFETY GATES
# =============================================================================

if (
    automation_df is not None
    and
    "gold_risk_level" in automation_df.columns
    and
    "automation_decision" in automation_df.columns
):

    print("\n" + "=" * 80)
    print("SAFETY GATES")
    print("=" * 80)

    gold_risk = (
        automation_df[
            "gold_risk_level"
        ]
        .astype(str)
        .str.upper()
    )

    predicted_action = (
        automation_df[
            "automation_decision"
        ]
        .astype(str)
        .str.upper()
    )

    high_auto = int(
        (
            (gold_risk == "HIGH")
            &
            (predicted_action == "AUTO")
        ).sum()
    )

    medium_auto = int(
        (
            (gold_risk == "MEDIUM")
            &
            (predicted_action == "AUTO")
        ).sum()
    )

    print(
        f"HIGH-risk AUTO   : {high_auto}"
    )

    print(
        f"MEDIUM-risk AUTO : {medium_auto}"
    )

    report_rows.extend([
        metric_row(
            "Safety",
            "HIGH-risk AUTO cases",
            high_auto,
            notes="Must remain zero"
        ),

        metric_row(
            "Safety",
            "MEDIUM-risk AUTO cases",
            medium_auto,
            notes="Must remain zero"
        ),
    ])

    assert high_auto == 0
    assert medium_auto == 0

    print(
        "\nPASS — No HIGH or MEDIUM risk case was automatically handled."
    )


# =============================================================================
# 9. FINAL REPORT DATAFRAME
# =============================================================================

final_report_df = pd.DataFrame(
    report_rows
)


# =============================================================================
# SAVE CSV
# =============================================================================

report_path = (
    EVAL_DIR
    / "final_evaluation_report_v2.csv"
)

final_report_df.to_csv(
    report_path,
    index=False
)


# =============================================================================
# BUILD SUMMARY
# =============================================================================

summary = {
    "evaluation_version": "v2",

    "golden_cases": (
        int(len(golden_df))
        if golden_df is not None
        else None
    ),

    "grounding": {
        "grounded": (
            int(
                grounding_df[
                    "grounding_decision"
                ]
                .astype(str)
                .str.upper()
                .eq("GROUNDED")
                .sum()
            )
            if grounding_df is not None
            else None
        ),

        "conditional": (
            int(
                grounding_df[
                    "grounding_decision"
                ]
                .astype(str)
                .str.upper()
                .eq("CONDITIONAL")
                .sum()
            )
            if grounding_df is not None
            else None
        ),

        "unsafe": (
            int(
                grounding_df[
                    "grounding_decision"
                ]
                .astype(str)
                .str.upper()
                .eq("UNSAFE")
                .sum()
            )
            if grounding_df is not None
            else None
        ),
    },

    "automation": {
        "auto_count": auto_count,
        "human_count": human_count,
        "coverage": (
            auto_count / total
            if auto_count is not None
            and total
            else None
        ),
        "precision": auto_precision,
        "recall": auto_recall,
        "f1": auto_f1,
        "accuracy": accuracy,
    },

    "safety": {
        "high_risk_auto": high_auto,
        "medium_risk_auto": medium_auto,
    },

    "trust_checker_artifact_available": (
        trust_df is not None
    ),
}


# =============================================================================
# SAVE JSON
# =============================================================================

summary_path = (
    EVAL_DIR
    / "final_evaluation_summary_v2.json"
)

with open(summary_path, "w") as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("\n" + "=" * 80)
print("FINAL EVALUATION SUMMARY")
print("=" * 80)

print(
    json.dumps(
        summary,
        indent=2
    )
)

print("\n" + "=" * 80)
print("ARTIFACTS SAVED")
print("=" * 80)

print(
    f"CSV : {report_path}"
)

print(
    f"JSON: {summary_path}"
)

print("\n" + "=" * 80)
print("FINAL EVALUATION HARNESS V2 COMPLETE")
print("=" * 80)

LOADING FINAL EVALUATION ARTIFACTS
WARNING — Optional artifact not found: trust_checker_evaluation.csv

ARTIFACT STATUS
AVAILABLE     : golden_evaluation_set_frozen.csv shape=(200, 16)
AVAILABLE     : retrieval_benchmark.csv shape=(5, 5)
AVAILABLE     : top1_evidence_review.csv shape=(200, 10)
AVAILABLE     : response_grounding_batch_01_v2.csv shape=(7, 8)
AVAILABLE     : final_automation_policy_evaluation.csv shape=(174, 7)
NOT AVAILABLE : trust_checker_evaluation.csv

GOLDEN SET
Golden cases: 200

RETRIEVAL EVALUATION


,method,recall_at_1,recall_at_3,recall_at_5,recall_at_10
0,TF-IDF,0.160,0.25,0.310,NaN
1,BM25,0.170,0.29,0.325,NaN
2,BGE-large,0.215,0.32,0.365,NaN
3,BM25+BGE RRF,0.220,0.33,0.380,0.425
4,RRF+CrossEncoder,0.250,0.35,0.390,0.440



HUMAN EVIDENCE REVIEW
WARNING — Could not identify evidence review label column.

TRUST CHECKER
Trust evaluation CSV is unavailable.
This is expected because the trust experiment artifact was not persisted.
Trust Checker results will therefore not be included as a standalone final metric.

GROUNDING CHECK
grounding_decision
GROUNDED    7

AUTOMATION POLICY
WARNING — automation_decision column not found.

FINAL EVALUATION SUMMARY
{
  "evaluation_version": "v2",
  "golden_cases": 200,
  "grounding": {
    "grounded": 7,
    "conditional": 0,
    "unsafe": 0
  },
  "automation": {
    "auto_count": null,
    "human_count": null,
    "coverage": null,
    "precision": null,
    "recall": null,
    "f1": null,
    "accuracy": null
  },
  "safety": {
    "high_risk_auto": null,
    "medium_risk_auto": null
  },
  "trust_checker_artifact_available": false
}

ARTIFACTS SAVED
CSV : /content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/final_evaluation_report_v2.csv
JSON: /conte

In [32]:
# =============================================================================
# HIVER SUPPORT AGENT — FINAL EVALUATION HARNESS V3
# Uses actual checkpoint schemas + recreates missing Trust evaluation
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json


# =============================================================================
# PATHS
# =============================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

EVAL_DIR = BASE_DIR / "evaluation"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"


# =============================================================================
# LOAD CORE ARTIFACTS
# =============================================================================

print("=" * 80)
print("LOADING FINAL EVALUATION ARTIFACTS")
print("=" * 80)

golden_df = pd.read_csv(
    EVAL_DIR / "golden_evaluation_set_frozen.csv"
)

retrieval_benchmark_df = pd.read_csv(
    EVAL_DIR / "retrieval_benchmark.csv"
)

evidence_review_df = pd.read_csv(
    EVAL_DIR / "top1_evidence_review.csv"
)

grounding_df = pd.read_csv(
    EVAL_DIR / "response_grounding_batch_01_v2.csv"
)

automation_df = pd.read_csv(
    EVAL_DIR / "final_automation_policy_evaluation.csv"
)


# =============================================================================
# SHOW ACTUAL SCHEMAS
# =============================================================================

print("\n" + "=" * 80)
print("ACTUAL ARTIFACT SCHEMAS")
print("=" * 80)

print("\nGolden:")
print(list(golden_df.columns))

print("\nRetrieval:")
print(list(retrieval_benchmark_df.columns))

print("\nEvidence review:")
print(list(evidence_review_df.columns))

print("\nGrounding:")
print(list(grounding_df.columns))

print("\nAutomation:")
print(list(automation_df.columns))


# =============================================================================
# REPORT STORAGE
# =============================================================================

report_rows = []


def add_metric(
    component,
    metric,
    value,
    denominator=None,
    notes=""
):
    report_rows.append({
        "component": component,
        "metric": metric,
        "value": value,
        "denominator": denominator,
        "notes": notes,
    })


# =============================================================================
# 1. DATASET
# =============================================================================

golden_count = len(golden_df)

add_metric(
    "Dataset",
    "Golden cases",
    golden_count,
    notes=(
        "Frozen manually reviewed/annotated evaluation set"
    )
)


# =============================================================================
# 2. RETRIEVAL
# =============================================================================

print("\n" + "=" * 80)
print("RETRIEVAL BENCHMARK")
print("=" * 80)

display(
    retrieval_benchmark_df
)

for _, row in retrieval_benchmark_df.iterrows():

    method = row["method"]

    for k in [1, 3, 5, 10]:

        column = f"recall_at_{k}"

        if column in retrieval_benchmark_df.columns:

            value = row[column]

            if not pd.isna(value):

                add_metric(
                    "Retrieval",
                    f"{method} Recall@{k}",
                    float(value),
                    golden_count,
                    (
                        "Intent-agreement proxy; "
                        "not human relevance"
                    )
                )


# =============================================================================
# 3. HUMAN EVIDENCE REVIEW
# =============================================================================

print("\n" + "=" * 80)
print("HUMAN EVIDENCE REVIEW")
print("=" * 80)

print("Columns:")
print(list(evidence_review_df.columns))

# -------------------------------------------------------------------------
# Find the actual review-label column.
# -------------------------------------------------------------------------

review_label_candidates = [
    "evidence_review",
    "evidence_label",
    "review_label",
    "relevance",
    "label",
    "human_label",
    "assessment",
]

review_label_col = None

for column in review_label_candidates:

    if column in evidence_review_df.columns:

        review_label_col = column
        break


# -------------------------------------------------------------------------
# If the schema contains one of the known human-review columns,
# use it.
# -------------------------------------------------------------------------

if review_label_col is not None:

    review_labels = (
        evidence_review_df[
            review_label_col
        ]
        .astype(str)
        .str.upper()
        .str.strip()
    )

else:

    # ---------------------------------------------------------------------
    # The established review artifact is known to contain the manual
    # RELEVANT / PARTIAL / IRRELEVANT assessment. If the label column
    # has a different name, inspect object columns automatically.
    # ---------------------------------------------------------------------

    object_columns = (
        evidence_review_df
        .select_dtypes(include=["object"])
        .columns
        .tolist()
    )

    detected_column = None

    for column in object_columns:

        values = set(
            evidence_review_df[column]
            .dropna()
            .astype(str)
            .str.upper()
            .str.strip()
            .unique()
        )

        if (
            "RELEVANT" in values
            and "PARTIAL" in values
            and "IRRELEVANT" in values
        ):

            detected_column = column
            break

    review_label_col = detected_column

    if review_label_col is not None:

        review_labels = (
            evidence_review_df[
                review_label_col
            ]
            .astype(str)
            .str.upper()
            .str.strip()
        )


# -------------------------------------------------------------------------
# Calculate evidence metrics.
# -------------------------------------------------------------------------

if review_label_col is not None:

    print(
        f"Review label column detected: {review_label_col}"
    )

    review_counts = (
        review_labels
        .value_counts()
    )

    print("\nReview distribution:")
    print(
        review_counts.to_string()
    )

    total_reviewed = len(
        review_labels
    )

    relevant = int(
        review_counts.get(
            "RELEVANT",
            0
        )
    )

    partial = int(
        review_counts.get(
            "PARTIAL",
            0
        )
    )

    irrelevant = int(
        review_counts.get(
            "IRRELEVANT",
            0
        )
    )

    useful = relevant + partial

    relevant_rate = (
        relevant / total_reviewed
    )

    useful_rate = (
        useful / total_reviewed
    )

    irrelevant_rate = (
        irrelevant / total_reviewed
    )

    add_metric(
        "Evidence",
        "Top-1 relevant rate",
        relevant_rate,
        total_reviewed,
        "Human-reviewed evidence"
    )

    add_metric(
        "Evidence",
        "Top-1 useful evidence rate",
        useful_rate,
        total_reviewed,
        "RELEVANT + PARTIAL"
    )

    add_metric(
        "Evidence",
        "Top-1 irrelevant rate",
        irrelevant_rate,
        total_reviewed,
        "Human-reviewed evidence"
    )

else:

    print(
        "WARNING — Could not automatically detect "
        "RELEVANT/PARTIAL/IRRELEVANT column."
    )


# =============================================================================
# 4. GROUNDING
# =============================================================================

print("\n" + "=" * 80)
print("GROUNDING EVALUATION")
print("=" * 80)

grounding_labels = (
    grounding_df[
        "grounding_decision"
    ]
    .astype(str)
    .str.upper()
    .str.strip()
)

grounding_counts = (
    grounding_labels
    .value_counts()
)

print(
    grounding_counts.to_string()
)

grounding_total = len(
    grounding_labels
)

grounded_count = int(
    grounding_counts.get(
        "GROUNDED",
        0
    )
)

conditional_count = int(
    grounding_counts.get(
        "CONDITIONAL",
        0
    )
)

unsafe_count = int(
    grounding_counts.get(
        "UNSAFE",
        0
    )
)

add_metric(
    "Grounding",
    "Grounded response rate",
    grounded_count / grounding_total,
    grounding_total,
    "Response grounding checker V2"
)

add_metric(
    "Grounding",
    "Conditional response rate",
    conditional_count / grounding_total,
    grounding_total,
    "Response grounding checker V2"
)

add_metric(
    "Grounding",
    "Unsafe response rate",
    unsafe_count / grounding_total,
    grounding_total,
    "Response grounding checker V2"
)


# =============================================================================
# 5. AUTOMATION POLICY
# =============================================================================

print("\n" + "=" * 80)
print("AUTOMATION POLICY EVALUATION")
print("=" * 80)

print(
    automation_df.to_string(index=False)
)

print("\nAutomation columns:")
print(
    list(automation_df.columns)
)


# =============================================================================
# IDENTIFY GOLD / PREDICTION COLUMNS
# =============================================================================

gold_automation_col = None
predicted_automation_col = None

# Known schema from our final policy artifact
gold_candidates = [
    "gold_automation_decision",
    "automation_decision_gold",
    "gold_label",
    "gold_decision",
]

prediction_candidates = [
    "predicted_automation_decision",
    "predicted_decision",
    "policy_decision",
    "prediction",
]


for column in gold_candidates:

    if column in automation_df.columns:

        gold_automation_col = column
        break


for column in prediction_candidates:

    if column in automation_df.columns:

        predicted_automation_col = column
        break


# =============================================================================
# FALLBACK: DISCOVER AUTO/HUMAN COLUMNS
# =============================================================================

if (
    gold_automation_col is None
    or
    predicted_automation_col is None
):

    auto_human_columns = []

    for column in automation_df.columns:

        values = set(
            automation_df[column]
            .dropna()
            .astype(str)
            .str.upper()
            .str.strip()
            .unique()
        )

        if (
            "AUTO" in values
            and "HUMAN" in values
        ):

            auto_human_columns.append(
                column
            )

    print(
        "\nColumns containing AUTO/HUMAN:"
    )

    for column in auto_human_columns:

        print(
            f"  - {column}"
        )

    # Use semantic names when possible.
    if gold_automation_col is None:

        for column in auto_human_columns:

            name = column.lower()

            if (
                "gold" in name
                or
                "human_label" in name
            ):

                gold_automation_col = column
                break

    if predicted_automation_col is None:

        for column in auto_human_columns:

            name = column.lower()

            if (
                "pred" in name
                or
                "decision" in name
                or
                "policy" in name
            ):

                if column != gold_automation_col:

                    predicted_automation_col = column
                    break


# =============================================================================
# AUTOMATION METRICS
# =============================================================================

if (
    gold_automation_col is not None
    and
    predicted_automation_col is not None
):

    print(
        f"\nGold column      : {gold_automation_col}"
    )

    print(
        f"Prediction column: {predicted_automation_col}"
    )

    gold_actions = (
        automation_df[
            gold_automation_col
        ]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    predicted_actions = (
        automation_df[
            predicted_automation_col
        ]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    confusion = pd.crosstab(
        gold_actions,
        predicted_actions,
        rownames=["GOLD"],
        colnames=["PREDICTED"],
        dropna=False
    )

    print("\nConfusion matrix:")
    print(
        confusion
    )

    true_auto = int(
        (
            (gold_actions == "AUTO")
            &
            (predicted_actions == "AUTO")
        ).sum()
    )

    false_auto = int(
        (
            (gold_actions == "HUMAN")
            &
            (predicted_actions == "AUTO")
        ).sum()
    )

    false_human = int(
        (
            (gold_actions == "AUTO")
            &
            (predicted_actions == "HUMAN")
        ).sum()
    )

    true_human = int(
        (
            (gold_actions == "HUMAN")
            &
            (predicted_actions == "HUMAN")
        ).sum()
    )

    total_cases = len(
        automation_df
    )

    auto_count = int(
        (predicted_actions == "AUTO").sum()
    )

    human_count = int(
        (predicted_actions == "HUMAN").sum()
    )

    auto_precision = (
        true_auto / (true_auto + false_auto)
        if (true_auto + false_auto) > 0
        else 0.0
    )

    auto_recall = (
        true_auto / (true_auto + false_human)
        if (true_auto + false_human) > 0
        else 0.0
    )

    auto_f1 = (
        2 * auto_precision * auto_recall
        /
        (auto_precision + auto_recall)
        if (auto_precision + auto_recall) > 0
        else 0.0
    )

    accuracy = (
        (true_auto + true_human)
        /
        total_cases
    )

    coverage = (
        auto_count / total_cases
    )

    print(
        f"\nAUTO count      : {auto_count}"
    )

    print(
        f"HUMAN count     : {human_count}"
    )

    print(
        f"AUTO coverage   : {coverage:.4f}"
    )

    print(
        f"AUTO precision  : {auto_precision:.4f}"
    )

    print(
        f"AUTO recall     : {auto_recall:.4f}"
    )

    print(
        f"AUTO F1         : {auto_f1:.4f}"
    )

    print(
        f"Overall accuracy: {accuracy:.4f}"
    )

    add_metric(
        "Automation",
        "AUTO coverage",
        coverage,
        total_cases,
        "Frozen conservative automation policy"
    )

    add_metric(
        "Automation",
        "AUTO precision",
        auto_precision,
        notes="Frozen policy evaluation"
    )

    add_metric(
        "Automation",
        "AUTO recall",
        auto_recall,
        notes="Frozen policy evaluation"
    )

    add_metric(
        "Automation",
        "AUTO F1",
        auto_f1,
        notes="Frozen policy evaluation"
    )

    add_metric(
        "Automation",
        "Overall accuracy",
        accuracy,
        notes="Frozen policy evaluation"
    )

else:

    print(
        "\nWARNING — Could not identify the gold/prediction "
        "automation columns."
    )

    print(
        "Automation metrics cannot be calculated safely."
    )


# =============================================================================
# 6. SAFETY GATES
# =============================================================================

print("\n" + "=" * 80)
print("SAFETY GATES")
print("=" * 80)

gold_risk_col = None

for column in [
    "gold_risk_level",
    "risk_level",
    "gold_risk",
]:

    if column in automation_df.columns:

        gold_risk_col = column
        break


if (
    gold_risk_col is not None
    and
    predicted_automation_col is not None
):

    risk_values = (
        automation_df[
            gold_risk_col
        ]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    predicted_values = (
        automation_df[
            predicted_automation_col
        ]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    high_auto = int(
        (
            (risk_values == "HIGH")
            &
            (predicted_values == "AUTO")
        ).sum()
    )

    medium_auto = int(
        (
            (risk_values == "MEDIUM")
            &
            (predicted_values == "AUTO")
        ).sum()
    )

    print(
        f"HIGH-risk AUTO   : {high_auto}"
    )

    print(
        f"MEDIUM-risk AUTO : {medium_auto}"
    )

    add_metric(
        "Safety",
        "HIGH-risk AUTO cases",
        high_auto,
        notes="Hard safety gate"
    )

    add_metric(
        "Safety",
        "MEDIUM-risk AUTO cases",
        medium_auto,
        notes="Hard safety gate"
    )

else:

    print(
        "WARNING — Risk/action columns unavailable."
    )


# =============================================================================
# 7. RECREATE TRUST CHECKER EVALUATION
# =============================================================================
#
# The standalone CSV was not persisted.
#
# We can recreate the evaluation from:
#   - frozen golden set
#   - saved reranked evidence
#   - runtime evidence function
#   - trust_check_v3()
#
# This is a development evaluation artifact, not a runtime feature.
# =============================================================================

print("\n" + "=" * 80)
print("TRUST CHECKER V3 RE-EVALUATION")
print("=" * 80)

trust_rows = []

for _, case in golden_df.iterrows():

    golden_id = str(
        case["golden_id"]
    )

    intent = str(
        case["intent"]
    )

    risk = str(
        case["risk_level"]
    )

    evidence = retrieve_runtime_evidence(
        golden_id=golden_id,
        top_k=5
    )

    trust_result = trust_check_v3(
        evidence_df=evidence,
        predicted_intent=intent,
        risk_level=risk
    )

    trust_rows.append({

        "golden_id": golden_id,

        "gold_intent": intent,

        "gold_risk_level": risk,

        "gold_automation_decision":
            case["automation_decision"],

        "trust_score":
            trust_result.get(
                "trust_score"
            ),

        "top1_ce_score":
            trust_result.get(
                "top1_ce_score"
            ),

        "evidence_tier":
            trust_result.get(
                "evidence_tier"
            ),

        "intent_consistency":
            trust_result.get(
                "intent_consistency"
            ),

        "semantic_consistency":
            trust_result.get(
                "semantic_consistency"
            ),

        "evidence_decision":
            trust_result.get(
                "evidence_decision"
            ),

        "automation_decision":
            trust_result.get(
                "automation_decision"
            ),

        "reason":
            trust_result.get(
                "reason",
                ""
            ),
    })


trust_eval_df = pd.DataFrame(
    trust_rows
)


# =============================================================================
# SAVE TRUST EVALUATION
# =============================================================================

trust_output_path = (
    EVAL_DIR
    / "trust_checker_evaluation_v3_recreated.csv"
)

trust_eval_df.to_csv(
    trust_output_path,
    index=False
)


print(
    f"Saved recreated Trust evaluation:"
)

print(
    trust_output_path
)


# =============================================================================
# TRUST SUMMARY
# =============================================================================

trust_decision_counts = (
    trust_eval_df[
        "evidence_decision"
    ]
    .value_counts()
)

print("\nEvidence decisions:")
print(
    trust_decision_counts.to_string()
)

for decision, count in trust_decision_counts.items():

    add_metric(
        "Trust Checker",
        f"{decision} count",
        int(count),
        len(trust_eval_df),
        (
            "Recreated from saved evidence artifacts; "
            "evidence-quality signal only"
        )
    )


# =============================================================================
# 8. CONSISTENCY CHECKS
# =============================================================================

print("\n" + "=" * 80)
print("FINAL CONSISTENCY CHECKS")
print("=" * 80)

checks = []


# Golden set
checks.append(
    (
        "Golden set = 200",
        len(golden_df) == 200
    )
)


# RAG evidence review
checks.append(
    (
        "Evidence review = 200",
        len(evidence_review_df) == 200
    )
)


# Grounding batch
checks.append(
    (
        "Grounding batch = 7",
        len(grounding_df) == 7
    )
)


# Trust evaluation
checks.append(
    (
        "Trust evaluation = 200",
        len(trust_eval_df) == 200
    )
)


# Safety gates if available
if high_auto is not None:

    checks.append(
        (
            "HIGH-risk AUTO = 0",
            high_auto == 0
        )
    )


if medium_auto is not None:

    checks.append(
        (
            "MEDIUM-risk AUTO = 0",
            medium_auto == 0
        )
    )


for description, passed in checks:

    print(
        f"{'PASS' if passed else 'FAIL'} — {description}"
    )


# =============================================================================
# 9. FINAL REPORT
# =============================================================================

final_report_df = pd.DataFrame(
    report_rows
)


report_path = (
    EVAL_DIR
    / "final_evaluation_report_v3.csv"
)

final_report_df.to_csv(
    report_path,
    index=False
)


# =============================================================================
# FINAL JSON SUMMARY
# =============================================================================

summary = {

    "evaluation_version": "v3",

    "golden_cases":
        int(len(golden_df)),

    "retrieval":
        {
            "benchmark_cases":
                int(len(golden_df)),
            "best_method":
                "RRF+CrossEncoder",
            "recall_at_1":
                0.25,
            "recall_at_3":
                0.35,
            "recall_at_5":
                0.39,
            "recall_at_10":
                0.44,
            "metric_type":
                "Intent-agreement proxy",
        },

    "evidence":
        {
            "review_cases":
                int(len(evidence_review_df)),
            "top1_relevant_rate":
                (
                    relevant_rate
                    if review_label_col is not None
                    else None
                ),
            "top1_useful_evidence_rate":
                (
                    useful_rate
                    if review_label_col is not None
                    else None
                ),
            "top1_irrelevant_rate":
                (
                    irrelevant_rate
                    if review_label_col is not None
                    else None
                ),
        },

    "grounding":
        {
            "cases":
                int(len(grounding_df)),
            "grounded":
                grounded_count,
            "conditional":
                conditional_count,
            "unsafe":
                unsafe_count,
        },

    "automation":
        {
            "auto_count":
                auto_count,
            "human_count":
                human_count,
            "coverage":
                coverage,
            "precision":
                auto_precision,
            "recall":
                auto_recall,
            "f1":
                auto_f1,
            "accuracy":
                accuracy,
        },

    "safety":
        {
            "high_risk_auto":
                high_auto,
            "medium_risk_auto":
                medium_auto,
        },

    "trust_checker":
        {
            "recreated":
                True,
            "evaluation_cases":
                int(len(trust_eval_df)),
            "artifact":
                str(trust_output_path),
            "role":
                "Evidence-quality signal, not primary automation classifier",
        },
}


summary_path = (
    EVAL_DIR
    / "final_evaluation_summary_v3.json"
)


with open(summary_path, "w") as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("FINAL EVALUATION SUMMARY")
print("=" * 80)

print(
    json.dumps(
        summary,
        indent=2
    )
)


print("\n" + "=" * 80)
print("ARTIFACTS SAVED")
print("=" * 80)

print(
    f"Evaluation CSV : {report_path}"
)

print(
    f"Summary JSON   : {summary_path}"
)

print(
    f"Trust CSV      : {trust_output_path}"
)


print("\n" + "=" * 80)
print("FINAL EVALUATION HARNESS V3 COMPLETE")
print("=" * 80)

LOADING FINAL EVALUATION ARTIFACTS

ACTUAL ARTIFACT SCHEMAS

Golden:
['golden_id', 'case_id', 'root_tweet_id', 'turn_count', 'customer_turns', 'support_turns', 'duration_hours', 'max_gap_hours', 'customer_query', 'support_response', 'provisional_bucket', 'risk_level', 'annotation_notes', 'intent', 'resolution_status', 'automation_decision']

Retrieval:
['method', 'recall_at_1', 'recall_at_3', 'recall_at_5', 'recall_at_10']

Evidence review:
['golden_id', 'customer_query', 'intent', 'case_id', 'ce_score', 'customer_problem', 'support_action', 'resolution_type', 'evidence_relevance', 'review_notes']

Grounding:
['golden_id', 'customer_query', 'generated_response', 'grounding_decision', 'violation_count', 'warning_count', 'violations', 'warnings']

Automation:
['golden_id', 'risk_level', 'investigation_rate', 'query_words', 'query_chars', 'predicted_automation', 'gold_automation']

RETRIEVAL BENCHMARK


,method,recall_at_1,recall_at_3,recall_at_5,recall_at_10
0,TF-IDF,0.160,0.25,0.310,NaN
1,BM25,0.170,0.29,0.325,NaN
2,BGE-large,0.215,0.32,0.365,NaN
3,BM25+BGE RRF,0.220,0.33,0.380,0.425
4,RRF+CrossEncoder,0.250,0.35,0.390,0.440



HUMAN EVIDENCE REVIEW
Columns:
['golden_id', 'customer_query', 'intent', 'case_id', 'ce_score', 'customer_problem', 'support_action', 'resolution_type', 'evidence_relevance', 'review_notes']
WARNING — Could not automatically detect RELEVANT/PARTIAL/IRRELEVANT column.

GROUNDING EVALUATION
grounding_decision
GROUNDED    7

AUTOMATION POLICY EVALUATION
golden_id risk_level  investigation_rate  query_words  query_chars predicted_automation gold_automation
GOLD-0001     MEDIUM                 0.8           21          140                HUMAN           HUMAN
GOLD-0002        LOW                 0.6           22          137                HUMAN           HUMAN
GOLD-0003        LOW                 0.4           21          114                HUMAN           HUMAN
GOLD-0004        LOW                 0.0            9           52                 AUTO            AUTO
GOLD-0005        LOW                 0.2           12           66                HUMAN           HUMAN
GOLD-0006        LOW  

In [34]:
# =============================================================================
# DIAGNOSTIC — INSPECT ACTUAL EVIDENCE REVIEW VALUES
# =============================================================================

from pathlib import Path
import pandas as pd

EVAL_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation"
)

evidence_df = pd.read_csv(
    EVAL_DIR / "top1_evidence_review.csv"
)

print("=" * 80)
print("EVIDENCE REVIEW DIAGNOSTIC")
print("=" * 80)

print("\nShape:")
print(evidence_df.shape)

print("\nColumns:")
for i, col in enumerate(evidence_df.columns):
    print(f"{i}: {col}")

print("\n" + "=" * 80)
print("evidence_relevance")
print("=" * 80)

print(
    evidence_df["evidence_relevance"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 80)
print("FIRST 20 ROWS — RELEVANCE + NOTES")
print("=" * 80)

print(
    evidence_df[
        [
            "golden_id",
            "evidence_relevance",
            "review_notes"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\n" + "=" * 80)
print("DATATYPE")
print("=" * 80)

print(
    evidence_df["evidence_relevance"].dtype
)

print("\n" + "=" * 80)
print("RAW UNIQUE VALUES")
print("=" * 80)

for value in evidence_df["evidence_relevance"].unique():

    print(
        repr(value)
    )

print("\n" + "=" * 80)
print("NON-NULL SAMPLE")
print("=" * 80)

non_null = evidence_df[
    evidence_df["evidence_relevance"].notna()
]

print(
    non_null[
        [
            "golden_id",
            "evidence_relevance",
            "review_notes"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

EVIDENCE REVIEW DIAGNOSTIC

Shape:
(200, 10)

Columns:
0: golden_id
1: customer_query
2: intent
3: case_id
4: ce_score
5: customer_problem
6: support_action
7: resolution_type
8: evidence_relevance
9: review_notes

evidence_relevance
evidence_relevance
NaN    200

FIRST 20 ROWS — RELEVANCE + NOTES
golden_id  evidence_relevance  review_notes
GOLD-0001                 NaN           NaN
GOLD-0002                 NaN           NaN
GOLD-0003                 NaN           NaN
GOLD-0004                 NaN           NaN
GOLD-0005                 NaN           NaN
GOLD-0006                 NaN           NaN
GOLD-0007                 NaN           NaN
GOLD-0008                 NaN           NaN
GOLD-0009                 NaN           NaN
GOLD-0010                 NaN           NaN
GOLD-0011                 NaN           NaN
GOLD-0012                 NaN           NaN
GOLD-0013                 NaN           NaN
GOLD-0014                 NaN           NaN
GOLD-0015                 NaN           N

In [35]:
# =============================================================================
# HIVER SUPPORT AGENT — HUMAN EVIDENCE REVIEW SETUP
# CELL 1: LOAD + PREPARE REVIEW DATA
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

EVAL_DIR = BASE_DIR / "evaluation"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"

# -----------------------------------------------------------------------------
# Load golden set
# -----------------------------------------------------------------------------

golden_df = pd.read_csv(
    EVAL_DIR / "golden_evaluation_set_frozen.csv"
)

# -----------------------------------------------------------------------------
# Load top-1 evidence
#
# This contains the retrieved historical evidence for each golden case.
# -----------------------------------------------------------------------------

reranked_df = pd.read_csv(
    CHECKPOINT_DIR / "cross_encoder_reranked_results.csv"
)

rag_corpus_df = pd.read_csv(
    CHECKPOINT_DIR / "structured_rag_corpus.csv"
)

# -----------------------------------------------------------------------------
# Validate required columns
# -----------------------------------------------------------------------------

required_golden = [
    "golden_id",
    "customer_query",
    "intent",
    "case_id",
]

required_reranked = [
    "golden_id",
    "case_id",
    "ce_rank",
    "ce_score",
]

required_rag = [
    "case_id",
    "customer_problem",
    "support_action",
    "resolution_type",
    "evidence_quality",
]

for col in required_golden:
    assert col in golden_df.columns, f"Missing golden column: {col}"

for col in required_reranked:
    assert col in reranked_df.columns, f"Missing reranked column: {col}"

for col in required_rag:
    assert col in rag_corpus_df.columns, f"Missing RAG column: {col}"

# -----------------------------------------------------------------------------
# Select TOP-1 retrieved evidence
# -----------------------------------------------------------------------------

top1_df = (
    reranked_df[
        reranked_df["ce_rank"] == 1
    ]
    .copy()
)

# -----------------------------------------------------------------------------
# Merge top-1 retrieval with historical evidence
# -----------------------------------------------------------------------------

top1_df = top1_df.merge(
    rag_corpus_df[
        [
            "case_id",
            "customer_problem",
            "support_action",
            "resolution_type",
            "evidence_quality",
        ]
    ],
    on="case_id",
    how="left",
)

# -----------------------------------------------------------------------------
# Keep only runtime/review-safe fields
# -----------------------------------------------------------------------------

top1_df = top1_df[
    [
        "golden_id",
        "case_id",
        "ce_rank",
        "ce_score",
        "customer_problem",
        "support_action",
        "resolution_type",
        "evidence_quality",
    ]
].copy()

# -----------------------------------------------------------------------------
# Merge with golden customer query
# -----------------------------------------------------------------------------

review_df = golden_df[
    [
        "golden_id",
        "customer_query",
        "intent",
    ]
].merge(
    top1_df,
    on="golden_id",
    how="left",
)

# -----------------------------------------------------------------------------
# Add annotation columns
#
# These are intentionally blank.
# They will be filled by YOUR human review.
# -----------------------------------------------------------------------------

review_df["evidence_relevance"] = np.nan
review_df["review_notes"] = np.nan

# -----------------------------------------------------------------------------
# Sort by golden ID
# -----------------------------------------------------------------------------

review_df = (
    review_df
    .sort_values("golden_id")
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

print("=" * 80)
print("HUMAN EVIDENCE REVIEW DATASET")
print("=" * 80)

print(
    f"Golden cases        : {len(golden_df)}"
)

print(
    f"Review rows         : {len(review_df)}"
)

print(
    f"Top-1 evidence rows : {len(top1_df)}"
)

print(
    f"Missing top-1       : {review_df['customer_problem'].isna().sum()}"
)

print("\nColumns:")
print(
    review_df.columns.tolist()
)

print("\nFirst 5 cases:")
display(
    review_df.head()
)

# -----------------------------------------------------------------------------
# Assertions
# -----------------------------------------------------------------------------

assert len(golden_df) == 200
assert len(review_df) == 200

assert (
    review_df["golden_id"].nunique()
    == 200
)

assert (
    review_df["evidence_relevance"].isna().all()
)

# -----------------------------------------------------------------------------
# Save initial review working file
# -----------------------------------------------------------------------------

working_path = (
    EVAL_DIR /
    "top1_evidence_review_working.csv"
)

review_df.to_csv(
    working_path,
    index=False
)

print("\n" + "=" * 80)
print("SETUP COMPLETE")
print("=" * 80)

print(
    f"Working review file:\n{working_path}"
)

HUMAN EVIDENCE REVIEW DATASET
Golden cases        : 200
Review rows         : 200
Top-1 evidence rows : 200
Missing top-1       : 0

Columns:
['golden_id', 'customer_query', 'intent', 'case_id', 'ce_rank', 'ce_score', 'customer_problem', 'support_action', 'resolution_type', 'evidence_quality', 'evidence_relevance', 'review_notes']

First 5 cases:


,golden_id,customer_query,intent,case_id,ce_rank,ce_score,customer_problem,support_action,resolution_type,evidence_quality,evidence_relevance,review_notes
0,GOLD-0001,Contact @115821 regarding payment on an order....,PAYMENT_BILLING,AMZ-073280,1,0.615286,[user] will someone please contact mw urgently...,[USER] Oh no! Sorry to hear this! Without prov...,INVESTIGATION_OR_FOLLOWUP,HIGH,NaN,NaN
1,GOLD-0002,@AmazonHelp LOL...you've missed the delivery d...,ORDER_DELIVERY,AMZ-032817,1,0.998420,"[user] yes, u missed the provided delivery date",[USER] Thanks for confirming! Who's the carrie...,INVESTIGATION_OR_FOLLOWUP,HIGH,NaN,NaN
2,GOLD-0003,So bummed that my package was delayed even wit...,ORDER_DELIVERY,AMZ-077675,1,0.991000,hey [user] i don’t pay for prime 2 day shippin...,"[USER] Hi Madison, without going into personal...",GENERAL_SUPPORT,HIGH,NaN,NaN
3,GOLD-0004,I love @115821 for my Christmas shopping gift ...,GENERAL_SOCIAL,AMZ-047124,1,0.957855,. [user] has to be the best thing ever for chr...,[USER] Can't beat skipping out on waiting in l...,GENERAL_SUPPORT,HIGH,NaN,NaN
4,GOLD-0005,@AmazonHelp Bleg on the phone 😕 what version s...,DEVICE_TECHNICAL,AMZ-001463,1,0.859899,[user] will it work on my phone?,[USER] We appreciate you reaching out! Here ar...,GENERAL_SUPPORT,MEDIUM,NaN,NaN



SETUP COMPLETE
Working review file:
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/top1_evidence_review_working.csv


In [36]:
# =============================================================================
# HIVER SUPPORT AGENT — INTERACTIVE HUMAN EVIDENCE REVIEWER
# CELL 2: REVIEW ONE CASE AT A TIME
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets


# =============================================================================
# PATHS
# =============================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

EVAL_DIR = BASE_DIR / "evaluation"

WORKING_PATH = (
    EVAL_DIR /
    "top1_evidence_review_working.csv"
)

FINAL_PATH = (
    EVAL_DIR /
    "top1_evidence_review.csv"
)


# =============================================================================
# LOAD WORKING REVIEW FILE
# =============================================================================

review_df = pd.read_csv(
    WORKING_PATH
)

# Normalize annotation columns
if "evidence_relevance" not in review_df.columns:
    review_df["evidence_relevance"] = np.nan

if "review_notes" not in review_df.columns:
    review_df["review_notes"] = np.nan


# =============================================================================
# REVIEW STATE
# =============================================================================

current_index = 0


# =============================================================================
# FIND FIRST UNREVIEWED CASE
# =============================================================================

unreviewed_indices = review_df[
    review_df["evidence_relevance"].isna()
].index.tolist()

if len(unreviewed_indices) > 0:
    current_index = unreviewed_indices[0]

else:
    current_index = 0


# =============================================================================
# UI ELEMENTS
# =============================================================================

case_header = widgets.HTML()

customer_box = widgets.HTML()

historical_box = widgets.HTML()

action_box = widgets.HTML()

metadata_box = widgets.HTML()

notes_box = widgets.Textarea(
    value="",
    placeholder="Optional: briefly explain your decision...",
    description="Notes:",
    layout=widgets.Layout(
        width="100%",
        height="80px"
    )
)

relevant_button = widgets.Button(
    description="RELEVANT",
    button_style="success",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

partial_button = widgets.Button(
    description="PARTIAL",
    button_style="warning",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

irrelevant_button = widgets.Button(
    description="IRRELEVANT",
    button_style="danger",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

previous_button = widgets.Button(
    description="← Previous",
    layout=widgets.Layout(
        width="130px"
    )
)

next_button = widgets.Button(
    description="Next →",
    layout=widgets.Layout(
        width="130px"
    )
)

save_button = widgets.Button(
    description="Save Progress",
    button_style="info",
    layout=widgets.Layout(
        width="150px"
    )
)

progress_box = widgets.HTML()


# =============================================================================
# DISPLAY CASE
# =============================================================================

def display_case():

    global current_index

    # ---------------------------------------------------------
    # Keep index valid
    # ---------------------------------------------------------

    if current_index < 0:
        current_index = 0

    if current_index >= len(review_df):
        current_index = len(review_df) - 1

    row = review_df.iloc[current_index]

    # ---------------------------------------------------------
    # Progress
    # ---------------------------------------------------------

    reviewed_count = int(
        review_df["evidence_relevance"]
        .notna()
        .sum()
    )

    total_count = len(review_df)

    progress_box.value = f"""
    <div style="
        padding:12px;
        border-radius:8px;
        background:#f1f5f9;
        margin-bottom:12px;
        font-size:16px;
    ">
        <b>Progress:</b>
        {reviewed_count} / {total_count}
        reviewed
        &nbsp;&nbsp;|&nbsp;&nbsp;
        Case {current_index + 1} / {total_count}
    </div>
    """

    # ---------------------------------------------------------
    # Header
    # ---------------------------------------------------------

    case_header.value = f"""
    <h2>
        {row['golden_id']}
    </h2>
    """

    # ---------------------------------------------------------
    # Customer query
    # ---------------------------------------------------------

    customer_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>👤 Customer Query</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['customer_query']}
    </p>

    </div>
    """

    # ---------------------------------------------------------
    # Historical customer problem
    # ---------------------------------------------------------

    historical_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>📚 Historical Customer Problem</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['customer_problem']}
    </p>

    </div>
    """

    # ---------------------------------------------------------
    # Historical support action
    # ---------------------------------------------------------

    action_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>🛠️ Historical Support Action</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['support_action']}
    </p>

    </div>
    """

    # ---------------------------------------------------------
    # Metadata
    # ---------------------------------------------------------

    metadata_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:15px;
    ">

    <b>Intent:</b> {row['intent']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Cross-Encoder Score:</b> {float(row['ce_score']):.4f}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Resolution Type:</b> {row['resolution_type']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Evidence Quality:</b> {row['evidence_quality']}

    </div>
    """

    # ---------------------------------------------------------
    # Existing notes
    # ---------------------------------------------------------

    existing_notes = row["review_notes"]

    if pd.isna(existing_notes):
        notes_box.value = ""
    else:
        notes_box.value = str(existing_notes)

    # ---------------------------------------------------------
    # Highlight existing label
    # ---------------------------------------------------------

    existing_label = row["evidence_relevance"]

    relevant_button.description = "RELEVANT"
    partial_button.description = "PARTIAL"
    irrelevant_button.description = "IRRELEVANT"

    if pd.notna(existing_label):

        label = str(existing_label).upper()

        if label == "RELEVANT":
            relevant_button.description = "✓ RELEVANT"

        elif label == "PARTIAL":
            partial_button.description = "✓ PARTIAL"

        elif label == "IRRELEVANT":
            irrelevant_button.description = "✓ IRRELEVANT"


# =============================================================================
# SAVE CURRENT DECISION
# =============================================================================

def save_decision(label):

    global current_index

    review_df.loc[
        review_df.index[current_index],
        "evidence_relevance"
    ] = label

    review_df.loc[
        review_df.index[current_index],
        "review_notes"
    ] = notes_box.value

    # Save immediately
    review_df.to_csv(
        WORKING_PATH,
        index=False
    )

    display_case()

    print(
        f"Saved {review_df.iloc[current_index]['golden_id']} "
        f"→ {label}"
    )


# =============================================================================
# BUTTON CALLBACKS
# =============================================================================

def on_relevant(button):
    save_decision("RELEVANT")


def on_partial(button):
    save_decision("PARTIAL")


def on_irrelevant(button):
    save_decision("IRRELEVANT")


def on_previous(button):

    global current_index

    if current_index > 0:

        current_index -= 1

        display_case()


def on_next(button):

    global current_index

    if current_index < len(review_df) - 1:

        current_index += 1

        display_case()


def on_save(button):

    review_df.to_csv(
        WORKING_PATH,
        index=False
    )

    print(
        "Progress saved successfully."
    )


# =============================================================================
# CONNECT BUTTONS
# =============================================================================

relevant_button.on_click(
    on_relevant
)

partial_button.on_click(
    on_partial
)

irrelevant_button.on_click(
    on_irrelevant
)

previous_button.on_click(
    on_previous
)

next_button.on_click(
    on_next
)

save_button.on_click(
    on_save
)


# =============================================================================
# REVIEW INSTRUCTIONS
# =============================================================================

instructions = widgets.HTML(
    """
    <div style="
        padding:15px;
        border-radius:8px;
        background:#f8fafc;
        border:1px solid #cbd5e1;
        margin-bottom:15px;
    ">

    <h3>How to review</h3>

    <p>
    Judge whether the <b>historical customer problem + support action</b>
    is useful for the current customer query.
    </p>

    <p>
    <b>RELEVANT</b> — Same or very similar problem, and the historical
    action is useful/applicable.
    </p>

    <p>
    <b>PARTIAL</b> — Related problem/evidence, but not completely
    applicable or requires adaptation.
    </p>

    <p>
    <b>IRRELEVANT</b> — Superficially similar, but the historical
    evidence is not useful for this case.
    </p>

    <p>
    <b>Important:</b> Do not judge only from the Cross-Encoder score.
    Read the customer query and historical evidence.
    </p>

    </div>
    """
)


# =============================================================================
# RENDER
# =============================================================================

display(
    instructions
)

display(
    progress_box
)

display(
    case_header
)

display(
    customer_box
)

display(
    historical_box
)

display(
    action_box
)

display(
    metadata_box
)

display(
    notes_box
)

display(
    widgets.HBox(
        [
            relevant_button,
            partial_button,
            irrelevant_button
        ]
    )
)

display(
    widgets.HBox(
        [
            previous_button,
            next_button,
            save_button
        ]
    )
)

# Initial case
display_case()

print(
    "\nReviewer ready."
)
print(
    "Your decision is saved immediately after clicking a label."
)

HTML(value='\n    <div style="\n        padding:15px;\n        border-radius:8px;\n        background:#f8fafc;…

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

Textarea(value='', description='Notes:', layout=Layout(height='80px', width='100%'), placeholder='Optional: br…


Reviewer ready.
Your decision is saved immediately after clicking a label.
Saved GOLD-0001 → PARTIAL


/tmp/ipykernel_2461/1018032381.py:347: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'PARTIAL' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review_df.loc[
/tmp/ipykernel_2461/1018032381.py:352: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review_df.loc[


In [37]:
# =============================================================================
# HIVER SUPPORT AGENT — INTERACTIVE HUMAN EVIDENCE REVIEWER
# FIXED VERSION — SAFE TEXT COLUMNS + RESUME SUPPORT
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets


# =============================================================================
# PATHS
# =============================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

EVAL_DIR = BASE_DIR / "evaluation"

WORKING_PATH = (
    EVAL_DIR /
    "top1_evidence_review_working.csv"
)

FINAL_PATH = (
    EVAL_DIR /
    "top1_evidence_review.csv"
)


# =============================================================================
# LOAD EXISTING WORK
# =============================================================================

review_df = pd.read_csv(
    WORKING_PATH
)


# =============================================================================
# FIX ANNOTATION COLUMN TYPES
# =============================================================================
#
# Important:
# These columns must be object/string capable because they will contain:
#
# evidence_relevance -> RELEVANT / PARTIAL / IRRELEVANT
# review_notes       -> text
#
# =============================================================================

if "evidence_relevance" not in review_df.columns:
    review_df["evidence_relevance"] = pd.Series(
        [None] * len(review_df),
        dtype="object"
    )
else:
    review_df["evidence_relevance"] = (
        review_df["evidence_relevance"]
        .astype("object")
    )

if "review_notes" not in review_df.columns:
    review_df["review_notes"] = pd.Series(
        [None] * len(review_df),
        dtype="object"
    )
else:
    review_df["review_notes"] = (
        review_df["review_notes"]
        .astype("object")
    )


# =============================================================================
# NORMALIZE EXISTING LABELS
# =============================================================================

review_df["evidence_relevance"] = (
    review_df["evidence_relevance"]
    .apply(
        lambda x:
        None
        if pd.isna(x)
        else str(x).strip().upper()
    )
)


# =============================================================================
# REVIEW STATE
# =============================================================================

current_index = 0


# =============================================================================
# FIND FIRST UNREVIEWED CASE
# =============================================================================

unreviewed_indices = (
    review_df[
        review_df["evidence_relevance"].isna()
    ]
    .index
    .tolist()
)

if len(unreviewed_indices) > 0:

    current_index = unreviewed_indices[0]

else:

    current_index = 0


# =============================================================================
# UI ELEMENTS
# =============================================================================

progress_box = widgets.HTML()

case_header = widgets.HTML()

customer_box = widgets.HTML()

historical_box = widgets.HTML()

action_box = widgets.HTML()

metadata_box = widgets.HTML()

notes_box = widgets.Textarea(
    value="",
    placeholder="Optional: briefly explain your decision...",
    description="Notes:",
    layout=widgets.Layout(
        width="100%",
        height="80px"
    )
)

relevant_button = widgets.Button(
    description="RELEVANT",
    button_style="success",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

partial_button = widgets.Button(
    description="PARTIAL",
    button_style="warning",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

irrelevant_button = widgets.Button(
    description="IRRELEVANT",
    button_style="danger",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

previous_button = widgets.Button(
    description="← Previous",
    layout=widgets.Layout(
        width="130px"
    )
)

next_button = widgets.Button(
    description="Next →",
    layout=widgets.Layout(
        width="130px"
    )
)

save_button = widgets.Button(
    description="Save Progress",
    button_style="info",
    layout=widgets.Layout(
        width="150px"
    )
)


# =============================================================================
# DISPLAY CURRENT CASE
# =============================================================================

def display_case():

    global current_index

    # ---------------------------------------------------------
    # Bounds
    # ---------------------------------------------------------

    current_index = max(
        0,
        min(
            current_index,
            len(review_df) - 1
        )
    )

    row = review_df.iloc[current_index]

    # ---------------------------------------------------------
    # Progress
    # ---------------------------------------------------------

    reviewed_count = int(
        review_df["evidence_relevance"]
        .notna()
        .sum()
    )

    total_count = len(review_df)

    progress_box.value = f"""
    <div style="
        padding:12px;
        border-radius:8px;
        background:#f1f5f9;
        margin-bottom:12px;
        font-size:16px;
    ">
        <b>Progress:</b>
        {reviewed_count} / {total_count} reviewed
        &nbsp;&nbsp;|&nbsp;&nbsp;
        Case {current_index + 1} / {total_count}
    </div>
    """

    # ---------------------------------------------------------
    # Case ID
    # ---------------------------------------------------------

    case_header.value = f"""
    <h2>{row['golden_id']}</h2>
    """

    # ---------------------------------------------------------
    # Customer query
    # ---------------------------------------------------------

    customer_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>👤 Customer Query</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['customer_query']}
    </p>

    </div>
    """

    # ---------------------------------------------------------
    # Historical problem
    # ---------------------------------------------------------

    historical_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>📚 Historical Customer Problem</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['customer_problem']}
    </p>

    </div>
    """

    # ---------------------------------------------------------
    # Historical action
    # ---------------------------------------------------------

    action_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>🛠️ Historical Support Action</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['support_action']}
    </p>

    </div>
    """

    # ---------------------------------------------------------
    # Metadata
    # ---------------------------------------------------------

    ce_score = (
        float(row["ce_score"])
        if pd.notna(row["ce_score"])
        else 0.0
    )

    metadata_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:15px;
    ">

    <b>Intent:</b> {row['intent']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Cross-Encoder Score:</b> {ce_score:.4f}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Resolution Type:</b> {row['resolution_type']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Evidence Quality:</b> {row['evidence_quality']}

    </div>
    """

    # ---------------------------------------------------------
    # Existing notes
    # ---------------------------------------------------------

    existing_notes = row["review_notes"]

    if pd.isna(existing_notes):
        notes_box.value = ""
    else:
        notes_box.value = str(existing_notes)

    # ---------------------------------------------------------
    # Existing decision
    # ---------------------------------------------------------

    existing_label = row["evidence_relevance"]

    relevant_button.description = "RELEVANT"
    partial_button.description = "PARTIAL"
    irrelevant_button.description = "IRRELEVANT"

    if pd.notna(existing_label):

        label = str(existing_label).upper()

        if label == "RELEVANT":
            relevant_button.description = "✓ RELEVANT"

        elif label == "PARTIAL":
            partial_button.description = "✓ PARTIAL"

        elif label == "IRRELEVANT":
            irrelevant_button.description = "✓ IRRELEVANT"


# =============================================================================
# SAVE DECISION
# =============================================================================

def save_decision(label):

    global current_index

    row_index = review_df.index[current_index]

    # ---------------------------------------------------------
    # Store decision
    # ---------------------------------------------------------

    review_df.at[
        row_index,
        "evidence_relevance"
    ] = str(label)

    # ---------------------------------------------------------
    # Store notes
    # ---------------------------------------------------------

    note_value = (
        notes_box.value.strip()
        if notes_box.value
        else ""
    )

    review_df.at[
        row_index,
        "review_notes"
    ] = note_value

    # ---------------------------------------------------------
    # Force correct column types
    # ---------------------------------------------------------

    review_df["evidence_relevance"] = (
        review_df["evidence_relevance"]
        .astype("object")
    )

    review_df["review_notes"] = (
        review_df["review_notes"]
        .astype("object")
    )

    # ---------------------------------------------------------
    # Save immediately
    # ---------------------------------------------------------

    review_df.to_csv(
        WORKING_PATH,
        index=False
    )

    print(
        f"Saved "
        f"{review_df.iloc[current_index]['golden_id']} "
        f"→ {label}"
    )

    # ---------------------------------------------------------
    # Automatically move to next UNREVIEWED case
    # ---------------------------------------------------------

    remaining = (
        review_df[
            review_df["evidence_relevance"].isna()
        ]
        .index
        .tolist()
    )

    if len(remaining) > 0:

        current_index = remaining[0]

    elif current_index < len(review_df) - 1:

        current_index += 1

    display_case()


# =============================================================================
# BUTTON CALLBACKS
# =============================================================================

def on_relevant(button):
    save_decision("RELEVANT")


def on_partial(button):
    save_decision("PARTIAL")


def on_irrelevant(button):
    save_decision("IRRELEVANT")


def on_previous(button):

    global current_index

    if current_index > 0:

        current_index -= 1

        display_case()


def on_next(button):

    global current_index

    if current_index < len(review_df) - 1:

        current_index += 1

        display_case()


def on_save(button):

    review_df.to_csv(
        WORKING_PATH,
        index=False
    )

    print(
        "Progress saved successfully."
    )


# =============================================================================
# CONNECT BUTTONS
# =============================================================================

relevant_button.on_click(
    on_relevant
)

partial_button.on_click(
    on_partial
)

irrelevant_button.on_click(
    on_irrelevant
)

previous_button.on_click(
    on_previous
)

next_button.on_click(
    on_next
)

save_button.on_click(
    on_save
)


# =============================================================================
# INSTRUCTIONS
# =============================================================================

instructions = widgets.HTML(
    """
    <div style="
        padding:15px;
        border-radius:8px;
        background:#f8fafc;
        border:1px solid #cbd5e1;
        margin-bottom:15px;
    ">

    <h3>How to review</h3>

    <p>
    Judge whether the <b>historical customer problem + support action</b>
    is useful for the current customer query.
    </p>

    <p>
    <b>RELEVANT</b> — Same or very similar problem, and the historical
    action is useful/applicable.
    </p>

    <p>
    <b>PARTIAL</b> — Related problem/evidence, but not completely
    applicable or requires adaptation.
    </p>

    <p>
    <b>IRRELEVANT</b> — Superficially similar, but the historical
    evidence is not useful for this case.
    </p>

    <p>
    <b>Important:</b> Do not judge only from the Cross-Encoder score.
    Read the customer query and historical evidence.
    </p>

    </div>
    """
)


# =============================================================================
# DISPLAY
# =============================================================================

display(instructions)

display(progress_box)

display(case_header)

display(customer_box)

display(historical_box)

display(action_box)

display(metadata_box)

display(notes_box)

display(
    widgets.HBox(
        [
            relevant_button,
            partial_button,
            irrelevant_button
        ]
    )
)

display(
    widgets.HBox(
        [
            previous_button,
            next_button,
            save_button
        ]
    )
)


# =============================================================================
# INITIAL DISPLAY
# =============================================================================

display_case()


# =============================================================================
# STATUS
# =============================================================================

reviewed_count = int(
    review_df["evidence_relevance"]
    .notna()
    .sum()
)

print(
    f"\nReviewer ready."
)

print(
    f"Existing reviewed cases: "
    f"{reviewed_count}/{len(review_df)}"
)

print(
    "Decisions are saved immediately."
)

HTML(value='\n    <div style="\n        padding:15px;\n        border-radius:8px;\n        background:#f8fafc;…

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

Textarea(value='', description='Notes:', layout=Layout(height='80px', width='100%'), placeholder='Optional: br…


Reviewer ready.
Existing reviewed cases: 1/200
Decisions are saved immediately.
Saved GOLD-0002 → RELEVANT
Saved GOLD-0003 → RELEVANT


In [40]:
# =============================================================================
# HIVER SUPPORT AGENT
# HUMAN EVIDENCE REVIEW — 50 CASE SAMPLE
# FINAL FIXED VERSION
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets


# =============================================================================
# PATHS
# =============================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

EVAL_DIR = BASE_DIR / "evaluation"

SOURCE_PATH = (
    EVAL_DIR / "top1_evidence_review_working.csv"
)

GOLDEN_PATH = (
    EVAL_DIR / "golden_evaluation_set_frozen.csv"
)

SAMPLE_PATH = (
    EVAL_DIR / "top1_evidence_review_50_sample.csv"
)


# =============================================================================
# LOAD SOURCE REVIEW FILE
# =============================================================================

source_df = pd.read_csv(
    SOURCE_PATH
)

golden_df = pd.read_csv(
    GOLDEN_PATH
)


# =============================================================================
# VALIDATION
# =============================================================================

assert len(source_df) == 200
assert len(golden_df) == 200

assert "golden_id" in source_df.columns
assert "golden_id" in golden_df.columns

assert "risk_level" in golden_df.columns
assert "intent" in golden_df.columns


# =============================================================================
# PRESERVE EXISTING ANNOTATIONS
# =============================================================================

source_df["evidence_relevance"] = (
    source_df["evidence_relevance"]
    .astype("object")
)

source_df["review_notes"] = (
    source_df["review_notes"]
    .astype("object")
)


# =============================================================================
# ADD RISK LEVEL FROM FROZEN GOLDEN SET
# =============================================================================

golden_metadata = golden_df[
    [
        "golden_id",
        "risk_level",
        "intent",
    ]
].copy()


# Remove duplicate metadata if any
golden_metadata = (
    golden_metadata
    .drop_duplicates(
        subset=["golden_id"]
    )
)


# Merge risk + authoritative intent
source_df = source_df.drop(
    columns=[
        "risk_level"
    ],
    errors="ignore"
)


source_df = source_df.merge(
    golden_metadata,
    on="golden_id",
    how="left",
    suffixes=(
        "",
        "_golden"
    )
)


# =============================================================================
# VALIDATE MERGE
# =============================================================================

missing_risk = int(
    source_df["risk_level"]
    .isna()
    .sum()
)

print("=" * 80)
print("RISK METADATA CHECK")
print("=" * 80)

print(
    f"Missing risk levels: {missing_risk}"
)

assert missing_risk == 0


# =============================================================================
# EXISTING REVIEWED CASES
# =============================================================================

existing_ids = [
    "GOLD-0001",
    "GOLD-0002",
    "GOLD-0003",
]


existing_df = source_df[
    source_df["golden_id"]
    .isin(existing_ids)
].copy()


print("\n" + "=" * 80)
print("EXISTING ANNOTATIONS")
print("=" * 80)

display(
    existing_df[
        [
            "golden_id",
            "risk_level",
            "intent",
            "evidence_relevance",
            "review_notes",
        ]
    ]
)


# =============================================================================
# REMAINING CASES
# =============================================================================

remaining_df = source_df[
    ~source_df["golden_id"]
    .isin(existing_ids)
].copy()


# =============================================================================
# TARGET SAMPLE
# =============================================================================

TARGET_SAMPLE_SIZE = 50

ALREADY_REVIEWED = len(
    existing_df
)

ADDITIONAL_N = (
    TARGET_SAMPLE_SIZE
    - ALREADY_REVIEWED
)


# =============================================================================
# RISK-STRATIFIED SAMPLING
# =============================================================================
#
# We preserve the 3 existing annotations.
#
# The remaining 47 cases are sampled approximately according to the
# risk distribution of the remaining golden cases.
#
# =============================================================================

risk_counts = (
    remaining_df[
        "risk_level"
    ]
    .value_counts()
)

total_remaining = len(
    remaining_df
)


# Calculate proportional target
raw_targets = {}

for risk in risk_counts.index:

    proportion = (
        risk_counts[risk]
        / total_remaining
    )

    raw_targets[risk] = (
        proportion
        * ADDITIONAL_N
    )


# Floor targets
risk_targets = {}

for risk, value in raw_targets.items():

    risk_targets[risk] = int(
        np.floor(value)
    )


# Distribute remaining slots
assigned = sum(
    risk_targets.values()
)

slots_left = (
    ADDITIONAL_N
    - assigned
)


fractional_order = sorted(
    raw_targets.keys(),
    key=lambda risk:
        raw_targets[risk]
        - risk_targets[risk],
    reverse=True
)


for i in range(slots_left):

    risk = fractional_order[i]

    risk_targets[risk] += 1


print("\n" + "=" * 80)
print("RISK SAMPLING TARGET")
print("=" * 80)

for risk in [
    "LOW",
    "MEDIUM",
    "HIGH"
]:

    if risk in risk_targets:

        print(
            f"{risk:<10}: "
            f"{risk_targets[risk]}"
        )


# =============================================================================
# SAMPLE WITHIN EACH RISK LEVEL
# =============================================================================

sample_parts = []

RANDOM_STATE = 42


for risk, target in risk_targets.items():

    pool = remaining_df[
        remaining_df["risk_level"]
        == risk
    ].copy()

    target = min(
        target,
        len(pool)
    )

    if target <= 0:
        continue

    sampled = pool.sample(
        n=target,
        random_state=RANDOM_STATE
    )

    sample_parts.append(
        sampled
    )


sampled_df = pd.concat(
    sample_parts,
    ignore_index=True
)


# =============================================================================
# SAFETY FILL IF NECESSARY
# =============================================================================

if len(sampled_df) < ADDITIONAL_N:

    selected_ids = set(
        sampled_df["golden_id"]
    )

    fill_pool = remaining_df[
        ~remaining_df["golden_id"]
        .isin(selected_ids)
    ].copy()

    needed = (
        ADDITIONAL_N
        - len(sampled_df)
    )

    fill = fill_pool.sample(
        n=needed,
        random_state=RANDOM_STATE
    )

    sampled_df = pd.concat(
        [
            sampled_df,
            fill
        ],
        ignore_index=True
    )


# =============================================================================
# COMBINE WITH EXISTING ANNOTATIONS
# =============================================================================

review_df = pd.concat(
    [
        existing_df,
        sampled_df
    ],
    ignore_index=True
)


# =============================================================================
# FINAL SAMPLE VALIDATION
# =============================================================================

assert len(review_df) == 50

assert (
    review_df["golden_id"]
    .nunique()
    == 50
)

assert (
    set(existing_ids)
    .issubset(
        set(review_df["golden_id"])
    )
)


# =============================================================================
# PRESERVE EXISTING LABELS
# =============================================================================

existing_check = review_df[
    review_df["golden_id"]
    .isin(existing_ids)
][
    [
        "golden_id",
        "evidence_relevance"
    ]
]


assert (
    existing_check
    .set_index("golden_id")
    .loc["GOLD-0001", "evidence_relevance"]
    == "PARTIAL"
)

assert (
    existing_check
    .set_index("golden_id")
    .loc["GOLD-0002", "evidence_relevance"]
    == "RELEVANT"
)

assert (
    existing_check
    .set_index("golden_id")
    .loc["GOLD-0003", "evidence_relevance"]
    == "RELEVANT"
)


# =============================================================================
# SAVE SAMPLE
# =============================================================================

review_df.to_csv(
    SAMPLE_PATH,
    index=False
)


# =============================================================================
# SAMPLE SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("50-CASE REVIEW SAMPLE CREATED")
print("=" * 80)

print(
    f"Total cases       : {len(review_df)}"
)

print(
    f"Already reviewed  : "
    f"{review_df['evidence_relevance'].notna().sum()}"
)

print(
    f"Remaining         : "
    f"{review_df['evidence_relevance'].isna().sum()}"
)

print("\nRisk distribution:")

print(
    review_df[
        "risk_level"
    ]
    .value_counts()
    .to_string()
)

print("\nIntent distribution:")

print(
    review_df[
        "intent"
    ]
    .value_counts()
    .to_string()
)

print(
    f"\nSaved sample:\n{SAMPLE_PATH}"
)


# =============================================================================
# INTERACTIVE REVIEWER
# =============================================================================

review_df = pd.read_csv(
    SAMPLE_PATH
)

review_df["evidence_relevance"] = (
    review_df["evidence_relevance"]
    .astype("object")
)

review_df["review_notes"] = (
    review_df["review_notes"]
    .astype("object")
)


# =============================================================================
# FIND FIRST UNREVIEWED CASE
# =============================================================================

unreviewed = (
    review_df[
        review_df["evidence_relevance"]
        .isna()
    ]
    .index
    .tolist()
)

if unreviewed:

    current_index = unreviewed[0]

else:

    current_index = 0


# =============================================================================
# UI
# =============================================================================

progress_box = widgets.HTML()

case_header = widgets.HTML()

customer_box = widgets.HTML()

historical_box = widgets.HTML()

action_box = widgets.HTML()

metadata_box = widgets.HTML()

notes_box = widgets.Textarea(
    value="",
    placeholder="Optional: briefly explain your decision...",
    description="Notes:",
    layout=widgets.Layout(
        width="100%",
        height="80px"
    )
)

relevant_button = widgets.Button(
    description="RELEVANT",
    button_style="success",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

partial_button = widgets.Button(
    description="PARTIAL",
    button_style="warning",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

irrelevant_button = widgets.Button(
    description="IRRELEVANT",
    button_style="danger",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

previous_button = widgets.Button(
    description="← Previous",
    layout=widgets.Layout(
        width="130px"
    )
)

next_button = widgets.Button(
    description="Next →",
    layout=widgets.Layout(
        width="130px"
    )
)

save_button = widgets.Button(
    description="Save Progress",
    button_style="info",
    layout=widgets.Layout(
        width="150px"
    )
)


# =============================================================================
# DISPLAY CASE
# =============================================================================

def display_case():

    global current_index

    current_index = max(
        0,
        min(
            current_index,
            len(review_df) - 1
        )
    )

    row = review_df.iloc[
        current_index
    ]

    reviewed_count = int(
        review_df[
            "evidence_relevance"
        ]
        .notna()
        .sum()
    )

    total_count = len(
        review_df
    )

    progress_box.value = f"""
    <div style="
        padding:12px;
        border-radius:8px;
        background:#f1f5f9;
        margin-bottom:12px;
        font-size:16px;
    ">
        <b>Progress:</b>
        {reviewed_count} / {total_count} reviewed
        &nbsp;&nbsp;|&nbsp;&nbsp;
        Case {current_index + 1} / {total_count}
    </div>
    """

    case_header.value = f"""
    <h2>{row['golden_id']}</h2>
    """

    customer_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>👤 Customer Query</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['customer_query']}
    </p>

    </div>
    """

    historical_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>📚 Historical Customer Problem</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['customer_problem']}
    </p>

    </div>
    """

    action_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>🛠️ Historical Support Action</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['support_action']}
    </p>

    </div>
    """

    ce_score = (
        float(row["ce_score"])
        if pd.notna(row["ce_score"])
        else 0.0
    )

    metadata_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:15px;
    ">

    <b>Intent:</b> {row['intent']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Cross-Encoder Score:</b> {ce_score:.4f}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Resolution Type:</b> {row['resolution_type']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Evidence Quality:</b> {row['evidence_quality']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Risk:</b> {row['risk_level']}

    </div>
    """

    if pd.isna(
        row["review_notes"]
    ):

        notes_box.value = ""

    else:

        notes_box.value = str(
            row["review_notes"]
        )

    relevant_button.description = (
        "RELEVANT"
    )

    partial_button.description = (
        "PARTIAL"
    )

    irrelevant_button.description = (
        "IRRELEVANT"
    )

    existing_label = (
        row["evidence_relevance"]
    )

    if pd.notna(existing_label):

        label = str(
            existing_label
        ).upper()

        if label == "RELEVANT":

            relevant_button.description = (
                "✓ RELEVANT"
            )

        elif label == "PARTIAL":

            partial_button.description = (
                "✓ PARTIAL"
            )

        elif label == "IRRELEVANT":

            irrelevant_button.description = (
                "✓ IRRELEVANT"
            )


# =============================================================================
# SAVE DECISION
# =============================================================================

def save_decision(label):

    global current_index

    row_index = review_df.index[
        current_index
    ]

    review_df.at[
        row_index,
        "evidence_relevance"
    ] = label

    review_df.at[
        row_index,
        "review_notes"
    ] = (
        notes_box.value.strip()
        if notes_box.value
        else ""
    )

    review_df.to_csv(
        SAMPLE_PATH,
        index=False
    )

    print(
        f"Saved "
        f"{review_df.iloc[current_index]['golden_id']}"
        f" → {label}"
    )

    remaining = (
        review_df[
            review_df[
                "evidence_relevance"
            ].isna()
        ]
        .index
        .tolist()
    )

    if remaining:

        current_index = remaining[0]

    display_case()


# =============================================================================
# BUTTON CALLBACKS
# =============================================================================

def on_relevant(button):

    save_decision(
        "RELEVANT"
    )


def on_partial(button):

    save_decision(
        "PARTIAL"
    )


def on_irrelevant(button):

    save_decision(
        "IRRELEVANT"
    )


def on_previous(button):

    global current_index

    if current_index > 0:

        current_index -= 1

        display_case()


def on_next(button):

    global current_index

    if current_index < len(review_df) - 1:

        current_index += 1

        display_case()


def on_save(button):

    review_df.to_csv(
        SAMPLE_PATH,
        index=False
    )

    print(
        "Progress saved successfully."
    )


# =============================================================================
# CONNECT BUTTONS
# =============================================================================

relevant_button.on_click(
    on_relevant
)

partial_button.on_click(
    on_partial
)

irrelevant_button.on_click(
    on_irrelevant
)

previous_button.on_click(
    on_previous
)

next_button.on_click(
    on_next
)

save_button.on_click(
    on_save
)


# =============================================================================
# INSTRUCTIONS
# =============================================================================

instructions = widgets.HTML(
    """
    <div style="
        padding:15px;
        border-radius:8px;
        background:#f8fafc;
        border:1px solid #cbd5e1;
        margin-bottom:15px;
    ">

    <h3>Human Evidence Review — 50 Case Sample</h3>

    <p>
    Judge whether the <b>historical customer problem + support action</b>
    is useful for the current customer query.
    </p>

    <p>
    <b>RELEVANT</b> — Same or very similar problem, and the historical
    action is useful/applicable.
    </p>

    <p>
    <b>PARTIAL</b> — Related problem/evidence, but not completely
    applicable or requires adaptation.
    </p>

    <p>
    <b>IRRELEVANT</b> — Superficially similar, but the historical
    evidence is not useful for this case.
    </p>

    <p>
    <b>Important:</b> Do not judge only from the Cross-Encoder score.
    Read the customer query and historical evidence.
    </p>

    <p>
    Your decision is saved immediately after clicking a label.
    </p>

    </div>
    """
)


# =============================================================================
# DISPLAY
# =============================================================================

display(instructions)

display(progress_box)

display(case_header)

display(customer_box)

display(historical_box)

display(action_box)

display(metadata_box)

display(notes_box)

display(
    widgets.HBox(
        [
            relevant_button,
            partial_button,
            irrelevant_button
        ]
    )
)

display(
    widgets.HBox(
        [
            previous_button,
            next_button,
            save_button
        ]
    )
)


# =============================================================================
# INITIAL DISPLAY
# =============================================================================

display_case()


# =============================================================================
# FINAL STATUS
# =============================================================================

print(
    "\nReviewer ready."
)

print(
    f"Review sample: {len(review_df)} cases"
)

print(
    f"Existing annotations preserved: "
    f"{review_df['evidence_relevance'].notna().sum()}"
)

print(
    "Decisions are saved immediately."
)

RISK METADATA CHECK
Missing risk levels: 0

EXISTING ANNOTATIONS


,golden_id,risk_level,intent,evidence_relevance,review_notes
0,GOLD-0001,MEDIUM,PAYMENT_BILLING,PARTIAL,NaN
1,GOLD-0002,LOW,ORDER_DELIVERY,RELEVANT,NaN
2,GOLD-0003,LOW,ORDER_DELIVERY,RELEVANT,NaN



RISK SAMPLING TARGET
LOW       : 25
MEDIUM    : 16
HIGH      : 6

50-CASE REVIEW SAMPLE CREATED
Total cases       : 50
Already reviewed  : 3
Remaining         : 47

Risk distribution:
risk_level
LOW       27
MEDIUM    17
HIGH       6

Intent distribution:
intent
ORDER_DELIVERY        17
GENERAL_SUPPORT        9
PAYMENT_BILLING        4
RETURNS_REFUNDS        4
GENERAL_SOCIAL         3
DEVICE_TECHNICAL       2
PURCHASE_CONTROL       2
ORDER_CANCELLATION     2
PREORDER_DELIVERY      2
ACCOUNT_ACCESS         2
DIGITAL_CONTENT        1
ORDER_TRACKING         1
ACCOUNT_SECURITY       1

Saved sample:
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/top1_evidence_review_50_sample.csv


HTML(value='\n    <div style="\n        padding:15px;\n        border-radius:8px;\n        background:#f8fafc;…

HTML(value='')

...

HTML(value='')

HTML(value='')

HTML(value='')

Textarea(value='', description='Notes:', layout=Layout(height='80px', width='100%'), placeholder='Optional: br…


Reviewer ready.
Review sample: 50 cases
Existing annotations preserved: 3
Decisions are saved immediately.
Saved GOLD-0055 → IRRELEVANT


In [42]:
# =============================================================================
# DIAGNOSTIC — CHECK 50-CASE REVIEW COMPLETION
# =============================================================================

import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

EVAL_DIR = BASE_DIR / "evaluation"

SAMPLE_PATH = (
    EVAL_DIR /
    "top1_evidence_review_50_sample.csv"
)

df = pd.read_csv(
    SAMPLE_PATH
)

print("=" * 80)
print("50-CASE REVIEW STATUS")
print("=" * 80)

print(
    f"Total cases: {len(df)}"
)

print("\nRaw evidence_relevance values:")

print(
    df["evidence_relevance"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n" + "=" * 80)
print("ANNOTATION COUNTS")
print("=" * 80)

relevant = int(
    (
        df["evidence_relevance"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        == "RELEVANT"
    ).sum()
)

partial = int(
    (
        df["evidence_relevance"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        == "PARTIAL"
    ).sum()
)

irrelevant = int(
    (
        df["evidence_relevance"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        == "IRRELEVANT"
    ).sum()
)

unreviewed = (
    len(df)
    - relevant
    - partial
    - irrelevant
)

print(
    f"RELEVANT   : {relevant}"
)

print(
    f"PARTIAL    : {partial}"
)

print(
    f"IRRELEVANT : {irrelevant}"
)

print(
    f"UNREVIEWED : {unreviewed}"
)

print("\n" + "=" * 80)
print("UNREVIEWED CASES")
print("=" * 80)

if unreviewed > 0:

    unanswered = df[
        ~df["evidence_relevance"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(
            [
                "RELEVANT",
                "PARTIAL",
                "IRRELEVANT"
            ]
        )
    ]

    print(
        unanswered[
            [
                "golden_id",
                "intent",
                "risk_level"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "All 50 cases are reviewed."
    )

print("\n" + "=" * 80)
print("STATUS")
print("=" * 80)

if unreviewed == 0:

    print(
        "✅ 50/50 REVIEW COMPLETE"
    )

else:

    print(
        f"⚠️ {unreviewed} cases still need review."
    )

50-CASE REVIEW STATUS
Total cases: 50

Raw evidence_relevance values:
evidence_relevance
NaN           46
RELEVANT       2
PARTIAL        1
IRRELEVANT     1

ANNOTATION COUNTS
RELEVANT   : 2
PARTIAL    : 1
IRRELEVANT : 1
UNREVIEWED : 46

UNREVIEWED CASES
golden_id             intent risk_level
GOLD-0122     GENERAL_SOCIAL        LOW
GOLD-0119    DIGITAL_CONTENT        LOW
GOLD-0099     ORDER_DELIVERY        LOW
GOLD-0080   PURCHASE_CONTROL        LOW
GOLD-0170     ORDER_DELIVERY        LOW
GOLD-0168    GENERAL_SUPPORT        LOW
GOLD-0084     ORDER_DELIVERY        LOW
GOLD-0018  PREORDER_DELIVERY        LOW
GOLD-0004     GENERAL_SOCIAL        LOW
GOLD-0030 ORDER_CANCELLATION        LOW
GOLD-0056   DEVICE_TECHNICAL        LOW
GOLD-0162     ORDER_TRACKING        LOW
GOLD-0177    PAYMENT_BILLING        LOW
GOLD-0144    GENERAL_SUPPORT        LOW
GOLD-0010     ORDER_DELIVERY        LOW
GOLD-0151    GENERAL_SUPPORT        LOW
GOLD-0060     ORDER_DELIVERY        LOW
GOLD-0022    GENERAL_SUPP

In [43]:
# =============================================================================
# HIVER SUPPORT AGENT
# RESUME EXISTING 50-CASE HUMAN EVIDENCE REVIEW
# =============================================================================

import pandas as pd
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets


# =============================================================================
# PATH
# =============================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
)

EVAL_DIR = BASE_DIR / "evaluation"

SAMPLE_PATH = (
    EVAL_DIR /
    "top1_evidence_review_50_sample.csv"
)


# =============================================================================
# LOAD EXISTING 50-CASE SAMPLE
# =============================================================================

review_df = pd.read_csv(
    SAMPLE_PATH
)

assert len(review_df) == 50

# Make annotation columns text-compatible
review_df["evidence_relevance"] = (
    review_df["evidence_relevance"]
    .astype("object")
)

review_df["review_notes"] = (
    review_df["review_notes"]
    .astype("object")
)


# =============================================================================
# NORMALIZE EXISTING LABELS
# =============================================================================

review_df["evidence_relevance"] = (
    review_df["evidence_relevance"]
    .apply(
        lambda x:
        None
        if pd.isna(x)
        else str(x).strip().upper()
    )
)


# =============================================================================
# FIND FIRST UNREVIEWED CASE
# =============================================================================

unreviewed_indices = (
    review_df[
        review_df["evidence_relevance"].isna()
    ]
    .index
    .tolist()
)

if len(unreviewed_indices) > 0:

    current_index = unreviewed_indices[0]

else:

    current_index = 0


# =============================================================================
# CURRENT STATUS
# =============================================================================

reviewed_count = int(
    review_df[
        "evidence_relevance"
    ]
    .notna()
    .sum()
)

remaining_count = (
    len(review_df)
    - reviewed_count
)


print("=" * 80)
print("RESUMING HUMAN EVIDENCE REVIEW")
print("=" * 80)

print(
    f"Total cases       : {len(review_df)}"
)

print(
    f"Already reviewed  : {reviewed_count}"
)

print(
    f"Remaining         : {remaining_count}"
)

print(
    f"Starting index    : {current_index + 1}"
)

print(
    f"Starting case     : "
    f"{review_df.iloc[current_index]['golden_id']}"
)


# =============================================================================
# UI ELEMENTS
# =============================================================================

progress_box = widgets.HTML()

case_header = widgets.HTML()

customer_box = widgets.HTML()

historical_box = widgets.HTML()

action_box = widgets.HTML()

metadata_box = widgets.HTML()

notes_box = widgets.Textarea(
    value="",
    placeholder="Optional: briefly explain your decision...",
    description="Notes:",
    layout=widgets.Layout(
        width="100%",
        height="80px"
    )
)

relevant_button = widgets.Button(
    description="RELEVANT",
    button_style="success",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

partial_button = widgets.Button(
    description="PARTIAL",
    button_style="warning",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

irrelevant_button = widgets.Button(
    description="IRRELEVANT",
    button_style="danger",
    layout=widgets.Layout(
        width="180px",
        height="45px"
    )
)

previous_button = widgets.Button(
    description="← Previous",
    layout=widgets.Layout(
        width="130px"
    )
)

next_button = widgets.Button(
    description="Next →",
    layout=widgets.Layout(
        width="130px"
    )
)

save_button = widgets.Button(
    description="Save Progress",
    button_style="info",
    layout=widgets.Layout(
        width="150px"
    )
)


# =============================================================================
# DISPLAY CURRENT CASE
# =============================================================================

def display_case():

    global current_index

    current_index = max(
        0,
        min(
            current_index,
            len(review_df) - 1
        )
    )

    row = review_df.iloc[
        current_index
    ]

    reviewed_count = int(
        review_df[
            "evidence_relevance"
        ]
        .notna()
        .sum()
    )

    total_count = len(
        review_df
    )

    progress_box.value = f"""
    <div style="
        padding:12px;
        border-radius:8px;
        background:#f1f5f9;
        margin-bottom:12px;
        font-size:16px;
    ">
        <b>Progress:</b>
        {reviewed_count} / {total_count} reviewed
        &nbsp;&nbsp;|&nbsp;&nbsp;
        Case {current_index + 1} / {total_count}
    </div>
    """

    case_header.value = f"""
    <h2>{row['golden_id']}</h2>
    """

    customer_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>👤 Customer Query</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['customer_query']}
    </p>

    </div>
    """

    historical_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>📚 Historical Customer Problem</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['customer_problem']}
    </p>

    </div>
    """

    action_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:12px;
    ">

    <h3>🛠️ Historical Support Action</h3>

    <p style="
        font-size:17px;
        line-height:1.6;
    ">
        {row['support_action']}
    </p>

    </div>
    """

    ce_score = (
        float(row["ce_score"])
        if pd.notna(row["ce_score"])
        else 0.0
    )

    metadata_box.value = f"""
    <div style="
        border:1px solid #cbd5e1;
        border-radius:8px;
        padding:15px;
        margin-bottom:15px;
    ">

    <b>Intent:</b> {row['intent']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Cross-Encoder Score:</b> {ce_score:.4f}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Resolution Type:</b> {row['resolution_type']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Evidence Quality:</b> {row['evidence_quality']}
    &nbsp;&nbsp; | &nbsp;&nbsp;

    <b>Risk:</b> {row['risk_level']}

    </div>
    """

    if pd.isna(row["review_notes"]):

        notes_box.value = ""

    else:

        notes_box.value = str(
            row["review_notes"]
        )

    # Reset button labels
    relevant_button.description = "RELEVANT"
    partial_button.description = "PARTIAL"
    irrelevant_button.description = "IRRELEVANT"

    # Highlight existing decision
    existing_label = row[
        "evidence_relevance"
    ]

    if pd.notna(existing_label):

        label = str(
            existing_label
        ).upper()

        if label == "RELEVANT":

            relevant_button.description = (
                "✓ RELEVANT"
            )

        elif label == "PARTIAL":

            partial_button.description = (
                "✓ PARTIAL"
            )

        elif label == "IRRELEVANT":

            irrelevant_button.description = (
                "✓ IRRELEVANT"
            )


# =============================================================================
# SAVE DECISION
# =============================================================================

def save_decision(label):

    global current_index

    row_index = review_df.index[
        current_index
    ]

    # Save label
    review_df.at[
        row_index,
        "evidence_relevance"
    ] = label

    # Save notes
    review_df.at[
        row_index,
        "review_notes"
    ] = (
        notes_box.value.strip()
        if notes_box.value
        else ""
    )

    # Save immediately
    review_df.to_csv(
        SAMPLE_PATH,
        index=False
    )

    print(
        f"Saved "
        f"{review_df.iloc[current_index]['golden_id']}"
        f" → {label}"
    )

    # Find next unanswered case
    remaining = (
        review_df[
            review_df[
                "evidence_relevance"
            ].isna()
        ]
        .index
        .tolist()
    )

    if remaining:

        current_index = remaining[0]

    display_case()


# =============================================================================
# BUTTON CALLBACKS
# =============================================================================

def on_relevant(button):

    save_decision(
        "RELEVANT"
    )


def on_partial(button):

    save_decision(
        "PARTIAL"
    )


def on_irrelevant(button):

    save_decision(
        "IRRELEVANT"
    )


def on_previous(button):

    global current_index

    if current_index > 0:

        current_index -= 1

        display_case()


def on_next(button):

    global current_index

    if current_index < len(review_df) - 1:

        current_index += 1

        display_case()


def on_save(button):

    review_df.to_csv(
        SAMPLE_PATH,
        index=False
    )

    print(
        "Progress saved successfully."
    )


# =============================================================================
# CONNECT BUTTONS
# =============================================================================

relevant_button.on_click(
    on_relevant
)

partial_button.on_click(
    on_partial
)

irrelevant_button.on_click(
    on_irrelevant
)

previous_button.on_click(
    on_previous
)

next_button.on_click(
    on_next
)

save_button.on_click(
    on_save
)


# =============================================================================
# INSTRUCTIONS
# =============================================================================

instructions = widgets.HTML(
    """
    <div style="
        padding:15px;
        border-radius:8px;
        background:#f8fafc;
        border:1px solid #cbd5e1;
        margin-bottom:15px;
    ">

    <h3>Human Evidence Review — Resume</h3>

    <p>
    <b>RELEVANT</b> — Same or very similar problem, and the historical
    action is useful/applicable.
    </p>

    <p>
    <b>PARTIAL</b> — Related evidence, but not completely applicable
    or requires adaptation.
    </p>

    <p>
    <b>IRRELEVANT</b> — Superficially similar, but the historical
    evidence is not useful for this case.
    </p>

    <p>
    Your decision is saved immediately.
    The reviewer automatically moves to the next unanswered case.
    </p>

    </div>
    """
)


# =============================================================================
# DISPLAY
# =============================================================================

display(instructions)

display(progress_box)

display(case_header)

display(customer_box)

display(historical_box)

display(action_box)

display(metadata_box)

display(notes_box)

display(
    widgets.HBox(
        [
            relevant_button,
            partial_button,
            irrelevant_button
        ]
    )
)

display(
    widgets.HBox(
        [
            previous_button,
            next_button,
            save_button
        ]
    )
)


# =============================================================================
# INITIAL DISPLAY
# =============================================================================

display_case()


print("\nReviewer ready.")
print(
    f"Existing reviewed cases: "
    f"{reviewed_count}/50"
)
print(
    f"Remaining cases: "
    f"{remaining_count}"
)

RESUMING HUMAN EVIDENCE REVIEW
Total cases       : 50
Already reviewed  : 4
Remaining         : 46
Starting index    : 5
Starting case     : GOLD-0122


HTML(value='\n    <div style="\n        padding:15px;\n        border-radius:8px;\n        background:#f8fafc;…

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

HTML(value='')

Textarea(value='', description='Notes:', layout=Layout(height='80px', width='100%'), placeholder='Optional: br…


Reviewer ready.
Existing reviewed cases: 4/50
Remaining cases: 46
Saved GOLD-0122 → RELEVANT
Saved GOLD-0119 → RELEVANT
Saved GOLD-0099 → RELEVANT
Saved GOLD-0080 → RELEVANT
Saved GOLD-0170 → PARTIAL
Saved GOLD-0168 → RELEVANT
Saved GOLD-0084 → RELEVANT
Saved GOLD-0018 → RELEVANT
Saved GOLD-0004 → RELEVANT
Saved GOLD-0030 → PARTIAL
Saved GOLD-0056 → RELEVANT
Saved GOLD-0162 → RELEVANT
Saved GOLD-0177 → RELEVANT
Saved GOLD-0144 → RELEVANT
Saved GOLD-0010 → RELEVANT
Saved GOLD-0151 → RELEVANT
Saved GOLD-0060 → RELEVANT
Saved GOLD-0022 → RELEVANT
Saved GOLD-0051 → RELEVANT
Saved GOLD-0185 → RELEVANT
Saved GOLD-0102 → RELEVANT
Saved GOLD-0036 → PARTIAL
Saved GOLD-0143 → PARTIAL
Saved GOLD-0079 → RELEVANT
Saved GOLD-0109 → PARTIAL
Saved GOLD-0048 → RELEVANT
Saved GOLD-0023 → RELEVANT
Saved GOLD-0038 → RELEVANT
Saved GOLD-0137 → RELEVANT
Saved GOLD-0128 → RELEVANT
Saved GOLD-0186 → RELEVANT
Saved GOLD-0025 → PARTIAL
Saved GOLD-0192 → PARTIAL
Saved GOLD-0043 → PARTIAL
Saved GOLD-0077 → RELEV

In [44]:
# ============================================================
# FINAL HUMAN EVIDENCE REVIEW METRICS — 50 CASES
# ============================================================

import os
import json
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")

SAMPLE_PATH = os.path.join(EVAL_DIR, "top1_evidence_review_50_sample.csv")
OUTPUT_CSV = os.path.join(EVAL_DIR, "top1_evidence_review_50_final.csv")
SUMMARY_JSON = os.path.join(EVAL_DIR, "human_evidence_review_50_summary.json")

# ------------------------------------------------------------
# 1. Load completed 50-case review
# ------------------------------------------------------------

df = pd.read_csv(SAMPLE_PATH)

print("Loaded:", SAMPLE_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. Normalize review labels
# ------------------------------------------------------------

LABEL_COL = "evidence_relevance"

if LABEL_COL not in df.columns:
    raise ValueError(
        f"Expected '{LABEL_COL}' column not found.\n"
        f"Available columns: {df.columns.tolist()}"
    )

df[LABEL_COL] = (
    df[LABEL_COL]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

VALID_LABELS = {"RELEVANT", "PARTIAL", "IRRELEVANT"}

reviewed = df[df[LABEL_COL].isin(VALID_LABELS)].copy()
unreviewed = df[~df[LABEL_COL].isin(VALID_LABELS)].copy()

print("\n" + "=" * 60)
print("HUMAN EVIDENCE REVIEW STATUS")
print("=" * 60)

print(f"Total cases     : {len(df)}")
print(f"Reviewed cases  : {len(reviewed)}")
print(f"Unreviewed      : {len(unreviewed)}")

if len(unreviewed) > 0:
    print("\nWARNING: Some cases are still unreviewed.")
    print(unreviewed[["golden_id", LABEL_COL]].head(20))
    raise ValueError("Review is not complete. Expected 50 reviewed cases.")

# ------------------------------------------------------------
# 3. Calculate final metrics
# ------------------------------------------------------------

counts = reviewed[LABEL_COL].value_counts()

relevant = int(counts.get("RELEVANT", 0))
partial = int(counts.get("PARTIAL", 0))
irrelevant = int(counts.get("IRRELEVANT", 0))
total = len(reviewed)

useful_evidence = relevant + partial

relevant_rate = relevant / total
useful_rate = useful_evidence / total
irrelevant_rate = irrelevant / total
partial_rate = partial / total

# ------------------------------------------------------------
# 4. Print results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL 50-CASE HUMAN EVIDENCE REVIEW")
print("=" * 60)

print(f"RELEVANT       : {relevant:2d} ({relevant_rate:.2%})")
print(f"PARTIAL        : {partial:2d} ({partial_rate:.2%})")
print(f"IRRELEVANT     : {irrelevant:2d} ({irrelevant_rate:.2%})")
print("-" * 60)
print(f"USEFUL EVIDENCE: {useful_evidence:2d} ({useful_rate:.2%})")
print(f"TOTAL REVIEWED : {total:2d}")
print("=" * 60)

# ------------------------------------------------------------
# 5. Save clean reviewed artifact
# ------------------------------------------------------------

reviewed.to_csv(OUTPUT_CSV, index=False)

summary = {
    "evaluation_name": "Human Evidence Relevance Review",
    "sample_size": total,
    "reviewed_cases": total,
    "labels": {
        "RELEVANT": relevant,
        "PARTIAL": partial,
        "IRRELEVANT": irrelevant
    },
    "metrics": {
        "top1_relevant_rate": round(relevant_rate, 4),
        "useful_evidence_rate": round(useful_rate, 4),
        "partial_rate": round(partial_rate, 4),
        "irrelevant_rate": round(irrelevant_rate, 4)
    },
    "definitions": {
        "RELEVANT": "Historical case addresses a similar customer problem and the historical support action is directly useful.",
        "PARTIAL": "Historical case is related, but the evidence requires adaptation or does not fully address the current problem.",
        "IRRELEVANT": "Historical case is not useful for handling the current customer problem."
    },
    "note": "50-case stratified human evidence review. These labels are manually reviewed annotations, not automatically generated labels."
}

with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved:")
print(OUTPUT_CSV)
print(SUMMARY_JSON)

print("\n✅ HUMAN EVIDENCE REVIEW COMPLETE — 50/50")

Loaded: /content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/top1_evidence_review_50_sample.csv
Shape: (50, 14)
Columns: ['golden_id', 'customer_query', 'intent', 'case_id', 'ce_rank', 'ce_score', 'customer_problem', 'support_action', 'resolution_type', 'evidence_quality', 'evidence_relevance', 'review_notes', 'risk_level', 'intent_golden']

HUMAN EVIDENCE REVIEW STATUS
Total cases     : 50
Reviewed cases  : 50
Unreviewed      : 0

FINAL 50-CASE HUMAN EVIDENCE REVIEW
RELEVANT       : 36 (72.00%)
PARTIAL        : 13 (26.00%)
IRRELEVANT     :  1 (2.00%)
------------------------------------------------------------
USEFUL EVIDENCE: 49 (98.00%)
TOTAL REVIEWED : 50

Saved:
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/top1_evidence_review_50_final.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/human_evidence_review_50_summary.json

✅ HUMAN EVIDENCE REVIEW COMPLETE — 50/50


In [45]:
# ============================================================
# HIVER SUPPORT AGENT — FINAL EVALUATION REPORT
# ============================================================

import os
import json
import pandas as pd
import numpy as np

BASE_DIR = "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")

os.makedirs(EVAL_DIR, exist_ok=True)

FINAL_CSV = os.path.join(EVAL_DIR, "FINAL_AGENT_EVALUATION.csv")
FINAL_JSON = os.path.join(EVAL_DIR, "FINAL_AGENT_EVALUATION_SUMMARY.json")

print("=" * 75)
print("HIVER SUPPORT AGENT — FINAL EVALUATION")
print("=" * 75)


# ============================================================
# 1. RETRIEVAL EVALUATION
# ============================================================

retrieval_path = os.path.join(EVAL_DIR, "retrieval_benchmark.csv")

retrieval_df = None
retrieval_results = []

if os.path.exists(retrieval_path):
    retrieval_df = pd.read_csv(retrieval_path)

    print("\n[1] RETRIEVAL EVALUATION")
    print("-" * 75)
    print("Shape:", retrieval_df.shape)
    print("Columns:", retrieval_df.columns.tolist())

    print("\nRetrieval benchmark:")
    print(retrieval_df.to_string(index=False))

    for _, row in retrieval_df.iterrows():
        result = {
            "method": str(row.iloc[0])
        }

        for col in retrieval_df.columns[1:]:
            value = row[col]

            if pd.notna(value):
                try:
                    result[str(col)] = float(value)
                except:
                    result[str(col)] = str(value)

        retrieval_results.append(result)

else:
    print("\n[1] RETRIEVAL EVALUATION")
    print("WARNING: retrieval_benchmark.csv not found.")


# ============================================================
# 2. HUMAN EVIDENCE REVIEW
# ============================================================

evidence_path = os.path.join(
    EVAL_DIR,
    "top1_evidence_review_50_final.csv"
)

evidence_results = {}

if os.path.exists(evidence_path):

    evidence_df = pd.read_csv(evidence_path)

    LABEL_COL = "evidence_relevance"

    evidence_df[LABEL_COL] = (
        evidence_df[LABEL_COL]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    total = len(evidence_df)

    relevant = int(
        (evidence_df[LABEL_COL] == "RELEVANT").sum()
    )

    partial = int(
        (evidence_df[LABEL_COL] == "PARTIAL").sum()
    )

    irrelevant = int(
        (evidence_df[LABEL_COL] == "IRRELEVANT").sum()
    )

    useful = relevant + partial

    evidence_results = {
        "sample_size": total,
        "relevant": relevant,
        "partial": partial,
        "irrelevant": irrelevant,
        "relevant_rate": round(relevant / total, 4),
        "useful_evidence_rate": round(useful / total, 4),
        "irrelevant_rate": round(irrelevant / total, 4)
    }

    print("\n[2] HUMAN EVIDENCE REVIEW")
    print("-" * 75)
    print(f"Sample size          : {total}")
    print(f"Relevant             : {relevant} ({relevant/total:.2%})")
    print(f"Partial              : {partial} ({partial/total:.2%})")
    print(f"Irrelevant            : {irrelevant} ({irrelevant/total:.2%})")
    print(f"Useful evidence      : {useful} ({useful/total:.2%})")

else:
    print("\n[2] HUMAN EVIDENCE REVIEW")
    print("WARNING: final human evidence artifact not found.")


# ============================================================
# 3. RESPONSE GROUNDING / SAFETY
# ============================================================

grounding_candidates = [
    os.path.join(EVAL_DIR, "response_grounding_batch_01_v2.csv"),
    os.path.join(EVAL_DIR, "response_grounding_batch_01.csv")
]

grounding_path = None

for path in grounding_candidates:
    if os.path.exists(path):
        grounding_path = path
        break

grounding_results = {}

if grounding_path:

    grounding_df = pd.read_csv(grounding_path)

    print("\n[3] RESPONSE GROUNDING / SAFETY")
    print("-" * 75)
    print("Artifact:", grounding_path)
    print("Shape:", grounding_df.shape)
    print("Columns:", grounding_df.columns.tolist())

    # Detect likely final status column
    possible_status_cols = [
        "grounding_status",
        "status",
        "final_status",
        "grounding_result"
    ]

    status_col = None

    for col in possible_status_cols:
        if col in grounding_df.columns:
            status_col = col
            break

    if status_col:

        statuses = (
            grounding_df[status_col]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.upper()
        )

        total = len(statuses)
        grounded = int((statuses == "GROUNDED").sum())
        conditional = int((statuses == "CONDITIONAL").sum())
        unsafe = int((statuses == "UNSAFE").sum())

        grounding_results = {
            "sample_size": total,
            "grounded": grounded,
            "conditional": conditional,
            "unsafe": unsafe,
            "grounded_rate": round(grounded / total, 4),
            "unsafe_rate": round(unsafe / total, 4)
        }

        print(f"Sample size          : {total}")
        print(f"GROUNDED             : {grounded} ({grounded/total:.2%})")
        print(f"CONDITIONAL          : {conditional} ({conditional/total:.2%})")
        print(f"UNSAFE               : {unsafe} ({unsafe/total:.2%})")

    else:
        print("WARNING: Could not identify grounding status column.")

else:
    print("\n[3] RESPONSE GROUNDING / SAFETY")
    print("WARNING: grounding artifact not found.")


# ============================================================
# 4. AUTOMATION POLICY
# ============================================================

automation_path = os.path.join(
    EVAL_DIR,
    "final_automation_policy_evaluation.csv"
)

automation_results = {}

if os.path.exists(automation_path):

    automation_df = pd.read_csv(automation_path)

    print("\n[4] AUTOMATION POLICY")
    print("-" * 75)
    print("Shape:", automation_df.shape)
    print("Columns:", automation_df.columns.tolist())

    print("\nPolicy evaluation artifact:")
    print(automation_df.to_string(index=False))

    # Known validated results from the frozen policy
    automation_results = {
        "policy_name": "conservative_v1",
        "risk_gate": "LOW only",
        "high_risk_action": "HUMAN",
        "medium_risk_action": "HUMAN",
        "auto_precision": 0.811,
        "auto_recall": 0.577,
        "auto_f1": 0.674,
        "overall_accuracy": 0.833,
        "auto_coverage_non_high_risk": 0.213,
        "false_auto": 7,
        "false_human": 22,
        "high_risk_auto": 0,
        "medium_risk_auto": 0
    }

    print("\nFrozen conservative policy:")
    print(f"AUTO precision       : 81.1%")
    print(f"AUTO recall          : 57.7%")
    print(f"AUTO F1              : 67.4%")
    print(f"Overall accuracy     : 83.3%")
    print(f"AUTO coverage        : 21.3%")
    print(f"False AUTO           : 7")
    print(f"False HUMAN          : 22")
    print(f"HIGH-risk AUTO       : 0")
    print(f"MEDIUM-risk AUTO     : 0")

else:
    print("\n[4] AUTOMATION POLICY")
    print("WARNING: automation policy artifact not found.")


# ============================================================
# 5. INTENT CLASSIFIER RESULTS
# ============================================================

intent_results = {
    "majority_baseline_accuracy": 0.345,
    "majority_baseline_macro_f1": 0.030,
    "majority_baseline_weighted_f1": 0.180,

    "weak_label_validation_accuracy": 0.976,
    "weak_label_validation_macro_f1": 0.9096,
    "weak_label_validation_weighted_f1": 0.9763,

    "golden_accuracy": 0.435,
    "golden_macro_f1": 0.2855,
    "golden_weighted_f1": 0.3469,

    "golden_sample_size": 200,

    "important_note":
        "Weak-label validation metrics are not human-ground-truth "
        "accuracy because the validation labels were generated by "
        "the same weak-label rules."
}

print("\n[5] INTENT CLASSIFIER")
print("-" * 75)

print("Majority baseline:")
print("  Accuracy            : 34.5%")
print("  Macro F1            : 3.0%")

print("\nTF-IDF + Logistic Regression:")
print("  Golden accuracy     : 43.5%")
print("  Golden macro F1     : 28.55%")
print("  Golden weighted F1  : 34.69%")

print("\nInterpretation:")
print("  +9.0 percentage points accuracy over majority baseline")
print("  +25.55 percentage points macro F1 over majority baseline")


# ============================================================
# 6. FINAL SYSTEM SUMMARY
# ============================================================

final_summary = {
    "project": "Hiver Evidence-Grounded Adaptive Support Agent",

    "architecture": [
        "SLM Router",
        "Risk Router",
        "Intent Classification",
        "Hybrid RAG",
        "BM25",
        "BGE-large embeddings",
        "Reciprocal Rank Fusion",
        "Cross-Encoder Reranker",
        "Historical Resolution Evidence",
        "Response Generation",
        "Grounding/Safety Checker",
        "Conservative Automation Policy"
    ],

    "dataset": {
        "source": "Customer Support on Twitter",
        "support_cases_used": 83218,
        "golden_evaluation_cases": 200,
        "human_evidence_review_cases": 50
    },

    "retrieval": retrieval_results,

    "human_evidence_review": evidence_results,

    "response_grounding": grounding_results,

    "automation_policy": automation_results,

    "intent_classifier": intent_results,

    "key_findings": [
        "Hybrid retrieval improves intent-agreement Recall@K over individual retrieval methods.",
        "Cross-Encoder reranking achieved the strongest recorded retrieval proxy.",
        "50 manually reviewed Top-1 evidence cases produced a 98% useful-evidence rate.",
        "Response grounding tests detected unsupported claims and internal-information leakage.",
        "The conservative automation policy prevents HIGH-risk and MEDIUM-risk cases from being auto-handled.",
        "Intent classification improves over the majority baseline but remains a known limitation.",
        "Golden-set evaluation is kept separate from RAG training and weak-label generation."
    ],

    "limitations": [
        "Retrieval Recall@K is an intent-agreement proxy, not human relevance accuracy.",
        "Human evidence review covers a 50-case stratified sample rather than all 200 golden cases.",
        "Intent classifier performance on the golden set remains modest.",
        "Runtime retrieval artifact is benchmark-specific by golden_id and should not be presented as arbitrary-query production retrieval.",
        "Weak-label validation metrics should not be interpreted as human-ground-truth performance.",
        "The end-to-end generated-response demonstration uses golden intent for integration testing."
    ]
}


# ============================================================
# 7. CREATE FLAT FINAL CSV
# ============================================================

flat_rows = []

# Intent
flat_rows.extend([
    {
        "category": "Intent Classification",
        "metric": "Majority Baseline Accuracy",
        "value": 0.345,
        "unit": "rate"
    },
    {
        "category": "Intent Classification",
        "metric": "Golden Accuracy",
        "value": 0.435,
        "unit": "rate"
    },
    {
        "category": "Intent Classification",
        "metric": "Golden Macro F1",
        "value": 0.2855,
        "unit": "rate"
    },
    {
        "category": "Intent Classification",
        "metric": "Golden Weighted F1",
        "value": 0.3469,
        "unit": "rate"
    }
])

# Human evidence
if evidence_results:
    flat_rows.extend([
        {
            "category": "Human Evidence Review",
            "metric": "Relevant Rate",
            "value": evidence_results["relevant_rate"],
            "unit": "rate"
        },
        {
            "category": "Human Evidence Review",
            "metric": "Useful Evidence Rate",
            "value": evidence_results["useful_evidence_rate"],
            "unit": "rate"
        },
        {
            "category": "Human Evidence Review",
            "metric": "Irrelevant Rate",
            "value": evidence_results["irrelevant_rate"],
            "unit": "rate"
        }
    ])

# Grounding
if grounding_results:
    flat_rows.extend([
        {
            "category": "Response Safety",
            "metric": "Grounded Rate",
            "value": grounding_results["grounded_rate"],
            "unit": "rate"
        },
        {
            "category": "Response Safety",
            "metric": "Unsafe Rate",
            "value": grounding_results["unsafe_rate"],
            "unit": "rate"
        }
    ])

# Automation
if automation_results:
    flat_rows.extend([
        {
            "category": "Automation Policy",
            "metric": "AUTO Precision",
            "value": automation_results["auto_precision"],
            "unit": "rate"
        },
        {
            "category": "Automation Policy",
            "metric": "AUTO Recall",
            "value": automation_results["auto_recall"],
            "unit": "rate"
        },
        {
            "category": "Automation Policy",
            "metric": "AUTO F1",
            "value": automation_results["auto_f1"],
            "unit": "rate"
        },
        {
            "category": "Automation Policy",
            "metric": "Overall Accuracy",
            "value": automation_results["overall_accuracy"],
            "unit": "rate"
        },
        {
            "category": "Automation Policy",
            "metric": "AUTO Coverage",
            "value": automation_results["auto_coverage_non_high_risk"],
            "unit": "rate"
        },
        {
            "category": "Automation Policy",
            "metric": "HIGH-risk AUTO",
            "value": automation_results["high_risk_auto"],
            "unit": "count"
        }
    ])

final_csv_df = pd.DataFrame(flat_rows)

final_csv_df.to_csv(FINAL_CSV, index=False)

with open(FINAL_JSON, "w") as f:
    json.dump(final_summary, f, indent=2)


# ============================================================
# 8. FINAL SCORECARD
# ============================================================

print("\n" + "=" * 75)
print("FINAL SCORECARD")
print("=" * 75)

print("""
DATA
--------------------------------------------------
Historical RAG cases              : 83,218
Golden evaluation cases            : 200
Human evidence review              : 50

INTENT
--------------------------------------------------
Majority baseline accuracy         : 34.50%
Golden accuracy                    : 43.50%
Golden macro F1                    : 28.55%
Golden weighted F1                 : 34.69%

HUMAN EVIDENCE
--------------------------------------------------
Relevant                           : 72.00%
Partial                            : 26.00%
Useful evidence                    : 98.00%
Irrelevant                         :  2.00%

AUTOMATION
--------------------------------------------------
AUTO precision                     : 81.10%
AUTO recall                        : 57.70%
AUTO F1                            : 67.40%
Overall accuracy                   : 83.30%
AUTO coverage                      : 21.30%
HIGH-risk AUTO                     : 0
MEDIUM-risk AUTO                   : 0
""")

if grounding_results:
    print("RESPONSE SAFETY")
    print("-" * 50)
    print(
        f"Grounded responses                 : "
        f"{grounding_results['grounded_rate']:.2%}"
    )
    print(
        f"Unsafe responses                   : "
        f"{grounding_results['unsafe_rate']:.2%}"
    )

print("\n" + "=" * 75)
print("FINAL ARTIFACTS SAVED")
print("=" * 75)

print(FINAL_CSV)
print(FINAL_JSON)

print("\n✅ FINAL EVALUATION COMPLETE")
print("✅ All validated metrics consolidated into one report.")

HIVER SUPPORT AGENT — FINAL EVALUATION

[1] RETRIEVAL EVALUATION
---------------------------------------------------------------------------
Shape: (5, 5)
Columns: ['method', 'recall_at_1', 'recall_at_3', 'recall_at_5', 'recall_at_10']

Retrieval benchmark:
          method  recall_at_1  recall_at_3  recall_at_5  recall_at_10
          TF-IDF        0.160         0.25        0.310           NaN
            BM25        0.170         0.29        0.325           NaN
       BGE-large        0.215         0.32        0.365           NaN
    BM25+BGE RRF        0.220         0.33        0.380         0.425
RRF+CrossEncoder        0.250         0.35        0.390         0.440

[2] HUMAN EVIDENCE REVIEW
---------------------------------------------------------------------------
Sample size          : 50
Relevant             : 36 (72.00%)
Partial              : 13 (26.00%)
Irrelevant            : 1 (2.00%)
Useful evidence      : 49 (98.00%)

[3] RESPONSE GROUNDING / SAFETY
---------------------

In [46]:
# ============================================================
# FIX + SAVE RESPONSE GROUNDING METRICS
# ============================================================

import os
import json
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/hiver_support_agent_checkpoint/data"
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")

GROUNDING_PATH = os.path.join(
    EVAL_DIR,
    "response_grounding_batch_01_v2.csv"
)

SUMMARY_PATH = os.path.join(
    EVAL_DIR,
    "response_grounding_final_summary.json"
)

# ------------------------------------------------------------
# 1. Load grounding results
# ------------------------------------------------------------

df = pd.read_csv(GROUNDING_PATH)

print("=" * 65)
print("RESPONSE GROUNDING / SAFETY EVALUATION")
print("=" * 65)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. Use the actual column
# ------------------------------------------------------------

LABEL_COL = "grounding_decision"

if LABEL_COL not in df.columns:
    raise ValueError(
        f"'{LABEL_COL}' not found. Available columns: "
        f"{df.columns.tolist()}"
    )

df[LABEL_COL] = (
    df[LABEL_COL]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

# ------------------------------------------------------------
# 3. Count decisions
# ------------------------------------------------------------

total = len(df)

grounded = int(
    (df[LABEL_COL] == "GROUNDED").sum()
)

conditional = int(
    (df[LABEL_COL] == "CONDITIONAL").sum()
)

unsafe = int(
    (df[LABEL_COL] == "UNSAFE").sum()
)

unknown = total - grounded - conditional - unsafe

# ------------------------------------------------------------
# 4. Calculate rates
# ------------------------------------------------------------

grounded_rate = grounded / total if total else 0
conditional_rate = conditional / total if total else 0
unsafe_rate = unsafe / total if total else 0

# ------------------------------------------------------------
# 5. Violation / warning statistics
# ------------------------------------------------------------

violation_count = int(
    pd.to_numeric(
        df["violation_count"],
        errors="coerce"
    ).fillna(0).sum()
)

warning_count = int(
    pd.to_numeric(
        df["warning_count"],
        errors="coerce"
    ).fillna(0).sum()
)

# ------------------------------------------------------------
# 6. Print final results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GROUNDING RESULTS")
print("-" * 65)

print(f"Total responses      : {total}")
print(f"GROUNDED             : {grounded} ({grounded_rate:.2%})")
print(f"CONDITIONAL          : {conditional} ({conditional_rate:.2%})")
print(f"UNSAFE               : {unsafe} ({unsafe_rate:.2%})")
print(f"UNKNOWN              : {unknown}")

print("\n" + "-" * 65)
print("SAFETY SIGNALS")
print("-" * 65)

print(f"Total violations     : {violation_count}")
print(f"Total warnings       : {warning_count}")

# ------------------------------------------------------------
# 7. Show decision distribution
# ------------------------------------------------------------

print("\nDecision distribution:")
print(
    df[LABEL_COL]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 8. Check individual cases
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CASE-LEVEL RESULTS")
print("-" * 65)

display(
    df[
        [
            "golden_id",
            LABEL_COL,
            "violation_count",
            "warning_count"
        ]
    ]
)

# ------------------------------------------------------------
# 9. Save summary
# ------------------------------------------------------------

summary = {
    "evaluation_name": "Response Grounding and Safety Evaluation",
    "artifact": "response_grounding_batch_01_v2.csv",
    "sample_size": total,
    "grounded": grounded,
    "conditional": conditional,
    "unsafe": unsafe,
    "unknown": unknown,
    "grounded_rate": round(grounded_rate, 4),
    "conditional_rate": round(conditional_rate, 4),
    "unsafe_rate": round(unsafe_rate, 4),
    "total_violations": violation_count,
    "total_warnings": warning_count,
    "interpretation": {
        "GROUNDED": "Response passed the grounding and safety checks.",
        "CONDITIONAL": "Response contained warnings requiring review.",
        "UNSAFE": "Response violated a grounding or safety rule."
    },
    "note": (
        "This evaluation covers the generated response batch "
        "available in the saved artifact and is not a claim about "
        "all possible model outputs."
    )
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 65)
print("SAVED")
print("=" * 65)

print(SUMMARY_PATH)

print("\n✅ Grounding metric extraction fixed.")
print("✅ Final grounding summary saved.")

RESPONSE GROUNDING / SAFETY EVALUATION
Shape: (7, 8)
Columns: ['golden_id', 'customer_query', 'generated_response', 'grounding_decision', 'violation_count', 'warning_count', 'violations', 'warnings']

-----------------------------------------------------------------
GROUNDING RESULTS
-----------------------------------------------------------------
Total responses      : 7
GROUNDED             : 7 (100.00%)
CONDITIONAL          : 0 (0.00%)
UNSAFE               : 0 (0.00%)
UNKNOWN              : 0

-----------------------------------------------------------------
SAFETY SIGNALS
-----------------------------------------------------------------
Total violations     : 0
Total warnings       : 0

Decision distribution:
grounding_decision
GROUNDED    7

-----------------------------------------------------------------
CASE-LEVEL RESULTS
-----------------------------------------------------------------


,golden_id,grounding_decision,violation_count,warning_count
0,GOLD-0001,GROUNDED,0,0
1,GOLD-0002,GROUNDED,0,0
2,GOLD-0003,GROUNDED,0,0
3,GOLD-0004,GROUNDED,0,0
4,GOLD-0005,GROUNDED,0,0
5,GOLD-0006,GROUNDED,0,0
6,GOLD-0007,GROUNDED,0,0



SAVED
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/response_grounding_final_summary.json

✅ Grounding metric extraction fixed.
✅ Final grounding summary saved.
